# Cross-entropy over a large vocabulary, mean with ignore_index

The last layer of a language model produces a `(M, V)` matrix of logits, one row per token, with $V$ in the tens of thousands. The training loss is the mean token cross-entropy, skipping padding positions:

$$\ell_i = \log\sum_{j=0}^{V-1} e^{z_{ij}} \;-\; z_{i, t_i}, \qquad L = \frac{1}{|\mathcal V|}\sum_{i \in \mathcal V} \ell_i, \qquad \mathcal V = \{\, i : t_i \ne \texttt{ignore\_index} \,\}$$

Write `cross_entropy(logits, target, ignore_index)` with a Triton kernel that never materialises the softmax.

## Inputs
- `logits`: contiguous `float32` CUDA tensor `(M, V)`, $M \ge 1$, $V \ge 1$, finite (values can be large, e.g. $\pm 100$).
- `target`: `int64` CUDA tensor `(M,)`. Each entry is either in $[0, V)$ or equal to `ignore_index`.
- `ignore_index`: Python int (PyTorch's default is `-100`).

## Output
- A 0-d `float32` tensor $L$. If **every** target is ignored, return NaN (that is what `torch.nn.functional.cross_entropy` does: $0/0$).

## Requirements
- One program per row with a fixed `BLOCK` (at most 4096). Stream the row in chunks, keeping a running max $m$ and running sum $s$ of $e^{z - m}$ (the online log-sum-exp), so any $V$ works.
- The kernel writes a per-row loss (0 for ignored rows) and a per-row valid flag. The host divides `loss.sum()` by `valid.sum()`. Do not use atomics (see the explanation for why).
- Never index the row with an ignored target: clamp it to a safe column before loading.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def cross_entropy(logits, target, ignore_index):
    """logits: (M, V) float32, target: (M,) int64, CUDA. Return the mean loss over non-ignored rows (0-d float32)."""
    raise NotImplementedError


@triton.jit
def _ce_kernel(logits_ptr, target_ptr, loss_ptr, valid_ptr, V, stride_row, ignore_index, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[2.0, 1.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [0, 2], "dtype": "int64"}, -100], "e": {"__nd__": 0.7531090974807739, "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 3.0], [5.0, -1.0], [0.5, 0.5]], "dtype": "float32"}, {"__nd__": [1, -100, 0], "dtype": "int64"}, -100], "e": {"__nd__": 0.4100376069545746, "dtype": "float32"}}, {"a": [{"__nd__": [[4.0], [-7.0]], "dtype": "float32"}, {"__nd__": [0, 0], "dtype": "int64"}, -100], "e": {"__nd__": 0.0, "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 2.0], [3.0, 4.0]], "dtype": "float32"}, {"__nd__": [-100, -100], "dtype": "int64"}, -100], "e": {"__nd__": {"__f__": "nan"}, "dtype": "float32"}}, {"a": [{"__nd__": [[100.0, -100.0, 50.0], [-80.0, 90.0, 89.0]], "dtype": "float32"}, {"__nd__": [2, 1], "dtype": "int64"}, -100], "e": {"__nd__": 25.156631469726562, "dtype": "float32"}}, {"a": [{"__nd__": [[1.6597816944122314, 0.6528018712997437, -0.17396996915340424, -6.956808090209961, 1.2944824695587158, -6.378839492797852, 2.7297637462615967, 1.8178966045379639, 2.4901700019836426, 2.483095169067383, 0.8955433368682861, -1.6050041913986206, -0.9211869239807129, 4.524216651916504, -1.7466881275177002, -0.6843714118003845, -2.1735434532165527, -1.5517480373382568, -0.9196736812591553, 0.7696549892425537, -0.881568193435669, -1.064125418663025, -1.8504455089569092, 0.26875340938568115, -4.032498359680176, 0.1565546989440918, 3.930077075958252, -2.300738573074341, -0.1361752152442932, 8.451626777648926, -2.9780232906341553, -4.863424777984619, -1.2804274559020996, 5.42081356048584, 6.1053466796875, -3.6479616165161133, 1.7104218006134033, 0.013564294204115868, -1.7651121616363525, -5.837093353271484, 3.073751926422119, -2.5816149711608887, 0.89702308177948, 0.24568189680576324, 5.610544204711914, 3.527984142303467, -3.466008186340332, -0.7871854305267334, 3.1105427742004395, -3.999044895172119, -1.1768649816513062, 0.6827273964881897, -4.84747838973999, -3.706374406814575, -5.112231254577637, 0.8609277009963989, -1.684543490409851, 2.785944938659668, 0.3413827419281006, 1.7160518169403076, -0.25102752447128296, 2.2557713985443115, -0.3099514842033386, -2.9466047286987305, 3.548305034637451, 2.9400243759155273, 6.800079345703125, -0.4266027808189392, -1.7397007942199707, -4.341127395629883, -11.577832221984863, -3.9251952171325684, 1.4604556560516357, -2.6565933227539062, -7.784472465515137, 1.8247792720794678, -1.7749725580215454, -1.0830122232437134, -1.294318437576294, -0.4040951728820801, 1.4233322143554688, -5.203549861907959, -0.6813523769378662, -1.5257673263549805, 3.4101741313934326, -3.1058921813964844, 4.912234306335449, 1.8240995407104492, 4.87393045425415, -2.1134560108184814, 2.417856454849243, 1.089311957359314, -4.493011474609375, -0.7827537059783936, -1.3580889701843262, -2.89846134185791, -2.3044486045837402, -4.3194804191589355, -1.4015216827392578, 1.0024089813232422, -2.3365159034729004, -4.689455986022949, 3.1359901428222656, -1.1124439239501953, 0.0822741836309433, -0.701830267906189, -1.6203556060791016, 1.8052761554718018, -1.5897185802459717, 0.0861818864941597, -4.33482551574707, -4.916297435760498, -1.168670892715454, 5.7320146560668945, 0.5437877774238586, 5.8008904457092285, 1.3565536737442017, 1.3603780269622803, -1.7499711513519287, -0.06725269556045532, 5.373857498168945, 3.5555686950683594, -1.8785679340362549, -1.409450888633728, 4.011725425720215, -1.544355869293213, 2.7592215538024902, -2.728686809539795, 1.8840497732162476, 4.578606128692627, 2.5050182342529297, 0.7303320169448853, 0.4211999177932739, -0.2536686062812805, -0.06918402016162872, 3.5364670753479004, 3.461714744567871, 1.2234365940093994, -3.7776408195495605, -1.5619758367538452, 0.8922026753425598, 0.1544865518808365, -4.058899879455566, 1.992027997970581, 7.796492576599121, -3.6796226501464844, 0.1913212537765503, 4.859365463256836, -1.2520443201065063, -1.2232252359390259, 0.5087257027626038, 1.7009449005126953, -1.2335871458053589, 2.9394431114196777, -5.714991092681885, 1.9662655591964722, 1.106313705444336, -0.4767269790172577, 1.162102222442627, 0.1436333805322647, 3.9894561767578125, -1.3851349353790283, -0.12425678968429565, -4.162964820861816, -3.374328374862671, -0.7229487895965576, -4.668085098266602, -2.5418641567230225, -3.9780635833740234, -1.4808058738708496, 0.35231518745422363, 2.830610990524292, -5.821823596954346, -0.6010169386863708, 2.5971217155456543, 4.140390872955322, 4.218213081359863, -4.302979469299316, -1.539419174194336, 1.6925623416900635, 2.8161704540252686, 0.8460565209388733, 0.7967592477798462, 0.7232956886291504, -2.3020825386047363, 0.4838712811470032, 2.4367105960845947, -1.0172111988067627, 3.5185060501098633, -8.526403427124023, -0.9795350432395935, 5.100660800933838, 2.6206607818603516, 4.332518577575684, 0.4383755922317505, -3.545172691345215, -1.5313529968261719, 4.566591262817383, -5.509744644165039, -2.8608951568603516, -5.14180326461792, -1.600904941558838, -1.2637139558792114, 1.7398090362548828, -2.4717462062835693, 5.140500068664551, 1.943182110786438, -1.4127155542373657, -2.973193645477295, -4.493569850921631, 0.37008410692214966, 5.951143741607666, -0.4813939332962036, -3.078429937362671, -0.1676449328660965, -1.5417430400848389, -2.704409122467041, -5.680562973022461, -0.1602482795715332, 4.466684341430664, 3.6944475173950195, 2.962718963623047, 0.6446929574012756, 1.4323985576629639, 3.0515527725219727, -0.1432797908782959, 0.8679587841033936, -1.1742717027664185, 3.422349452972412, -2.754625082015991, 2.0703001022338867, 2.595059633255005, 2.538166046142578, 2.1641745567321777, -1.3623487949371338, -1.6611151695251465, -1.2042396068572998, -1.0923826694488525, -3.345184564590454, -1.3033292293548584, -1.8761404752731323, -2.8106160163879395, 1.1270246505737305, -2.2327635288238525, 0.7874981164932251, 4.460329055786133, 5.108610153198242, 0.1021275669336319, 3.1145145893096924, 0.7999712228775024, 7.232879161834717, 4.996791839599609, 3.410388946533203, -0.41503840684890747, -0.6971752643585205, 1.6990320682525635, -0.041820403188467026, -0.686052143573761, 2.147031784057617, -3.700681686401367, -5.9678955078125, -3.009850025177002, -3.4343700408935547, -0.2040296196937561, -2.577805519104004, -0.5481928586959839, -6.052778244018555, 4.469785213470459, 1.2320923805236816, -2.5662217140197754, 0.9886152148246765, 3.7457735538482666, 2.5725364685058594, -0.23509164154529572, 1.8888304233551025, 2.7046172618865967, -3.608347177505493, -4.306295394897461, -1.7576597929000854, 3.002861738204956, 4.488968372344971, 0.4389409124851227, 4.57669734954834, 3.0483877658843994, -0.17910295724868774, -2.211998462677002, -1.9042407274246216, 0.2814900875091553, 4.2770161628723145, 2.571453094482422, -3.9009711742401123, -3.6694116592407227, 2.49271559715271, 0.6308108568191528, -2.818985939025879, 2.085184335708618, 4.165224075317383, 2.4514427185058594, 0.3664741516113281, 4.081788063049316, -1.1206742525100708, -1.741349697113037, -2.7389369010925293, 5.765109539031982, 6.45989990234375, -0.07284439355134964, 1.8421902656555176, 1.422055959701538, 2.882275342941284, 3.6591849327087402, 2.116446018218994, -0.21615484356880188, -1.5009710788726807, -3.0147790908813477, -7.828374862670898, -0.22729651629924774, -2.4515299797058105, 0.1542632281780243, -0.2542897164821625, -3.0374655723571777, -1.450150728225708, 1.309666633605957, 0.5691533088684082, -2.599355697631836, -2.213602304458618, 1.7785618305206299, 0.7757238149642944, 0.9877721071243286, -0.43424564599990845, 2.2887322902679443, 0.34236106276512146, 7.671336650848389, -1.3635797500610352, -2.061410903930664, 5.029923439025879, -0.16225630044937134, -2.292290687561035, 1.358330488204956, -6.918756484985352, 1.3948235511779785, 0.10797382891178131, 0.8548263311386108, -2.3709211349487305, 4.003075122833252, 0.9923645853996277, -0.012269994243979454, 1.1346831321716309, -2.468153715133667, -0.3436390459537506, -4.474331855773926, -3.280097484588623, 0.91998291015625, 0.38862982392311096, 2.263132333755493, -1.961759090423584, -0.3063362240791321, -2.03838849067688, 0.7918740510940552, 2.3702292442321777, 3.2279791831970215, 1.7293514013290405, -2.369576930999756, 0.6394896507263184, -3.1094655990600586, -5.764346599578857, 0.5194733738899231, 0.24625930190086365, 2.2233004570007324, 2.463981866836548, -3.7164316177368164, -2.7439985275268555, -0.03912776708602905, 0.1909714639186859, 1.1975051164627075, -3.9534335136413574, -0.790256917476654, 1.886702060699463, 5.648495674133301, -3.8064801692962646, 4.815793037414551, -0.905814528465271, -4.3809709548950195, 3.940436840057373, -3.258152484893799, -3.561758279800415, 2.094940185546875, 3.044830322265625, 0.13628044724464417, -2.2320308685302734, -4.544607162475586, 0.24801430106163025, -7.0911126136779785, -4.446932792663574, 3.092520236968994, 1.7693023681640625, 3.5684242248535156, 1.1301043033599854, -3.526048183441162, 1.3107928037643433, -5.536111831665039, 1.9745614528656006, 3.9785051345825195, -3.1216671466827393, 2.9586291313171387, -0.008729638531804085, -0.6116723418235779, -6.051601409912109, -0.6012730002403259, 2.4691121578216553, -5.279755592346191, -1.5851171016693115, -2.2521891593933105, -5.0237579345703125, 2.9920787811279297, -0.0819377452135086, -0.01499955728650093, 2.8093316555023193, -2.3949546813964844, 9.625776290893555, -1.5160406827926636, -3.6235392093658447, 1.7740583419799805, 1.0200109481811523, -2.9887356758117676, -2.810201644897461, 2.4075980186462402, 0.8006123304367065, 1.1723514795303345, 2.5486338138580322, -1.8652973175048828, 3.7405219078063965, 0.05128736048936844, 2.1326053142547607, -2.7734296321868896, 2.240488290786743, 0.8313528299331665, 7.428621768951416, -0.45002514123916626, 4.596617698669434, 1.9292032718658447, 4.730559349060059, 0.7460200786590576, 1.4840123653411865, -6.605594635009766, -3.7844130992889404, 3.179793119430542, 1.319109320640564, 2.3667709827423096, -0.7321099042892456, 2.4055230617523193, -1.0149279832839966, -0.11687888205051422, 3.697498083114624, 3.285214424133301, -1.2519354820251465, 0.2074337601661682, 0.7635263800621033, -0.3686719834804535, -0.24633222818374634, -2.7693569660186768, 1.7528514862060547, -3.0777761936187744, -2.8339571952819824, 3.856794834136963, 5.523412227630615, 5.581498622894287, 1.1090859174728394, -1.553858757019043, 2.2730283737182617, 4.693066596984863, 0.40121206641197205, -1.157360553741455, -4.940616607666016, -5.001967430114746, 5.190302848815918, -4.348938465118408, -1.957822561264038, -0.8560405969619751, 0.769568681716919, -3.843923568725586, 1.7343488931655884, 1.7552049160003662, 0.23827806115150452, -1.838576316833496, -1.0906809568405151, -5.755791664123535, 1.8851690292358398, -1.5605610609054565, -4.0947771072387695, -0.11524756252765656, 1.4762283563613892, 1.0567740201950073, 1.8494722843170166, 3.4735817909240723, -1.5972747802734375, 0.5348014831542969, 0.7281631231307983, -4.3969621658325195, 3.2128348350524902, -1.0337706804275513, 0.6810972094535828, 0.8322590589523315, -2.505570650100708, 2.383906841278076, 2.8406050205230713, 0.43702420592308044, -8.73448371887207, 3.998788356781006, -0.5192900896072388, -2.0126259326934814, -1.827942132949829, -3.3203601837158203, -1.1835453510284424, 1.7652136087417603, -3.391860008239746, 0.5466873049736023, -1.8641154766082764, -0.8566882610321045, -1.1898736953735352, 0.8336743712425232, 2.943359375, -0.11770665645599365, 2.6030187606811523, 4.365993499755859, -4.415975570678711, 2.2663967609405518, -0.9745904803276062, -0.6307781934738159, 2.840585708618164, -0.7700090408325195, -1.9499599933624268, 3.7207632064819336, 3.088041305541992, 1.197422742843628, 8.417440414428711, -2.97706937789917, -4.886493682861328, -0.6588575839996338, -3.4145455360412598, 1.0982028245925903, -1.7731664180755615, -1.7997877597808838, 0.020194152370095253, -2.5226974487304688, -5.75186014175415, -2.1802892684936523, 2.100198268890381, 0.7021394968032837, 2.788294792175293, 5.034026622772217, 2.6256232261657715, -0.17639262974262238, -2.2486157417297363, 1.3840738534927368, 4.804327964782715, 4.256616592407227, 2.798588752746582, -2.401860475540161, 1.7431261539459229, 2.1187963485717773, 3.6696691513061523, -3.7899279594421387, -2.0819802284240723, 0.41423726081848145, 0.3745265007019043, -1.820727825164795, 1.4991624355316162, -0.8782624006271362, 3.315342664718628, -1.455509901046753, 0.5953337550163269, -6.272098541259766, 0.6094835996627808, -1.9865753650665283, 2.124096393585205, -0.5128461122512817, 3.6800804138183594, -1.5884056091308594, -4.29481840133667, 0.7182806134223938, 1.7385118007659912, -2.226616382598877, -3.8768978118896484, 2.2655792236328125, 3.1517436504364014, 2.1914565563201904, 0.2726648449897766, 0.00375316571444273, 4.889359474182129, 2.4047257900238037, -2.7076339721679688, -3.212343692779541, 1.8781170845031738, 2.1974778175354004, -1.8231335878372192, -2.2577829360961914, 2.898649215698242, -0.9308275580406189, 2.5114450454711914, -3.291799306869507, -2.9406847953796387, -5.100180149078369, -0.08098497986793518, -1.1901005506515503, -5.007127285003662, 0.47633999586105347, -1.5695130825042725, -0.6528447866439819, 4.191365718841553, 2.9079513549804688, 1.484165906906128, -0.9806662201881409, -3.5488779544830322, -1.9554123878479004, -1.1153712272644043, 3.764869451522827, -2.7610878944396973, -1.623000144958496, -0.025359902530908585, 1.1712582111358643, 3.2394180297851562, 3.973767042160034, -0.44353052973747253, 5.765070915222168, 2.012711524963379, -2.3482682704925537, -2.9218058586120605, -1.2516274452209473, -0.7548023462295532, 2.358623743057251, -2.446366786956787, -1.7094433307647705, -2.5466504096984863, 0.7937350273132324, -4.35361385345459, 1.6409668922424316, -2.3733482360839844, -0.6563760638237, 1.285540223121643, 0.8836402893066406, 8.31830883026123, -2.7291934490203857, -6.247684478759766, -3.931551933288574, 2.468594789505005, -3.210402011871338, 3.0614185333251953, -0.6614169478416443, 2.212895631790161, -4.587802410125732, 3.1031899452209473, 3.451249837875366, -2.7985804080963135, 3.939878463745117, 7.872281074523926, -0.012407897040247917, -2.8116626739501953, -1.7964881658554077, -0.8693034648895264, 3.0164499282836914, 5.470417499542236, -1.2515218257904053, -0.6804841160774231, -4.157873630523682, -1.5700016021728516, 1.4068210124969482, -6.015852928161621, 2.0590837001800537, 2.676527500152588, 3.6099867820739746, -1.713332176208496, -4.698602676391602, -0.8384621143341064, -3.103666305541992, -5.280416011810303, -2.2815029621124268, -4.174489974975586, 5.763059616088867, 2.234066963195801, 2.6314311027526855, -3.3274359703063965, 4.105548858642578, -1.1270767450332642, 2.5463364124298096, -3.0572762489318848, 1.8664945363998413, -5.02164363861084, -0.5229219794273376, -0.7309958934783936, -4.097463130950928, 0.07620833069086075, -1.7079927921295166, -1.1888595819473267, 2.2041375637054443, 3.8747470378875732, -0.5982404947280884, 3.8409981727600098, 2.996410846710205, 2.2286462783813477, 4.752690315246582, -4.506484508514404, 3.983203887939453, 1.1300500631332397, -1.1689777374267578, -0.5001272559165955, 3.050553798675537, 3.6012678146362305, 2.155240535736084, -3.4610118865966797, -4.36488676071167, 0.14947561919689178, 1.9229938983917236, -2.042560577392578, 2.663969039916992, 1.8542542457580566, 1.479007363319397, -2.246515989303589, 3.7590572834014893, 2.939676284790039, 4.099185943603516, 1.1794911623001099, -3.737008810043335, 1.1867153644561768, 2.6422572135925293, -0.36798393726348877, -3.371873617172241, 1.7184762954711914, 1.9682245254516602, -3.7039437294006348, -2.129948616027832, -4.294042587280273, 1.0089752674102783, 0.5626753568649292, 5.223376750946045, -2.71687388420105, 3.998101234436035, -0.6397974491119385, 0.8194122314453125, -3.683882713317871, 0.08236902207136154, -1.576886773109436, -0.32601702213287354, 0.555806040763855, 1.4576210975646973, 2.4997851848602295, -0.05704501271247864, -2.662848472595215, 3.0756425857543945, 0.25335943698883057, -1.7996420860290527, 4.112689018249512, 0.8779941201210022, 2.1806578636169434, -5.050073623657227, -1.8463966846466064, 6.043726444244385, 1.5192842483520508, 1.4627199172973633, -2.1011478900909424, -0.39533209800720215, 1.1206088066101074, -0.21165695786476135, -0.3280695378780365, -5.243209362030029, -2.3811137676239014, -1.3491524457931519, 1.683646559715271, 1.5394327640533447, -0.05066661536693573, -2.180570602416992, -2.18335223197937, 0.7173937559127808, -4.069744110107422, -0.6898677349090576, -0.9697175025939941, -0.7980331778526306, 4.915793418884277, -0.47526997327804565, 2.5292370319366455, 4.719995975494385, 1.490951657295227, -0.041862260550260544, -3.7946290969848633, 4.792356491088867, -3.3670499324798584, -6.158184051513672, 3.1216344833374023, -4.446251392364502, -3.6554067134857178, -2.2176153659820557, -3.6419334411621094, -6.617664813995361, 0.4566330909729004, 1.3218052387237549, 2.1305956840515137, -0.22662848234176636, -1.3383355140686035, -1.734256625175476, 3.484055995941162, 2.360900402069092, -1.469497799873352, 0.7611777782440186, -3.689443588256836, 1.331329107284546, -3.37764573097229, -0.609111487865448, -0.875106930732727, -8.188484191894531, 1.8195550441741943, -0.2746589779853821, -2.529897928237915, -2.3433642387390137, -0.5125163793563843, -0.7035270929336548, 1.124187707901001, 0.22576886415481567, 1.5927014350891113, 0.5105465650558472, -4.369809150695801, -4.390178680419922, 1.1876025199890137, 0.9663619995117188, -0.21527548134326935, -4.090817928314209, -2.2957444190979004, -2.6040091514587402, 1.788590908050537, -2.588928461074829, -4.219501972198486, -2.552558660507202, 0.7337385416030884, 4.518085479736328, -1.6480884552001953, 3.15788197517395, -1.498490810394287, -1.4466097354888916, -2.6904759407043457, 2.1931042671203613, -1.5896027088165283, 2.6678617000579834, -1.3377268314361572, 5.044787406921387, 0.5336146354675293, 0.052698180079460144, -3.6359660625457764, 3.7429652214050293, 0.8162176609039307, 0.15900570154190063, 0.0036307298578321934, -1.5239613056182861, -1.588563323020935, 0.9840811491012573, -2.413770914077759, -0.4522296190261841, 6.694908142089844, -0.396811842918396, -0.8561351895332336, 2.657236099243164, -2.912729263305664, 4.146501541137695, 0.012526958249509335, -3.4671928882598877, -0.4288783669471741, -2.0641984939575195, 2.0531387329101562, -1.378206729888916, 2.140723943710327, -5.718091011047363, -1.3042017221450806, 1.6836823225021362, 1.4966440200805664, 0.1792110651731491, 2.225104808807373, 2.3097472190856934, -0.009773013181984425, 0.6246733069419861, 2.668626546859741, 0.4921603202819824, -7.617253303527832, -4.690858364105225, -1.951350212097168, 2.026503562927246, 0.18626590073108673, 1.918215274810791, 2.9778590202331543, 1.606564998626709, -2.8192567825317383, 2.479072093963623, -4.039393424987793, -3.8541948795318604, -1.363852620124817, -6.820242881774902, 4.887123107910156, 6.934262275695801, -0.6170458793640137, -1.2627562284469604, -6.36093807220459, 1.4159146547317505, -2.2562668323516846, -3.7788257598876953, -4.329171180725098, -0.573739230632782, -0.8734899759292603, -0.6233519315719604, -2.0741734504699707, -5.39192008972168, 4.015392303466797, 2.1640982627868652, -2.014699697494507, -4.294033050537109, -3.2749176025390625, 1.7895824909210205, 1.243013858795166, 0.576205849647522, 3.2924928665161133, -1.9086642265319824, 4.001765727996826, 0.2430427074432373, -2.494478940963745, 1.2720835208892822, 0.7047544717788696, -2.332362174987793, 5.094791412353516, -5.135990142822266, 0.5270770788192749, 0.9799684286117554, -2.356217384338379, -0.07080555707216263, 3.68509578704834, -3.548339366912842, -3.4419026374816895, -1.2186061143875122, 1.310453176498413, 3.1386046409606934, 1.5378832817077637, -3.5350499153137207, 3.127986431121826, 1.9354963302612305, 2.8724799156188965, -5.139228343963623, -5.269990921020508, -2.2617359161376953, -1.614769697189331, -0.32232505083084106, -3.5649499893188477, -3.4985718727111816, 0.9534798860549927, -6.656258583068848, -3.858433723449707, -0.2653605341911316, 1.8922905921936035, 0.48023611307144165, 4.414271831512451, 0.5652776956558228, 1.7689108848571777, 2.0830233097076416, -0.21856534481048584, -4.27519416809082, -1.2036484479904175, -0.9868116974830627, -2.974729299545288, 1.5976663827896118, -1.9817464351654053, -4.850305557250977, -0.45729678869247437, -3.4343338012695312, 1.1392754316329956, -8.241589546203613, -5.511065483093262, -0.42450737953186035, 0.2945636212825775, -0.20926079154014587, 4.066559314727783, -1.1877524852752686, -1.9781666994094849, 2.024268865585327, -1.4126733541488647, 0.731452226638794, 3.3200833797454834, -0.9144634008407593, -0.7508013248443604, -2.232724189758301, 0.7521042823791504, 1.4446759223937988, 2.6893091201782227, 1.6044448614120483, 5.57612419128418, 0.116435706615448, 0.05770421773195267, -1.1617014408111572, 3.4855003356933594, -0.04913858324289322, -1.1357812881469727, -4.139517784118652, -0.6520068645477295, 2.300755262374878, -7.919780731201172, -3.544616222381592, -2.961428642272949, -2.6655831336975098, -0.09671278297901154, -1.3583213090896606, 0.17548111081123352, 0.6401816010475159, 3.7991061210632324, -0.9458403587341309, 0.8307527303695679, -0.8555095195770264, 0.2958018183708191, -1.194145679473877, -3.581326961517334, 1.0038082599639893, 3.0485808849334717, -2.9404377937316895, -0.25076937675476074, -0.2543535828590393, -6.439948081970215, 5.583858013153076, 4.171799182891846, 3.33522367477417, 2.7927398681640625, -2.684551239013672, -7.399791717529297, -0.34012386202812195, 0.08973178267478943, -0.9586988687515259, -1.4615263938903809, 2.7615575790405273, 2.143766403198242, -0.22204555571079254, -1.652068018913269, 6.573814392089844, 2.289344310760498, 0.1452069878578186, -4.021742820739746, -4.746062755584717, -8.355728149414062, -0.9155464768409729, 0.6971532702445984, -0.6902385354042053, -0.17219236493110657, 0.5550922155380249, 2.313145399093628, -5.421067237854004, -3.856013774871826, 2.469362735748291, 0.6673036813735962, 1.3272502422332764, -3.131460428237915, 4.3169379234313965, 0.20414091646671295, -0.8344371318817139, -2.760213851928711, 0.4311819076538086, 0.3659118413925171, -0.467387318611145, 4.691143989562988, 3.4220218658447266, -0.1284666657447815, -1.8113404512405396, -1.392906665802002, 0.9347697496414185, -8.326705932617188, 0.8493810296058655, -0.7133079171180725, 6.275110244750977, 1.2683056592941284, -0.5310542583465576, 0.12858720123767853, -1.8879811763763428, -2.0766897201538086, -0.1949615180492401, -5.955564498901367, 0.01854170672595501, -4.982301235198975, -2.124948501586914, -2.1627085208892822, 2.082322597503662, 4.662936210632324, 4.251497745513916, -1.4189441204071045, -0.6639929413795471, 0.13244056701660156, 0.4891934394836426, 3.363553762435913, 2.0420753955841064, 0.014473019167780876, 5.767175674438477, 2.7237701416015625, 5.5226922035217285, -0.1149076521396637, 1.7771999835968018, 1.0606372356414795, 0.6045821905136108, -3.0312259197235107, 0.8377490043640137, -4.2537055015563965, 1.2376807928085327, 5.225399971008301, 3.487722396850586, 0.9930065274238586, -0.7596660852432251, 2.381344795227051, -2.997861623764038, -3.494194507598877, -3.7808518409729004, 1.258185863494873, -3.834014415740967, 1.849106788635254, 3.1158077716827393, 2.123877763748169, -0.22879046201705933, 1.3403816223144531, -3.4892492294311523, 0.6743093132972717, 5.065617084503174, 4.421451568603516, -4.638818264007568, -0.1472853124141693, -0.9379781484603882, 2.7738165855407715, 0.8672463893890381, -0.2397102415561676, -1.852852463722229, 0.5060802698135376, -0.38154953718185425, 0.2255207896232605, -0.4064415693283081, -3.6259312629699707, 1.258795976638794, 1.3808495998382568, 0.6883258819580078, 0.8077685832977295, 3.6624512672424316, 3.271806478500366, 4.968419075012207, 2.111201763153076, 2.5629427433013916, 0.2092709243297577, -3.0460128784179688, -3.4305777549743652, 1.590618371963501, 0.4262657165527344, 1.6691434383392334, -6.822213172912598, -0.6370316743850708, 0.3149755001068115, -1.1513500213623047, -0.8544777631759644, -5.430084705352783, -0.43563443422317505, 3.632143974304199, -3.5271034240722656, 1.4913609027862549, -2.2954444885253906, 0.3855275511741638, 4.09912109375, 0.021694492548704147, 0.9688630104064941, 1.786385178565979, -0.21738025546073914, 3.619175910949707, -6.075315475463867, -1.0703907012939453, -5.7964982986450195, -4.643226623535156, -0.15020520985126495, -3.717437982559204, -1.3287400007247925, 3.9150545597076416, 1.508610725402832, 11.076630592346191, -0.40709829330444336, 0.10886994004249573, -7.455438137054443, -0.15597862005233765, -1.4587059020996094, 1.3336856365203857, 0.02479773387312889, -2.0153753757476807, -5.927724361419678, 2.1883580684661865, -2.203861713409424, -5.268948554992676, 1.044012427330017, -6.115964889526367, -2.002098798751831, -6.3595476150512695, -1.0617083311080933, -5.262483596801758, 8.503463745117188, 3.1466166973114014, 1.1848808526992798, 0.8105619549751282, 1.733003854751587, 2.310946464538574, 0.5874680280685425, -3.516141891479492, 0.2539474666118622, -4.9386162757873535, -1.3293952941894531, 0.9714125990867615, -3.5145797729492188, 1.0791531801223755, -1.6921067237854004, 6.917366027832031, -1.376258373260498, 1.2583820819854736, 1.6285922527313232, 0.19492001831531525, 0.7456789016723633, -1.0108057260513306, 0.20408549904823303, 0.34022843837738037, 2.304835319519043, 3.65598726272583, -0.17934711277484894, 4.131533622741699, -0.1040075495839119, -2.9148662090301514, -1.3680503368377686, -2.040370225906372, -0.7100181579589844, 0.17231501638889313, 0.6460603475570679, 3.514808177947998, 2.2313404083251953, -1.2847154140472412, 1.6546604633331299, -0.2047901451587677, 2.3257832527160645, 3.728437900543213, -0.3300831615924835, 2.3159852027893066, 4.144944667816162, 1.1365138292312622, -3.2587337493896484, -0.31167975068092346, -2.6638238430023193, 3.0098483562469482, 0.6265519857406616, 0.9811766147613525, 5.344698429107666, 2.2647953033447266, 0.43828970193862915, -5.403841972351074, -1.5570719242095947, -2.0680785179138184, -0.5667392611503601, -3.725602626800537, 1.8695240020751953, 3.7478504180908203, -1.0748755931854248, -4.395318508148193, 0.3075859546661377, -0.7232592105865479, 2.969193458557129, -0.19244155287742615, -2.2458205223083496, -4.171452522277832, 0.012563559226691723, -7.02573299407959, -3.3307549953460693, 2.13104248046875, -0.49036669731140137, -0.15896302461624146, -0.7384941577911377, -4.180572986602783, -1.190727949142456, 1.35176420211792, 1.5271493196487427, -3.143251895904541, 0.2060084044933319, 2.738563060760498, 0.7293833494186401, 4.516046524047852, 0.011403351090848446, -4.928899765014648, -0.05774115025997162, 0.45095711946487427, 1.9881007671356201, 1.4885003566741943, 4.530345916748047, 1.7949371337890625, 0.5769680738449097, -1.566849946975708, -0.7089946269989014, 3.9023373126983643, 5.691215991973877, 1.0625656843185425, 0.6013264060020447, -0.5822903513908386, 0.7312407493591309, 0.18461939692497253, -1.4629310369491577, 2.709348678588867, -3.2654738426208496, 0.45698976516723633, 0.5068361759185791, 1.9780923128128052, -0.4587201476097107, -2.836219549179077, -0.2848503589630127, -3.0660042762756348, -1.0588535070419312, -2.281632423400879, 7.243075370788574, -6.485471725463867, -3.5977935791015625, 3.670231819152832, -0.6240267753601074, -4.112348556518555, 1.8627815246582031, 6.285582542419434, 0.09637358039617538, 2.609076976776123, -4.006746292114258, -1.742155909538269, 0.6053575277328491, 0.5396820902824402, -2.015162944793701, -1.7439851760864258, -1.850404977798462, -2.351935863494873, -1.6453845500946045, -1.7069413661956787, 1.3381184339523315, -4.247828483581543, 1.4704538583755493, 5.217331886291504, 0.47924888134002686, -1.406974196434021, -2.7947604656219482, -1.5537775754928589, -3.618767261505127, 0.7397656440734863, -0.311540424823761, 2.26452374458313, 0.8092610836029053, 2.404167413711548, 4.535159587860107, 2.940706729888916, 8.308937072753906, 3.5236706733703613, -1.2379815578460693, 1.9807323217391968, 2.4509501457214355, 0.5244573354721069, 2.4245731830596924, 1.9461205005645752, 4.213172912597656, -0.11416785418987274, -2.553460121154785, -1.0575710535049438, -5.538500785827637, -7.759768486022949, -2.965639114379883, -1.4379769563674927, -1.9698693752288818, 3.6477408409118652, -3.1813697814941406, -3.565103530883789, 6.020263671875, -4.524310111999512, 2.377735137939453, -0.24301865696907043, -7.579043388366699, 3.0319108963012695, -0.9018938541412354, 1.2261888980865479, 0.7678177356719971, 1.1145975589752197, 2.8131420612335205, 6.637143135070801, 0.9208661913871765, -0.45255398750305176, 2.8296384811401367, 1.52109694480896, -7.470393180847168, 1.4627540111541748, -1.0681440830230713, -0.6103106737136841, -4.242166042327881, 5.806650161743164, 1.5887837409973145, -1.261763095855713, -1.1769732236862183, -0.4772107005119324, -0.24201726913452148, 2.441922187805176, 1.2437632083892822, -1.4537595510482788, -0.651531994342804, 1.8986821174621582, -2.8542912006378174, -1.1586718559265137, 2.9547367095947266, 0.08766479790210724, 4.410243034362793, -3.314359664916992, -1.2402653694152832, -2.2981276512145996, -2.460845470428467, 2.4185190200805664, 3.336214542388916, 0.46787750720977783, -0.1555696725845337, 5.552865028381348, 5.047530174255371, -5.956450939178467, -0.011063920333981514, -5.83906888961792, -4.39408540725708, -1.7504451274871826, -3.220527172088623, 2.020984649658203, -1.0317063331604004, -0.4726378917694092, -0.023894764482975006, 5.445995807647705, 1.1489564180374146, 2.9804329872131348, 1.8112070560455322, 3.88065767288208, 0.3885606527328491, 0.06100388243794441, 4.084568977355957, 0.7475809454917908, -4.545999526977539, -3.540675640106201, -1.0684800148010254, 1.7260048389434814, -2.0802602767944336, 4.724133491516113, 1.4941351413726807, -0.5488376617431641, -3.4209094047546387, 2.337594747543335, 2.0763092041015625, -3.251582622528076, 4.173746109008789, 3.874135971069336, 4.628378868103027, 0.11219360679388046, 2.990607500076294, -5.946407318115234, 1.4316871166229248, -1.0738439559936523, -0.1292763650417328, 1.501678228378296, -1.962843894958496, -0.42459505796432495, -2.8429088592529297, -2.2574543952941895, -3.2591195106506348, 1.1343075037002563, 3.8552236557006836, -2.8047420978546143, -4.692436218261719, -0.5734715461730957, 1.2556133270263672, 0.8210431337356567, -3.4449005126953125, 0.044380366802215576, -2.492946147918701, -1.2299644947052002, -0.37232622504234314, -1.2713267803192139, 1.3771090507507324, -1.5052928924560547, 4.52097749710083, 0.35955095291137695, -3.357604503631592, 2.934401512145996, 2.565825939178467, 0.8024005889892578, 0.006957505829632282, 1.024669885635376, 2.562457799911499, -0.36070913076400757, 2.4773952960968018, -1.6804561614990234, 1.4993871450424194, -2.945542335510254, 1.050499439239502, 1.720228910446167, -1.584115982055664, 1.100638508796692, -5.507787704467773, -0.27236872911453247, -0.7558683156967163, 1.8783371448516846, -0.09650935232639313, 3.0471982955932617, 5.675603866577148, 0.32965564727783203, 2.6427130699157715, 5.006522178649902, 2.8763210773468018, 3.3042426109313965, -0.5432730913162231, 1.2467454671859741, -2.8684349060058594, 2.6886584758758545, 1.3943287134170532, 1.4087495803833008, -5.342229843139648, 3.189591407775879, 4.689504146575928, 4.311817169189453, 1.5627955198287964, 4.154167175292969, 0.3539872467517853, -1.6656205654144287, 0.1087193414568901, -1.8611745834350586, 5.419425010681152, -6.985729694366455, 0.9646903276443481, -2.274886131286621, -1.3944776058197021, 2.3163058757781982, 0.003218619618564844, 3.4194066524505615, -0.22652709484100342, -0.18803688883781433, 0.4576491415500641, 3.813192844390869, -1.8574974536895752, -0.7834648489952087, -1.498587965965271, -1.2374905347824097, -3.5094656944274902, -0.8130841255187988, 1.3680651187896729, 4.20865535736084, 1.437400460243225, -1.6454172134399414, 4.290450572967529, -4.988924503326416, 1.4816983938217163, -3.494600296020508, -0.3409213423728943, -3.8865180015563965, -4.421584129333496, -1.578199863433838, 5.229053497314453, 1.073590636253357, 2.6650166511535645, -0.9355372190475464, 0.10551069676876068, -2.1941680908203125, -1.6633594036102295, -0.9064892530441284, 2.8255906105041504, 1.0064611434936523, -2.094973564147949, -0.4468398094177246, -2.481409788131714, 1.6224327087402344, 0.83443284034729, 0.8052354454994202, 2.1383728981018066, 1.728107213973999, 4.355424880981445, -2.0714216232299805, 2.2695486545562744, -0.3884308934211731, 2.835127115249634, -1.3316893577575684, -4.671191692352295, -1.0904409885406494, -1.6941107511520386, -3.014530658721924, -0.9529843926429749, 3.4974708557128906, -0.8307775259017944, 2.1309332847595215, 4.147943496704102, 3.145778179168701, 1.5161371231079102, 2.8411571979522705, -1.0897325277328491, -3.6342320442199707, -4.247061729431152, 2.8612565994262695, 4.110622406005859, -0.722771942615509, 0.22116267681121826, -6.586236476898193, 2.2481398582458496, -0.17765899002552032, -5.279428005218506, 1.5763163566589355, 1.8853731155395508, 2.575875759124756, -0.18422819674015045, 0.8858429193496704, -2.3951990604400635, 4.244101047515869, -0.8219747543334961, 3.8415892124176025, 1.7421574592590332, -0.4816552698612213, 4.014500617980957, -1.7125530242919922, 1.2918120622634888, 0.6354297995567322, -0.15220415592193604, 0.7829462885856628, 2.301882743835449, -0.8603485822677612, -1.2080274820327759, -0.3408432602882385, -1.3758740425109863, -2.6523969173431396, 1.0059688091278076, -4.866120338439941, -0.11269892752170563, -1.265390157699585, 0.535999596118927, 1.4076789617538452, 0.7792848944664001, 0.009305227547883987, 3.880843162536621, -1.4892009496688843, -3.453700304031372, -0.663047194480896, 2.0704598426818848, 5.294898986816406, 6.141761779785156, 1.1673316955566406, 3.488450288772583, 4.85366153717041, -1.2494632005691528, 4.5081305503845215, 1.4680484533309937, -0.16421741247177124, 1.7890465259552002, 2.100761890411377, -5.653833389282227, 0.40997427701950073, 0.805217981338501, -1.8768081665039062, 0.6739833354949951, -1.006333589553833, -2.2094736099243164, -0.2322489470243454, -5.683755874633789, 1.3310937881469727, 1.652553677558899, 1.001947045326233, 1.6945574283599854, -1.184190273284912, -1.5789222717285156, -5.062070369720459, -4.999876499176025, -2.144233226776123, 2.70742130279541, -0.039586037397384644, 3.9291749000549316, 2.2708170413970947, -2.937521457672119, -1.7008888721466064, -3.1597537994384766, -4.217219352722168, 0.7426520586013794, 1.9652700424194336, -2.615067481994629, 3.104745864868164, 2.597212791442871, -1.6214759349822998, 1.044971227645874, -2.76183819770813, 0.10811339318752289, 1.5937683582305908, -5.344539642333984, 4.35197639465332, -4.890632152557373, 0.3426331579685211, -1.3323800563812256, 2.6664412021636963, -2.4602346420288086, -1.3195182085037231, 9.251188278198242, 5.113487720489502, 2.6815404891967773, -1.9300020933151245, 1.166195034980774, -4.708244323730469, -2.184354066848755, 1.3984885215759277, 2.4469707012176514, -0.8880857229232788, 2.274818181991577, 2.5807716846466064, -2.7741212844848633, 0.8387133479118347, -0.03692111745476723, 4.317644119262695, -3.5572636127471924, 1.4254491329193115, -0.37060803174972534, 6.086496353149414, -6.664645195007324, 5.1075849533081055, -0.4789496064186096, 1.6612704992294312, -4.337481498718262, -3.7990357875823975, -0.688129186630249, 0.276744544506073, -7.183052062988281, -1.913243055343628, -1.3237107992172241, -0.456687867641449, 2.0177454948425293, 0.9351760149002075, -0.763105034828186, -3.6229047775268555, -4.7752509117126465, 0.2950174808502197, 4.337585926055908, 0.04159902036190033, 2.7525486946105957, -3.346404790878296, 2.755420207977295, -0.5843466520309448, 0.34890878200531006, 2.0000104904174805, 2.296222448348999, 0.8247789144515991, 1.1961495876312256, 1.0217702388763428, -2.053849220275879, -0.8592352867126465, 0.08253434300422668, 4.888270854949951, -3.4584178924560547, 0.3374021649360657, 3.001882553100586, 2.0830888748168945, -2.6864142417907715, -3.670255184173584, -1.5450119972229004, 0.9313230514526367, 2.44600772857666, -0.5773817300796509, 0.39644312858581543, -1.125211477279663, -2.4167375564575195, 3.6341521739959717, 5.21439266204834, -0.39599964022636414, -2.3706469535827637, 1.9713342189788818, -4.2930588722229, -3.898090362548828, 0.4764269292354584, 5.416867256164551, -1.092016339302063, 0.22095496952533722, 4.6092352867126465, -3.4341750144958496, -4.5584611892700195, 6.442558288574219, 3.0398831367492676, -1.116047978401184, 1.5841526985168457, 2.7389039993286133, -0.7309300899505615, 7.515102386474609, -4.800204753875732, 4.828066825866699, 1.2163283824920654, 0.609943151473999, 0.8358203172683716, -1.8297569751739502, -0.5127352476119995, -1.4092445373535156, -4.211747169494629, 1.8145623207092285, -3.891164779663086, -2.8172976970672607, 5.804681777954102, -1.493873953819275, 4.592962265014648, 3.9558305740356445, 0.6691718697547913, 4.657226085662842, -1.658717393875122, -2.611665964126587, -1.3753681182861328, -1.0606698989868164, 1.4764201641082764, 5.825225830078125, -2.500304698944092, -0.8363796472549438, -0.08870545029640198, 0.7709264755249023, -0.9877560138702393, 4.055084705352783, -1.1930525302886963, -2.8126115798950195, 4.985439300537109, 4.176733493804932, -0.04712097346782684, -2.6483492851257324, 0.4830224812030792, 2.1611409187316895, 4.6830291748046875, 4.9561896324157715, -0.7979366183280945, 0.30070558190345764, -0.5050419569015503, 0.25361067056655884, -1.2703251838684082, -1.0005500316619873, -0.38930743932724, 0.42011430859565735, -5.320704460144043, -1.1120319366455078, 2.282789945602417, 1.425845980644226, 2.164787530899048, 2.3139920234680176, 1.692551851272583, 3.2189011573791504, 0.18219339847564697, -3.530660390853882, 3.826457977294922, 1.150173306465149, -5.006545066833496, 2.623852014541626, 2.440790891647339, -3.794851779937744, 1.3724231719970703, -0.048917800188064575, 1.169058918952942, -1.6245436668395996, 5.08104133605957, -2.747084856033325, -3.0718941688537598, -1.248555302619934, -0.11416739225387573, -1.6801121234893799, 4.428708553314209, -3.0016369819641113, 6.910859107971191, -4.2311201095581055, 1.3558650016784668, 5.215219020843506, 6.955543041229248, -0.5901838541030884, -4.367282390594482, -2.053525924682617, -1.4639068841934204, 2.0697214603424072, -2.7609941959381104, 2.9082605838775635, -2.666349411010742, 2.5103750228881836, 7.695417881011963, 2.406514883041382, 3.6880745887756348, 0.78121018409729, -0.6862534284591675, 1.804551601409912, 2.0612590312957764, -8.270191192626953, -2.119793653488159, 6.4912004470825195, 2.0229663848876953, 0.8676618933677673, -0.8999179601669312, -0.8666748404502869, -0.19260317087173462, -9.395572662353516, 5.353600025177002, 1.377345323562622, -2.1587607860565186, 4.140957832336426, -0.18948514759540558, 4.385085105895996, -1.6700339317321777, 2.645812511444092, -4.768075466156006, 2.7583158016204834, -4.3136887550354, 0.21376070380210876, 1.3895854949951172, -4.674524307250977, 0.5736345052719116, -1.178044080734253, 0.5331182479858398, 3.107011556625366, 0.20040300488471985, 3.9461822509765625, -2.272474765777588, -2.676919937133789, 3.023068428039551, -2.94305157661438, 1.25198233127594, -2.382098913192749, -0.03471599519252777, 1.3554329872131348, -2.077637195587158, -1.9292497634887695, -4.1987528800964355, 2.5500431060791016, -2.568399429321289, -2.778137445449829, -0.16691651940345764, -1.034165620803833, -2.428331136703491, 0.976660430431366, 0.265966534614563, -2.0809359550476074, -1.521224021911621, 3.705759286880493, 0.234685018658638, -1.643275260925293, -0.39416152238845825, 1.520981788635254, 5.576923847198486, -3.6061055660247803, 1.1909183263778687, -1.737212061882019, 0.8265863656997681, -1.7279210090637207, -2.2496917247772217, -2.3112103939056396, -1.5490515232086182, 0.7726319432258606, 5.606538772583008, -3.0523757934570312, -2.474658966064453, -1.4052419662475586, 4.516907691955566, 4.365685939788818, 4.211977958679199, 1.6107285022735596, 0.01335565373301506, -3.5160884857177734, 0.22224359214305878, 0.9312770962715149, -1.31966233253479, -1.883969783782959, -3.5116374492645264, 1.2301809787750244, -0.36718180775642395, 2.7156128883361816, -1.8963139057159424, -1.7734662294387817, 3.49190616607666, 0.24825510382652283, -4.627529144287109, -2.207515001296997, -3.0489158630371094, -0.6040106415748596, -5.663498878479004, 1.9563796520233154, 1.9381974935531616, -0.33814212679862976, -0.30574336647987366, -0.3077241778373718, -1.5138698816299438, 3.1992623805999756, -3.010066032409668, 7.04368782043457, -3.211024284362793, 0.8217564821243286, -2.8870480060577393, -2.301926851272583, -0.5291194915771484, -4.4944024085998535, 1.5930812358856201, 0.554235577583313, 1.6482019424438477, 2.2355353832244873, -2.3960084915161133, -0.5503613352775574, -0.6724377870559692, 3.3524045944213867, -0.33590251207351685, -2.521167278289795, 1.0572779178619385, -1.2118446826934814, 2.588775396347046, 4.098433494567871, 2.9683239459991455, -2.2707388401031494, 2.694416046142578, -2.5704591274261475, -1.0421046018600464, 2.8962340354919434, 0.715714693069458, 5.559758186340332, 2.5204215049743652, 0.6305813789367676, 3.4261302947998047, 0.3271656930446625, 3.938237190246582, -6.54415225982666, -0.24997854232788086, -2.7319464683532715, -0.42021507024765015, -1.6973216533660889, 5.766551971435547, 1.7144811153411865, -0.32117167115211487, 2.610584259033203, -7.40861177444458, 4.027003288269043, -6.188961029052734, 0.3769094944000244, 5.770224571228027, -3.467693567276001, -2.73504376411438, -6.730630874633789, -5.6123151779174805, 0.7264286875724792, 0.43056437373161316, -1.4494794607162476, 1.0106945037841797, -1.4323880672454834, -2.729590654373169, -1.2588014602661133, -3.6068239212036133, -6.978874206542969, -0.8196278214454651, 1.1777968406677246, 3.497037649154663, 1.8568435907363892, -0.9263771772384644, -2.5320498943328857, -0.31977415084838867, 0.95949786901474, -0.1943141222000122, -1.5297024250030518, 1.128549337387085, -6.889150619506836, 1.9232418537139893, 5.131021022796631, -3.7534594535827637, 1.8251597881317139, -1.1773568391799927, -2.2447938919067383, 2.0418858528137207, 1.9057984352111816, -0.4918906092643738, 5.29400634765625, 1.642608642578125, -5.373655796051025, 4.667180061340332, 0.4403526186943054, 4.879424571990967, -2.8669214248657227, -2.6387245655059814, -1.2827399969100952, 1.0853406190872192, 2.2671420574188232, -2.154587507247925, 10.386581420898438, -2.736105442047119, -1.1636666059494019, 6.143556594848633, 5.729231357574463, 4.634359359741211, 1.226062297821045, 0.4661344289779663, 6.819146633148193, -3.7473015785217285, -1.7179341316223145, 0.16956880688667297, 0.6304166316986084, 3.208036422729492, 0.36049211025238037, 0.9489009380340576, 1.4206831455230713, -0.8805990219116211, 0.7254533171653748, -3.463977098464966, -2.758558511734009, 0.020441755652427673, -1.0612951517105103, 1.8350751399993896, 0.11704932153224945, 0.8054017424583435, 2.849501848220825, -5.217404365539551, -3.3318653106689453, -0.9746372699737549, 5.3781561851501465, -1.2893201112747192, 5.366927146911621, 3.0177154541015625, 0.3329430818557739, 1.2451865673065186, 1.7572360038757324, -1.263096809387207, 4.409807205200195, 1.3787206411361694, 0.024134576320648193, -1.776871919631958, 1.7258394956588745, 1.7342071533203125, 2.5791566371917725, 6.86804723739624, 1.9628552198410034, -1.7769582271575928, -1.8932937383651733, -3.66231632232666, -2.8204784393310547, -3.775028705596924, 2.9375600814819336, -6.526028633117676, 4.340118885040283, -1.8041621446609497, -3.9971842765808105, -2.412172317504883, -0.12237397581338882, 0.19484812021255493, -1.460888147354126, 3.152132511138916, -3.842087745666504, -2.74550461769104, 2.1581926345825195, -3.8450093269348145, -2.2982513904571533, 1.628727912902832, 1.8477519750595093, 1.7999653816223145, 2.019111156463623, 0.00751982256770134, 0.6673827171325684, 0.5001609921455383, -0.09288058429956436, 2.864894390106201, 0.1598898470401764, 2.6692397594451904, 0.1278490126132965, 2.8257033824920654, -6.8319196701049805, 6.117388725280762, -2.486116409301758, -0.8853265643119812, -1.3204779624938965, 5.777920722961426, -4.160806655883789, -0.8648808002471924, -0.6592804193496704, 7.175647735595703, 4.286582946777344, -2.489621162414551, -0.6951310038566589, 0.5254107713699341, -2.2329602241516113, 4.402311325073242, 1.0679230690002441, -4.92984676361084, 0.32664623856544495, -0.061882540583610535, -1.3356318473815918, 0.8614034056663513, 2.210664749145508, -0.6692755818367004, -1.7699477672576904, 1.1679952144622803, -5.084948539733887, 3.0539181232452393, -3.89218807220459, 0.5150594711303711, -0.4640008807182312, 1.683051586151123, -1.639106273651123, 1.2815040349960327, -4.846012592315674, 4.2197041511535645, 1.743708610534668, 2.904388427734375, -1.4957540035247803, 0.7867389917373657, -1.172346591949463, -0.00434483215212822, -3.188901424407959, -0.16131040453910828, -7.763367176055908, 3.346309185028076, -3.6420912742614746, 2.6762871742248535, 4.186532974243164, 2.146036386489868, -3.360586643218994, 1.038316249847412, 2.221480369567871, 2.4185867309570312, 1.3453171253204346, 0.32443201541900635, -0.8543681502342224, -0.7064683437347412, 0.5945195555686951, -0.1453438699245453, -0.8302440643310547, -0.9912971258163452, 0.23975211381912231, -1.6582834720611572, 2.864557981491089, 0.26289984583854675, 3.6121535301208496, -2.053957223892212, -2.5754733085632324, -1.5370943546295166, -0.26471009850502014, 0.07503670454025269, 3.030395984649658, -6.108079433441162, 0.8241664171218872, -4.814393043518066, -2.712141990661621, -0.4851667881011963, 0.21494528651237488, 4.020448684692383, -0.5967482328414917, 3.7349367141723633, 1.957685947418213, -0.48366352915763855, 2.862184524536133, -1.226309895515442, 1.7802162170410156, 5.7809858322143555, 2.2283782958984375, -0.04259299486875534, -1.8617839813232422, -0.720524787902832, -1.7514642477035522, 5.464859962463379, 2.666454792022705, 0.6310151815414429, 1.5390145778656006, 6.063115119934082, -0.4738492965698242, -0.7624338865280151, 4.841484069824219, 2.4086084365844727, -2.824186325073242, 2.1109204292297363, 0.20522716641426086, -0.47556567192077637, -0.8655990958213806, -1.1719419956207275, 0.7102252244949341, 2.7645626068115234, -2.5289459228515625, 1.6085269451141357, -4.213456153869629, -0.49925467371940613, 3.176926374435425, 0.2533848285675049, -0.7108563780784607, 4.736408233642578, 1.9894130229949951, 1.2343484163284302, 2.0628092288970947, -3.5271992683410645, -0.5047760605812073, 5.700165748596191, 2.602412462234497, 5.300039768218994, -2.3183891773223877, -0.7977052927017212, 2.8660197257995605, 3.052888870239258, -5.598690986633301, 0.5328680872917175, -0.420015424489975, -1.5421631336212158, 1.4342889785766602, -2.4725639820098877, 1.1717331409454346, 1.754764437675476, -1.8330090045928955, -0.12579992413520813, -7.30766487121582, 1.110740303993225, -0.42082470655441284, -5.516775131225586, -5.570956230163574, -0.8307846784591675, 5.85241174697876, 0.0754907950758934, -3.833043098449707, -1.3311899900436401, -6.665778636932373, -2.382864475250244, -2.938141107559204, -0.1771961897611618, 3.021359920501709, -1.3808948993682861, 5.4079976081848145, -0.6594986319541931, 0.3534061014652252, -0.9668142199516296, -3.536465644836426, -2.446303367614746, -1.7708669900894165, 4.070115089416504, -0.8887150287628174, -4.725127220153809, 0.5173125863075256, 0.5783786773681641, -3.6145570278167725, 0.8916379809379578, 0.251857727766037, -4.230997085571289, 2.426985263824463, -4.4845452308654785, -2.8725290298461914, 1.6057859659194946, -4.986064910888672, -8.017345428466797, -2.2138102054595947, 0.1308642476797104, -0.27968400716781616, -1.6523401737213135, -4.309603691101074, 2.974749803543091, 2.4777021408081055, 0.25993531942367554, 1.1800119876861572, -5.6434831619262695, 1.778731346130371, 3.814328193664551, 2.210618257522583, 2.4941775798797607, 0.5198894143104553, 0.36531147360801697, -6.692404747009277, -1.7399029731750488, 0.4984890818595886, -2.973686695098877, -4.618836402893066, -1.330179214477539, -2.811655044555664, 3.484813928604126, -1.2567909955978394, -0.5249308943748474, 3.881312370300293, -3.2730703353881836, 0.7479809522628784, 4.5930562019348145, -2.6425421237945557, -1.2673708200454712, 2.892958164215088, 2.637913465499878, -1.7221086025238037, -1.6064643859863281, -1.6904754638671875, -2.3989193439483643, 2.0814638137817383, -0.9091944694519043, -2.1821084022521973, -2.2031941413879395, 2.53389835357666, -8.267668724060059, -1.2344200611114502, 0.7200709581375122, -2.029613733291626, -2.8967809677124023, 1.5180613994598389, 0.712022602558136, 5.782918930053711, -1.8862684965133667, 1.2604954242706299, -3.0271410942077637, 3.630767583847046, 3.0053811073303223, 1.9027774333953857, 3.13218355178833, 2.183770179748535, 1.419163703918457, 0.07859030365943909, 2.810587167739868, 1.7344796657562256, -0.8695336580276489, -0.04242872819304466, 0.6668605804443359, 4.591405391693115, 0.30780699849128723, -0.24483127892017365, -3.569758415222168, 2.511061191558838, 0.9117288589477539, 0.7113198041915894, 0.7122722268104553, 1.8056176900863647, -4.626980304718018, -2.4954826831817627, -0.6159717440605164, 2.504742383956909, 1.492462158203125, 2.8150453567504883, 1.1911072731018066, 1.970858097076416, -1.8604191541671753, -1.3868610858917236, -0.47991397976875305, -0.6119589805603027, 2.1260452270507812, 0.21025840938091278, 0.2617129981517792, -0.003847200656309724, 4.328250885009766, -1.1836199760437012, 2.0380704402923584, -1.395167350769043, 1.1625515222549438, 1.1978445053100586, -0.44817790389060974, -2.805555820465088, 1.1411880254745483, 0.7476984858512878, -0.5638875365257263, -0.27725133299827576, -0.7532037496566772, 6.948698997497559, -1.1429545879364014, -4.363278865814209, -4.617559432983398, 2.177316904067993, -5.06963586807251, 1.773888349533081, -1.4023489952087402, 0.6755030155181885, 0.06895267963409424, 2.6413488388061523, -1.3561122417449951, -0.2680532932281494, -2.559631109237671, -6.264357089996338, 3.6400060653686523, -4.119464874267578, 1.5111005306243896, 2.784391403198242, 1.0831241607666016, -2.7215917110443115, 4.8159637451171875, -3.4780189990997314, 2.916898250579834, 3.6523869037628174, -3.7718560695648193, 2.130711317062378, -2.376579761505127, 1.6916309595108032, -5.627514362335205, 5.5626983642578125, 2.4107799530029297, 0.5479602217674255, -0.6573649048805237, 4.797977924346924, 1.903968095779419, 1.6564509868621826, -0.6945914030075073, -0.43357789516448975, 0.19932463765144348, -1.9467254877090454, 1.1873284578323364, 5.857563018798828, 2.188511371612549, 3.085836887359619, 2.0794565677642822, 7.6152448654174805, -0.748328685760498, -4.7463483810424805, -8.109304428100586, -0.7444981336593628, 1.6127647161483765, 3.929837226867676, -3.57114315032959, -0.8749943971633911, 5.273569107055664, -3.993967056274414, -2.058328628540039, -0.24996468424797058, -2.42924427986145, 0.5651423931121826, -5.141977310180664, -1.5281333923339844, -1.3164265155792236, 3.4260857105255127, 1.9870845079421997, 2.1594645977020264, -2.7990756034851074, 4.15952205657959, -0.20959016680717468, 4.231685638427734, -4.477293014526367, -3.200448513031006, -1.7690074443817139, 4.476011276245117, -2.996784210205078, 0.01935734413564205, -1.2446699142456055, 2.8905818462371826, -0.26991698145866394, 5.149232387542725, -2.653592586517334, 6.655020713806152, 0.2507984936237335, -3.456958770751953, 2.423038959503174, -0.03300590440630913, -2.458303213119507, 4.259052753448486, -1.2355881929397583, -3.349860906600952, 1.1624648571014404, 2.537724494934082, 2.354706048965454, -4.2484660148620605, 5.000319957733154, 0.8977272510528564, 1.9600496292114258, -3.1792123317718506, -2.361321449279785, -1.5633211135864258, 0.17749300599098206, 6.764020919799805, 0.8089392185211182, 0.578702449798584, 4.34767484664917, 4.961559772491455, 2.92781662940979, -1.104051113128662, -4.7761945724487305, -6.686727046966553, -0.9499024152755737, -1.9956326484680176, 2.6642563343048096, 6.098775863647461, -1.196887731552124, -0.49849367141723633, 0.28782588243484497, -5.540301322937012, 0.3915616571903229, 3.2055444717407227, 8.790404319763184, -3.7601773738861084, 3.1908369064331055, 5.507821559906006, -2.8060643672943115, 0.029441379010677338, 4.956027984619141, -1.2254830598831177, -0.2405276745557785, 1.6515474319458008, 4.73173189163208, 2.1518850326538086, -0.6053450107574463, 2.8840174674987793, -2.1100220680236816, -3.3064942359924316, -0.23033608496189117, 0.14967331290245056, 1.1513190269470215, 1.3640679121017456, -0.890438437461853, -5.130709171295166, -4.837421894073486, -0.48825603723526, 3.006925582885742, -2.2218916416168213, -1.1451456546783447, 2.5191280841827393, -3.3389315605163574, 2.7604262828826904, -0.9209916591644287, -1.0519907474517822, 2.467799186706543, -1.370574712753296, 3.5493686199188232, -2.770326852798462, -2.0603103637695312, -0.9699299335479736, 2.757746696472168, -2.448823928833008, 1.3533680438995361, 1.7513998746871948, 3.8976922035217285, -1.2696163654327393, 1.6861735582351685, 3.2056796550750732, 6.491445064544678, 0.4550279974937439, 1.8465452194213867, 0.9980483055114746, -9.697568893432617, 0.6916589140892029, -0.10808822512626648, -3.3070969581604004, -2.457318067550659, -0.39366859197616577, 0.458431601524353, -0.0473051443696022, -1.392484188079834, 0.2809523344039917, 2.934784173965454, -0.2499513328075409, 4.06783390045166, 4.985625267028809, 3.4674875736236572, -1.1446850299835205, -0.34371712803840637, -7.8975019454956055, -0.630179762840271, -1.1275075674057007, -2.700974225997925, -7.249587059020996, 2.397892713546753, 3.134472370147705, -0.8150675296783447, 2.473947286605835, -1.245166540145874, -6.3629865646362305, -0.42610299587249756, -1.6219487190246582, -1.05472993850708, -0.580915629863739, -3.6410300731658936, -5.229311943054199, 2.958632230758667, 0.13846194744110107, 1.074138879776001, -3.7171549797058105, -8.363102912902832, -4.868255615234375, 3.617926597595215, 1.3845230340957642, -6.535262107849121, -2.491147994995117, -4.027738094329834, -0.058454498648643494, 5.372337818145752, -1.9835447072982788, 1.1828895807266235, -0.5460717678070068, -1.4994720220565796, -3.177786111831665, -2.0590052604675293, 1.7748795747756958, 1.4051098823547363, 2.2737114429473877, -2.3203234672546387, -2.0156948566436768, 0.5026163458824158, -0.4457554221153259, -2.673490285873413, -1.4969396591186523, 5.0238423347473145, -0.42076143622398376, 1.8057729005813599, 2.9918127059936523, -0.26830992102622986, 5.6516642570495605, 1.1015032529830933, 2.704451084136963, -1.0169315338134766, 2.4591376781463623, 0.14894506335258484, 2.199732542037964, -1.5808056592941284, -1.5314934253692627, -0.01685134507715702, 2.537085771560669, -3.763014316558838, 0.5036770701408386, 0.0536905936896801, 3.637810707092285, -6.675723552703857, -4.285887718200684, 2.6924901008605957, 1.3743449449539185, 1.934133529663086, 0.4320828914642334, 1.3462146520614624, 1.729939341545105, 3.09513521194458, -3.5653724670410156, -2.146125555038452, 3.7128043174743652, 3.6744565963745117, -2.5872299671173096, 5.570269584655762, 2.01334547996521, 3.095396041870117, -0.35679373145103455, -2.2832374572753906, 0.3717454671859741, 0.08250166475772858, -3.115853786468506, 0.5568225979804993, 0.3657523989677429, -4.793513298034668, -2.2592880725860596, -0.19343622028827667, -2.202782392501831, 2.235626459121704, 3.7853143215179443, -3.015531539916992, 0.8373115062713623, 1.3945499658584595, 2.805265426635742, -4.528248310089111, -2.9093732833862305, 1.2460600137710571, 3.5976297855377197, 1.0597788095474243, -2.4756569862365723, 1.555506706237793, -2.1062421798706055, -4.307600498199463, 0.5259764194488525, 2.90394926071167, 3.173201084136963, 0.776964545249939, -2.1109535694122314, -3.461060047149658, 2.1182773113250732, -2.8692450523376465, 0.9141871333122253, 1.2114912271499634, -1.972175121307373, -0.4796537756919861, -0.11562111228704453, 2.250182628631592, -1.8795483112335205, -0.26866286993026733, 0.7433636784553528, -3.5435633659362793, -1.357582688331604, -1.4361894130706787, 0.5910828113555908, -3.3178672790527344, -0.07205156981945038, 1.0147743225097656, 4.300663948059082, 0.2485610842704773, 4.7500457763671875, 1.4599583148956299, 2.196432590484619, -2.0010361671447754, -1.6284366846084595, 1.5496233701705933, -1.729689121246338, 4.936856269836426, 0.6495477557182312, 8.084184646606445, -1.5619144439697266, 2.0152740478515625, 1.7074601650238037, 0.6728634238243103, 1.350814700126648, -6.136946678161621, 0.37659019231796265, 3.314347743988037, -0.05922103673219681, -3.7771196365356445, -1.0533033609390259, 0.2349156141281128, 2.761070966720581, -1.8644903898239136, 3.4433445930480957, -2.133192539215088, 2.9558539390563965, 0.8250714540481567, 6.963400840759277, 2.0434534549713135, 3.7403125762939453, -0.9043541550636292, -5.76878023147583, 0.138761967420578, 0.2732764780521393, -2.0907063484191895, -0.08048029243946075, -5.576114177703857, -0.8459174036979675, -1.897656798362732, 4.154900550842285, 0.05221203714609146, 0.1073053628206253, -0.06591282784938812, -5.890035629272461, -3.117208957672119, 3.5600781440734863, 1.5774152278900146, -4.302385330200195, 0.5157992839813232, -1.468468189239502, 1.0652151107788086, 0.03626609593629837, -0.3863644301891327, -4.286555290222168, -1.284691333770752, -2.843655586242676, -1.1427741050720215, 2.3601531982421875, 0.8161393404006958, 1.316002368927002, 1.9964121580123901, -3.774325370788574, -1.108214259147644, 2.6877846717834473, 3.305694580078125, -6.868706703186035, -3.4229297637939453, 0.7877650260925293, 8.782577514648438, -2.444004535675049, -0.8387154340744019, 0.25610417127609253, 2.239595651626587, -4.54754638671875, -2.2788329124450684, -1.8297619819641113, -2.436785936355591, -2.048210620880127, 1.1208040714263916, 3.380302906036377, 1.4354815483093262, 0.3731970489025116, -0.500973105430603, -2.392193078994751, -4.089657783508301, -3.9792873859405518, 3.025529384613037, 3.287148952484131, 1.3248231410980225, 0.6263795495033264, -0.15596485137939453, 1.1238579750061035, -1.5347669124603271, -3.039196014404297, -4.451912879943848, 0.0897960513830185, -3.1734938621520996, -5.913171291351318, -1.5353840589523315, -2.8034632205963135, 2.100128412246704, 4.115719318389893, 1.5817632675170898, 2.469883441925049, 5.735170364379883, -3.9396567344665527, -3.7838823795318604, -0.0014668912626802921, -1.1948511600494385, 0.7000595927238464, 3.4009084701538086, -0.10734997689723969, -5.285292625427246, 8.424520492553711, -3.18792986869812, -3.870642900466919, 3.294055938720703, -5.1097917556762695, -3.5631861686706543, 0.050832413136959076, -2.4610085487365723, 0.44626951217651367, 3.3724985122680664, 0.2770649492740631, -1.1919299364089966, 1.6426901817321777, 0.9436880350112915, 8.728849411010742, -2.88822603225708, 3.9295692443847656, -0.802895188331604, 1.28279447555542, -6.085681915283203, -3.3429312705993652, -3.797788381576538, -5.263134002685547, -2.0071229934692383, 4.527755260467529, -5.145656108856201, 4.059610366821289, -0.6572115421295166, -0.19311770796775818, -4.504199504852295, 0.3568471372127533, 4.125539779663086, -2.0304160118103027, -0.01343998871743679, -0.3053247332572937, -1.5292786359786987, -3.2956881523132324, 3.285191774368286, 2.1610031127929688, -2.344552516937256, 3.3801205158233643, -2.889657974243164, -0.009486999362707138, -5.1214728355407715, -6.121512413024902, -1.8196444511413574, 5.045029640197754, 0.1684776097536087, 3.1479034423828125, 5.151659965515137, 0.928112804889679, -2.6096386909484863, 0.21698923408985138, -1.6470777988433838, 1.5079758167266846, -1.8092477321624756, 4.1033172607421875, 3.3022232055664062, 0.6676655411720276, 0.28326717019081116, 1.4408485889434814, -0.16258570551872253, 0.5723700523376465, 2.927112340927124, -2.0510356426239014, -1.204682469367981, -0.6408458948135376, -6.761760711669922, 4.048547267913818, -0.9367622137069702, 4.537022113800049, -6.105169773101807, -6.385697364807129, -1.0947242975234985, 0.20295634865760803, 2.971503257751465, 2.477684259414673, 4.144073486328125, 0.2599012851715088, 7.741042137145996, 2.5961248874664307, -5.307064056396484, -2.4481687545776367, -2.4956769943237305, -0.341646671295166, -2.107077121734619, 0.7944737076759338, 3.82662296295166, -4.656075477600098, -1.9375855922698975, 5.465847015380859, 5.3969807624816895, 5.741310119628906, 0.8621336817741394, 3.9387524127960205, 1.3539845943450928, -5.02574348449707, -1.9992554187774658, 2.194096565246582, 0.32399702072143555, -2.785815715789795, -3.097524404525757, 1.2755237817764282, 0.7308282852172852, -1.7200291156768799, 4.71987247467041, -1.513361930847168, -1.4523656368255615, 0.07748237252235413, -0.26455533504486084, -0.4241548776626587, -6.173983097076416, 7.354460716247559, 1.844205379486084, 1.123155951499939, -1.834124207496643, -0.42515257000923157, 1.4001920223236084, -1.9738280773162842, 2.4962379932403564, 2.024229049682617, -0.9538483619689941, -2.255472421646118, 2.2982099056243896, 7.104372978210449, 0.7685821652412415, 3.0570666790008545, -0.35863393545150757, -2.445230007171631, 3.591602325439453, -0.8825499415397644, 6.72405481338501, -1.9240543842315674, -4.6262359619140625, 0.9724349975585938, 2.55210018157959, 2.6181042194366455, 0.33775103092193604, -3.6938066482543945, 3.459113121032715, -3.0554661750793457, 6.909707069396973, -1.6767754554748535, -0.5777999758720398, -0.8125600218772888, 4.31986141204834, 5.999420166015625, 1.4834617376327515, 1.5512034893035889, -2.8785648345947266, 3.0460846424102783, -7.088004112243652, 3.4910998344421387, 1.409209966659546, -2.5551352500915527, 2.521416187286377, 2.3803958892822266, 5.823943138122559, 5.149150848388672, 0.5941753387451172, 1.4502415657043457, -3.7869837284088135, 2.5332789421081543, -3.67614483833313, -1.712156057357788, -4.327173709869385, -1.6579957008361816, 1.5926120281219482, -2.5612576007843018, 0.19389846920967102, -3.762730836868286, 1.4454370737075806, 2.8096327781677246, 0.27128928899765015, -0.1292967051267624, -2.8089818954467773, 2.087496042251587, 3.6283092498779297, -0.6520634889602661, -0.06793783605098724, -1.432011604309082, -3.877645969390869, -0.3387937843799591, -4.800739765167236, 3.5040149688720703, 5.415378093719482, 2.8072140216827393, 1.0436806678771973, -0.3423055112361908, 0.3153015077114105, -3.7485036849975586, 0.7967909574508667, 0.2804521918296814, 2.0260045528411865, 5.464475154876709, 1.01813542842865, 5.128018379211426, 2.8043339252471924, -1.836189866065979, 2.3708503246307373, 0.5743811130523682, 2.1002187728881836, 2.1416537761688232, 1.66029691696167, 0.05994537100195885, 2.4162282943725586, -0.21163791418075562, -1.4965213537216187, 0.6342907547950745, -3.9381699562072754, 2.4277548789978027, 3.1533899307250977, 0.203786700963974, 4.815512657165527, 3.2833242416381836, 1.3348060846328735, 2.4890530109405518, 3.3704776763916016, -0.7138413190841675, 2.130495548248291, 2.7244670391082764, 2.214925765991211, -0.5825342535972595, 4.428563594818115, -5.828376293182373, -0.59039306640625, 0.5663408637046814, 5.247045993804932, -4.081376075744629, -2.995788097381592, -3.0832464694976807, -0.45888903737068176, -2.112851619720459, -3.0412838459014893, 2.8692567348480225, 1.7134898900985718, -0.03033190220594406, -1.968857765197754, -0.7991369962692261, -0.3334876000881195, 6.53054666519165, -3.1720714569091797, -1.4145402908325195, -1.4319828748703003, -0.41948527097702026, 2.3198301792144775, -4.043166637420654, -3.0457515716552734, 3.5510129928588867, 0.8950622081756592, 4.686794281005859, -2.001826047897339, -4.472990036010742, -1.700934648513794, -2.011687994003296, 2.079974889755249, -2.3721511363983154, 0.36740314960479736, 2.388578176498413, 0.5240899324417114, -0.8660933375358582, -0.7996355295181274, 0.8443524241447449, 2.1849205493927, -2.9304275512695312, -4.037013530731201, 4.7937822341918945, -4.444400787353516, -0.8208663463592529, -2.5592637062072754, 3.9585323333740234, 0.8484587669372559, 4.148434638977051, 4.02545690536499, -1.9304959774017334, 0.7460548877716064, -4.738768577575684, -1.4565337896347046, 2.425973653793335, 1.514341950416565, 1.3665612936019897, 0.33437997102737427, -1.1779377460479736, 2.2106823921203613, 0.7225756049156189, 0.17281368374824524, 3.3045907020568848, -1.034726619720459, 5.865722179412842, -0.3814869523048401, 5.314264297485352, 0.8167893290519714, -1.1769521236419678, -0.35272449254989624, 2.4101040363311768, 1.7952450513839722, 2.222447395324707, 1.0120458602905273, -0.4129715859889984, -1.5602519512176514, -1.9300072193145752, -0.5483628511428833, -5.569794654846191, 0.8762952089309692, 0.3612491190433502, -0.19510701298713684, -2.443176031112671, 0.07686176896095276, 2.159947156906128, -3.0435190200805664, -2.9615511894226074, -0.8655437231063843, 0.7607476711273193, -3.4011549949645996, 2.179469585418701, -5.885104656219482, 5.087337493896484, -4.149683475494385, -3.1964657306671143, -1.781522274017334, -4.098760604858398, -0.620858907699585, -2.1644153594970703, 4.516632080078125, 0.08264294266700745, 2.734591007232666, 0.3870755732059479, 3.580965995788574, 1.4090232849121094, -4.1059250831604, -1.0513064861297607, 1.2895088195800781, 1.4049768447875977, -6.1286301612854, -2.7814502716064453, -4.84021520614624, -4.636353969573975, 0.30289143323898315, 0.26002734899520874, -6.954874038696289, -3.838674545288086, 3.0082297325134277, 3.5226666927337646, 2.450882911682129, -3.5228397846221924, 0.4268127679824829, 3.218996286392212, -0.23952291905879974, -1.3033314943313599, -2.0151071548461914, 0.9102728962898254, 2.730915069580078, 5.334861755371094, -0.34376728534698486, 0.21503300964832306, -4.615403652191162, 3.3839216232299805, -1.9953882694244385, 4.233131408691406, -2.3155179023742676, -0.30563420057296753, 1.9322952032089233, 0.7068777680397034, -7.114555358886719, -0.7500590085983276, 0.4556494653224945, 7.400458335876465, 4.133204460144043, 4.854126930236816, 2.172698974609375, 4.714848041534424, 0.4748043417930603, -3.8633503913879395, -2.0042569637298584, 1.4573936462402344, -1.6442124843597412, -0.11164945363998413, -0.44718995690345764, 4.43486213684082, -7.961745262145996, -3.898484706878662, 0.4848604202270508, -0.7636438608169556, 6.235790252685547, 2.9068033695220947, 0.7429648041725159, -1.7063913345336914, 0.477467805147171, -0.37315404415130615, -4.533694744110107, -2.6465859413146973, 1.7527213096618652, 1.127030372619629, 0.28034135699272156, 5.228947639465332, -2.744246244430542, -0.9682223796844482, 2.525290012359619, 3.682560443878174, -2.0981149673461914, 1.1218743324279785, -0.8632521629333496, -2.839836597442627, 0.33556920289993286, -2.0387940406799316, 2.936638593673706, 1.3109302520751953, -0.555238664150238, 3.7208099365234375, 4.439793109893799, -1.8561762571334839, -4.78564453125, -2.042302131652832, 0.41825157403945923, -7.055650234222412, -6.338194847106934, 3.5532593727111816, -0.00980444997549057, 1.5635014772415161, 0.17060410976409912, -1.572461485862732, 0.9790525436401367, -0.6769448518753052, 3.1826024055480957, -0.09969902783632278, 0.8204603791236877, -0.6845489740371704, -3.487886905670166, -2.0972039699554443, 1.9630002975463867, -2.788125514984131, 0.03649142384529114, -1.6618916988372803, -1.1070318222045898, 4.080337047576904, -3.8480937480926514, -3.035277843475342, -2.860383987426758, 3.611128807067871, 1.4854973554611206, -4.763971328735352, -2.8315834999084473, -0.9683437347412109, -1.2820749282836914, -2.12506103515625, 2.5233659744262695, -2.9333837032318115, 0.9991697072982788, -0.22446656227111816, -0.6144410371780396, -5.65725040435791, -3.0126256942749023, 3.764925003051758, 1.2695984840393066, -4.626637935638428, 0.6204768419265747, -5.254411697387695, 2.2542102336883545, -2.255279541015625, 4.85964822769165, 3.7111568450927734, 2.2322750091552734, -3.4752068519592285, 0.0847572535276413, -3.0600485801696777, -1.6778745651245117, -2.5665929317474365, -0.6459380984306335, 1.7013280391693115, 2.119311809539795, -6.508397102355957, -3.1750967502593994, -3.7278800010681152, 2.899068832397461, -3.2295851707458496, 1.6357452869415283, 0.30914098024368286, 2.3887250423431396, -1.0150495767593384, -1.6120104789733887, 2.459975004196167, 3.274111270904541, -0.4235384166240692, -1.1992089748382568, -1.5269274711608887, -0.7609471678733826, 2.834507942199707, -2.3886303901672363, 3.7522201538085938, -3.970088005065918, 0.47558829188346863, -2.420854330062866, 1.0242425203323364, 7.183432579040527, -2.1353001594543457, 3.615067481994629, -3.9994635581970215, 1.4591643810272217, 7.297143936157227, 1.0928239822387695, -4.803017616271973, -1.4662339687347412, 0.5209717750549316, -3.162287950515747, 4.927124500274658, 2.354907989501953, -2.194884777069092, 2.283726215362549, 1.0275583267211914, -5.0243706703186035, 5.168188571929932, 3.4333882331848145, -5.910398006439209, 3.9220824241638184, -0.11955497413873672, 2.326525926589966, -0.21847596764564514, -3.744095802307129, 0.29580196738243103, 4.512395858764648, -5.710634231567383, -0.325882226228714, -2.2071352005004883, 1.6637753248214722, -4.213650703430176, 1.308173418045044, -1.0757598876953125, 1.779820442199707, -0.20838569104671478, -3.1732540130615234, -0.2981736660003662, 1.0763553380966187, 0.5471369028091431, 4.729122161865234, -0.9160124659538269, 1.1345417499542236, 2.7562763690948486, 4.863527774810791, -1.3388710021972656, -0.9849520325660706, -0.45616573095321655, -2.7124381065368652, -3.447599411010742, 2.7993273735046387, 1.8026680946350098, 3.6428213119506836, 4.432980060577393, -1.9890294075012207, -0.46790623664855957, -2.089090585708618, -0.5010238885879517, 4.301692962646484, -3.8727550506591797, 4.073102951049805, -1.9942498207092285, 1.8085391521453857, 6.395342826843262, 3.827479362487793, -3.848745346069336, 0.5218247175216675, 4.0489420890808105, -3.793189525604248, -3.8739514350891113, 0.4435831308364868, -3.8642659187316895, -3.879373550415039, -0.3210386037826538, 3.4946374893188477, 0.9550182223320007, 3.4072458744049072, 1.1345402002334595, 1.0177984237670898, 2.6218504905700684, -1.459275722503662, -0.4840925931930542, -0.5529942512512207, -0.9328112602233887, -3.0093016624450684, -0.9459026455879211, -1.051745891571045, 3.118594169616699, -2.1468007564544678, 2.4799976348876953, -1.6928441524505615, 5.419940948486328, -2.970942974090576, -9.199453353881836, 2.0120115280151367, 1.4386870861053467, 1.3515053987503052, 4.6110334396362305, 3.5002403259277344, -1.466349482536316, -2.5836105346679688, 0.524182915687561, -1.9963593482971191, -1.5647943019866943, -1.4789996147155762, 3.0945396423339844, 3.417020320892334, 0.4544880986213684, 3.139364242553711, -1.9346950054168701, -0.2304268479347229, 2.3894004821777344, -1.8415749073028564, 4.370361804962158, -0.36070701479911804, 0.4579821228981018, 2.6009392738342285, -4.246002197265625, 1.440232753753662, -1.0635918378829956, -5.07960319519043, -0.8867959976196289, 1.9829374551773071, -6.408688545227051, 1.0405387878417969, -6.303423881530762, -2.136645555496216, -0.6793689727783203, 4.712959289550781, 0.6270684599876404, 0.48814743757247925, 1.82175874710083, 2.4628689289093018, -0.019544079899787903, 0.5985321402549744, 1.2326278686523438, -1.1202698945999146, 3.15313982963562, 1.4989614486694336, 2.654454231262207, 2.9902524948120117, -2.1675078868865967, -1.0093799829483032, 0.9734163880348206, -1.023787498474121, -0.8709572553634644, -0.7548038959503174, -0.4526875615119934, 2.5839760303497314, -6.287066459655762, 1.9305641651153564, -3.2165913581848145, 4.964658260345459, 1.1445584297180176, 2.879730224609375, 1.9146606922149658, 0.5208349227905273, -1.6082050800323486, 0.37073618173599243, 3.491395950317383, -1.543007254600525, -0.9764347076416016, -1.2215297222137451, 0.6103104948997498, -0.750292181968689, 2.830432415008545, -3.397742986679077, -3.0449843406677246, 2.1834957599639893, 2.927086591720581, 1.9140527248382568, 5.531630992889404, 3.2777838706970215, -2.0431928634643555, -3.0854897499084473, -1.0565820932388306, -1.330636978149414, 1.020995020866394, -1.2186408042907715, -1.1457487344741821, -0.6361892819404602, 0.642918586730957, 1.977759838104248, 1.0855183601379395, 3.7312536239624023, 1.5621707439422607, 1.2387299537658691, 3.9203109741210938, 6.93364143371582, 0.07239115983247757, -2.4549076557159424, -3.350271224975586, 3.0600006580352783, -4.71980619430542, 0.9661836624145508, 3.9946093559265137, -5.833829879760742, 0.981685996055603, 2.592205047607422, 1.4263594150543213, 1.0425031185150146, -5.674142360687256, 0.2931091785430908, -4.1076741218566895, -1.9130254983901978, 0.6063982248306274, 2.5550270080566406, 1.4932026863098145, 4.006112098693848, -2.8011021614074707, 2.3596885204315186, 0.4723634123802185, -1.235741376876831, -0.5609197020530701, 2.6089162826538086, 3.802656412124634, -3.5550622940063477, 5.075231075286865, -2.0979199409484863, 0.6474978923797607, 1.3386943340301514, -0.43340492248535156, -5.668698787689209, -0.7530378103256226, 2.0627126693725586, -0.5528292655944824, 2.1983180046081543, 0.6476491689682007, 2.0695395469665527, 2.998013734817505, 7.284992694854736, 2.4979214668273926, 4.607135772705078, 0.039220042526721954, 2.6358180046081543, -2.8631136417388916, 2.9786715507507324, 3.500260353088379, -2.6744747161865234, 2.7468833923339844, 1.7701869010925293, 3.3481311798095703, 1.7116516828536987, 0.14498673379421234, -2.457458734512329, 0.3099973797798157, 1.703244686126709, -2.6767096519470215, -1.8318214416503906, -6.263891220092773, 0.044224709272384644, 2.0879971981048584, 3.079728841781616, -5.053199768066406, 0.8646367788314819, -3.142070770263672, 4.338613510131836, -2.8793797492980957, -3.9296798706054688, 3.4647750854492188, -2.2753520011901855, -0.194393128156662, -4.79631233215332, 6.295961380004883, -4.072157859802246, -2.4634170532226562, -2.6875414848327637, 0.12543584406375885, -3.8513753414154053, -0.44795364141464233, 1.425447702407837, 7.101717948913574, 6.0351362228393555, -1.4126975536346436, 1.2135858535766602, -0.7519705295562744, -5.261448860168457, -2.6410043239593506, -0.36629581451416016, -1.1441712379455566, -0.12837080657482147, 3.1062893867492676, 2.2660224437713623, 1.1897339820861816, -1.9307332038879395, -0.2173905372619629, 3.266659736633301, 1.8266047239303589, -4.371100902557373, 1.5359998941421509, -0.18845808506011963, -1.0621492862701416, 2.8173341751098633, 0.0035728481598198414, -1.4803516864776611, 0.9860209822654724, -2.087364912033081, 0.5081250667572021, -3.3959603309631348, 2.5151290893554688, -4.096804618835449, 0.6997780799865723, -3.355365753173828, 2.7998082637786865, 1.7422783374786377, -1.851036787033081, 0.6331477761268616, 2.447505474090576, -2.6370744705200195, -1.343996524810791, 1.7737637758255005, -5.880128860473633, -6.510551452636719, 0.3662383556365967, -1.4392235279083252, -5.423959732055664, -2.5838918685913086, 1.504263997077942, 0.654597282409668, -4.096362113952637, 3.1267175674438477, 1.0522866249084473, -1.0757718086242676, 2.6655662059783936, 2.6070189476013184, -0.6789492964744568, -2.7681496143341064, 2.9350006580352783, 4.121293544769287, 0.9097042679786682, -3.7127861976623535, -2.0064172744750977, 5.983834743499756, -2.5073962211608887, -0.6565704345703125, 0.30573025345802307, -1.7305829524993896, 2.0026798248291016, 1.8895686864852905, -1.7116937637329102, 1.5789070129394531, 1.8252387046813965, 5.372069835662842, 6.424346923828125, -3.1077654361724854, -0.48706671595573425, 1.446720838546753, 3.336491107940674, -2.253244400024414, -4.919094085693359, 3.106891632080078, -1.6715869903564453, -1.2235174179077148, -0.7870014309883118, -1.1390571594238281, 3.444640636444092, -5.9979424476623535, 3.345761299133301, 3.6065640449523926, 0.6797792911529541, 1.6429612636566162, -3.333650588989258, 1.1432554721832275, -0.0012289633741602302, -3.1957435607910156, -0.7143978476524353, -0.34440484642982483, 1.4611166715621948, -0.8249890804290771, -2.776456594467163, -0.6424596905708313, -2.9278063774108887, 2.150670051574707, 1.4274060726165771, 2.035290479660034, 5.192376613616943, 3.5489840507507324, 0.22739584743976593, -2.1803321838378906, 0.6127755045890808, 13.75918960571289, 0.30413079261779785, 1.4651286602020264, -0.9632712006568909, 3.173818588256836, 1.1619763374328613, 3.3811612129211426, 0.775010347366333, -0.8870410323143005, 6.7486162185668945, 1.6738110780715942, 1.4476923942565918, 0.8907853960990906, -1.8448097705841064, -2.5931522846221924, 1.9329848289489746, -2.6088218688964844, -3.128060817718506, -2.78255033493042, -0.5579278469085693, -7.905552864074707, 0.3639400601387024, -1.3682310581207275, 5.288823127746582, 1.498401165008545, -5.110146522521973, 5.96867561340332, 0.3080344796180725, -3.2135820388793945, 2.1223692893981934, -3.011284828186035, 4.093041896820068, -5.365435600280762, -3.530623435974121, -1.1908879280090332, 1.0613560676574707, 2.7310848236083984, -2.2721824645996094, -3.812089204788208, -3.316042900085449, 0.9985329508781433, 2.4834117889404297, -6.6549224853515625, 0.47183024883270264, 3.3356902599334717, -0.11031292378902435, -1.5713982582092285, 1.5108747482299805, 2.755664587020874, 0.22889813780784607, -3.358079433441162, 2.0161960124969482, 7.269454002380371, 2.917818069458008, 2.214320659637451, -3.569100856781006, -1.8828322887420654, 1.3520991802215576, 1.0295002460479736, -0.5913525819778442, -6.640752792358398, -1.4285346269607544, 1.7221102714538574, 8.37233829498291, 0.98195481300354, 0.41224581003189087, 4.091499328613281, -0.772993803024292, -0.3811767101287842, 1.691838026046753, 1.804713249206543, 2.4906110763549805, 2.8263392448425293, 1.3349677324295044, -1.2462230920791626, -0.43303579092025757, 1.9363818168640137, -1.0453351736068726, -2.3588860034942627, 1.024857521057129, -7.775726795196533, 2.6858723163604736, 0.37266895174980164, -1.8008427619934082, -1.6380833387374878, -1.6303707361221313, 2.4290382862091064, -2.174874782562256, 2.0183985233306885, 0.5359520316123962, -0.12945103645324707, 1.9085156917572021, -1.9384685754776, 8.568320274353027, 1.642398476600647, -0.71352219581604, -4.605754375457764, -1.171010971069336, -2.110790729522705, -1.9456822872161865, -1.064229965209961, 5.735286235809326, 2.2914178371429443, 3.3923771381378174, 0.7201794385910034, -3.269904613494873, -0.640809953212738, -2.578946590423584, 0.7305049896240234, 1.0802384614944458, 5.811493873596191, -3.242828607559204, 3.2966742515563965, -0.49001049995422363, 1.444031000137329, -2.48057222366333, 6.061091423034668, 1.1058496236801147, 2.6262524127960205, 6.032759666442871, 0.5592308044433594, -0.12703436613082886, 1.5750800371170044, -1.1783021688461304, -3.2435407638549805, 2.5445263385772705, 1.2687243223190308, -3.306321859359741, 2.4237308502197266, -2.2978475093841553, -3.7698991298675537, -0.6269897222518921, 0.5252600908279419, -0.867129385471344, -0.7749685049057007, 2.9252285957336426, 3.160581350326538, 0.4832209646701813, -5.634134769439697, -1.3338067531585693, -7.64406681060791, -5.251045227050781, -0.7232096791267395, 1.7554042339324951, 3.789515495300293, 1.1657856702804565, 4.356806755065918, 4.502626895904541, -2.7405738830566406, -1.3679206371307373, 0.6197890639305115, -3.194558620452881, 0.21496975421905518, -1.9622021913528442, -2.8506035804748535, -7.26295547792688e-05, 4.135442733764648, -4.482789993286133, -1.768623948097229, -0.7347208261489868, 1.9823366403579712, -0.2686211168766022, -4.242335796356201, 2.1787326335906982, 1.5919737815856934, 2.154531478881836, 2.7591135501861572, 1.446336269378662, -6.615321636199951, -3.7593884468078613, -0.7104146480560303, 0.6581434011459351, 1.4203131198883057, 1.9179050922393799, -2.4995546340942383, 0.8331028819084167, -0.67234206199646, 1.5057556629180908, 5.354123115539551, -2.1607613563537598, 2.2176737785339355, -1.3964290618896484, -0.34273970127105713, -1.962140679359436, 0.18658730387687683, 1.5304396152496338, -5.923877716064453, 5.405308246612549, 4.141708850860596, 5.852313041687012, 2.0766220092773438, 2.9713449478149414, -0.9792580604553223, 1.8277170658111572, -1.728241205215454, 4.935532093048096, -0.08850023150444031, 3.1993274688720703, 0.7165915966033936, 3.8092153072357178, 1.9397989511489868, -3.6852340698242188, -5.678923606872559, -1.2568857669830322, -0.44675809144973755, -3.8027567863464355, 2.654559850692749, 3.0558836460113525, 2.178593158721924, 2.048750638961792, 1.1314600706100464, -2.1026902198791504, 1.6848475933074951, -5.256486892700195, 0.603830099105835, 2.6071090698242188, -5.369296550750732, 5.919525623321533, -2.85095477104187, 3.293336868286133, -0.8982611894607544, 0.3068472743034363, -2.0860109329223633, 0.05627059191465378, -0.147230327129364, -1.7724897861480713, 3.238830089569092, 1.922174334526062, 0.9899671077728271, -3.2840309143066406, 1.796297550201416, 3.9885878562927246, 3.293059825897217, 3.486945629119873, 1.5985889434814453, 1.938256025314331, -2.6971588134765625, 2.153735637664795, -0.864534854888916, -2.980086326599121, 1.0819015502929688, 1.011648178100586, 1.6964346170425415, -2.55366849899292, 0.8355200290679932, 4.0046186447143555, 2.0405635833740234, 1.1427364349365234, 1.2752951383590698, -1.1144874095916748, 1.9604462385177612, -5.712375164031982, -3.114182472229004, -4.274670124053955, -6.852761268615723, 1.8067595958709717, -1.1716009378433228, -0.07356846332550049, -0.04903123527765274, -1.8466633558273315, 1.8697954416275024, -2.190950870513916, 0.8367296457290649, -0.8925321102142334, -3.025686740875244, 1.1235246658325195, 4.851652145385742, 3.2296364307403564, 4.587552070617676, -2.0082666873931885, 4.848823547363281, -5.489505767822266, 0.8377349376678467, 1.3409984111785889, 1.957517385482788, 1.617954969406128, -0.5859065055847168, -1.4175658226013184, 1.0950653553009033, -0.8947178721427917, -5.018566131591797, 6.590353488922119, -1.0869543552398682, 3.7149791717529297, 0.15877088904380798, -1.46224045753479, -0.24911029636859894, -0.7725542783737183, -1.684922695159912, -0.9187454581260681, -2.2341256141662598, 0.9345773458480835, 0.466061532497406, 0.7099207639694214, -4.42098331451416, 1.0084017515182495, 4.9770355224609375, -3.389889717102051, 0.38828667998313904, 3.920344591140747, -0.16172006726264954, 3.695742130279541, -1.0364936590194702, -6.881416320800781, 6.473623275756836, 2.645740509033203, -4.2763261795043945, -0.22869408130645752, 0.002098959172144532, -3.270556926727295, -6.91298770904541, 0.7834093570709229, 0.9332190752029419, -4.313344955444336, -5.510045051574707, 3.3713643550872803, 0.7613904476165771, -1.273672342300415, 2.566087484359741, -0.019657723605632782, -3.9129910469055176, 0.08537939190864563, 6.959698677062988, -2.2948334217071533, 2.5012009143829346, -0.4801536798477173, 1.4772180318832397, 1.197969913482666, 2.6810860633850098, 3.541245460510254, -0.5162999629974365, 4.6185760498046875, 1.2240846157073975, -3.5414071083068848, 8.787013053894043, 1.028868556022644, -2.1449313163757324, 2.2310972213745117, 1.721643090248108, 5.198605537414551, 4.802455425262451, 2.093501091003418, 1.789965033531189, 2.828962564468384, 2.883676052093506, 2.741591453552246, 2.617225170135498, -1.8098547458648682, 3.437519073486328, 3.7587037086486816, 3.32368540763855, -1.6841740608215332, 0.21083301305770874, -7.562616348266602, -6.631097316741943, 0.4715315103530884, 1.6352366209030151, -0.07243143767118454, 0.9011164903640747, 2.327533721923828, -2.1428792476654053, 2.237982749938965, 2.956651210784912, -1.770922064781189, 2.9929592609405518, -1.3650703430175781, 0.5464259386062622, 0.29132938385009766, -0.0374864861369133, -6.560369968414307, 5.153707027435303, -3.903130531311035, 0.988350510597229, -0.39704465866088867, 0.17660115659236908, -2.5826170444488525, -0.3838149309158325, 0.4493730068206787, 2.0549235343933105, 3.5351734161376953, 0.7159815430641174, -1.4042978286743164, 1.5214686393737793, 1.2889231443405151, 2.592782497406006, -2.0810739994049072, 1.735579252243042, -5.90532112121582, 2.716796636581421, -0.1298724114894867, 3.5922746658325195, 0.8621349334716797, 3.107788324356079, 4.220473766326904, -3.503988742828369, -3.9905076026916504, 2.619563341140747, 1.4349956512451172, -2.621636390686035, -1.834325909614563, -1.9376174211502075, 1.7412798404693604, -3.499143123626709, -4.239444732666016, -1.4178282022476196, -2.6642439365386963, 1.5216848850250244, 1.1564230918884277, -3.2919821739196777, 1.8096259832382202, 2.127880573272705, -0.7984544038772583, -2.4601902961730957, -4.275071620941162, 0.8278396725654602, 3.8940200805664062, 3.200733184814453, 1.1528035402297974, 8.329410552978516, -3.6143641471862793, 2.3651750087738037, -2.041489601135254, 1.3057141304016113, -1.9379160404205322, -0.9602981805801392, 2.1156911849975586, -3.9681522846221924, -1.7299261093139648, -3.239260673522949, 0.7473962306976318, 1.3978891372680664, 1.9272340536117554, -4.742293357849121, -4.425704002380371, 0.743776798248291, 0.47623202204704285, -2.9255075454711914, 0.45885151624679565, 0.7792177200317383, -1.4203730821609497, 8.745429039001465, -2.5787577629089355, 3.202526569366455, 0.11694951355457306, 0.2761520445346832, -2.7910284996032715, -1.2347917556762695, -0.8324283361434937, -1.4225760698318481, 1.5359926223754883, 2.171976327896118, 0.3705250918865204, 2.081470489501953, -0.733849048614502, -2.2943813800811768, -2.1587672233581543, -2.7309629917144775, -1.4588783979415894, -1.1687276363372803, 1.039677619934082, -1.1871038675308228, 3.219461441040039, 2.4367949962615967, -0.3920082449913025, -3.6486289501190186, 0.05872737988829613, 0.6173148155212402, 2.047720432281494, 2.2246365547180176, 1.5299193859100342, 1.3909902572631836, 2.277130603790283, 5.392853260040283, 0.3689517378807068, -1.0505547523498535, 5.529492378234863, 2.095825672149658, -2.4465880393981934, 0.14068840444087982, -1.412631869316101, 4.210240364074707, -2.5199027061462402, -1.6669528484344482, -3.028177261352539, -4.732202529907227, 2.4883999824523926, -6.807187557220459, 1.747525930404663, -1.2178727388381958, 3.427412509918213, -0.4931415319442749, 4.037416458129883, 4.4326300621032715, 4.914935111999512, 2.0473175048828125, 1.983829379081726, 2.335440158843994, -6.142233848571777, 6.157347679138184, -0.17162594199180603, -0.48374059796333313, 1.0690791606903076, 1.9296777248382568, 0.22755856812000275, -2.3317184448242188, -1.4062697887420654, 3.8094191551208496, 0.5097700357437134, -1.0381029844284058, -2.4217700958251953, -3.005354404449463, -1.4490457773208618, 1.0269131660461426, -0.044303469359874725, -4.397285461425781, -2.856745958328247, -5.885978698730469, -0.21382994949817657, 1.9432032108306885, 1.1461076736450195, 4.859134197235107, -1.372785210609436, 5.290509223937988, -3.4672203063964844, 0.6710311770439148, -1.2580642700195312, 0.265890896320343, 1.8159546852111816, -2.307363271713257, -5.985575199127197, -2.9015607833862305, -1.780233383178711, -2.7118940353393555, 4.876611232757568, -1.365529179573059, -2.4447760581970215, 2.0526769161224365, 0.8173143863677979, -1.2458159923553467, -1.678330898284912, 1.3239550590515137, 0.8610730171203613, 2.052551746368408, 1.4120081663131714, -3.542060136795044, 2.337697744369507, -5.252186298370361, -4.94460916519165, -5.363371849060059, -1.5672557353973389, 5.701196193695068, -0.3938942551612854, -0.0632687658071518, -1.0817975997924805, 3.714606285095215, -5.892847061157227, 0.8794360756874084, 1.8139182329177856, 4.471473693847656, -0.012733438983559608, -0.8893151879310608, -1.1922516822814941, -5.173192977905273, -0.9764641523361206, 0.07231304794549942, 1.3100378513336182, 2.238567590713501, -4.315731525421143, -3.287916898727417, 1.019972324371338, -4.094533920288086, 5.551475524902344, -4.5967912673950195, -0.638713538646698, -2.6308515071868896, 1.9604912996292114, -2.3247289657592773, 0.5792096257209778, -0.9148069024085999, -1.5091357231140137, 0.6013149619102478, 3.4288878440856934, 1.3910244703292847, -0.7786325216293335, -4.098464488983154, -2.452515125274658, -5.1351494789123535, 3.008378028869629, -3.329012870788574, 1.5878723859786987, 3.577322244644165, 6.122114181518555, 3.169887065887451, -2.521348237991333, 0.32135286927223206, -0.8778862953186035, -3.9391674995422363, -6.17479133605957, 3.2846169471740723, -1.0796643495559692, 1.7231229543685913, 1.306506633758545, -0.9831881523132324, -0.604171872138977, 3.3988699913024902, 3.8470394611358643, 3.689035177230835, 0.5854009389877319, 1.6894197463989258, -0.5848245024681091, -5.0235161781311035, 3.229637622833252, -3.5673258304595947, -1.3590826988220215, -3.5933923721313477, 0.9095853567123413, 5.096177577972412, 4.087162017822266, -1.1138532161712646, 3.8428258895874023, -4.8185930252075195, -0.6378847360610962, -2.951510429382324, -1.3023474216461182, 1.457840919494629, -4.929436683654785, 4.093023300170898, 3.2722272872924805, 2.1980600357055664, 0.13631123304367065, 2.6200716495513916, 3.4550037384033203, -0.7648136615753174, -5.118715286254883, -1.4913253784179688, -0.1121048778295517, -5.809477806091309, 1.5404000282287598, -1.1981343030929565, -1.1760190725326538, 3.642179489135742, -2.0193374156951904, -2.903062582015991, 1.2182890176773071, -2.663133144378662, 2.719882011413574, -2.103498697280884, -1.0875940322875977, 2.3033785820007324, -1.4243088960647583, -2.771969795227051, 2.834261417388916, -5.028210163116455, -2.8598780632019043, 2.044370651245117, -0.3960680365562439, 2.5347352027893066, -2.25126051902771, 1.1626118421554565, -1.146109700202942, 4.420820236206055, 3.354918956756592, 4.298416614532471, 2.7339086532592773, 0.9920935034751892, 0.706993043422699, -1.0358901023864746, 1.1207321882247925, -2.106468915939331, -0.3465058207511902, -5.216803073883057, 1.5449740886688232, 0.29158225655555725, -0.5265811085700989, -1.1198023557662964, 0.3202555179595947, -2.447624921798706, -2.2389116287231445, 4.088808059692383, -7.060935020446777, 2.0560503005981445, -3.045775890350342, 1.0440305471420288, -3.729228973388672, -1.410394310951233, 5.132673263549805, -3.149538516998291, -2.2362093925476074, 0.880591630935669, -0.6773179769515991, -2.176632881164551, -6.381626129150391, -2.2438926696777344, -2.3162946701049805, 1.5169012546539307, 0.5646969676017761, -5.444185256958008, 0.31014472246170044, 2.525928497314453, 2.6172945499420166, 0.7019385099411011, 5.392297744750977, -4.709995269775391, -5.1869964599609375, 5.071333885192871, -0.41705721616744995, 1.3201984167099, -1.3128793239593506, -0.5334244966506958, 7.141348838806152, 5.10115385055542, 4.092454433441162, -0.543920636177063, 0.19216981530189514, -6.405994892120361, -4.192666053771973, -0.18547865748405457, 0.42738670110702515, -0.5737430453300476, 1.3120296001434326, 4.379448890686035, 1.1476531028747559, -2.7813892364501953, 1.2364240884780884, -0.12024872750043869, 2.7317376136779785, 0.9807796478271484, -2.1161911487579346, 0.6522577404975891, 5.295943260192871, 7.333163261413574, 4.700099945068359, -1.4918880462646484, 0.5793287754058838, 1.3901609182357788, -2.292938470840454, 0.5870069861412048, -2.454834461212158, -6.510903358459473, -0.03853287920355797, -1.290858268737793, 6.313950061798096, 2.6470139026641846, -1.605067491531372, -2.151181936264038, -2.04721999168396, 7.931980133056641, 3.7592132091522217, -0.7779240608215332, -3.9385743141174316, 4.046929359436035, -1.4064334630966187, -4.142358779907227, 1.5378952026367188, 0.5908442735671997, 1.0475636720657349, 0.807686984539032, -0.2621593177318573, 0.5913141965866089, -1.5580694675445557, 3.55036997795105, -3.8295364379882812, 3.0732526779174805, 0.2643736004829407, -3.933576822280884, 2.8401503562927246, 0.13954707980155945, 3.1378047466278076, 7.389458656311035, -0.5403811931610107, -1.3361287117004395, -3.7391605377197266, -4.407276153564453, -2.8590168952941895, 2.927257537841797, 3.420747756958008, -3.0151872634887695, 3.8204855918884277, 3.4291398525238037, -1.8498940467834473, -2.143484592437744, -0.6226019859313965, -3.0087783336639404, -4.207091808319092, 0.5140079259872437, -1.2283046245574951, -6.8400373458862305, -3.397162437438965, -1.2437734603881836, 7.007020950317383, 1.9609391689300537, 0.49747657775878906, -0.6787528395652771, 1.5881335735321045, -1.2500977516174316, -2.693458080291748, 6.803912162780762, -4.809196472167969, -0.14169266819953918, 7.221031665802002, 3.2893314361572266, 4.941437721252441, -6.345349311828613, 5.372971057891846, 2.7521591186523438, -4.384771823883057, 7.54117488861084, 5.623429298400879, -2.4030227661132812, -0.32165372371673584, 1.2693562507629395, -4.374439239501953, -1.2876852750778198, 2.955335855484009, 0.0006991742411628366, -0.28869104385375977, 0.5043078064918518, 0.8841979503631592, 1.127396583557129, -2.9985313415527344, 7.000984191894531, 1.9830983877182007, 1.172369122505188, -1.7235034704208374, -2.9038007259368896, 2.195847988128662, -3.5782477855682373, -3.5651135444641113, -1.369148850440979, 4.32562255859375, -0.5267982482910156, 2.249056339263916, -2.648470878601074, -1.7200548648834229, -2.6528851985931396, 0.40626585483551025, -4.402046203613281, -1.0634223222732544, -4.140668869018555, 2.2080488204956055, 2.2046966552734375, 1.408408522605896, -3.133021831512451, -3.456331729888916, -3.7487905025482178, 2.1113221645355225, -1.0343600511550903, 0.47513407468795776, 2.8083419799804688, 6.188894748687744, 0.3797253370285034, -0.1828819215297699, -1.9363571405410767, 3.3560357093811035, -2.416769504547119, 5.763148307800293, -0.921720564365387, -1.0014780759811401, 0.9327683448791504, 2.269859790802002, -0.17457625269889832, 0.9680496454238892, -4.005962371826172, 1.6682556867599487, -2.109140396118164, -0.4359847903251648, -1.0887880325317383, 1.269838809967041, 1.3074206113815308, -1.3088198900222778, 3.5802059173583984, -4.228392124176025, -10.214178085327148, 0.6872766017913818, 6.392617225646973, -6.375166893005371, -1.6363366842269897, -0.38637518882751465, -0.8815299272537231, -0.2132151871919632, -0.12579940259456635, 0.11613103747367859, -1.0530920028686523, 1.5290272235870361, -2.3175978660583496, -2.84897780418396, -4.1534104347229, -3.0744032859802246, -6.3623785972595215, -0.4941999614238739, -0.3360646367073059, 1.7633814811706543, -2.613447666168213, 1.6993293762207031, -0.6769549250602722, 3.9345812797546387, 0.7483916282653809, 0.7098761796951294, -2.094163656234741, 2.506070852279663, -1.3711661100387573, 4.242236614227295, 2.305055856704712, -2.6603903770446777, -0.08771839737892151, 1.1296491622924805, 4.518080711364746, 1.2183353900909424, 0.41126033663749695, 3.7594170570373535, -2.523651361465454, -3.241905927658081, -3.471348762512207, 0.9308483004570007, -4.102119445800781, 5.650815963745117, 0.5018149614334106, 2.3441414833068848, 1.049243450164795, -2.2755889892578125, -2.0638010501861572, -2.4191040992736816, 2.253239154815674, -0.4442339539527893, -8.497435569763184, 0.6659086346626282, 3.4691338539123535, 2.0279293060302734, 0.4239545464515686, 4.866848468780518, 0.43445876240730286, 2.5774590969085693, 0.3916206657886505, 1.286906123161316, 0.5518091917037964, -4.572941303253174, 0.350776344537735, 1.1401021480560303, 0.34017565846443176, 2.6023027896881104, 2.7051360607147217, -4.221421718597412, 2.0616776943206787, 6.71721076965332, 0.9910488128662109, 1.0680458545684814, -4.032674789428711, 1.3344507217407227, 2.851369857788086, 10.651620864868164, 4.368653774261475, 3.816845655441284, -0.26555848121643066, -0.8756940364837646, 1.0597703456878662, -1.8607360124588013, 1.8605321645736694, 3.971001625061035, -0.9248174428939819, 5.0261430740356445, -0.0677119791507721, 1.8938251733779907, -1.4362977743148804, -2.339801549911499, 1.8422520160675049, -1.4656486511230469, 1.8475282192230225, 1.5048298835754395, -3.313312530517578, -0.17551273107528687, 2.4447696208953857, 4.724374771118164, 2.30096435546875, 0.38236820697784424, -1.732293963432312, -3.5479230880737305, 1.1733074188232422, 1.6424177885055542, 1.7427067756652832, 1.626190423965454, 3.6656413078308105, -0.22205746173858643, 1.0904985666275024, 3.924048900604248, -2.447709798812866, 0.5688925385475159, 0.8763870000839233, -3.9839835166931152, -5.145785331726074, -4.296929359436035, 2.0275135040283203, -0.5482865571975708, -3.328845500946045, -3.1643481254577637, -0.8501532077789307, 4.029229164123535, 5.681818008422852, 0.9199042320251465, -0.9200607538223267, 1.1000287532806396, -1.3217017650604248, -1.9407696723937988, -3.9841370582580566, -3.3794007301330566, 0.5555740594863892, -1.1280075311660767, -0.15392445027828217, 3.639197587966919, -3.241687059402466, 1.8895983695983887, -2.288184881210327, -3.752084970474243, 5.356621265411377, -2.5171666145324707, -1.1184906959533691, -0.3564474880695343, 2.7785451412200928, -0.2841658890247345, 3.3191027641296387, -4.268868446350098, 0.5059667825698853, -2.763676643371582, 0.45379650592803955, 0.09801557660102844, -2.8281900882720947, 1.7798019647598267, -0.15178857743740082, 2.6836085319519043, -0.47391021251678467, -1.6330883502960205, -3.2375056743621826, -3.6712732315063477, -6.410469055175781, 0.41231369972229004, -0.10376548022031784, -2.7513535022735596, 1.0187840461730957, -0.05253073573112488, -3.109830856323242, 1.7789511680603027, 1.7088840007781982, 1.7653460502624512, 1.2935292720794678, 4.0193352699279785, 0.41797685623168945, 4.450338363647461, 4.504762649536133, 3.077078342437744, 1.7859282493591309, -4.30633544921875, 2.279165744781494, -0.10066545009613037, 5.8846964836120605, 1.9688606262207031, -0.32076525688171387, -5.129922866821289, -3.752539873123169, -1.9293426275253296, -0.9156836271286011, 1.9660532474517822, 2.000800132751465, -1.762162446975708, -6.417743682861328, -2.392609119415283, 2.4786293506622314, 1.521724820137024, 2.6293954849243164, -0.5435784459114075, 1.0657743215560913, 4.631656646728516, 9.194009780883789, -2.1103532314300537, 0.33417439460754395, 8.729308128356934, -1.3481780290603638, 4.212933540344238, 1.1212584972381592, -4.792494773864746, -3.268824577331543, 1.2207558155059814, 1.9130827188491821, 1.4261806011199951, -0.2920357584953308, 0.006446996703743935, -0.837761402130127, -5.1646647453308105, 6.22979736328125, 6.614543437957764, 5.5883941650390625, -3.4008898735046387, 1.1329617500305176, -4.507474422454834, -1.9136919975280762, 6.048585891723633, -3.503878593444824, -3.5489463806152344, 2.5059075355529785, 3.496778726577759, -1.2933659553527832, -0.43071985244750977, -1.1166632175445557, -6.926568984985352, -0.009282197803258896, -1.819704294204712, 1.7096190452575684, 1.0556728839874268, -5.5514397621154785, -0.9893993139266968, -1.7687790393829346, 2.0996906757354736, 1.3424491882324219, 0.4175640642642975, -0.5414116978645325, 0.7690063714981079, 1.2903436422348022, -2.229039430618286, -1.291685938835144, -3.97308611869812, -3.113016128540039, 4.566077709197998, 0.8246400356292725, 1.065601110458374, 0.09775009006261826, -6.566844463348389, -1.623279094696045, 3.704850912094116, 1.8030985593795776, -1.5428520441055298, 2.5000274181365967, -1.7244147062301636, 4.058333873748779, -0.9561725854873657, 4.770166873931885, -2.0118446350097656, -3.7711329460144043, -0.2281646579504013, 0.6471035480499268, -7.636713027954102, -1.7799317836761475, -2.2715988159179688, -2.5671191215515137, -1.0677908658981323, -3.0979621410369873, -1.9217959642410278, 0.3088381588459015, -2.0587289333343506, 4.7650017738342285, -3.9648995399475098, -3.7479751110076904, -3.0676541328430176, 1.0960510969161987, -4.353443145751953, 6.462325096130371, 2.738151788711548, -0.11423727124929428, -2.559638500213623, 0.3822840750217438, -2.117048978805542, -0.8198637962341309, 4.650486946105957, -0.11005225777626038, 2.2193808555603027, 1.5160492658615112, 4.0506978034973145, -1.482772946357727, 4.848470211029053, 7.466175079345703, -1.7874845266342163, 2.566525459289551, -0.5919333100318909, 0.9840072393417358, -1.2510782480239868, -1.5824182033538818, -0.005579294171184301, -1.2861136198043823, -0.4175679087638855, -2.4326298236846924, -5.955031871795654, 4.772276878356934, -0.16874249279499054, -1.8135426044464111, -4.304165840148926, 3.35191011428833, -2.4983770847320557, 2.6954360008239746, -0.8499594926834106, 6.234125137329102, -0.9759369492530823, 2.5083446502685547, -2.0792980194091797, 0.07802905142307281, 6.929327487945557, -0.81975257396698, 0.966548502445221, -4.01559591293335, -3.479783535003662, 0.48875510692596436, 2.1162824630737305, 1.534449577331543, -1.3981704711914062, -0.3190898001194, 3.0107781887054443, -1.4635169506072998, 0.5078245997428894, 0.42558833956718445, -2.062304735183716, -2.579279661178589, 1.0131953954696655, 1.9965558052062988, 3.651381015777588, -1.3108468055725098, 1.1055392026901245, 4.167355537414551, 3.9164786338806152, -3.697139024734497, -1.3105732202529907, -2.0902342796325684, 2.28116512298584, -2.1276960372924805, 3.4845876693725586, 1.5425784587860107, -4.014419078826904, -2.98545503616333, 2.1649281978607178, 0.3796173334121704, 0.43401870131492615, -2.0505878925323486, -0.9599127173423767, -0.4226696491241455, 0.9866334199905396, 3.043931007385254, -2.730088710784912, 3.7396903038024902, 1.2272346019744873, -2.555483102798462, 3.8127944469451904, -1.2103209495544434, 4.589405059814453, 1.1332939863204956, 2.597203493118286, 1.3739694356918335, 2.2371702194213867, -5.302380561828613, 1.0411877632141113, 1.7892647981643677, 0.5489218831062317, -3.171421766281128, 6.023531913757324, 6.2757182121276855, -2.0819625854492188, -1.9646068811416626, -3.069951057434082, 3.875560760498047, -2.0469441413879395, -2.9271585941314697, -4.156772136688232, -0.18394474685192108, -1.144325613975525, -5.237617492675781, -1.215406060218811, 1.1647685766220093, -3.184575319290161, -0.6708059310913086, 1.650254726409912, 1.8949215412139893, 1.8596137762069702], [-3.0736777782440186, 1.8288671970367432, -1.485060691833496, -0.15496648848056793, 1.9435341358184814, 0.800237774848938, 0.6142604351043701, 1.2626450061798096, -2.0684525966644287, -5.261699199676514, 3.460261821746826, -1.9491221904754639, 0.5964778661727905, -5.319682598114014, 6.611321449279785, 0.5737919807434082, -1.1142725944519043, 2.4337639808654785, 0.891722559928894, -1.4979723691940308, -0.3050956428050995, 4.954324245452881, 4.3224592208862305, -2.098339557647705, 3.6291470527648926, 4.320390701293945, -0.9368798732757568, 4.276354789733887, -5.404055595397949, -0.6467560529708862, -0.8417689800262451, -1.4926557540893555, 2.2452235221862793, 0.5208325982093811, -2.2501344680786133, -2.661525011062622, -2.30302095413208, -1.175214171409607, -4.703420162200928, -0.6387234926223755, 2.4340553283691406, 1.1859756708145142, -5.986726760864258, -4.30168342590332, -0.5450454950332642, 1.9698221683502197, 8.773506164550781, -0.3790128529071808, 1.7856205701828003, -4.181045055389404, 0.08352014422416687, -3.01645565032959, -0.05451885610818863, 3.506068706512451, -1.398831844329834, 5.600584506988525, 1.3720468282699585, 1.4187238216400146, 4.418426036834717, -1.4957597255706787, 3.103355884552002, -2.2797765731811523, 0.2668662369251251, 1.0604512691497803, -2.14536714553833, 2.918222427368164, -0.2698395848274231, 3.3846609592437744, 1.298866868019104, -5.620731830596924, -1.4965962171554565, -1.3296387195587158, -3.6694583892822266, 1.8536221981048584, -3.6457700729370117, 2.0206751823425293, 2.522942066192627, -0.8972939252853394, -1.9020512104034424, -2.0601449012756348, -3.761500358581543, 0.6391070485115051, -2.928109884262085, 4.52139949798584, 1.5431549549102783, 5.024940013885498, -2.186429023742676, -1.7068523168563843, 1.3433654308319092, 5.907376766204834, 0.11924727261066437, -5.693603515625, -1.7612996101379395, -2.196550130844116, -4.254006385803223, -3.694298028945923, 1.127860426902771, 0.6890595555305481, -10.9976224899292, 1.1885261535644531, -3.80413556098938, 0.6566835641860962, 5.922166347503662, -3.962899684906006, -1.3250106573104858, 4.737888813018799, 0.7623608708381653, 1.8093284368515015, -1.5522644519805908, 3.078134059906006, -6.33342170715332, 0.4374270737171173, -6.312016010284424, 0.41187459230422974, -0.17371846735477448, 1.1238818168640137, -2.2432966232299805, 1.1851733922958374, 0.9308926463127136, 1.477891206741333, -2.125493049621582, -2.047351360321045, -1.1333907842636108, -2.320648193359375, 1.1362448930740356, -0.6200692653656006, 1.1871919631958008, 2.1720314025878906, -0.28705623745918274, 2.5280871391296387, 1.0160201787948608, -6.780491828918457, 6.663721561431885, 3.0694878101348877, -2.6188442707061768, 6.395609378814697, -2.920940637588501, -0.03291689604520798, 2.1711325645446777, -0.16249297559261322, -0.5526059865951538, 2.8372395038604736, 1.7509765625, 4.402758598327637, 1.5645902156829834, -1.194925308227539, 0.9731361865997314, 5.501440048217773, 0.29191312193870544, 4.842108249664307, 3.971120595932007, -0.16365750133991241, -4.846889495849609, -1.8998955488204956, -3.1724953651428223, 0.7027264833450317, -1.0813119411468506, -0.7764787673950195, 2.3196403980255127, 4.168628692626953, 0.5596846342086792, 6.642454147338867, -5.278988838195801, 1.184625506401062, 9.859821319580078, -0.20776250958442688, -2.0996439456939697, -2.7390880584716797, 0.361290842294693, 1.3691225051879883, 3.5308728218078613, -4.016262054443359, -2.181636333465576, -0.7690585851669312, 0.25932934880256653, -1.751651644706726, 3.6341733932495117, 0.35465142130851746, -2.7997851371765137, -0.8471608757972717, 2.139174461364746, 2.2414519786834717, 2.6325864791870117, 0.5270743370056152, 1.4532625675201416, -3.231362819671631, -0.412834107875824, -0.7434456944465637, 2.7119839191436768, -9.224726676940918, -1.7826966047286987, 0.4520992636680603, -3.8306100368499756, -4.16878604888916, 0.6508116722106934, 0.9276427030563354, -3.033987522125244, 2.4348256587982178, 1.1225059032440186, -0.3243589401245117, -0.7305175065994263, -7.387808799743652, 6.32994270324707, 3.0062804222106934, 1.799455165863037, -0.445801317691803, 2.596611261367798, -2.709205150604248, -3.0630476474761963, -0.9057510495185852, 5.225815773010254, -1.2528845071792603, 0.7684575319290161, -3.6004035472869873, 3.272038459777832, -2.7281503677368164, -1.7797510623931885, -0.4453362226486206, -4.409781455993652, -2.592836856842041, 0.21251091361045837, -0.836703896522522, 3.23085355758667, -4.39100456237793, -1.8536663055419922, 1.9325577020645142, 3.5634074211120605, 3.464844226837158, 1.5350449085235596, 1.0915241241455078, 2.4851551055908203, 4.920079231262207, -6.361380577087402, -0.033304329961538315, -5.063840389251709, -4.796360015869141, -7.014081001281738, -1.5563818216323853, 6.869779586791992, -0.2598957419395447, -1.101528286933899, -3.041858196258545, -0.12539392709732056, 3.690207004547119, -4.329310417175293, -0.37446221709251404, 1.3726379871368408, 1.7927777767181396, -3.109499931335449, -1.8469877243041992, 0.36949801445007324, -2.8720645904541016, 1.7527976036071777, 0.9578288197517395, 5.589077949523926, -1.7885202169418335, 2.665714740753174, -4.2274065017700195, -1.1371662616729736, -4.453930377960205, -0.03113442286849022, 2.5509400367736816, 4.039159297943115, -0.6609815359115601, 0.2555531859397888, -5.018405914306641, -1.0880579948425293, 1.618025779724121, -0.733536958694458, 1.5796068906784058, -3.38558292388916, -5.084902763366699, -0.7345908284187317, 6.507342338562012, -0.20717743039131165, 2.066025733947754, -2.31715726852417, -2.7515735626220703, -6.078433036804199, -0.7792913317680359, 2.720736503601074, -2.3945813179016113, -0.5547336339950562, 4.060601711273193, 3.6187899112701416, 0.7744672298431396, 1.7835972309112549, 4.797648906707764, -1.5810890197753906, -4.905465602874756, -0.2212139368057251, 0.931322455406189, 2.335216760635376, -5.479039669036865, -2.191526174545288, -1.6607887744903564, -0.3942958116531372, 5.608633995056152, 0.47220659255981445, -0.9008609056472778, -2.6017730236053467, 5.653793811798096, -1.246498465538025, -5.820745944976807, -3.5797934532165527, -0.04410259425640106, -2.4611997604370117, 0.14560651779174805, -0.721159815788269, 0.8528399467468262, -0.6241562962532043, -1.3837409019470215, 3.521317720413208, -2.477029800415039, -3.1787545680999756, 2.884420871734619, 3.4412941932678223, -6.457383155822754, 1.642155408859253, 3.363640308380127, -0.8716712594032288, -1.5250385999679565, -0.7237106561660767, 1.6247538328170776, -0.9669970273971558, -0.8163852095603943, -0.4891189634799957, 0.7327430844306946, 0.5679662823677063, 3.381877899169922, 0.20853517949581146, -2.519623041152954, -6.850299835205078, -5.364319801330566, 1.8716429471969604, -3.2639379501342773, -0.8897849321365356, -0.04035535827279091, 2.821023941040039, -4.854015827178955, 5.075464248657227, -3.351490020751953, -3.9921860694885254, 4.567877292633057, -0.459179162979126, 2.410060405731201, -1.9087493419647217, 2.403470993041992, -1.630985140800476, 1.3393049240112305, 1.1791517734527588, -1.6294846534729004, 4.6434807777404785, 1.3674806356430054, 1.5267183780670166, 2.4338860511779785, -2.1193056106567383, -0.2463289499282837, 2.6422412395477295, -6.297216415405273, 2.503190517425537, 2.5608184337615967, -0.10249315202236176, -0.03084125742316246, -3.7628698348999023, 2.485476016998291, -4.5628252029418945, 1.9710205793380737, -4.8176727294921875, -0.3597675561904907, -5.582540988922119, 2.1279361248016357, -0.3847794532775879, -0.9923205971717834, -2.3503103256225586, -4.267666816711426, -3.533740997314453, -0.5375590324401855, 0.77968829870224, 2.7180495262145996, -1.51972496509552, -6.341242790222168, -1.1544468402862549, -4.746171951293945, 3.429535388946533, -0.9562180042266846, -1.9315948486328125, -2.226936101913452, 1.481885313987732, 1.6620359420776367, -5.87985372543335, 3.8059592247009277, -2.8895833492279053, -2.3912649154663086, 0.4722592830657959, -0.4839630722999573, 3.6365952491760254, 2.640547752380371, 2.185605049133301, -1.497697114944458, 4.435105323791504, -4.451313495635986, -2.025411605834961, -3.9051754474639893, 2.5681796073913574, -0.5435709953308105, 1.3591595888137817, -4.536710739135742, 3.91209077835083, -1.780484676361084, 3.007882833480835, 2.4646782875061035, -1.5581333637237549, -1.4167615175247192, -0.3843681812286377, 2.63200044631958, -2.6633710861206055, -5.18196964263916, 4.031296253204346, 3.01585054397583, -5.858551025390625, -0.9567791223526001, -0.18912678956985474, 0.7899976372718811, -1.593463659286499, 1.7390265464782715, 1.5553133487701416, 1.5565109252929688, 4.478954315185547, 2.680684804916382, 1.3072636127471924, -1.192291259765625, -1.2464320659637451, 0.0831453800201416, 0.6105793118476868, 4.111446380615234, 4.0941267013549805, -0.9228668212890625, -0.25291508436203003, 2.5340218544006348, 1.9470158815383911, -2.243117570877075, 5.155647277832031, 1.5754791498184204, -0.6701478362083435, -2.697810649871826, 1.99137282371521, 0.4733937084674835, 0.13031242787837982, 4.377730369567871, 1.726872444152832, 2.9199059009552, 4.9548797607421875, -1.3839284181594849, -0.37817156314849854, 0.9010602235794067, 0.6308266520500183, 0.3103867471218109, 3.5166513919830322, -2.8531389236450195, -1.5072758197784424, -0.41318029165267944, -0.7882003784179688, -7.132124900817871, -2.349194288253784, 6.114093780517578, -1.0653958320617676, 8.6245698928833, -7.43666410446167, 0.5744343996047974, -1.43600332736969, 1.8434946537017822, 3.548140048980713, 4.896805286407471, 0.5940413475036621, -1.5636076927185059, 2.4823243618011475, -1.126793384552002, 0.9006658792495728, -1.730022668838501, -3.588878631591797, -3.629254102706909, 4.393600940704346, 0.703272819519043, 1.752152681350708, -0.053835660219192505, 1.004612684249878, 1.4556694030761719, 0.8077499866485596, -2.058462381362915, -0.8350962400436401, 2.1016995906829834, -7.896544456481934, -2.7009692192077637, 1.685307264328003, -0.7026280164718628, -3.408416748046875, -0.2663598656654358, -2.703702688217163, -0.46908748149871826, 0.7779381275177002, -4.138057231903076, 2.624676465988159, 4.910740852355957, -2.3162550926208496, -4.796507358551025, -0.02969559095799923, -1.1833527088165283, -1.1650660037994385, -0.7481798529624939, 1.0340561866760254, -6.118878364562988, 1.6579127311706543, 3.042928695678711, -0.3697925806045532, -2.281891107559204, -6.175667762756348, 1.1742136478424072, -0.826304018497467, 2.4176254272460938, 2.8973255157470703, 1.2122650146484375, -3.099698543548584, 1.1659188270568848, 1.3928908109664917, 2.6736812591552734, 5.7520833015441895, 2.5240652561187744, 5.964409828186035, -0.9550696611404419, -0.8982342481613159, -0.1577700972557068, 1.491834044456482, 0.11976289004087448, -4.730505466461182, -4.386133193969727, -1.0009148120880127, -2.5580241680145264, 1.4697015285491943, 5.637352466583252, -0.28876906633377075, -3.7828493118286133, -1.1094143390655518, 3.4415478706359863, -1.034074068069458, 1.5648832321166992, 3.417100667953491, -0.7658532857894897, 4.947530746459961, -2.499474048614502, -2.849815607070923, 0.18268367648124695, 1.9651718139648438, -6.476898193359375, -0.11029504239559174, 0.3113923966884613, 2.2327020168304443, -1.5548717975616455, 2.0229098796844482, -3.765759229660034, -2.6348764896392822, -0.6192306280136108, 0.6453127264976501, 5.727895259857178, -0.4580968916416168, 4.185316562652588, -3.1418917179107666, -2.0595624446868896, -1.4769186973571777, 2.078469753265381, 0.8027236461639404, 3.252572536468506, -0.9048722982406616, 0.13589759171009064, -1.7760883569717407, -2.1539018154144287, -1.983635663986206, 4.5655903816223145, -2.0723440647125244, 0.5726742744445801, -0.7481809258460999, -0.607785701751709, 2.18914794921875, 8.449769020080566, -4.290957450866699, -1.4848767518997192, -2.923762559890747, 2.1221776008605957, 0.6967846155166626, 1.806878685951233, 0.4443959593772888, -1.1836097240447998, -8.015680313110352, -0.5040938258171082, -3.0769357681274414, 1.4625968933105469, 3.2193422317504883, -4.207621097564697, 0.9925925731658936, -3.0281505584716797, -6.547542572021484, -0.32025477290153503, 3.6525049209594727, 2.8574252128601074, 0.7816622257232666, -3.17090106010437, 4.1806254386901855, -1.624600887298584, -4.9029154777526855, -1.4488719701766968, 1.1737117767333984, -2.398038864135742, 0.495562344789505, -0.9312942624092102, 3.2645511627197266, -0.11784014850854874, -0.21865659952163696, -4.855926036834717, 2.8984861373901367, 2.9986557960510254, 1.0761042833328247, 0.707440197467804, -1.3047866821289062, -1.4503867626190186, 1.3614959716796875, 0.6531245708465576, -1.8335084915161133, -3.1976704597473145, -1.5505836009979248, 2.583397626876831, -1.9851624965667725, 6.524311065673828, 0.04674758389592171, -3.44610595703125, 0.792583703994751, 8.484132766723633, 4.182353973388672, -0.6202735900878906, -3.1880013942718506, 3.562990188598633, -2.896538734436035, 0.5601401329040527, 1.684199333190918, -3.0000829696655273, -0.9928643703460693, -2.459028720855713, -2.693991184234619, -6.566967964172363, 4.23944616317749, -0.595707356929779, -4.083736896514893, 0.6266114711761475, 0.11684386432170868, 1.1492524147033691, -1.4599725008010864, 3.134474277496338, -5.508909702301025, 0.8591424822807312, 6.993533134460449, 1.6708087921142578, -3.3157060146331787, -1.5200304985046387, 1.0262305736541748, -3.1703505516052246, -6.004258632659912, -2.0228402614593506, -4.9270501136779785, 0.39196112751960754, -7.055907726287842, -2.5084145069122314, 6.252715110778809, 0.02364739030599594, 3.5087363719940186, 1.1888394355773926, -6.363398551940918, -1.4701558351516724, -2.327061176300049, 1.692743182182312, -5.073441982269287, 1.1953771114349365, -3.713629722595215, 2.8506956100463867, 1.2786211967468262, 1.732640027999878, 5.816222667694092, -0.6794984936714172, 0.37858277559280396, -3.1498050689697266, -1.081108570098877, -2.8815455436706543, 1.0686559677124023, -2.795320510864258, 0.12718263268470764, 5.321720123291016, -1.589475154876709, 0.032166339457035065, -3.0130069255828857, 3.2651586532592773, 3.1709604263305664, 1.9510059356689453, 2.36474609375, 3.8548922538757324, 0.17095863819122314, -2.2170071601867676, 0.7153393030166626, -5.807514190673828, -3.177577495574951, -3.749466896057129, -2.414600133895874, 1.913909912109375, -6.178933143615723, 2.9209353923797607, -4.699238300323486, 2.2125673294067383, -4.240048885345459, -0.10809215903282166, -4.08220911026001, 2.0211639404296875, -0.32951393723487854, -7.002614974975586, 3.2202916145324707, -1.3449115753173828, -0.6693205237388611, -4.848940849304199, 1.832711935043335, -0.9164692163467407, -4.5558929443359375, 0.17011770606040955, -0.9299461841583252, -2.7429122924804688, 0.6380672454833984, -6.079136848449707, -0.01887522079050541, -0.8308330774307251, 1.0960277318954468, -2.079622268676758, -1.3371933698654175, 1.2142539024353027, 1.7596051692962646, 2.575127124786377, 0.2305212914943695, 2.9744772911071777, -3.8504397869110107, 3.9318275451660156, 6.246086597442627, -2.341499090194702, -3.5593671798706055, -2.633925437927246, 3.586660861968994, 1.0526816844940186, -5.720293045043945, -4.913280963897705, 1.7226738929748535, -1.3694982528686523, -3.200890064239502, -2.710855484008789, -2.309051990509033, -6.217831134796143, -3.4714527130126953, -0.9764454960823059, 1.6871856451034546, 1.8997188806533813, -4.468328952789307, 1.8227310180664062, -3.951096296310425, 4.520870208740234, 2.067195177078247, 4.92588996887207, 0.47957244515419006, -3.4855105876922607, 2.157437801361084, -0.13128533959388733, -0.439431756734848, -3.5655264854431152, 5.120511054992676, 1.0045380592346191, 3.0093679428100586, 2.941884756088257, -1.0389540195465088, 4.014409065246582, 0.42099279165267944, 1.9111745357513428, -2.05080509185791, 2.3061177730560303, -2.1343836784362793, 4.8959641456604, -0.549965500831604, -5.313190460205078, 4.813287258148193, -3.284289836883545, -0.9885417222976685, 0.2236522138118744, 0.662406861782074, 0.6581758260726929, 1.4463446140289307, 4.05220890045166, -3.1578733921051025, 0.5681817531585693, 1.7045519351959229, 0.4755600392818451, -1.998058557510376, 3.136308193206787, -1.857047438621521, 0.5258464813232422, 0.6113552451133728, 2.4653308391571045, -1.6186292171478271, -0.485857218503952, 1.6958627700805664, -1.5582046508789062, 2.8514931201934814, 1.1683275699615479, -0.7870244979858398, -3.2236409187316895, 0.04253847897052765, 1.211738109588623, -2.1402716636657715, 2.2668261528015137, -0.1800675094127655, 3.200730323791504, -0.37756866216659546, -4.945619106292725, -0.2730819880962372, -3.943605422973633, 1.1517342329025269, 0.7954071164131165, -6.448143005371094, -1.9362554550170898, 2.897312641143799, 2.144005537033081, 1.948940634727478, 1.0719246864318848, -4.042557716369629, -4.254125595092773, 5.286230087280273, -3.5959420204162598, 2.6756393909454346, 2.485525131225586, -2.129058361053467, -0.05299369990825653, 0.19440817832946777, 0.7850337624549866, -2.304841995239258, -1.1923139095306396, -3.2015938758850098, 0.6216641664505005, 1.7648289203643799, -0.4011414051055908, 3.641733169555664, 1.97965407371521, -2.2009382247924805, -3.497173547744751, -0.5514439940452576, 0.8338818550109863, -1.7072092294692993, 0.45186537504196167, 1.281396508216858, -2.6246442794799805, -0.9412026405334473, 2.4644360542297363, -2.6265885829925537, -0.4158310294151306, 1.369806170463562, 2.329723596572876, -1.8621091842651367, -7.320332050323486, -7.0951924324035645, 0.21114397048950195, 0.9685909152030945, -4.320135593414307, 1.3362165689468384, -0.11420982331037521, -3.9181485176086426, -2.5992326736450195, 3.311737060546875, 0.14977745711803436, 2.4021241664886475, -0.9224038124084473, 0.9922350645065308, 1.5073052644729614, 5.938990116119385, 4.241550445556641, -2.365544319152832, 1.7966704368591309, -1.2088537216186523, 0.08117779344320297, -0.605065107345581, 1.4857773780822754, 0.3562946021556854, -4.602738380432129, -0.171209916472435, 2.8246917724609375, -2.05749249458313, -0.5138994455337524, 1.1010550260543823, -1.5447160005569458, 6.503420829772949, -1.0862312316894531, -2.429190158843994, -5.574937343597412, 4.694399356842041, 3.7312495708465576, 7.164588928222656, -2.0953786373138428, -0.4078061878681183, -0.3877734839916229, 3.9669291973114014, -4.729676246643066, -0.5489987134933472, -4.7322587966918945, -3.145341396331787, -5.060153007507324, 1.1622140407562256, -2.956434965133667, -0.938873291015625, -7.996227264404297, 3.3093173503875732, -0.6694313287734985, 1.0770058631896973, -0.017968812957406044, -1.4026596546173096, -0.16163641214370728, 3.291783332824707, 0.6959003210067749, -3.220278024673462, 2.299802541732788, -6.890172958374023, 2.557279109954834, -5.19802713394165, 1.785400629043579, 2.2234983444213867, -4.319164276123047, -4.773386478424072, 3.366609573364258, -3.0421535968780518, -3.106100559234619, -1.1931476593017578, -2.3649985790252686, -1.8893053531646729, -2.0740721225738525, -3.9241485595703125, 0.3798214793205261, -0.0020078395027667284, 0.8858011960983276, 2.773721218109131, 1.6504765748977661, 2.345945358276367, 0.01654798910021782, 2.4110941886901855, 5.220097064971924, -1.5631294250488281, 0.2542325258255005, 0.22833171486854553, 1.063847541809082, 3.1792330741882324, -4.4712677001953125, 2.616950035095215, -1.2991958856582642, 0.2542822062969208, -0.15220044553279877, -2.776989459991455, -0.8882006406784058, 1.603435754776001, -0.6668367981910706, 1.1999924182891846, 0.04594968631863594, 2.554316282272339, 1.8061208724975586, 4.203701972961426, -0.5094650983810425, -5.997220993041992, -2.7617719173431396, -0.4882148504257202, 1.6161632537841797, -0.5700455904006958, 4.8729939460754395, -0.665441632270813, -0.3494192659854889, 0.27011838555336, 0.3119322955608368, 1.875971794128418, -3.582538604736328, -4.697908878326416, -0.11862123012542725, -0.6151278018951416, 0.04251010715961456, -5.166127681732178, 3.6080081462860107, 7.478975772857666, 1.1106843948364258, -5.411985397338867, 4.874098777770996, 1.9306342601776123, 0.8177865743637085, 1.3544708490371704, 0.9272893667221069, 1.3071649074554443, -0.17226991057395935, 4.928326606750488, -0.5693272948265076, -1.9238717555999756, 2.046046733856201, 5.703972816467285, 2.133258104324341, -4.036017417907715, 3.0406289100646973, -4.086237907409668, -3.257930278778076, -1.7042609453201294, 0.5089730024337769, -2.5847675800323486, 2.9182286262512207, -5.403649806976318, -2.1237738132476807, -1.1689918041229248, -4.05345344543457, -4.837532043457031, 0.020057639107108116, -2.247614860534668, 6.23061466217041, -0.2119956910610199, -3.7450225353240967, 0.6212426424026489, 5.639945030212402, -0.7789496183395386, -0.9999575614929199, 0.7650545835494995, 2.347684860229492, -0.17080175876617432, 1.4172000885009766, 2.9665136337280273, -1.2028939723968506, -6.3638410568237305, 3.1476235389709473, -1.2213153839111328, 1.3261806964874268, -0.524567186832428, 0.5441896915435791, 2.937441825866699, -4.214008331298828, 5.659986972808838, 0.803868293762207, -4.488144874572754, -1.0221240520477295, -0.384097158908844, 3.316531181335449, -2.9303574562072754, 1.0698250532150269, 1.710323452949524, -0.37570691108703613, -5.763847827911377, 0.044717904180288315, 2.483410358428955, -1.7254589796066284, -0.185097336769104, -0.5802181363105774, -2.557676315307617, -2.5826621055603027, 1.6359364986419678, 2.036100387573242, 1.5885193347930908, -1.5813579559326172, -0.2802234888076782, 1.5551873445510864, -5.993749141693115, -1.5824543237686157, -0.8537335395812988, -3.613780975341797, -1.4423115253448486, -3.3367557525634766, 2.509402275085449, -4.331327438354492, -0.32637661695480347, 0.46541160345077515, 4.057611465454102, 2.540670871734619, -2.818871021270752, 5.872037410736084, 3.119774341583252, 2.0418572425842285, -6.189847469329834, -1.1814168691635132, -2.835700273513794, 2.801537036895752, -0.3347615897655487, 1.2687153816223145, 0.32532575726509094, -1.7292948961257935, -7.147713661193848, 2.8698654174804688, -2.3579697608947754, 0.15641286969184875, 4.255479335784912, -0.45426151156425476, 0.5621623992919922, -4.293267250061035, 0.6738375425338745, 1.269986867904663, 2.0562503337860107, -6.677971839904785, -0.2783104181289673, -1.1654837131500244, -5.482935428619385, -1.9435253143310547, -2.7820563316345215, 1.7871546745300293, -0.5033754706382751, 0.7756628394126892, 3.0710785388946533, 0.6951060891151428, 1.6872289180755615, 3.0618224143981934, -0.5537143349647522, 8.224173545837402, -0.13538868725299835, -0.049358516931533813, 3.440431594848633, -1.8584001064300537, 0.24474692344665527, -4.903410911560059, -7.121199607849121, 0.3988235592842102, -0.7197920680046082, 0.5430923700332642, -4.547430992126465, 2.760910749435425, 4.605004787445068, 2.8561525344848633, -2.972187042236328, 3.0236871242523193, -2.6316401958465576, 4.210384368896484, -2.0214643478393555, 2.78576397895813, 1.6378487348556519, 1.2232236862182617, 0.7433341145515442, -2.4344983100891113, -5.537054061889648, 0.3422919809818268, 1.7026381492614746, -1.399253487586975, 0.7457844614982605, 4.5775909423828125, -0.987628161907196, 2.4105329513549805, -0.4292600154876709, 0.12039852142333984, -3.8495888710021973, 1.1139780282974243, -6.387209892272949, -3.615504264831543, 2.061857223510742, -3.2935845851898193, -3.9603402614593506, 0.45261067152023315, 1.9943585395812988, -2.944451332092285, 2.8516135215759277, 0.032230526208877563, -3.643251895904541, -2.491368532180786, 6.694302558898926, 0.4986315071582794, 2.751208782196045, 2.0517287254333496, 0.3702678978443146, 1.0463030338287354, 2.9180264472961426, -1.1364150047302246, 5.291186809539795, 3.0123252868652344, 0.5041666030883789, -0.6868366599082947, 1.2144620418548584, -1.0966745615005493, 4.166201114654541, 4.417280197143555, -2.333787679672241, -3.634726047515869, 1.928586721420288, -1.2587083578109741, 1.3166842460632324, -6.813782691955566, 0.8712726831436157, -2.3682689666748047, 4.55128812789917, -1.0677582025527954, -2.240237236022949, -0.7484159469604492, 2.476397752761841, 1.831095814704895, 1.8318032026290894, 0.3682931959629059, -4.9239912033081055, 0.18323327600955963, -1.245880126953125, 1.6007373332977295, -1.7611488103866577, 0.4021567702293396, -2.1480278968811035, -2.837200403213501, 1.3673821687698364, 4.308345794677734, -3.7108349800109863, -1.9661645889282227, -1.4143016338348389, 5.962574005126953, -0.5832721590995789, 1.930191993713379, 0.21498042345046997, 2.406829357147217, 0.18763361871242523, 3.9888854026794434, 1.2303050756454468, 1.8299535512924194, -3.3956189155578613, -4.420827865600586, 1.3104684352874756, -4.111385345458984, -2.557506561279297, 0.5035172700881958, 3.0248656272888184, -3.2177658081054688, -0.37747400999069214, 1.650681495666504, -4.434788703918457, 3.161475419998169, 4.346379280090332, 1.0288586616516113, -0.2819403409957886, 4.6321258544921875, -2.0395827293395996, -0.0011052985209971666, -1.3625699281692505, 1.7345690727233887, 0.5328664779663086, -0.05040779709815979, -0.19506075978279114, -4.7633466720581055, 2.948514938354492, -3.703004837036133, -2.6249399185180664, 5.482089042663574, 3.360680103302002, 0.27485278248786926, -2.579209089279175, 0.12981244921684265, 4.265904903411865, 4.772134304046631, 2.6748151779174805, -0.8439533114433289, -6.486220836639404, 1.2365833520889282, -2.942004680633545, -1.7472002506256104, -3.32667875289917, 3.20699405670166, -0.6632134914398193, 2.072566270828247, 1.0781866312026978, -2.8736376762390137, 4.574916362762451, -7.191732406616211, -0.9826396107673645, 0.8928327560424805, 4.62690544128418, 2.1243515014648438, -1.5398669242858887, 0.9190107583999634, 0.5799851417541504, 0.2671058475971222, 7.3843994140625, -3.2218732833862305, 1.6239979267120361, -0.028886903077363968, -2.41398286819458, -3.1397502422332764, 2.724048614501953, 1.6084175109863281, -2.7266504764556885, 5.346482276916504, -2.8937506675720215, 1.897249698638916, -3.3099682331085205, 4.740750312805176, 1.0776331424713135, 1.2767695188522339, -1.3765807151794434, 1.0227187871932983, -3.644061326980591, -1.4545080661773682, -4.004514694213867, 0.6553404331207275, 2.7818803787231445, -0.14189928770065308, 3.9832711219787598, -2.849501371383667, 4.645787239074707, 5.24227237701416, -2.254584550857544, 1.5138006210327148, 1.9369754791259766, -4.67745304107666, -2.9127602577209473, -2.8487701416015625, 3.2986106872558594, -0.4680720567703247, -1.9040826559066772, 6.920677185058594, -3.305173397064209, 2.1671431064605713, -0.96682208776474, -0.95802903175354, 1.0619025230407715, 0.04994148015975952, 2.2639827728271484, 1.2509360313415527, -1.664233684539795, -3.741262912750244, -11.099625587463379, -2.6623616218566895, -1.193547010421753, -2.440521240234375, 1.9298419952392578, -1.7008633613586426, 6.958401203155518, 3.387997627258301, 2.8389368057250977, 2.1162972450256348, -4.29747200012207, -2.620311737060547, -1.1221213340759277, -0.10523208975791931, 0.5858339071273804, 1.0482935905456543, -0.9069530367851257, 0.8540523648262024, -3.7899231910705566, -6.249745845794678, 0.25633227825164795, 3.8387253284454346, 0.8705800771713257, -1.6178616285324097, 3.3406267166137695, 4.820443153381348, -0.8474820256233215, -2.420396566390991, -2.8968420028686523, -3.8868165016174316, 5.069439888000488, -2.4677579402923584, 0.05681471526622772, -0.1948811262845993, -1.9698024988174438, 5.761903762817383, 1.1186473369598389, -3.9038124084472656, 0.3114335834980011, 6.661046981811523, -1.2617850303649902, 2.1256356239318848, 4.102694034576416, 0.24253490567207336, -1.3096988201141357, 0.12857586145401, -3.3139634132385254, 1.4472132921218872, -3.781606435775757, 1.0851730108261108, 0.7732571363449097, 2.4471209049224854, -5.491457939147949, -1.5392042398452759, -1.1152769327163696, 3.228855609893799, -2.503957509994507, -1.1322808265686035, -3.747215509414673, 2.7426068782806396, 0.6352986097335815, -1.0463008880615234, -3.2021803855895996, -2.536419630050659, -3.5054376125335693, -0.7160269021987915, -1.9460846185684204, -2.529093027114868, 3.180734634399414, 0.17156067490577698, 4.999847412109375, -2.1257288455963135, 5.864540100097656, 2.3777151107788086, -2.3641650676727295, -0.8342665433883667, 2.022303581237793, -6.400646209716797, 2.4433114528656006, -3.9283833503723145, -1.4914910793304443, 0.968108057975769, -2.2544612884521484, -0.9436956644058228, 1.198479175567627, 1.6455705165863037, -6.96241569519043, 2.6396539211273193, -2.0966506004333496, -1.3727190494537354, -5.133317947387695, -2.6733274459838867, 3.7826950550079346, -4.897487640380859, 0.15529567003250122, 4.602807521820068, 3.001495361328125, 0.14719747006893158, -3.275850772857666, -0.38236290216445923, -0.3178984820842743, -2.0943477153778076, -1.950310230255127, -1.1565625667572021, -2.086683988571167, 1.4195698499679565, 1.372904896736145, -4.223997116088867, 2.208930492401123, -1.7855136394500732, -5.11392879486084, 1.164501667022705, 0.5191364884376526, -3.67684006690979, 2.084768295288086, 4.092365264892578, -2.501739501953125, 4.430522441864014, -1.0328258275985718, -1.6766796112060547, -1.245758295059204, 0.6451231837272644, 1.7789801359176636, 2.3904953002929688, -2.4676032066345215, 0.7047979235649109, 2.673982620239258, -6.527500152587891, -2.7676682472229004, -0.0047294399701058865, 2.0942792892456055, -2.470515489578247, -1.1959707736968994, 2.1293869018554688, 1.842881441116333, 0.6189727783203125, -1.8680660724639893, -2.6927781105041504, 5.517634391784668, 1.7037312984466553, 3.0660529136657715, 2.245496988296509, 2.1995046138763428, -1.5134625434875488, 0.26347726583480835, 4.968229293823242, 1.5891577005386353, -0.5071861147880554, 3.3041505813598633, 3.5067367553710938, 0.17303858697414398, 11.359294891357422, -1.8339176177978516, 2.4496359825134277, -1.7810230255126953, 0.8153177499771118, 3.8739185333251953, 2.135113477706909, 1.2416058778762817, -0.9489825963973999, -5.678187847137451, -3.074195384979248, -4.730754375457764, -0.8855037689208984, -0.666508138179779, -2.465359926223755, -2.917914390563965, 5.388986110687256, -5.040975570678711, -3.90427303314209, 0.4217962622642517, -1.5272374153137207, -2.4813168048858643, -1.257858157157898, 0.03957148268818855, -1.9877886772155762, 1.666994333267212, -1.9981176853179932, -3.252659320831299, -5.42397403717041, 0.7190014123916626, -3.1039481163024902, -0.40163442492485046, -3.3727633953094482, -1.983606219291687, -2.830449104309082, -0.10251036286354065, -2.383636474609375, -5.012266635894775, -0.8829517364501953, 4.264565944671631, -3.7211341857910156, 1.8047754764556885, -1.0671610832214355, -3.3707833290100098, -2.3712410926818848, -2.4933626651763916, -3.110351085662842, -1.5268301963806152, -2.508378028869629, 1.3590344190597534, 0.7262754440307617, -0.6228786706924438, 4.723393440246582, 5.893772602081299, -0.6250157356262207, 1.1935054063796997, 5.347899436950684, 3.0775065422058105, 2.2952077388763428, -1.3624824285507202, -2.356663942337036, 7.417928695678711, -5.024901390075684, -0.8591042757034302, -3.099684238433838, 2.756194591522217, 0.7750988602638245, 1.5759308338165283, 3.9607930183410645, -1.3856558799743652, 6.5611724853515625, 2.234046697616577, 1.4566543102264404, -4.994541168212891, 5.988189697265625, 2.85447359085083, 4.539619445800781, 0.7404916286468506, -3.763800621032715, 1.3326598405838013, -0.5421625971794128, -2.3823864459991455, 3.131397247314453, -5.420820713043213, 0.47282832860946655, 1.891339659690857, -2.788541555404663, 3.090517997741699, -4.784127235412598, -0.4393923878669739, -6.382734298706055, 3.195038080215454, 2.4142982959747314, 2.195460557937622, 1.1945598125457764, 3.4084420204162598, -0.9633074998855591, -1.5162065029144287, 2.047679901123047, 1.7235569953918457, 0.9359923601150513, 0.9639964699745178, 0.24040153622627258, 5.36881685256958, -5.514983177185059, 0.19527532160282135, -0.5182386636734009, -0.287062406539917, 2.448019027709961, -4.723217487335205, 4.984479904174805, -0.676966667175293, -3.909351110458374, 1.4505460262298584, -7.325010299682617, -3.9311318397521973, -0.7179966568946838, 1.0602116584777832, -2.457592487335205, -0.9219083786010742, 4.055387020111084, -0.8364535570144653, 3.3977372646331787, 0.07388357073068619, -2.2718870639801025, -1.5926578044891357, -5.326562881469727, -1.9718270301818848, 1.7378625869750977, -0.45018619298934937, -0.8777185678482056, 0.32039493322372437, -0.6243401169776917, 0.7441943883895874, -0.15817490220069885, -1.1265101432800293, -0.21955564618110657, 5.743565559387207, 0.45313140749931335, -0.3338869512081146, -1.0916333198547363, 0.17708048224449158, 1.2370657920837402, 6.449947357177734, -7.351522445678711, 1.672637939453125, -0.3143549859523773, -1.0382341146469116, 6.22223424911499, -3.318471908569336, 0.978988766670227, 2.310229778289795, 8.108867645263672, -0.4407570958137512, 1.9255645275115967, 3.728193998336792, 0.5778820514678955, -1.1407545804977417, -0.766713559627533, 1.4065290689468384, -1.567272424697876, -2.9338583946228027, -3.3418800830841064, 0.4458519220352173, -6.813573360443115, -2.2943789958953857, -1.0801905393600464, 5.715517520904541, 1.4601879119873047, -2.6963207721710205, -0.5590248107910156, -1.4444574117660522, 2.431563377380371, 4.642305374145508, 1.045666217803955, 3.544776439666748, 0.1438959687948227, 0.6199857592582703, -0.9830015301704407, -5.786379814147949, 3.802034854888916, -0.021554194390773773, -3.4831933975219727, -3.742617130279541, 2.656832456588745, 1.5598137378692627, -1.2459406852722168, -2.9216089248657227, 4.428494453430176, 2.7879691123962402, 1.345205307006836, -0.22773611545562744, 2.700413227081299, 0.2407766729593277, -1.8763489723205566, -2.901298999786377, -2.990396738052368, -3.915107488632202, -2.386819362640381, 3.986858367919922, -5.722116470336914, -3.0097098350524902, -1.6349081993103027, 3.9225239753723145, -0.12256713211536407, -0.8652611970901489, 3.292182445526123, -1.1779671907424927, -6.107779502868652, 3.7281696796417236, -2.154055118560791, 4.849674224853516, -0.05394294485449791, 5.410721778869629, -0.7068154215812683, -2.0858564376831055, -4.649835586547852, 0.08959430456161499, 4.291084289550781, 5.716493129730225, 5.141141891479492, -1.4255802631378174, 1.7239110469818115, 4.622365474700928, 0.6194007396697998, -6.852074146270752, -0.9234545230865479, 2.702244281768799, 1.7213913202285767, 2.1869170665740967, 3.2917237281799316, 3.1420254707336426, -0.5460478067398071, -1.4120607376098633, 2.327484130859375, -2.4819486141204834, 3.028285503387451, 0.7720593214035034, -4.301811695098877, -1.7778375148773193, -1.6972250938415527, -0.2846863567829132, -2.180870771408081, 1.0788301229476929, 3.4504075050354004, 3.5110878944396973, -3.5294981002807617, -5.3175272941589355, 3.273256778717041, -3.987539291381836, -3.3185126781463623, -0.8567842245101929, 3.3913328647613525, 4.5248541831970215, -2.894183397293091, -11.77194881439209, 1.889682412147522, 0.7844139337539673, -4.040999889373779, 0.42001187801361084, 7.009390830993652, 4.6664862632751465, 0.9377418756484985, -1.686384677886963, 5.9764814376831055, -4.369952201843262, 1.9271782636642456, 2.2176685333251953, 2.6686089038848877, 4.668999671936035, 2.975857734680176, -6.764330863952637, 2.1842899322509766, 1.9731807708740234, 1.8142368793487549, 0.37764811515808105, 4.752859115600586, -1.4141485691070557, -2.4792826175689697, 0.7637162208557129, -0.7730218172073364, 1.121824860572815, 0.2744695842266083, 2.593291759490967, 0.4940846562385559, -0.282825231552124, -0.4433550238609314, 0.7317897081375122, 0.32025474309921265, -1.9418258666992188, 0.2883750796318054, -5.324873447418213, 2.3016433715820312, 0.009417427703738213, -0.1873534470796585, 1.2556395530700684, -1.9590463638305664, -2.3758788108825684, -5.118895053863525, 0.3438495397567749, -0.11298304796218872, -1.026243805885315, 4.696951866149902, -5.646001815795898, 4.228837013244629, -3.0654282569885254, 1.1107912063598633, -4.770486354827881, -1.5981334447860718, -3.5179672241210938, -0.3675960302352905, 0.9337493181228638, 2.728614091873169, 1.0904507637023926, 4.215073108673096, 0.6910591721534729, 4.70326042175293, 0.7499537467956543, -0.18542565405368805, 2.561431407928467, -7.2836737632751465, -1.213968276977539, 2.5681304931640625, -1.6098908185958862, -2.006334066390991, 5.515292167663574, 2.8689639568328857, -0.01878437027335167, -1.446866750717163, 2.0674941539764404, 0.7042574882507324, 1.1029902696609497, -2.819162607192993, -3.974972724914551, -1.2314369678497314, -1.2704870700836182, -4.344655513763428, -0.2792356014251709, -7.192670822143555, -3.2380642890930176, -1.4525337219238281, 0.03539328649640083, 1.4435067176818848, -0.9584218263626099, 1.8483450412750244, -2.135277032852173, 1.3274883031845093, 2.994410991668701, 5.7875566482543945, -2.190598487854004, 2.7235238552093506, 3.903271198272705, 1.7961770296096802, -3.65061616897583, 5.873969078063965, -3.3978939056396484, -3.1848583221435547, -2.9671013355255127, -0.21176473796367645, 0.6739248037338257, -2.610001564025879, 0.2887342870235443, 1.5080757141113281, 0.13862641155719757, -0.2067142277956009, -0.5003406405448914, 1.319455623626709, 0.6887674331665039, -1.3241320848464966, 2.015693187713623, -4.528715133666992, 6.7899298667907715, 0.2122231125831604, -2.3344099521636963, -0.8699824810028076, 0.8530845642089844, -0.7450258731842041, -2.4784317016601562, 1.0647435188293457, -0.2794370651245117, -3.492206573486328, 6.6416401863098145, -0.18531736731529236, -0.781111478805542, -3.2618987560272217, 1.5669853687286377, 6.233715057373047, -0.8450157642364502, 0.4451879560947418, 2.0713841915130615, -3.1270058155059814, -3.132007598876953, -3.5537071228027344, 1.221913456916809, 3.1163883209228516, 2.3794264793395996, -0.5941222906112671, -1.1157842874526978, 3.6181435585021973, -1.5994384288787842, 1.2644805908203125, 3.8819379806518555, -2.2869162559509277, 1.4225397109985352, -1.8983569145202637, 2.6932473182678223, 3.115218162536621, -0.6781288981437683, -0.03667036071419716, -3.8019232749938965, -2.50785493850708, -4.998250961303711, -0.6817044019699097, 1.8873851299285889, 2.47350811958313, 0.07409698516130447, -1.9747679233551025, 1.6171910762786865, -4.579605579376221, 0.8494800329208374, 0.5288299918174744, -6.935412406921387, 4.371813774108887, -1.4938528537750244, 0.3957562744617462, -3.0575385093688965, -0.09041880071163177, -1.755237340927124, -1.8516321182250977, -6.257706642150879, 5.647572040557861, 3.0517380237579346, -4.858763217926025, -0.4335681200027466, 6.520421504974365, 0.7577400207519531, 2.0112764835357666, -7.9507646560668945, -6.284292221069336, -0.24873748421669006, 2.2434885501861572, -3.576014518737793, -2.2394933700561523, 2.705939769744873, -0.6713420748710632, 3.6823232173919678, -3.3770041465759277, 3.751833915710449, -3.1702699661254883, 1.2141495943069458, -3.579139232635498, 2.0493249893188477, 2.298062801361084, 3.1590213775634766, -0.49542057514190674, -1.8598949909210205, -3.5119848251342773, 1.0575659275054932, -0.5489833950996399, 1.7507696151733398, 1.4828031063079834, 4.56887149810791, 4.113497734069824, -2.0987937450408936, -1.0410466194152832, 3.0690670013427734, -0.15971529483795166, -4.3089823722839355, 5.401021957397461, 4.632884979248047, -1.8957550525665283, -1.8596622943878174, -0.41569721698760986, -2.695028781890869, 1.7381010055541992, -1.9112210273742676, -0.11568784713745117, -3.815241813659668, -0.9684536457061768, -0.9928470849990845, -7.588696002960205, -4.537217617034912, -4.199419975280762, 1.137477993965149, 4.342000961303711, -0.6268684267997742, -3.0728578567504883, 0.14925435185432434, 3.4186041355133057, 1.5557852983474731, 7.589761734008789, -0.11586235463619232, 1.4263877868652344, 1.103929877281189, 7.379157066345215, -1.6233773231506348, -0.029128532856702805, 3.7113285064697266, 4.482351779937744, -0.2261778563261032, 0.7336592674255371, 1.9842021465301514, 5.769364833831787, -1.1688199043273926, -0.6251341104507446, 0.601911187171936, 0.10207550972700119, -1.6167004108428955, 0.5729418992996216, -2.642531633377075, -1.4029892683029175, -2.5525405406951904, 4.604920387268066, -4.326813220977783, -1.4036781787872314, -3.456967830657959, -1.4362214803695679, 5.114103317260742, -4.563038349151611, -0.975304126739502, -1.019230604171753, 1.8768857717514038, 0.7159274816513062, 1.888587474822998, -5.31599235534668, 0.4218488037586212, 2.1717660427093506, 0.27441051602363586, -3.7378005981445312, 0.32610952854156494, 1.8138799667358398, 1.0688878297805786, 1.2017157077789307, 1.8603763580322266, -0.6041611433029175, 1.263948917388916, -4.725430488586426, -6.180316925048828, 4.5758867263793945, -7.2301025390625, 3.6980788707733154, 3.3501713275909424, 6.275325775146484, 2.248842239379883, 1.1913468837738037, -2.320740222930908, 0.6338464617729187, 1.1834487915039062, -5.991084575653076, -0.815685510635376, 0.01710594817996025, 1.3196170330047607, 2.8164756298065186, 2.4568560123443604, 6.7474894523620605, -3.364393472671509, -2.665879726409912, -5.231089115142822, 1.478480339050293, -4.36460542678833, -3.654033660888672, 1.6386053562164307, 2.287165403366089, -3.7044119834899902, 1.9170243740081787, 1.7848531007766724, -0.023971103131771088, -0.022738821804523468, 1.732336163520813, 5.157479286193848, -0.6960687041282654, -4.571307182312012, -2.703946113586426, -2.54807448387146, -0.3775210976600647, 0.05435287579894066, -3.2272045612335205, 0.8377796411514282, -0.2593322992324829, -2.162940740585327, 2.8815414905548096, -1.4475934505462646, -1.009415626525879, 1.7553706169128418, -3.121242046356201, -3.619112014770508, -1.2094236612319946, 0.5133172273635864, 2.3664982318878174, -3.701902389526367, -4.745206832885742, 1.643524169921875, -0.2532203495502472, 1.7515044212341309, 4.031103134155273, -1.225888729095459, -6.261404514312744, -0.22661492228507996, -2.2972636222839355, 0.4296688735485077, -0.7602431774139404, -0.8020018339157104, 2.4022505283355713, 1.1528514623641968, -1.3919302225112915, -2.1183736324310303, 2.30311918258667, 0.7259894609451294, -2.7199158668518066, -4.655852317810059, 1.1354087591171265, 0.37440311908721924, 0.10254742205142975, -1.754692792892456, 7.039144515991211, 1.3960338830947876, -4.196795463562012, 0.6267269849777222, -2.987436294555664, -0.9547195434570312, 4.40128231048584, -2.4079158306121826, -1.0013548135757446, -3.7325384616851807, 1.9950315952301025, -1.1083965301513672, -3.8170166015625, 2.8461129665374756, 2.5049352645874023, -1.6854021549224854, 4.790022373199463, -0.7160181403160095, 4.303998947143555, -3.040694236755371, -3.057098150253296, -5.825373649597168, -4.291109561920166, 2.966484308242798, -2.362154006958008, -2.0451900959014893, 2.3069963455200195, 0.5241336822509766, 5.254616737365723, -1.5864691734313965, 1.0565749406814575, -0.5420886874198914, -2.655251979827881, 0.9311368465423584, 2.3406448364257812, 5.871945381164551, 0.3476925790309906, 3.3639655113220215, 3.297966241836548, -0.5310996770858765, -5.424839019775391, -0.28371429443359375, -1.783048391342163, 2.6038084030151367, 2.175445556640625, -4.5789690017700195, 0.3224423825740814, -4.127830505371094, -1.4172219038009644, -2.4681644439697266, -3.7738938331604004, -0.7364576458930969, -3.6277456283569336, -3.8856148719787598, -0.46943777799606323, -0.10797905921936035, -2.907693862915039, 5.536844253540039, 2.2109878063201904, -2.8901772499084473, 0.8999775648117065, 1.1027218103408813, -1.9011729955673218, 1.3535163402557373, 0.37838250398635864, -6.534950256347656, -8.390678405761719, 0.1608904004096985, -2.2193691730499268, 0.36255714297294617, -6.822938919067383, -2.4723145961761475, -4.98431396484375, 2.882143497467041, 3.8379499912261963, -2.523886203765869, 1.2306820154190063, -4.316465377807617, -0.46473366022109985, 0.7511199712753296, 0.13210412859916687, 1.6374483108520508, 2.217881679534912, 2.4498565196990967, 3.4329276084899902, -1.6459354162216187, 1.720681071281433, 0.6494484543800354, 2.3791046142578125, -0.2631266415119171, 0.8950297832489014, 2.6390180587768555, 1.6667194366455078, 2.6405954360961914, -4.119887351989746, 0.3477535843849182, 1.8633973598480225, -3.0847182273864746, -3.578322172164917, -0.5360879302024841, 4.354263782501221, -0.9317423105239868, -5.184761047363281, -4.592726707458496, -0.881411612033844, -0.8173642158508301, -0.46636274456977844, -5.814141273498535, 3.289297342300415, -0.5708312392234802, -2.578939914703369, 2.622258424758911, 2.4631364345550537, -2.679868459701538, -5.1330790519714355, -2.2078230381011963, 3.3756494522094727, 1.7280099391937256, 0.43276646733283997, -4.249924659729004, -0.33298322558403015, -5.595227241516113, -0.7099218368530273, 4.935308456420898, -1.823911428451538, -2.630235195159912, 1.2147513628005981, -3.8502438068389893, -0.4026985168457031, -5.031940460205078, -2.973752737045288, -1.6569795608520508, 2.1776390075683594, 0.6797786951065063, -2.2478623390197754, 0.06892433762550354, -3.0223960876464844, 3.5525550842285156, 4.4321794509887695, 2.759312152862549, 3.7610156536102295, -1.027122139930725, 3.559483528137207, -3.918879508972168, 3.073291301727295, 6.007630348205566, -0.193453848361969, 1.4332000017166138, -2.518512010574341, 1.8671311140060425, -1.580552101135254, -0.7563140392303467, -6.483834743499756, -0.05785215273499489, -0.6690158843994141, 0.7843377590179443, 2.5886285305023193, -3.6581063270568848, -0.42080920934677124, -0.11971582472324371, -2.73348069190979, 0.19458049535751343, 4.269458293914795, -4.6572370529174805, 5.780357360839844, 1.2894591093063354, -0.7182207703590393, -1.8879587650299072, 2.7537670135498047, 4.359162330627441, -3.92669677734375, 5.126923561096191, 3.6017417907714844, -1.0186176300048828, -2.46358585357666, -1.6299781799316406, 2.937345027923584, -0.09332890808582306, 1.3869006633758545, 0.74549400806427, -0.750458836555481, -2.454427719116211, 1.0825297832489014, 0.955369770526886, 4.072128772735596, 2.068218946456909, -2.212808609008789, 2.960594654083252, -0.27652543783187866, 2.9854445457458496, 0.12550798058509827, -0.2497338056564331, -4.023211479187012, 0.8763980269432068, -1.3019037246704102, 2.3162403106689453, 0.4752022325992584, -2.388185739517212, 3.4799559116363525, -6.892571926116943, 0.2833523452281952, -1.4288196563720703, 3.0775601863861084, 1.8870594501495361, -1.7723461389541626, 1.7405571937561035, 4.425246238708496, 2.232229709625244, -2.182607889175415, -0.3580043613910675, 0.2806572914123535, -1.8214385509490967, -0.7715515494346619, 0.7367655634880066, 0.7977743148803711, -0.7385234832763672, -4.914363861083984, 3.525501251220703, -2.9001712799072266, 1.61038076877594, -1.9102084636688232, -1.5329968929290771, -4.337355136871338, 5.36319637298584, 0.7802711725234985, 0.9691343307495117, 0.9856024980545044, -0.32416921854019165, -2.413654327392578, -0.18430224061012268, -6.192376136779785, -1.597154140472412, -1.6246830224990845, 0.2964957356452942, 3.7291882038116455, 1.9981695413589478, -2.491710901260376, 2.3668160438537598, -3.21193528175354, -0.08733689039945602, -4.121036052703857, -2.416769504547119, -0.6338621973991394, -1.5291227102279663, -1.8765860795974731, 1.3045586347579956, 2.4573161602020264, -3.418311595916748, -2.4329845905303955, -2.3389010429382324, 1.0161775350570679, 6.097002029418945, 0.44025322794914246, -0.5191253423690796, 3.5381240844726562, 4.752577781677246, 0.8346779346466064, -0.231184720993042, 5.220978736877441, 1.95443594455719, -2.2739715576171875, 3.6954360008239746, -0.5086402893066406, 2.74141001701355, -1.417270541191101, -8.336633682250977, -1.8785736560821533, 1.1486046314239502, -3.6883292198181152, -2.2693729400634766, -0.7190609574317932, 3.5551633834838867, -1.5629518032073975, 1.9940810203552246, 0.16042141616344452, 2.1192517280578613, -1.2717487812042236, 2.2919530868530273, -0.9211121797561646, 3.970160722732544, 0.22283269464969635, 3.821422576904297, 0.8922979831695557, -5.349785804748535, -8.354460716247559, 6.686531066894531, -0.1448565125465393, -8.535146713256836, 2.7202537059783936, 1.744489073753357, -1.3719922304153442, -1.4338297843933105, 2.1263785362243652, -6.871906757354736, -1.446589469909668, 1.8281176090240479, 1.2995681762695312, 1.2909126281738281, -0.17844270169734955, -0.9403239488601685, -4.716504096984863, -1.2757492065429688, -4.465816020965576, -0.8424673676490784, 2.8906307220458984, 2.431896448135376, 0.506511390209198, 1.4563651084899902, -5.498928070068359, 2.0081381797790527, -4.901732444763184, -3.8382225036621094, -1.5982027053833008, 1.7651242017745972, 2.5835416316986084, -2.6487388610839844, 2.2579236030578613, -0.8222163319587708, -0.23494577407836914, -1.2170891761779785, 3.7151050567626953, -0.537057101726532, -0.02273033931851387, -2.285989999771118, -2.638476610183716, 3.080371856689453, 0.6484562158584595, -1.7626805305480957, -3.344038486480713, 4.307393550872803, 8.363180160522461, 2.6563284397125244, 1.7128868103027344, 1.7741700410842896, 3.6291723251342773, 0.32857003808021545, 2.6397619247436523, 2.2616569995880127, 0.07051198184490204, -1.780980110168457, 5.441500663757324, -2.1619627475738525, 0.49379706382751465, 0.4343225061893463, -1.099083662033081, -3.903475284576416, -2.3723256587982178, 0.8327428102493286, -3.2276973724365234, -4.125908374786377, -0.09007135778665543, 0.598555326461792, 2.4191226959228516, -3.1749038696289062, -2.6085879802703857, 2.0880401134490967, 0.4185021221637726, 0.6598411798477173, 0.6722699999809265, -4.013548374176025, 3.572502851486206, 5.571868896484375, -2.091052532196045, -1.61331045627594, 0.9354240894317627, -8.036168098449707, -0.9048866033554077, 2.194141149520874, 0.7695459127426147, -3.6362223625183105, 6.061559677124023, 4.736441612243652, 3.9428651332855225, 2.272308349609375, 1.8456363677978516, -7.977956771850586, 3.553981304168701, 1.3477826118469238, 0.7919589281082153, 2.499191999435425, -4.021824359893799, 5.5843825340271, -2.8536903858184814, 5.301459789276123, -2.5371527671813965, 1.950703740119934, 1.530958890914917, 3.114651918411255, 5.21897029876709, -0.3899760842323303, 1.3736140727996826, -2.056182861328125, -2.7925641536712646, -3.226562023162842, 2.343630790710449, 2.037661552429199, -3.9886302947998047, -0.06259982287883759, -1.5478947162628174, 1.6291744709014893, -0.6522639989852905, -0.1949288249015808, 3.1560091972351074, 3.0204100608825684, -4.353288650512695, 2.5563652515411377, -3.3992199897766113, 3.5321030616760254, 0.7002419233322144, -0.7889689207077026, 3.037100315093994, 1.1033759117126465, -0.028611652553081512, 0.4817489683628082, 1.4079262018203735, 0.4953085780143738, 1.8012409210205078, -0.022638095542788506, -4.71121883392334, -2.576943874359131, -0.8528653383255005, -1.5584967136383057, -4.77921724319458, -1.7890338897705078, -2.2358102798461914, 3.2446160316467285, 2.511949062347412, 0.4838651418685913, -1.7215139865875244, -2.695364475250244, 4.718386650085449, 2.158243179321289, 2.1410460472106934, 2.1525895595550537, -0.24748951196670532, -2.575551986694336, -1.8176110982894897, 2.690913200378418, -0.6298924684524536, 6.345512866973877, 1.860867977142334, -0.5543017387390137, 0.8364474773406982, -2.2900338172912598, 2.1107516288757324, 3.420860767364502, -0.2767539620399475, 3.2336268424987793, 0.48536261916160583, 0.8594385385513306, -3.5953493118286133, -0.18099035322666168, 6.603591442108154, 0.9729183912277222, -0.11876808106899261, 0.9596741795539856, 4.500412940979004, -6.297914028167725, -2.4625625610351562, 1.564622402191162, -1.9817357063293457, 1.944087266921997, -0.4816724359989166, 2.9608941078186035, 3.655397891998291, -2.155573844909668, -3.038273811340332, -0.6557069420814514, 4.423367500305176, 2.8339452743530273, 3.393601655960083, -1.052624225616455, -6.781228065490723, 0.40117043256759644, 5.152740478515625, -3.442788600921631, 2.031139850616455, -6.903495788574219, 2.705073356628418, 1.8183964490890503, -1.3420965671539307, 3.929084300994873, 0.44461682438850403, 0.5025836229324341, 4.792511940002441, 1.9985642433166504, -7.538784027099609, 2.909515380859375, 2.782653331756592, -4.337926864624023, 3.997102975845337, -0.4037686586380005, -0.044905174523591995, 5.228178977966309, -2.60516357421875, -5.006799697875977, 5.393185615539551, 0.04184259474277496, 4.043149471282959, -3.403322219848633, 0.7639310359954834, 4.480724334716797, -1.414263129234314, -0.1312185674905777, -0.14001548290252686, -5.145812034606934, 6.336774826049805, 0.9187700748443604, 2.0915257930755615, -7.66988468170166, 3.694448232650757, -5.090110778808594, 3.0109026432037354, 1.6354167461395264, 6.025454521179199, 0.49727755784988403, -1.6463810205459595, -2.1804380416870117, 0.8605713844299316, -3.3373074531555176, 2.9660885334014893, -0.10858970880508423, -3.2199883460998535, -2.827857494354248, -4.419691562652588, -1.0745700597763062, 1.3308799266815186, -1.1408464908599854, -2.874809741973877, -6.865658283233643, 5.04415225982666, 1.3749659061431885, -1.683349609375, 2.119457960128784, -1.288118839263916, -3.383509635925293, -1.791018009185791, 1.3738083839416504, -1.242854118347168, -3.7697548866271973, 2.1357078552246094, -1.7363178730010986, 0.7080458998680115, 3.7244720458984375, -1.4576634168624878, 4.132285118103027, -1.4913442134857178, -3.956569194793701, -3.9670186042785645, -1.8364577293395996, -4.549642562866211, -3.8024163246154785, 1.9032485485076904, -2.1200616359710693, -1.6109895706176758, -3.5844736099243164, 3.4809799194335938, 3.6509246826171875, 3.985043525695801, -1.0549488067626953, 1.131231665611267, -0.9189047813415527, 1.8791110515594482, -0.8913940787315369, -1.5835721492767334, -1.99092698097229, -1.464036464691162, -3.8912909030914307, -0.6638311743736267, 2.5325236320495605, 5.608027458190918, -3.4777140617370605, -1.0937867164611816, -0.35129210352897644, 3.399771213531494, -1.6766912937164307, -0.9754147529602051, -1.5366778373718262, 3.6015007495880127, 3.607769250869751, 0.1954258382320404, 0.6788910627365112, 1.9140613079071045, 8.823654174804688, -1.5781242847442627, -0.5066076517105103, -1.666224718093872, 1.6876025199890137, 3.0774736404418945, 0.9798880219459534, -0.5959552526473999, -1.6486027240753174, 1.5165727138519287, 0.3976508378982544, -3.258103609085083, 1.6669385433197021, -1.8332276344299316, -0.507463276386261, 1.3984875679016113, 4.187763214111328, 0.4106614887714386, -0.964832067489624, -1.4158689975738525, 5.577592372894287, 2.723562479019165, 0.9802406430244446, 2.10013747215271, 0.39470136165618896, 1.2792882919311523, -0.8916208744049072, -3.31333065032959, -1.793808102607727, 1.4625295400619507, -1.1079014539718628, -2.773819923400879, -3.613563060760498, -0.1491069197654724, -5.565853118896484, 5.506389617919922, 0.03623761981725693, -1.1427500247955322, 3.8487229347229004, -1.171158790588379, 3.926598072052002, 0.23403431475162506, -3.3313372135162354, -2.582191228866577, -1.5372941493988037, -1.0982563495635986, 1.5469934940338135, 0.21688717603683472, -3.074624538421631, -0.22414544224739075, -2.5854461193084717, -0.001028135884553194, -0.04856127128005028, -3.013288736343384, 3.4290611743927, 3.5978686809539795, 1.9684149026870728, -0.31642359495162964, -0.005733659490942955, -0.3672483563423157, 4.708948135375977, 1.0151134729385376, 1.68212890625, 1.184765100479126, 1.5969654321670532, 0.7589287757873535, -5.790879249572754, -5.723013877868652, -2.793668746948242, -4.751394271850586, -0.47365301847457886, -0.5850663781166077, 2.7015573978424072, -0.9708543419837952, -2.2148499488830566, -4.226154804229736, -2.267881155014038, -1.0589897632598877, 3.3837053775787354, 1.6673641204833984, -1.3816535472869873, 4.900947570800781, -0.5379128456115723, -0.31809118390083313, -0.6320109367370605, -1.6008381843566895, 0.7705938220024109, -1.5326769351959229, -3.025911569595337, 1.8438282012939453, 2.0888004302978516, -2.5355284214019775, -1.9833844900131226, -2.126333713531494, 3.4821252822875977, 1.306337594985962, -1.5635898113250732, -5.906634330749512, -1.3464691638946533, -2.521744966506958, 2.040205955505371, 0.75067138671875, 2.5357799530029297, 0.12910646200180054, -1.7660880088806152, -0.670785129070282, -3.1690471172332764, 2.274747371673584, 2.673666477203369, -5.670861721038818, -2.199589252471924, 1.202405571937561, -3.635765552520752, 4.343731880187988, 1.5023036003112793, 1.7708537578582764, -1.9250218868255615, -1.239292025566101, 2.243299722671509, -4.337526321411133, 1.790079116821289, -0.6348779201507568, -2.7348597049713135, 4.245295524597168, 0.4614332318305969, 3.015979528427124, -2.3288562297821045, 3.0692801475524902, 1.1786859035491943, -1.0373578071594238, -4.294053077697754, 3.956418037414551, -2.2800004482269287, 5.730677127838135, -0.4526943862438202, 0.45662498474121094, -0.5364309549331665, -1.7133411169052124, 1.658643364906311, -0.35381489992141724, -1.5586252212524414, 0.24790263175964355, 0.7563899159431458, 0.49569252133369446, 3.315035343170166, -1.3096681833267212, -0.8934468030929565, 3.3460135459899902, 1.7160682678222656, -4.163285732269287, -0.8838433027267456, 1.803320288658142, 4.649674892425537, 2.361968755722046, 0.47001200914382935, -3.3299202919006348, -4.5517168045043945, 1.3013713359832764, 4.8519606590271, 3.6776537895202637, 1.2563406229019165, 1.1949986219406128, -0.7098316550254822, -4.818977355957031, 1.2449349164962769, -3.1866495609283447, 2.3427019119262695, 1.1400502920150757, 1.1583954095840454, 3.1652369499206543, -4.053075790405273, 0.8764928579330444, -4.649649620056152, -0.35153263807296753, 1.4434431791305542, 0.3002089858055115, 0.18603740632534027, 0.02134459838271141, 0.5231914520263672, 1.2680922746658325, 3.1403517723083496, 0.6209995746612549, 1.7411866188049316, -0.9380338191986084, -3.4684948921203613, -2.340041160583496, 2.8084425926208496, 2.807952642440796, -1.7694175243377686, -2.520538806915283, -0.5902906656265259, -1.1653927564620972, -5.178438663482666, -0.770750105381012, 0.44707566499710083, -1.3421962261199951, 3.6168065071105957, -3.6530492305755615, -5.504139423370361, 1.6267873048782349, -1.7268006801605225, -3.1387343406677246, -4.3315653800964355, -0.9942020773887634, 5.766239643096924, 4.401706695556641, -0.5546877384185791, -4.924478054046631, -5.429195404052734, -5.774294853210449, 2.7463207244873047, 2.950371742248535, -1.2159364223480225, 0.029572848230600357, -4.020774841308594, 3.0016121864318848, 0.2919432818889618, 1.6919376850128174, 1.4575551748275757, 4.430835723876953, -1.707012414932251, -0.7405552864074707, -1.6224110126495361, -0.566371738910675, 0.9040480852127075, -1.2104802131652832, -0.636131227016449, -1.5063016414642334, -3.924171209335327, -4.84549617767334, 1.6454590559005737, -6.8717145919799805, -5.779756546020508, 0.17865698039531708, 1.0182273387908936, -0.024286815896630287, -1.6034128665924072, -5.274688720703125, -3.99005126953125, 2.01304292678833, -3.4509334564208984, -2.1466002464294434, 0.6103649139404297, 0.815558910369873, -4.290491580963135, -1.1971430778503418, 0.05228903144598007, 1.869286298751831, -3.547414779663086, -1.0871859788894653, 2.2739157676696777, 1.0472735166549683, -2.219517707824707, -4.451726913452148, -6.466817378997803, -3.2298636436462402, -0.6539585590362549, -1.5297868251800537, 1.1790045499801636, -3.0741257667541504, 1.6044319868087769, -4.631840705871582, -0.9344850182533264, -2.006166934967041, -3.8544740676879883, 0.4885677099227905, 3.7686538696289062, 3.951514720916748, -1.7421846389770508, 4.562704086303711, -3.3697564601898193, -2.8031866550445557, 3.4668326377868652, -2.763605833053589, -1.0994162559509277, 0.1393539309501648, -0.10648322105407715, 2.963843584060669, 1.2624669075012207, -5.650678634643555, 1.8930892944335938, -6.535561561584473, 1.504612922668457, 0.49009740352630615, 1.5378429889678955, 3.6188111305236816, -4.2255048751831055, -1.5446457862854004, 4.080378532409668, 1.5178167819976807, 2.33099365234375, -3.6891636848449707, -0.6500070095062256, -5.815217018127441, 4.244576930999756, -8.952823638916016, 4.114426612854004, -4.284955024719238, -2.5354466438293457, 1.3591927289962769, -1.5985114574432373, 4.722168922424316, 1.127532958984375, 1.308428406715393, -0.07155241072177887, 4.799828052520752, -2.5341789722442627, 1.7221319675445557, 2.997826099395752, 1.382995843887329, 3.0181660652160645, 0.940872073173523, 0.537562370300293, -1.0861767530441284, 5.038701057434082, 3.3701562881469727, 4.349861145019531, 4.147535800933838, 4.9201765060424805, 4.20993709564209, 1.0100595951080322, -0.8424850702285767, 1.3987072706222534, 4.713290691375732, 0.2436593472957611, -3.7804603576660156, 1.057800531387329, -5.103835105895996, 0.3934909701347351, -1.3681128025054932, 4.041543006896973, -3.6524267196655273, 1.7904127836227417, -4.022263526916504, -0.7941069602966309, -0.29079148173332214, -0.6394368410110474, 1.870527744293213, -0.032548703253269196, -6.160330295562744, -3.90337872505188, 3.5721888542175293, -4.036022186279297, 2.112800121307373, -4.328495025634766, 3.667893648147583, 1.0113856792449951, -0.6107532382011414, -2.2522919178009033, -3.1886067390441895, -2.3354878425598145, -4.1277055740356445, -0.46657276153564453, 0.8620666265487671, -0.07569187134504318, -0.19926349818706512, -2.721400260925293, 0.253891259431839, -0.12568718194961548, 2.081294536590576, -4.829545497894287, 2.451403856277466, 0.4967857599258423, 4.926802158355713, 0.1077561154961586, 1.7787641286849976, 4.204455375671387, 4.886762619018555, 3.6855263710021973, 0.48693180084228516, -3.1981382369995117, 3.278542995452881, -5.166411876678467, -0.04914538934826851, 1.7847676277160645, 0.19285142421722412, -1.7126102447509766, 0.49080538749694824, -4.17362642288208, 0.8420358896255493, 0.4152613878250122, -1.540051817893982, -3.577877998352051, -4.535739421844482, 3.6079564094543457, 0.29576513171195984, -2.8760175704956055, -0.9228839874267578, -0.4455919861793518, -5.98420524597168, -3.942401885986328, 0.3703303933143616, -3.101933479309082, -1.3285378217697144, 0.6570104360580444, 0.8823232650756836, 2.394080638885498, -0.5460044741630554, 2.3988451957702637, -3.6005730628967285, 2.819612979888916, 0.2283288538455963, 3.44503116607666, 0.16144505143165588, -1.691648244857788, -0.18382373452186584, -0.3616681694984436, -0.12363298237323761, -1.2765867710113525, -6.404294967651367, -8.680679321289062, 0.6802368760108948, 0.7722277641296387, -3.611785888671875, -3.786611795425415, -6.244284152984619, -0.7361305356025696, -0.5644878149032593, 0.3552178740501404, 0.8874015808105469, -7.784369945526123, 3.965524435043335, -4.313967227935791, 1.0584468841552734, 1.2535803318023682, -0.8887086510658264, -3.356808662414551, -1.6463382244110107, -1.6952630281448364, -6.315996170043945, -4.800979137420654, 2.2080531120300293, -1.066444993019104, -3.479506492614746, 1.9374537467956543, 2.174189329147339, -1.5723111629486084, -0.4474683701992035, -3.3804564476013184, 4.106152534484863, 1.8936591148376465, 4.358683109283447, -2.545233964920044, 3.343130588531494, 1.116308569908142, 4.574185371398926, -0.7718168497085571, 2.736069440841675, 4.428737640380859, 3.453171730041504, 3.829033851623535, 0.09343580901622772, 1.3100323677062988, 0.6635507345199585, -3.811767101287842, -0.3693491220474243, 2.2961788177490234, 1.985090732574463, 1.6607093811035156, 1.721017599105835, -2.2635505199432373, -2.565675735473633, -0.890007495880127, 2.502581834793091, -4.49338960647583, -2.9638500213623047, -1.6055041551589966, 3.5878193378448486, 0.6271324157714844, -0.7345209717750549, -2.792848587036133, -0.758834958076477, 0.27821049094200134, 4.746166229248047, 0.3501763641834259, 0.4424097239971161, 4.644327163696289, -1.4868340492248535, -0.47480934858322144, 1.2477773427963257, -2.4235401153564453, 5.528095245361328, 0.5925568342208862, 2.639402151107788, 3.279252290725708, 1.4888958930969238, -5.030368804931641, 0.31596705317497253, 1.561995506286621, 5.856356620788574, -4.31382417678833, -6.301633834838867, 2.9937751293182373, 0.6496682167053223, -2.1068708896636963, -4.255335807800293, 3.4908394813537598, 2.963409900665283, -5.95875358581543, -0.6237192153930664, 1.5301011800765991, -0.9845702648162842, 3.1824264526367188, 0.08512338250875473, 1.2270663976669312, -7.08334493637085, -3.8013787269592285, -0.6418228149414062, 3.7525384426116943, 3.705763578414917, 0.6791445016860962, -1.2489984035491943, 2.931640625, 0.15255144238471985, -2.379519462585449, -0.6181424260139465, -1.080920934677124, -5.691786289215088, -2.5964975357055664, 3.2175469398498535, -1.1755380630493164, 2.3021857738494873, 2.3382253646850586, -2.825019121170044, 1.6018977165222168, -2.2676634788513184, -3.785658836364746, -5.667277812957764, 1.4691975116729736, -2.8373162746429443, -1.9839810132980347, -3.534289836883545, 3.265563726425171, -0.45459967851638794, 2.061203956604004, -1.6030981540679932, -0.8789708018302917, -2.474332809448242, 1.3163925409317017, -3.7547829151153564, 0.23966893553733826, 2.2296762466430664, 1.390781283378601, -3.601052761077881, 0.3038230538368225, -2.9963302612304688, 5.610959053039551, -1.1420128345489502, -0.6878180503845215, -0.2793836295604706, 2.52022123336792, 1.2924808263778687, 3.837838649749756, 0.41895791888237, 0.37195247411727905, -2.279102087020874, -2.4210667610168457, -3.7219176292419434, -1.3763667345046997, -1.9063278436660767, -0.04916568845510483, 0.8377653360366821, 1.2278305292129517, 0.6809109449386597, -1.593787431716919, 4.048213005065918, 2.7209935188293457, 1.631514549255371, -1.8939990997314453, 2.66494083404541, 4.807031631469727, 0.3779330253601074, -1.5668609142303467, -1.2250820398330688, 2.4303994178771973, -2.8930070400238037, -3.600048542022705, 1.5910574197769165, -2.2077183723449707, 0.6650180816650391, -5.607989311218262, -0.4023250937461853, -3.740772247314453, 2.3043880462646484, -0.5213938355445862, 0.9612966775894165, 5.852078437805176, 6.5055341720581055, 1.8295583724975586, 3.1496264934539795, -1.0542267560958862, -6.959305763244629, -1.6775672435760498, 0.8803153038024902, 3.205993175506592, -4.3560028076171875, 0.6104952692985535, -5.171680450439453, -0.5948560237884521, 1.856525182723999, 3.6602206230163574, 0.4977668523788452, -1.016215443611145, 0.2577657401561737, -0.40366625785827637, 0.38557201623916626, 5.475634574890137, 1.651909589767456, 4.745928764343262, 0.343542218208313, -0.09133241325616837, -0.5222474932670593, -5.00783109664917, 1.7641935348510742, -6.058660984039307, 2.6240062713623047, 0.7506901025772095, -1.3366291522979736, 0.8750865459442139, 0.5289347767829895, -0.14015907049179077, -1.0084238052368164, 4.735357284545898, 0.672555685043335, -2.0143444538116455, -3.8493261337280273, -0.27006369829177856, -1.0580730438232422, 0.7734078168869019, 4.1312761306762695, 2.3917453289031982, -0.12900850176811218, 2.161547899246216, -2.5209410190582275, -1.281448483467102, 3.170708179473877, 1.2195268869400024, -0.0810985416173935, -5.294391632080078, 2.141741991043091, 0.5372723937034607, 5.618747711181641, -0.3817543387413025, -0.68168044090271, 1.6826956272125244, -0.5390093326568604, -0.5314589142799377, -3.2541189193725586, 2.882200002670288, 5.359347343444824, -3.532869815826416, 3.0449602603912354, -0.6487178206443787, -3.934414863586426, 1.8481035232543945, -0.38378435373306274, 5.401391506195068, 0.6265320777893066, -2.450047731399536, -1.7699940204620361, -0.03585262596607208, 2.7339773178100586, -2.2966108322143555, 4.985832691192627, 5.066269874572754, 2.5076334476470947, 0.06892500817775726, 0.1612444967031479, -5.589169025421143, -1.8082911968231201, -6.658154487609863, 0.29080167412757874, -0.5643163919448853, 5.812313079833984, -2.4545326232910156, -0.7480292916297913, -3.317559242248535, -2.600681781768799, -1.462465763092041, 0.03695845231413841, 7.9248247146606445, 0.8080192804336548, 3.823862314224243, -3.630904197692871, -1.6740446090698242, 2.1490373611450195, -0.11778136342763901, 2.9065654277801514, -0.3538205921649933, 3.9814488887786865, 4.561287879943848, -0.7562609910964966, -0.2579125761985779, 0.44478654861450195, 0.0793645977973938, -1.5666782855987549, -0.6547621488571167, -0.9309669733047485, -2.8613638877868652, -0.2895588278770447, 2.4590067863464355, -2.82053804397583, -1.198377251625061, 4.916621208190918, 2.462660074234009, 3.9335827827453613, -0.43716609477996826, 0.7996279001235962, 2.910177230834961, -5.319609642028809, 4.1118574142456055, -3.7923195362091064, 6.946988105773926, -0.8276014924049377, 0.6052036881446838, -2.6456985473632812, 0.6357690095901489, 1.213989496231079, 1.9666180610656738, 1.4732342958450317, -1.0353388786315918, -0.6725887060165405, 2.0436043739318848, 2.018706798553467, 4.4786577224731445, -2.989955425262451, 2.3878700733184814, -5.596046447753906, -4.892147064208984, -7.075316429138184, -1.5215553045272827, 3.0286123752593994, 1.8249964714050293, -4.791776180267334, 0.5652857422828674, 2.3256876468658447, 4.851350784301758, 1.0649319887161255, -1.5207786560058594, -2.4574930667877197, -2.7826459407806396, 0.5781264305114746, 2.6673009395599365, -0.5136331915855408, -0.8671901822090149, 1.4069385528564453, -4.285152435302734, 4.640028953552246, 1.2010912895202637, -0.5111397504806519, -5.995543479919434, 2.774993419647217, -0.08916674554347992, -0.7746385931968689, 1.4880201816558838, 3.23720383644104, -4.906560897827148, -0.3544236421585083, 2.8045358657836914, -1.4765210151672363, -3.5589466094970703, -0.1022796705365181, -1.5530022382736206, 3.4067726135253906, 1.356308937072754, -3.6425607204437256, -2.0978705883026123, -3.7618181705474854, -0.7269735336303711, -1.1794120073318481, 4.009919166564941, 0.9851785898208618, -2.8755042552948, -2.6255767345428467, -7.3965606689453125, 1.4187939167022705, 0.14984562993049622, -4.746469497680664, -2.0233957767486572, -3.0392839908599854, -2.5969924926757812, -1.0860497951507568, -2.1235926151275635, -4.7860918045043945, 0.40524762868881226, 1.8883893489837646, 0.611175537109375, -3.0971264839172363, 1.4159907102584839, -4.030628204345703, 6.94736909866333, -5.239396572113037, 1.650701880455017, 3.106396198272705, -3.0714550018310547, 4.082827568054199, 1.5895562171936035, -2.185612440109253, -2.6552226543426514, 1.7464202642440796, 8.087906837463379, 0.10328366607427597, 1.6205382347106934, -0.440551221370697, 3.226536750793457, -1.063106656074524, -2.093745708465576, -2.2529008388519287, -0.1163279190659523, -3.4550461769104004, -4.344964981079102, -0.08358785510063171, -4.195075988769531, 1.5604023933410645, -1.5822758674621582, 1.663496971130371, 2.1776585578918457, 2.079777717590332, -2.681436777114868, 0.7131627202033997, 2.6762473583221436, 1.4777789115905762, 1.6204938888549805, -1.663137435913086, 0.45879900455474854, 2.9691569805145264, -1.3841934204101562, -5.070013999938965, -1.5068061351776123, -1.3457465171813965, 0.12202996015548706, 0.9676538705825806, -1.7038753032684326, 2.234647274017334, 6.328690052032471, -0.028668247163295746, -3.543710231781006, -1.9556097984313965, 0.4255185127258301, 1.882603406906128, -1.5827038288116455, 2.895662784576416, 0.6179615259170532, 2.0398683547973633, 3.1252005100250244, 3.336358070373535, 0.3439134955406189, -1.9536463022232056, -1.76478910446167, -2.3459949493408203, -2.719564914703369, 0.252427875995636, 0.1558188647031784, -4.343003273010254, -6.417795181274414, -1.784918189048767, 5.338135719299316, 3.9285030364990234, -6.131805896759033, -0.6188066005706787, 2.2741291522979736, 1.2680637836456299, 2.6550493240356445, 1.1000101566314697, 0.014734393917024136, 4.6852874755859375, 0.6757490038871765, -3.3267769813537598, -5.593968391418457, 0.5264933705329895, -0.3632471561431885, -1.0033808946609497, -0.49147945642471313, 0.6984114646911621, 1.895233154296875, 1.3325550556182861, -0.8279838562011719, -1.3763833045959473, -3.8553466796875, 1.453708291053772, 1.976347804069519, 0.6339066028594971, -3.5502567291259766, -1.5259026288986206, -0.8072406649589539, 1.0187453031539917, -1.055227279663086, -4.738816261291504, -0.6938280463218689, 2.724212169647217, 2.5788674354553223, 0.21059370040893555, -4.051334857940674, 0.392665833234787, -0.8823423385620117, -1.6722092628479004, 1.31706964969635, -2.6373753547668457, -6.5091400146484375, 2.478506565093994, 3.1291823387145996, 4.100982189178467, 1.960837483406067, 1.22427237033844, -2.26530122756958, -4.73384952545166, 3.5706915855407715, 3.2964470386505127, -6.853468894958496, -4.34654426574707, 0.7228000164031982, -3.504413604736328, -3.959235191345215, 2.348741292953491, 1.3613603115081787, 0.25229412317276, -2.4921607971191406, -4.423165798187256, 1.6483383178710938, -0.17540429532527924, 7.349871635437012, 4.875646114349365, 1.929429054260254, 0.477977454662323, -4.237002849578857, 1.1188740730285645, -3.8859610557556152, 0.9528716802597046, 2.0741732120513916, -2.4542617797851562, 3.1224822998046875, 1.5601468086242676, -1.4294934272766113, 1.2391371726989746, -6.322819709777832, -3.4298653602600098, 0.9574708938598633, 0.2119300663471222, 1.8530292510986328, 2.0003392696380615, -0.7749543786048889, 1.0016911029815674, 3.2624096870422363, -1.7538995742797852, 1.8068358898162842, -4.7995781898498535, 3.5326087474823, 1.857647180557251, -1.669915795326233, 2.1764421463012695, 5.953861236572266, -2.2498793601989746, 4.021418571472168, -4.048968315124512, 5.3638386726379395, 2.833279848098755, 0.12506930530071259, 2.2856338024139404, 2.0394797325134277, -4.011094093322754, -0.002816674765199423, 0.6398744583129883, 6.357430458068848, 8.517461776733398, 0.18626496195793152, 3.658724069595337, -2.256176471710205, -0.27043676376342773, -4.798359394073486, 2.346754789352417, 4.019370079040527, 0.3150717616081238, 0.3264558017253876, -1.1394093036651611, -1.7125686407089233, 7.639296054840088, -1.183081865310669, 3.219179153442383, -2.1933376789093018, -0.5590647459030151, 1.3431239128112793, -1.955509901046753, 3.3680155277252197, -1.8803448677062988, -1.589799404144287, 3.2284536361694336, 2.213122606277466, -2.5282607078552246, -2.7011263370513916, -2.8127992153167725, 1.8261923789978027, 3.792598247528076, 8.259539604187012, -1.615755319595337, 0.5597953200340271, -3.2471730709075928, 3.7474818229675293, -4.3844475746154785, 4.589881896972656, 3.174375057220459, -8.226274490356445, -0.8481734991073608, -5.090991020202637, -2.3828134536743164, -0.40128734707832336, -1.0318063497543335, -6.997247695922852, -0.40060746669769287, -0.2293674349784851, 3.9263768196105957, 1.1275604963302612, -3.8991730213165283, 3.2363710403442383, 1.4713078737258911, -3.7475643157958984, 0.5111408829689026, -2.4799532890319824, 2.7148451805114746, -4.1423139572143555, -3.1155238151550293, 1.1466660499572754, 0.18164657056331635, 2.9606475830078125, -4.86828088760376, -4.490482807159424, 3.9125194549560547, -0.089853435754776, -0.20463183522224426, 1.784057378768921, -2.6741764545440674, 0.7287091016769409, 2.3059346675872803, -3.1355695724487305, 1.0230238437652588, -2.6912238597869873, -5.545840263366699, -3.587636947631836, 1.0105774402618408, 0.5783438682556152, 2.209930896759033, 2.76712965965271, -1.2338191270828247, -1.8466424942016602, 1.4266057014465332, 0.9046372175216675, 1.1149516105651855, -7.200931549072266, -1.7809672355651855, 2.7720651626586914, 5.511800289154053, -3.0446157455444336, 1.491455316543579, 0.8919344544410706, -3.232794761657715, -0.04705461859703064, 1.0558364391326904, 2.6152656078338623, 0.9455862045288086, 0.8829793930053711, 1.5768725872039795, 6.438494682312012, 1.1168440580368042, 0.41328978538513184, 1.116170048713684, 0.8440680503845215, -0.7185472846031189, 2.2287545204162598, -1.6284345388412476, -3.3957958221435547, 1.677269697189331, -2.205251932144165, 1.3402844667434692, 0.5718756914138794, -1.1088664531707764, 0.9768612384796143, -0.2440544068813324, -1.3293275833129883, -0.10627768933773041, -2.3135275840759277, -2.1104938983917236, 1.1007277965545654, 0.814881443977356, 3.525141716003418, -0.9689631462097168, 3.0424647331237793, -0.7703023552894592, 1.7851243019104004, -0.01618758589029312, -2.1166186332702637, -3.815242052078247, -0.6212740540504456, -1.745061993598938, 2.906698703765869, 2.2515509128570557, 2.307504177093506, -2.9270241260528564, -3.695779800415039, -2.0085067749023438, 1.1043654680252075, -2.3030641078948975, 2.2105484008789062, 1.80463707447052, -0.8557668924331665, -4.60567569732666, -0.9204959869384766, 0.3107248544692993, 1.7264925241470337, -2.5711445808410645, -1.4405009746551514, 1.4785165786743164, 0.9239212274551392, 4.791037082672119, -0.6265157461166382, 2.3647303581237793, -2.4859957695007324, 1.7993062734603882, -1.7895913124084473, -2.2955527305603027, -1.4417059421539307, -0.6414055228233337, -2.0513057708740234, 0.8897924423217773, -1.3870577812194824, -3.0379066467285156, -0.19311591982841492, 2.24906063079834, 0.9954228401184082, 3.47910475730896, 1.090608835220337, -5.069615840911865, 1.3128585815429688, 6.012602806091309, -1.714263677597046, 2.398371934890747, -3.0478477478027344, 5.597201347351074, 1.0289511680603027, 0.847752034664154, 4.0148725509643555, -4.06897497177124, 2.2644948959350586, -2.3193085193634033, 6.144965171813965, -1.2155096530914307, -5.346038341522217, -4.385923862457275, -2.634713649749756, -0.6460156440734863, -0.7065329551696777, 2.0472259521484375, -5.462419509887695, -1.3584699630737305, -2.753293514251709, -0.0007377263391390443, -3.1090686321258545, -3.759554386138916, -0.48705044388771057, 0.4181271195411682, 0.13243836164474487, -0.06670661270618439, 1.2188656330108643, -2.129953384399414, 2.2038815021514893, -4.269888877868652, -3.591887950897217, -1.1582719087600708, 3.210000991821289, 0.4041212797164917, 0.8718129396438599, 2.4858827590942383, 1.1644477844238281, -0.6821353435516357, 2.7732479572296143, -2.712688446044922, 2.643031358718872, -3.847787857055664, 4.660645008087158, -1.4278167486190796, 2.5955164432525635, -2.26084303855896, -0.7655853033065796, 3.1750192642211914, 0.28233712911605835, 3.5408096313476562, 0.09610531479120255, 2.4081339836120605, 5.419520854949951, 2.881901741027832, 2.0806050300598145, 4.071190357208252, 0.2546267807483673, 2.868697166442871, -4.34730863571167, -0.4989502429962158, -1.8128981590270996, -3.6078879833221436, -1.3359949588775635, 0.6139246225357056, -0.3087935745716095, 2.8405654430389404, -1.4484624862670898, -3.1518137454986572, 3.570772171020508, 5.251781463623047, -2.473416328430176, -4.646655082702637, 0.7369292378425598, 1.5785548686981201, -0.8671870231628418, 2.6039352416992188, 0.22202597558498383, -0.4589136242866516, 0.18302810192108154, -2.5919089317321777, 0.8641026020050049, -2.4546570777893066, 1.3455013036727905, -2.4826948642730713, -3.864149808883667, -2.4981751441955566, -3.168394088745117, -7.00217342376709, 0.6779065728187561, -0.9945068359375, 3.6083288192749023, 3.5436911582946777, 0.8227313160896301, 1.728025197982788, 0.5534490346908569, -2.139431953430176, 0.011576903983950615, -1.0307495594024658, 1.7227495908737183, -0.2215600311756134, -0.42602500319480896, -0.9502092003822327, 0.4758848547935486, -1.256277322769165, -3.2052104473114014, 2.6524109840393066, 0.4925289452075958, 3.941040515899658, -0.18519197404384613, 0.46312975883483887, -1.4881504774093628, 4.497329235076904, -3.053077220916748, 4.118742942810059, -7.192700386047363, 1.724281668663025, -2.5070011615753174, -9.693357467651367, -2.88692045211792, 2.6234521865844727, 2.2517991065979004, -0.6901420950889587, -1.2678359746932983, 0.18775495886802673, 2.50028920173645, -2.1451218128204346, 0.057132538408041, 0.9132995009422302, -1.1894478797912598, 1.5540404319763184, 2.475585460662842, 2.93416690826416, 3.7996554374694824, 3.2044894695281982, -2.6436915397644043, -1.1465141773223877, 2.7899200916290283, -1.8197143077850342, 0.4677932560443878, 0.23570621013641357, -1.6360877752304077, 3.6576523780822754, -1.1592655181884766, 1.730362057685852, -0.22227050364017487, -3.2496581077575684, -1.6472339630126953, -2.6031365394592285, 1.1504113674163818, 2.6156983375549316, 4.994198322296143, 0.5538966655731201, -4.495514869689941, -0.38142579793930054, 3.3342270851135254, -0.28035372495651245, 1.006256341934204, 1.0715439319610596, -7.713898658752441, 2.0883171558380127, 2.563171625137329, -3.5071370601654053, 0.8205318450927734, 7.626315593719482, 0.5083534717559814, -2.172640085220337, 6.04248046875, -0.23977476358413696, -0.3161068856716156, -0.926042914390564, 1.35009765625, 2.0213875770568848, -3.1796140670776367, 1.307816743850708, 2.6792337894439697, -2.0632736682891846, 2.1970958709716797, 0.15643861889839172, 2.8684277534484863, -2.6837735176086426, 1.8359289169311523, -1.9256086349487305, -3.4839725494384766, -5.11143684387207, -0.49273961782455444, 1.0141737461090088, -3.198103427886963, 3.9555537700653076, -6.683791160583496, 1.314264178276062, 1.2047226428985596, -3.271681785583496, -0.8705329895019531, 1.906768560409546, -0.8510919213294983, 2.0856099128723145, -2.6966552734375, 0.8248702883720398, -0.5775384306907654, 3.0440878868103027, -0.04203561693429947, -2.350555896759033, -3.9958572387695312, -1.0844894647598267, 1.6224937438964844, -0.6174808740615845, 0.3919999599456787, -3.3002896308898926, -1.6334155797958374, 4.759598731994629, -1.4807133674621582, -0.1105106920003891, -1.6050755977630615, -1.1131973266601562, 1.7031701803207397, -3.63519287109375, 2.522101402282715, 6.600472450256348, 2.343902587890625, 0.8539279103279114, 5.2018351554870605, -0.3660883903503418, 3.6277260780334473, -0.6879099011421204, -3.2447972297668457, 0.6697533130645752, -0.23569245636463165, 6.130283355712891, -0.35886287689208984, -6.464232444763184, 0.8720957040786743, -4.7042646408081055, -6.5875043869018555, -1.902611494064331, 3.447014331817627, -1.8160117864608765, 2.139911413192749, 1.5938255786895752, 3.234567403793335, -0.7676748633384705, -2.3090925216674805, 0.8904241323471069, 1.873967170715332, 1.5639967918395996, 5.0603179931640625, -1.9436368942260742, -1.616078495979309, 2.6436638832092285, 3.176193952560425, -4.439818859100342, 0.4310750961303711, 6.046767234802246, -3.372169017791748, -2.3898894786834717, -4.524242401123047, -3.8128888607025146, -1.7354226112365723, 2.1626532077789307, 2.469468593597412, 1.2982406616210938, -3.3910868167877197, 2.0041565895080566, -1.1696674823760986, -0.60433429479599, -7.05711555480957, -0.9350755214691162, 2.0555624961853027, 3.745629072189331, -2.0187480449676514, 4.963823318481445, -2.393033266067505, 3.1628189086914062, -0.31544962525367737, -2.449091911315918, 1.9356969594955444, -1.327317237854004, -7.980981826782227, 1.598654866218567, -4.010746479034424, -0.10843989998102188, -0.09343168884515762, 0.39466622471809387, 0.7682306170463562, 8.184823989868164, -2.8083255290985107, 0.6487416625022888, 2.616292953491211, -5.5124711990356445, 3.3112916946411133, -0.581322193145752, -1.6498193740844727, 0.10988496243953705, -4.372857570648193, -2.6936607360839844, 4.779646873474121, 2.4662084579467773, 7.584840774536133, -0.6391758322715759, 1.029144525527954, 0.3972698450088501, -1.1466881036758423, 1.6499638557434082, 0.7887016534805298, 2.9552910327911377, -1.686293125152588, 3.9980340003967285, -0.9859644770622253, -5.3072614669799805, -1.6328253746032715, 4.045270919799805, 2.5987963676452637, -1.0543829202651978, 1.8918575048446655, -1.364760398864746, 1.570585012435913, -0.3002392649650574, 3.9202659130096436, -2.114762544631958, 2.416731357574463, -1.868239164352417, 2.741816520690918, -3.1035828590393066, -4.094080924987793, -0.9199776649475098, 7.144569396972656, -1.1361780166625977, -3.1200649738311768, 1.0852302312850952, 0.04687784984707832, -4.564133167266846, -0.503705620765686, -1.585843563079834, 1.8175137042999268, -1.0711462497711182, -4.117181777954102, 0.5342965722084045, 8.992576599121094, -0.3491383492946625, -1.2134231328964233, 1.9328001737594604, 3.804297924041748, 5.395328521728516, -1.8947067260742188, 0.6349627375602722, -8.384368896484375, 0.6672260761260986, -1.892430067062378, -3.20723032951355, -3.8617422580718994, -0.2651481628417969, 0.09362123906612396, 1.3187353610992432, -4.787083625793457, -4.716872215270996, -3.6273040771484375, -0.8512119054794312, -0.904244065284729, 1.4986565113067627, -0.4095616340637207, 0.6131584644317627, -0.599899172782898, -2.3631398677825928, -0.8566418886184692, 4.825512409210205, -1.9729496240615845, 7.685827732086182, -0.9955073595046997, -5.441128253936768, 3.0948636531829834, -1.5045909881591797, 2.9855706691741943, 0.7052472829818726, 2.2648277282714844, 1.8202232122421265, 3.741534948348999, -2.920438289642334, -0.1429242640733719, 2.1737821102142334, -0.4833563268184662, 4.3032379150390625, 0.2083255648612976, -0.872657060623169, -3.2195420265197754, -1.1387455463409424, 1.498022437095642, 5.134578704833984, -0.8418276309967041, 3.2937216758728027, -1.5491843223571777, -5.330063343048096, -1.2397668361663818, 3.886418581008911, -0.014409556984901428, -0.5575603246688843, 2.431727409362793, 0.05365341901779175, -2.035505533218384, 1.7203484773635864, -2.834778308868408, 3.090874195098877, -2.358414649963379, 2.909440040588379, -3.1094021797180176, -0.9492777585983276, -4.386055946350098, 1.522062063217163, -1.2931902408599854, 0.1299319565296173, -4.495024681091309, -0.27747854590415955, 0.029762525111436844, -2.573545455932617, -1.468612551689148, 1.1271315813064575, 4.1774115562438965, 0.9340392351150513, 6.150393962860107, -4.109519958496094, -2.072554111480713, -3.1623544692993164, 5.270700454711914, 0.47135961055755615, -1.4570133686065674, 5.33566951751709, 1.2990612983703613, -6.81560754776001, -4.617526531219482, 3.519716739654541, -0.4412498474121094, -5.012284278869629, -2.8338351249694824, 1.6234463453292847, -1.019819736480713, 2.4750423431396484, -2.607848644256592, -3.4038915634155273, 1.3349754810333252, 6.17719841003418, 2.7729296684265137, 2.5063300132751465, -1.5947859287261963, -5.268726825714111, 4.1296820640563965, 1.8603551387786865, -0.5625082850456238, -1.4657602310180664, 3.4083545207977295, -3.1869378089904785, -2.5001771450042725, -0.5217017531394958, -1.7773386240005493, 0.20973072946071625, -0.3222557604312897, 0.29617902636528015, -2.6888742446899414, -4.070501327514648, -0.22802630066871643, -2.588528633117676, 2.541144847869873, -1.0289109945297241, -2.4245073795318604, -0.7263152599334717, -0.5244518518447876, -2.5447821617126465, -2.484602689743042, -1.7776484489440918, -4.445522308349609, 0.11034980416297913, -1.9372460842132568, 0.08941513299942017, -3.608640670776367, -0.7661434412002563, -2.2039437294006348, 2.8442440032958984, 1.7892589569091797, -0.37353822588920593, 5.320257186889648, -3.253920078277588, 2.6272623538970947, 0.5447025895118713, 5.541485786437988, -8.84086799621582, 5.321418285369873, -4.141987323760986, 3.758915662765503, -3.835629940032959, 1.865186333656311, -0.3757550120353699, 5.225102424621582, -4.163810729980469, -4.237884521484375, 1.8252592086791992, 1.760812520980835, -4.540764331817627, -0.4254528284072876, 0.3550192713737488, 0.4967944025993347, -1.9913631677627563, 2.0115134716033936, 3.8790111541748047, -3.5844368934631348, 4.116212844848633, -2.3788397312164307, 0.3507881760597229, -1.0292000770568848, 0.6820104122161865, -3.245422840118408, -0.48145854473114014, 1.5436431169509888, -0.7832770347595215, 4.4098310470581055, -2.677596092224121, -0.7571669816970825, -0.7477697134017944, 1.5721246004104614, -2.4836130142211914, -2.7456469535827637, 1.0961941480636597, 0.5050770044326782, -1.3842631578445435, 1.9257087707519531, 0.6379063129425049, 0.9842404127120972, 1.4453468322753906, 5.438556671142578, 5.348921775817871, 1.9992939233779907, -5.833780765533447, -1.6383140087127686, 3.8037734031677246, 3.051352024078369, -1.9112975597381592, 0.46946942806243896, 2.6128413677215576, -0.2906067669391632, 0.5719125270843506, -0.7617393136024475, 0.3254459500312805, -0.8782693147659302, 3.272183895111084, -0.622463583946228, -0.43893447518348694, 2.1917991638183594, -0.12262994050979614, 0.18476685881614685, -4.952418804168701, 0.41350555419921875, 0.29053226113319397, 4.280111789703369, -2.2784812450408936, -0.3893597424030304, 2.707371234893799, -0.8890253305435181, -3.1702609062194824, -0.6431061625480652, 3.3025636672973633, 1.2799798250198364, 4.305462837219238, 6.800572395324707, -1.7017273902893066, 1.3924349546432495, -0.9119149446487427, -2.368572473526001, 1.6483469009399414, -6.281885147094727, 3.3341689109802246, -1.0320842266082764, 1.8985364437103271, 1.1044809818267822, -2.4409570693969727, -1.2403587102890015, 6.193312644958496, -5.557782173156738, 0.7965258359909058, 2.238597869873047, -5.032191753387451, -4.45848274230957, -1.7590408325195312, 1.9293360710144043, 0.16682764887809753, 0.06376497447490692, -0.3612353801727295, 3.8450775146484375, 2.4437661170959473, -1.0757451057434082, -1.4356838464736938, 2.3846347332000732, -2.149564743041992, 2.27876615524292, -3.322122097015381, -2.2169041633605957, 2.121634006500244, -0.026832029223442078, -0.114232137799263, -2.9900760650634766, 1.5543204545974731, -5.874040603637695, -0.13853712379932404, -4.601448059082031, -2.7587616443634033, -1.9086472988128662, -4.146779537200928, -1.0170069932937622, -1.0174083709716797, 1.3173103332519531, 1.6957286596298218, 0.9291509389877319, 4.65424919128418, -5.134554386138916, -0.03444279730319977, -0.1884940266609192, -5.691347599029541, -0.4276700019836426, -3.1654696464538574, -0.38390082120895386, -2.9876255989074707, 1.3970807790756226, -1.13773512840271, 1.958597183227539, -5.9235711097717285, -0.9408890604972839, -6.691158294677734, 5.989347457885742, -4.067767143249512, -1.5633103847503662, 1.6161303520202637, -0.8750451803207397, 1.133418083190918, -1.6771306991577148, -0.1451285481452942, 0.4318714141845703, -2.4425790309906006, -4.212713241577148, -3.9155335426330566, -1.0981159210205078, 0.1355391889810562, 2.6215901374816895, 1.2459793090820312, -2.6498920917510986, -1.9023501873016357, 5.859875202178955, -1.4371817111968994, 0.8309821486473083, 3.3507275581359863, -1.782780647277832, -5.620258331298828, 8.43028736114502, 2.342247486114502, -0.9352646470069885, -1.6395008563995361, -0.014615923166275024, -1.2070523500442505, -3.5499792098999023, -3.7748284339904785, -4.896178245544434, -1.42416512966156, 4.295438766479492, -0.05285000801086426, -2.349493980407715, -5.023772239685059, -0.8706262111663818, -1.0519979000091553, -3.947221279144287, 0.18674737215042114, -5.078471660614014, 0.688853919506073, -2.3844070434570312, -0.9193130731582642, 0.41317224502563477, 2.652304172515869, -4.889029502868652, -1.112257957458496, -2.0773704051971436, -1.9325541257858276, 3.3376412391662598, 0.5192519426345825, 1.2896971702575684, -2.691441774368286, 2.9518580436706543, -1.1636695861816406, -3.9113540649414062, -1.0176331996917725, -4.819951057434082, 4.794064044952393, -4.128739833831787, 0.329293429851532, 2.239452838897705, -0.3140857517719269, -4.2054123878479, 3.3257460594177246, -1.36759352684021, -4.074724197387695, 0.8972198367118835, 0.15107955038547516, 1.0723087787628174, -3.0495035648345947, 5.01861572265625, 1.0113790035247803, 3.4183597564697266, 2.4472413063049316, 3.2779417037963867, -1.950414776802063, 1.8648145198822021, -1.8504817485809326, 2.8922884464263916, 1.170286774635315, -4.057515621185303, 2.795527458190918, 1.8561420440673828, 3.2422993183135986, -0.26870429515838623, -2.0197701454162598, -1.8913850784301758, 4.7223968505859375, -2.773207426071167, 0.8901760578155518, 3.1850271224975586, 4.951353549957275, -6.494185447692871, 0.8287132978439331, 3.7216429710388184, 5.6561126708984375, 1.5141956806182861, -1.5509052276611328, -3.7128260135650635, -1.633138656616211, 2.4739606380462646, -3.742887496948242, 1.8856382369995117, -3.251866340637207, 1.1437270641326904, -1.5685887336730957, 4.0670342445373535, -0.5444621443748474, -1.833932876586914, -4.802731990814209, 4.065234184265137, 0.4759114980697632, -1.7170395851135254, 0.02538124844431877, 1.8729321956634521, 1.620229959487915, -0.4214051365852356, -0.4944496154785156, -0.34274595975875854, 2.392578601837158, -0.2691253125667572, -0.15877500176429749, -1.353514313697815, 2.643259048461914, 1.3560678958892822, 0.13662540912628174, 1.949575424194336, 1.2140177488327026, -0.6286112070083618, -2.1558122634887695, -0.6423039436340332, 0.7769814133644104, -3.848850727081299, -4.364289283752441, -0.6709499359130859, -2.6501307487487793, -1.5974125862121582, -3.013273000717163, 4.700254917144775, 6.757437705993652, 4.121639728546143, 4.9765095710754395, 0.04120376706123352, 4.687416076660156, -5.428565979003906, -8.4398193359375, -1.1768466234207153, -2.579343557357788, 2.1456220149993896, 0.016958504915237427, -0.13309569656848907, -1.3984549045562744, -0.724116325378418, -0.21269144117832184, 0.01239732839167118, 4.5995683670043945, 4.5449395179748535, 1.2215510606765747, -1.1847565174102783, -2.331580638885498, -1.6599910259246826, -1.9539847373962402, -4.300814151763916, -2.221346378326416, 2.0828495025634766, -2.1889820098876953, -1.9430222511291504, -1.7135270833969116, -1.519761562347412, 0.9992518424987793, -1.4254502058029175, -0.47841423749923706, 3.7459864616394043, 0.0862359032034874, 4.221166610717773, -2.691070556640625, -4.489090442657471, 0.10575639456510544, 1.1466721296310425, 5.026266098022461, 0.8740665912628174, -2.4205474853515625, 2.7761666774749756, -3.555452585220337, 0.17054197192192078, 0.5933413505554199, -1.1426621675491333, -0.49614137411117554, 5.720656394958496, -2.2299554347991943, 5.57332706451416, 2.73626446723938, 0.7057399153709412, 2.1962080001831055, 5.044799327850342, -4.463599681854248, -0.2611747980117798, -0.9846391677856445, 0.28038403391838074, 1.2318785190582275, -2.4959328174591064, 0.5838445425033569, -1.2977397441864014, -5.238851070404053, -3.879960060119629, -4.078681468963623, -5.643737316131592, 2.0247557163238525, 0.5447885990142822, -2.7843270301818848, 5.520113945007324, -2.553105115890503, -1.178511619567871, -0.9276915192604065, -2.4919490814208984, -2.8046481609344482, 3.4723856449127197, -2.712222099304199, -0.7669752836227417, 0.3889252543449402, 3.000136613845825, -0.6752423644065857, 0.05543679744005203, 1.8460843563079834, 6.363962173461914, -3.5877676010131836, -0.6959196329116821, 2.040910243988037, 0.33958595991134644, -5.558920383453369, 3.3120222091674805, 1.878453016281128, -0.23630809783935547, 3.849776029586792, 4.228365898132324, -1.6537688970565796, 1.0574052333831787, -4.2214579582214355, -2.876347064971924, -4.023997783660889, -1.5874767303466797, -4.714976787567139, 4.990801811218262, -0.03560476005077362, -1.5317163467407227, -1.4319546222686768, -2.712257146835327, -2.8716554641723633, -3.4179162979125977, 3.244615316390991, -1.3170597553253174, 2.025784730911255, 0.5499691963195801, 1.226503610610962, 1.2036243677139282, -4.0711798667907715, 5.495880126953125, 1.4256547689437866, 2.774308443069458, 2.1943793296813965, 6.448323726654053, -0.9783580303192139, -6.823502540588379, -4.078772068023682, -1.8616290092468262, 0.7747757434844971, 2.1225976943969727, 2.9149770736694336, 0.5855494737625122, -4.368590354919434, 1.6156296730041504, -1.5310553312301636, -1.2648084163665771, -4.91517448425293, -2.700239896774292, 1.2661224603652954, -1.4215080738067627, -6.070550918579102, -2.9224812984466553, 3.0132312774658203, -3.9704880714416504, -4.539246082305908, -3.2176430225372314, 3.4937844276428223, -3.2580466270446777, -3.860415458679199, 2.3579039573669434, 2.45589017868042, -2.687375068664551, 1.3910634517669678, -0.8419395089149475, 2.314457893371582, -1.9177218675613403, -1.3335535526275635, -0.23537130653858185, 0.13980847597122192, -0.3317639231681824, 2.5976178646087646, 3.456413745880127, -2.3315322399139404, 3.2248544692993164, 0.7502942085266113, -1.3721296787261963, 1.0229887962341309, -6.169870376586914, 0.29987117648124695, -0.6664308905601501, -0.26223596930503845, 3.080441951751709, -5.521214008331299, 0.06429348886013031, 0.009835740551352501, -1.15912926197052, -3.830690383911133, 0.42632877826690674, 0.3541637063026428, 0.8220815658569336, 5.951983451843262, 0.7893933057785034, -4.373778343200684, -7.100665092468262, -5.9365153312683105, -1.7078014612197876, -2.9519729614257812, -1.393784523010254, 1.712364673614502, -0.2934577465057373, -2.098940372467041, -0.6344588994979858, 1.4893347024917603, 3.814912796020508, 3.195624351501465, -6.877505302429199, 1.0961625576019287, 2.0830039978027344, 3.5550050735473633, 2.859793186187744, 0.6744480729103088, -0.9445722699165344, 1.1850718259811401, 4.1375932693481445, 5.612541198730469, 1.7106273174285889, -0.2810174524784088, 1.7158186435699463, -2.5794997215270996, -0.8644133806228638, -2.4117541313171387, -1.688030481338501, -1.3268589973449707, -0.5970415472984314, 0.6276094317436218, 0.6402987241744995, 0.35068875551223755, -7.053896903991699, 0.09390219300985336, -2.8150506019592285, 4.6180033683776855, 0.9255704879760742, -2.7362184524536133, 1.731271743774414, -4.811116695404053, 1.5935616493225098, -2.255890369415283, 1.3319706916809082, -3.917858362197876, 0.3995331823825836, -1.1613500118255615, 1.0047813653945923, -1.8101701736450195, 2.416398763656616, 3.598426580429077, 1.5306966304779053, 2.5223374366760254, 5.406529903411865, -4.352807998657227, -4.436291694641113, -0.5921935439109802, -1.3890905380249023, 1.2262330055236816, 0.5580441951751709, -2.880608558654785, -3.9040889739990234, 3.287437915802002, 2.7285349369049072, -3.993838310241699, 0.6979501247406006, 4.518685817718506, 1.8249151706695557, 2.146087646484375, 0.7137803435325623, -2.9537734985351562, -0.9288983345031738, -2.183976650238037, 3.004512310028076, -0.5181540250778198, -1.453122615814209, 2.151028871536255, -0.39225882291793823, 0.35526326298713684, -5.636903762817383, 1.6865648031234741, -1.2436813116073608, 0.43626725673675537, 0.8024909496307373, -1.8789420127868652, -2.7831435203552246, -3.309864044189453, 0.6589628458023071, -1.7554867267608643, 1.8179728984832764, -2.5854179859161377, 2.359102725982666, -2.3060545921325684, 1.293197751045227, 4.053774833679199, -4.608743190765381, -7.157960891723633, 3.1452150344848633, 3.361342430114746, -2.552342176437378, -0.2791455388069153, -0.9335330724716187, 5.127100944519043, -0.8832651376724243, 2.40120005607605, -4.792394161224365, 6.050830841064453, -0.9023234844207764, -0.18094328045845032, 2.3141496181488037, 4.554296493530273, 0.9087870121002197, -4.745324611663818, -3.593289852142334, -2.4989588260650635, 2.478759288787842, -3.9525904655456543, 3.5832014083862305, -2.044142484664917, 4.581259727478027, 1.8153877258300781, -1.3303757905960083, 0.29208892583847046, 4.05327033996582, -0.8853909969329834, 6.935938835144043, 6.334918975830078, -2.459364175796509, -6.708183288574219, 5.373534202575684, -2.3430070877075195, -2.438159465789795, -6.006071090698242, 0.42358285188674927, 2.068743944168091, -0.9191977977752686, -3.599269390106201, 2.6413371562957764, -4.383431434631348, -2.0145671367645264, 0.6504977941513062, -2.037705421447754, 3.7715601921081543, 6.693683624267578, -4.3324408531188965, 6.721273422241211, 0.03729655593633652, 1.197249174118042, 6.756868362426758, 0.6960866451263428, 1.5395653247833252, 0.03106960654258728, 1.4668618440628052, -0.4394010901451111, 7.661247253417969, 2.2499959468841553, 0.23588407039642334, 0.15485244989395142, -3.147439479827881, 1.3080554008483887, 2.8262035846710205, 1.2698845863342285, 2.0381035804748535, -6.30438232421875, -3.1783084869384766, 1.7367444038391113, -2.413557291030884, -0.4585244357585907, 1.4932377338409424, -2.6187877655029297, -0.7529115676879883, -5.933002948760986, -5.366351127624512, 0.717186689376831, -4.453237533569336, -1.8742222785949707, -2.443697929382324, -2.039884090423584, 1.4814327955245972, -1.7796211242675781, -1.9439401626586914, 0.5238485336303711, -1.602603793144226, -3.268908977508545, 0.29102152585983276, 1.7695472240447998, 4.284045219421387, 0.3622692823410034, -0.061891112476587296, -2.633505344390869, -2.696817636489868, -2.219639301300049, -5.579297065734863, -4.612334251403809, 2.306103467941284, 3.1177186965942383, -1.6772468090057373, 3.402113199234009, 4.513828754425049, 3.2142319679260254, 2.832289218902588, 1.2067277431488037, -1.6558313369750977, 0.18593357503414154, -0.9212186336517334, 2.0450966358184814, 3.5282888412475586, 0.8187866806983948, -1.7292355298995972, 4.472622871398926, -1.2234914302825928, -1.3501638174057007, 7.25318717956543, -3.715038299560547, -0.23943635821342468, -4.757413864135742, -0.5984180569648743, -3.5121006965637207, 5.260289669036865, 3.084559440612793, -0.4010767936706543, 0.4853852391242981, 1.0489928722381592, 0.5271977186203003, -2.895521640777588, -1.259751558303833, 0.005545398220419884, 0.050890300422906876, 0.5462404489517212, -4.522223472595215, 9.705289840698242, 1.0484744310379028, 2.255553960800171, -3.8899619579315186, -0.9490828514099121, 5.875861644744873, -2.062013626098633, -1.847011685371399, -2.481149435043335, -2.420498847961426, -5.103224277496338, -2.8977880477905273, -3.179067611694336, -4.197734832763672, -0.6195356249809265, -4.293347358703613, -1.1137809753417969, 0.26947399973869324, -1.039851188659668, 0.7901700139045715, 1.6150935888290405], [2.4800572395324707, 7.550140857696533, -0.5872778296470642, -4.243728160858154, 1.0429189205169678, -1.8020257949829102, -4.087725639343262, -0.010429253801703453, -1.025343894958496, 7.838314056396484, -1.0514105558395386, -3.7715930938720703, 2.801023483276367, -1.700685739517212, -1.316484808921814, -2.0646748542785645, 1.5390751361846924, 0.6637450456619263, -0.003383419942110777, -1.775680661201477, -2.0710315704345703, -5.3222479820251465, 1.826746940612793, -5.194084167480469, -6.648603916168213, -3.916107416152954, 6.0333251953125, 0.6336327195167542, 1.6121996641159058, -1.4789806604385376, -2.3019795417785645, -3.3639814853668213, 2.4956958293914795, 0.2043689787387848, 1.1797873973846436, 5.441311359405518, -5.411341667175293, -0.5744746923446655, -2.3578085899353027, 1.8355175256729126, 1.077431321144104, -0.11570166051387787, -2.289651393890381, -3.713296890258789, -2.925605058670044, -2.5682363510131836, -0.3718082308769226, 1.2923855781555176, 5.113873481750488, 6.447965621948242, -1.0613431930541992, 2.243445634841919, 0.7502189874649048, 0.346977174282074, -0.48476359248161316, -2.5055899620056152, -0.8148059844970703, 4.646165370941162, -1.7088944911956787, 1.9138789176940918, -2.3580029010772705, 2.1938257217407227, -1.7765552997589111, -0.6287800073623657, -0.7404971718788147, -6.453799724578857, 1.2960424423217773, -1.8972818851470947, -2.389561176300049, 3.7122111320495605, 1.8891348838806152, -2.466061592102051, 1.2608100175857544, -3.378048896789551, -2.8690764904022217, 3.9886879920959473, -6.461400985717773, 0.8038363456726074, 7.444002628326416, 0.9914054274559021, -1.364713191986084, -1.0127410888671875, 2.371772527694702, -2.374988555908203, 1.1583597660064697, 3.6146600246429443, -0.20198273658752441, -0.30155226588249207, 0.33591538667678833, 0.7175396680831909, 1.6807079315185547, -0.7596114873886108, 4.141720294952393, -2.8307735919952393, 1.2771003246307373, -0.5381009578704834, 5.2400407791137695, -1.4413458108901978, -1.4591286182403564, -1.5252656936645508, -1.6060054302215576, -4.7688069343566895, 0.9680802226066589, 1.1667101383209229, -0.8553155064582825, -1.4136683940887451, -2.193427324295044, -2.118954658508301, -2.4402594566345215, 2.19730544090271, 3.9859814643859863, -0.3608584403991699, -0.05039360001683235, -4.845527648925781, 2.009976387023926, 1.7048916816711426, -6.258059978485107, 0.8268663883209229, -2.045351982116699, -4.37297248840332, 3.155935287475586, 1.7327826023101807, -3.7477216720581055, -3.0512094497680664, -2.4390156269073486, 2.9901084899902344, -3.614689826965332, 1.1130621433258057, -0.16473177075386047, 2.4496920108795166, -1.9726946353912354, -1.569410800933838, -0.2210899293422699, 1.469577431678772, 1.5429785251617432, -6.962813377380371, -2.299032688140869, -1.1758413314819336, 3.207428455352783, 1.6968426704406738, -4.511434555053711, 2.770500898361206, -4.099676609039307, -3.232255697250366, -1.4886571168899536, 4.751767635345459, 1.1815615892410278, -0.21433857083320618, -0.6308630108833313, 0.9204386472702026, 3.881830930709839, 1.2228533029556274, 1.746044397354126, 6.04461669921875, 4.603757858276367, 0.7725024223327637, -4.6586737632751465, -2.72218656539917, -2.5492892265319824, -2.7053260803222656, 0.8161416053771973, 1.4991512298583984, -1.7480854988098145, -0.049957338720560074, 0.4985281825065613, -1.874737024307251, 3.0911028385162354, 2.492462635040283, 3.283144950866699, -0.2967470586299896, -1.183245062828064, 1.245532512664795, -3.735940456390381, 3.1111700534820557, 4.029764175415039, 1.3781594038009644, -3.74006986618042, 0.5967994928359985, 0.21482089161872864, 1.122248888015747, 2.529198169708252, 1.9573626518249512, -4.523999214172363, 3.3041176795959473, -2.0865087509155273, 5.485341548919678, -2.58734393119812, -1.91550612449646, 2.6101231575012207, -5.092209815979004, -2.8530826568603516, -0.1201046034693718, -2.8039920330047607, -1.4820010662078857, -4.5175371170043945, 1.5273996591567993, -4.5327253341674805, -0.5756809711456299, 1.3993239402770996, -0.6337014436721802, -0.7785013914108276, 0.114937424659729, 3.371273994445801, -1.5263924598693848, -1.0502771139144897, -2.3336472511291504, 2.9517297744750977, 0.89256352186203, -0.14057520031929016, 0.11503367125988007, 2.1066572666168213, 4.022849082946777, 2.8167624473571777, 4.118131637573242, 0.6815415620803833, -0.02071094885468483, 3.2154171466827393, 4.542040824890137, 5.842193126678467, 0.28490281105041504, 0.08496541529893875, -0.4295499324798584, 0.9000829458236694, -4.937889099121094, -2.367699146270752, 1.1544746160507202, 4.166437149047852, -1.0013813972473145, -2.999561309814453, 1.073857307434082, 4.80092716217041, -2.2426512241363525, -1.4732025861740112, 2.0219788551330566, -3.9813132286071777, 2.920069932937622, -6.236330986022949, 0.9914911985397339, -3.3043665885925293, -3.9234070777893066, -1.656274437904358, 0.6907885074615479, -2.0419857501983643, 1.1648427248001099, -0.07278721034526825, 0.9757019877433777, 4.177582740783691, 2.3409571647644043, 5.106642723083496, 3.1860055923461914, -3.9429168701171875, 0.974250078201294, 2.741727590560913, 2.0645625591278076, 3.796506881713867, 1.2164266109466553, 0.6411594748497009, 6.1142578125, -4.631808280944824, 1.0099494457244873, 6.474607944488525, -1.0220414400100708, -1.6498603820800781, 6.4369964599609375, 0.18883153796195984, -3.9853763580322266, 8.075668334960938, -1.7780272960662842, -0.9123982191085815, 2.6552417278289795, -0.9637212753295898, 1.091522216796875, -0.8668673038482666, 3.2968101501464844, -0.6177359819412231, 3.35086727142334, 0.5317694544792175, -2.412973165512085, -6.683429718017578, -3.92566180229187, 0.24345745146274567, 4.514048099517822, 1.676714301109314, -8.159741401672363, -2.3533291816711426, 5.579671382904053, -0.37399303913116455, 3.6083383560180664, -3.9948887825012207, -3.458284378051758, -0.6125223636627197, 2.555734157562256, 1.0393954515457153, -1.2609306573867798, -2.089510440826416, 1.887214183807373, 3.353069543838501, -3.691277503967285, -3.4651200771331787, 2.614932060241699, 1.164952039718628, 1.4196425676345825, -0.013001987710595131, -2.9418907165527344, 0.8668904900550842, -2.8026154041290283, -8.062807083129883, -4.267406463623047, -0.5119284391403198, -2.1214938163757324, 1.1138346195220947, -1.9181874990463257, 8.437955856323242, -0.20217415690422058, -0.838657021522522, -3.9427781105041504, 5.753896236419678, -2.693514823913574, 0.22022853791713715, -4.281515598297119, -2.0845959186553955, -0.4076108932495117, 0.8711808919906616, -0.09338726103305817, 4.646474838256836, 6.745420932769775, -6.395388603210449, -3.0910353660583496, 1.531470775604248, 1.138731598854065, 4.992300987243652, -3.688546657562256, -1.4901214838027954, 1.7913748025894165, -0.28133195638656616, -0.7640249133110046, -0.21847695112228394, 0.4098297953605652, 0.8124582767486572, 0.9092667102813721, -0.6257842183113098, -2.952090263366699, -0.26252901554107666, 0.2171803116798401, 0.7276685237884521, -2.1915440559387207, 2.9193789958953857, -0.8647464513778687, -0.21674036979675293, -2.752108573913574, 0.9291460514068604, -2.2027697563171387, -1.5208594799041748, -0.8914989233016968, 2.41774582862854, -0.7529734969139099, -5.824106693267822, -0.6088805198669434, -0.9954503774642944, 1.4126627445220947, -2.2428150177001953, -0.29283618927001953, 1.8458064794540405, -2.845958709716797, -1.5691416263580322, 3.0148637294769287, 2.084153413772583, -1.324577808380127, 2.1360883712768555, -0.761360228061676, -3.2786898612976074, 1.8576090335845947, -4.4540300369262695, 7.896459579467773, 4.616706848144531, -2.0769217014312744, 4.036481857299805, 0.05337224155664444, -1.8438661098480225, 2.065164089202881, -5.09589147567749, -2.908698797225952, -1.0137324333190918, -4.602480411529541, -6.245302200317383, 2.312474489212036, 4.573111534118652, -1.02263605594635, -0.6595236659049988, -1.4914556741714478, 4.342144966125488, 4.057199001312256, -4.404831886291504, 1.0446909666061401, -2.5063209533691406, -0.2540077567100525, -0.8167632818222046, -3.2524638175964355, 3.6147658824920654, 1.1023499965667725, -5.949624061584473, -4.689516544342041, 1.0492066144943237, -1.4252889156341553, 1.013458251953125, 0.5389853119850159, 0.37263956665992737, 2.2103121280670166, 1.1159751415252686, 1.4772329330444336, -2.142080783843994, 1.4436695575714111, 3.6839537620544434, -1.5682880878448486, 0.3423558473587036, 2.025280475616455, -1.2256138324737549, -2.126835346221924, 3.783700704574585, 1.700307846069336, 1.2295982837677002, 1.8757867813110352, 0.3789435625076294, -0.24836301803588867, -0.7244706153869629, 3.1337532997131348, -5.891794204711914, 2.0414469242095947, 3.1805152893066406, -0.7068068981170654, -0.9774536490440369, -2.2838807106018066, 2.594287872314453, 4.607574939727783, -0.4926973581314087, -2.509404182434082, 0.9983963370323181, 0.753762423992157, -2.940650224685669, -1.2213647365570068, 2.8365163803100586, -1.9863522052764893, -4.288562774658203, 4.227784156799316, -3.147078514099121, -1.3488539457321167, -7.599056243896484, 1.3990273475646973, -3.153219223022461, 0.48650312423706055, -1.7827835083007812, -0.8083798885345459, 4.647512912750244, 5.822626113891602, -4.430114269256592, -2.5965607166290283, 3.108934164047241, -2.233025312423706, -1.5425097942352295, -0.9109088182449341, -0.3376641273498535, 0.3387373983860016, 1.5297391414642334, 0.15776173770427704, -1.6811738014221191, 4.397173881530762, 3.6662485599517822, -0.06768184900283813, 0.3554946184158325, 2.3137311935424805, -0.8001525402069092, 1.5997700691223145, -3.4696404933929443, 3.6206607818603516, 1.5870662927627563, 2.599524974822998, 2.041788101196289, 5.861083507537842, -1.181860327720642, -2.4530375003814697, -0.34548985958099365, -2.9585747718811035, 1.951364517211914, 2.236703634262085, -1.3284637928009033, 6.9119873046875, 4.866201400756836, -5.053312301635742, -1.965029239654541, -1.4694080352783203, -2.1108741760253906, 0.11913466453552246, 3.892740249633789, 0.7210029363632202, -0.8299573659896851, 0.9119499921798706, -0.932976245880127, 2.299271583557129, -5.7902631759643555, 2.347297191619873, -1.589330792427063, -2.4992642402648926, 3.9222028255462646, 3.607243537902832, -0.19871766865253448, 0.39878180623054504, -1.1581488847732544, 0.4256337583065033, -1.9067692756652832, -1.5768349170684814, 3.697540283203125, 3.7426648139953613, -3.947072982788086, 0.32296761870384216, 2.7336721420288086, 5.060410499572754, 0.654641330242157, -2.4084439277648926, -2.1302123069763184, -1.0930691957473755, -3.217397689819336, 0.48021721839904785, 0.42979079484939575, 2.1207425594329834, 1.3640047311782837, -1.7554450035095215, -0.44998839497566223, 0.38649582862854004, -0.1869833767414093, -5.172702789306641, 2.0579490661621094, -6.850644111633301, -2.8158140182495117, 2.0352063179016113, 3.6642556190490723, 0.8455519676208496, -2.8611154556274414, -0.8865640163421631, 0.004505830816924572, -1.7220702171325684, 4.908493995666504, 0.7811206579208374, -0.030334090813994408, 4.737137794494629, -4.2957353591918945, -2.4221651554107666, -2.8298733234405518, -1.5415663719177246, 0.8811994791030884, 2.1916675567626953, 5.955549716949463, 2.7582290172576904, 1.1719388961791992, -5.892718315124512, 1.115854024887085, -6.420768737792969, -0.4646521806716919, -2.36861252784729, 1.345237374305725, 1.9404313564300537, -0.6973999738693237, 1.4137390851974487, 1.2773569822311401, -0.08751619607210159, -3.070629358291626, -1.052352786064148, -2.896770477294922, -1.7253212928771973, -2.434774160385132, -2.547668933868408, -0.48992669582366943, 1.389449954032898, -5.4391584396362305, -3.1782689094543457, 0.19884449243545532, 2.4143364429473877, 2.3171188831329346, -0.26635247468948364, 0.9789875745773315, 1.9919975996017456, 0.8425533771514893, -1.5423681735992432, 0.0942051038146019, -1.718018889427185, 2.2624921798706055, 2.2409861087799072, -0.30088093876838684, -2.40590238571167, 2.536412239074707, 0.5211130380630493, -3.480905532836914, 1.982149600982666, 7.438616752624512, 2.595853328704834, 5.38444185256958, 1.3164377212524414, 3.791698694229126, -4.801523685455322, 3.594302177429199, 3.0266733169555664, 0.23102498054504395, -1.810297966003418, 1.2057716846466064, 0.8397771120071411, -3.1392524242401123, 2.1774075031280518, 3.1259186267852783, -4.019200801849365, 0.39723676443099976, 0.989412784576416, 3.7565269470214844, -2.5184402465820312, -1.9785336256027222, 5.414034843444824, 1.1060044765472412, 1.2260971069335938, 0.9570235013961792, 1.0094980001449585, -0.1681147664785385, -3.2860379219055176, -2.2924466133117676, -0.03254088759422302, -3.864179849624634, 1.2852318286895752, -1.7329120635986328, 0.6290537118911743, 3.020376205444336, 2.814629077911377, -6.613536834716797, -0.7789860963821411, 0.17422184348106384, 1.0594472885131836, 0.5815771818161011, 0.38882362842559814, -4.033876895904541, 0.6102224588394165, 0.3235483765602112, 0.4810090661048889, -2.5307493209838867, 4.299223899841309, -2.669362783432007, 1.5572437047958374, -5.315304279327393, -0.7917487621307373, -0.9690389633178711, -0.5409128069877625, 0.7600457668304443, 2.448566198348999, 1.4393305778503418, 3.2589616775512695, 2.6634109020233154, -0.45391765236854553, -2.55031418800354, 1.138878345489502, -5.620169162750244, 2.227908134460449, 4.37451171875, 4.1710405349731445, 0.12153051048517227, -2.765448570251465, -1.3907184600830078, 1.0507673025131226, -4.275971412658691, 1.497019648551941, -2.205732822418213, 0.9908462762832642, 0.5295296907424927, 1.60772705078125, -1.246328353881836, -7.102993965148926, -0.5356885194778442, -2.420182228088379, -0.7839133143424988, -2.258167028427124, 4.586294651031494, 6.443364143371582, -0.076659195125103, 2.6283135414123535, -4.610354423522949, 1.1565910577774048, -1.7162127494812012, -1.2553601264953613, -1.5336709022521973, -1.3788880109786987, 2.101125955581665, -0.45656806230545044, -2.815018653869629, 1.8526692390441895, 4.965935707092285, -0.5058197379112244, 1.7097651958465576, -3.146747589111328, -1.0839024782180786, 1.6015446186065674, 2.4498281478881836, 1.8533539772033691, 2.1630122661590576, 8.854833602905273, 0.37142059206962585, -5.845757484436035, -0.3061853349208832, 1.2708392143249512, 0.4348991811275482, 3.803410053253174, -2.775174617767334, -5.279748439788818, -1.7264940738677979, -0.1433011144399643, 12.17760944366455, -5.629159450531006, 5.2431135177612305, 3.1685843467712402, -0.41905587911605835, 5.50616979598999, 3.669281244277954, -0.06423543393611908, 0.6904754042625427, -3.9787216186523438, 0.7681394219398499, -1.1259726285934448, -1.2540279626846313, -1.780165195465088, 7.552639007568359, 1.7804300785064697, 1.9749078750610352, 8.202150344848633, -2.1678366661071777, 2.2291455268859863, -0.8974812626838684, -3.015944480895996, 1.0192683935165405, 3.893557548522949, -2.7119100093841553, 0.06711915135383606, 1.1296359300613403, 1.5544421672821045, -0.5569172501564026, -3.9591546058654785, -0.9561086297035217, 2.7226004600524902, -2.6958796977996826, 5.550825119018555, -3.7424476146698, 2.816890001296997, -0.03764771297574043, 1.571803331375122, 2.787431240081787, -0.46309351921081543, 2.745119571685791, 4.435502052307129, 1.9395537376403809, -2.020380973815918, 4.9411773681640625, 6.648155212402344, 2.7352747917175293, -3.6942620277404785, 4.972011089324951, -3.444387912750244, -1.8926842212677002, -3.7487540245056152, -0.28652870655059814, 1.1228322982788086, 0.8322525024414062, 1.7071166038513184, 0.19028712809085846, 0.7848202586174011, 1.6790876388549805, 0.3866800367832184, -1.4485876560211182, 0.4669148325920105, 0.03629828244447708, -1.29971182346344, -2.7171084880828857, 4.69044828414917, 3.529588222503662, -1.9077340364456177, 1.5303608179092407, 2.8005425930023193, 4.901954174041748, 3.0446910858154297, -1.5996625423431396, -1.1247856616973877, -7.10529899597168, -4.668837547302246, 3.056659698486328, 2.8663432598114014, 5.583652973175049, 0.9803558588027954, -1.5956151485443115, -2.834510564804077, 0.5532756447792053, -0.24477118253707886, 2.849776029586792, -0.23579642176628113, -0.24164722859859467, -1.8057963848114014, -2.5883772373199463, 2.6486752033233643, -1.5522081851959229, 3.5665476322174072, -0.7634759545326233, -3.538219451904297, 4.495894908905029, 1.0627944469451904, -1.8084073066711426, 4.459278106689453, -1.3457062244415283, -2.806079626083374, 2.865638256072998, -5.406744956970215, 3.666347026824951, -0.3737662434577942, -3.1913886070251465, 4.030398368835449, -5.321502685546875, -2.199842929840088, -0.9292402267456055, 1.6374306678771973, -1.4630382061004639, 4.791691303253174, 0.22356583178043365, 3.582465171813965, 0.33687180280685425, 1.1256318092346191, 0.6224750876426697, -2.8709969520568848, -7.219852924346924, 5.5272111892700195, -1.3097753524780273, 2.603567123413086, -3.4292211532592773, -3.796187400817871, 2.032227039337158, 3.418797492980957, -1.1904807090759277, -4.37425422668457, -0.5599819421768188, 1.332257628440857, -4.959932804107666, -4.443890571594238, 0.8654913902282715, 2.6797738075256348, -1.7362048625946045, 8.37791919708252, 2.3894197940826416, -0.7826197147369385, 2.3326566219329834, 2.4595344066619873, -1.6980741024017334, 2.7074131965637207, 1.179375410079956, -2.3433570861816406, 0.6546704173088074, 2.290117025375366, 1.2887473106384277, 6.134977340698242, 5.122814655303955, 1.2347891330718994, -5.665004253387451, -5.118565559387207, -2.470186233520508, -5.186861991882324, -1.0609813928604126, 4.860464572906494, 3.279892921447754, 6.97920036315918, 1.3451478481292725, 1.6318771839141846, -0.25951650738716125, 5.937190532684326, -0.41657984256744385, -3.6317832469940186, -3.231614589691162, -0.1332653909921646, -2.909327507019043, -1.0960023403167725, -1.2227528095245361, 3.2229843139648438, -0.4344745874404907, -2.6850082874298096, -1.364099144935608, 4.2491374015808105, -2.7508316040039062, 5.579163551330566, -1.3159228563308716, -2.9374804496765137, -4.65843391418457, -0.14200012385845184, -0.73045414686203, -4.694860458374023, 1.1152093410491943, 3.0108284950256348, -7.649057388305664, -4.562578201293945, -0.13256052136421204, -2.238330125808716, -1.329217791557312, -0.6965583562850952, 1.4112985134124756, -0.9477640986442566, -1.8761855363845825, 3.161200523376465, -1.5537245273590088, 4.6986403465271, -5.23515510559082, 0.9344183206558228, 3.354604482650757, 0.4349442422389984, 2.4191267490386963, -5.3610382080078125, 0.9545866250991821, 2.748196601867676, 1.4756022691726685, -2.5571720600128174, -2.3114616870880127, 0.19183172285556793, 0.9648549556732178, -1.0931153297424316, 1.8868227005004883, -1.6124839782714844, -0.9043292999267578, -0.04614943265914917, 1.5308303833007812, -0.7910860776901245, 3.608471632003784, -4.040061950683594, 0.6146142482757568, -1.7231229543685913, 2.8860678672790527, 2.923799514770508, -2.562903881072998, 0.5114399194717407, 0.6234771609306335, -5.420092582702637, 2.0394954681396484, -3.5964913368225098, -4.55117130279541, 0.5270451307296753, -5.244082450866699, -1.3231048583984375, -0.5217182040214539, -6.0305657386779785, 1.1518999338150024, 1.3157638311386108, 4.24355411529541, -0.07071660459041595, 0.17084737122058868, 5.984767913818359, 2.8205814361572266, 0.15600015223026276, 3.228233814239502, 2.2501299381256104, 4.816864490509033, 1.4231574535369873, -0.6246359348297119, 1.6544597148895264, -0.3288421630859375, 0.3663802146911621, -0.5847919583320618, -2.579446792602539, 0.4128320813179016, -0.14758345484733582, 1.5014526844024658, 0.0161330234259367, 2.2135701179504395, -1.1646441221237183, 3.6436643600463867, -2.3384885787963867, 4.597639083862305, -3.4297056198120117, -1.5608484745025635, 4.580118179321289, 2.2841875553131104, 1.9960136413574219, -0.15582703053951263, 2.1187479496002197, 6.343633651733398, -2.8198354244232178, -3.681835412979126, -1.507117748260498, 2.223435878753662, -0.7754682302474976, 0.5087144374847412, -1.5189781188964844, -0.3061295747756958, 0.8763898015022278, -3.5087180137634277, -0.5011647939682007, -1.9449734687805176, -1.4020456075668335, -1.758496642112732, 4.136679172515869, -2.4165828227996826, 6.73103141784668, -3.191824436187744, 4.785999298095703, 4.688296318054199, 5.964364051818848, 1.9215762615203857, -2.0703539848327637, 6.084626197814941, 1.2227941751480103, 4.185371398925781, 3.8499598503112793, 1.5766915082931519, -0.25381070375442505, 5.1821699142456055, -0.8769620656967163, 3.2089858055114746, 0.5260114073753357, 4.501891136169434, -7.162258148193359, 0.8803601861000061, 6.161990165710449, 0.7529017925262451, 4.530379772186279, -4.042726516723633, 4.638586044311523, -4.010133743286133, -1.1361668109893799, 5.960622310638428, -1.3169543743133545, 2.1656835079193115, 0.6898089647293091, -0.8158133029937744, 3.284245014190674, -2.9034924507141113, 5.467350959777832, 0.2892107665538788, 2.87335205078125, -4.639965534210205, -4.036229610443115, 0.1933336704969406, 2.6096837520599365, -4.493338584899902, 4.649484157562256, -2.5800557136535645, 3.080411195755005, 7.066888809204102, -3.969330310821533, 3.2907261848449707, -0.5432475805282593, -0.057259250432252884, -3.102125644683838, -1.8493685722351074, -1.1987884044647217, -2.503697633743286, -3.041595697402954, -0.4785047173500061, 4.870309829711914, 0.24881726503372192, -2.5177669525146484, 1.2239431142807007, -0.5958631634712219, -1.079793930053711, 2.480966329574585, 0.586929440498352, 3.319363594055176, 0.14629417657852173, 0.8446661233901978, -1.0388383865356445, 1.5961092710494995, 5.376682758331299, -4.654545783996582, -0.3349694013595581, 0.5165234804153442, 1.577064871788025, -1.1091365814208984, 3.045287609100342, 3.6114957332611084, 0.23838263750076294, -2.895718574523926, -3.9091548919677734, -3.065511703491211, -1.2811667919158936, -3.7297182083129883, -2.799335479736328, -2.1062979698181152, -1.0545367002487183, -5.377085208892822, 1.727509617805481, -4.787055492401123, -3.3749990463256836, 1.792832612991333, 4.461302280426025, 4.898718357086182, 1.5448167324066162, 3.485755443572998, 2.030977487564087, -3.107919692993164, 2.7597641944885254, -0.07695913314819336, -1.8662853240966797, 1.967890977859497, -1.9527859687805176, -4.38953971862793, 2.9280214309692383, 1.4097285270690918, -0.9217042922973633, 1.2201385498046875, 5.1408796310424805, -2.4223930835723877, -1.8167767524719238, 6.624896049499512, -6.685873985290527, -0.5201352834701538, -3.3593978881835938, -0.1540399044752121, -0.527614414691925, -1.4022679328918457, 1.557502031326294, 3.1652920246124268, -0.1435984969139099, -1.783737301826477, 0.7370582818984985, -2.038177013397217, -6.898376941680908, -2.244145154953003, -0.42603737115859985, 2.5242044925689697, -0.3445412814617157, -2.5273492336273193, -0.3847625255584717, 4.324429988861084, -2.8947463035583496, -0.23783400654792786, 0.39068925380706787, 0.0828864797949791, 3.629551649093628, -2.106295585632324, 0.48641619086265564, 2.8192830085754395, -1.548462152481079, -3.309079170227051, -5.388167381286621, 1.0958024263381958, -3.7192835807800293, 1.6241989135742188, -2.3943135738372803, 2.6459743976593018, 0.3817814588546753, 2.11892032623291, -0.40705496072769165, -3.424032688140869, 2.548774242401123, 2.2238924503326416, -4.989893913269043, -2.4099996089935303, 1.3759245872497559, 6.645442962646484, 1.0230597257614136, -2.7709054946899414, -2.106555461883545, -1.170427680015564, 5.9641618728637695, 0.44707050919532776, 3.478562831878662, 2.760206699371338, -0.024905137717723846, -4.785902976989746, 3.592658758163452, -4.993819713592529, -3.6289308071136475, -3.2092442512512207, 3.405625820159912, -2.595614433288574, -2.9064602851867676, 1.6694608926773071, -1.8013126850128174, 3.7557506561279297, 1.520907998085022, 4.161930561065674, -2.880213737487793, 0.7586708664894104, 0.9217287302017212, -0.11208418011665344, -9.43548583984375, -2.9948935508728027, 0.5233426690101624, -0.28594771027565, -2.103485107421875, -0.6395420432090759, 1.1373143196105957, -2.4083638191223145, -4.803967475891113, 1.3328979015350342, 2.4511497020721436, 0.0973094031214714, 1.1904569864273071, -1.0542329549789429, 0.7574408054351807, -0.26705437898635864, -3.7377052307128906, -1.5319147109985352, -3.903564453125, 4.004576683044434, -3.706251382827759, 0.5638577342033386, 3.2714362144470215, 3.387856960296631, -2.2825677394866943, 0.4694288372993469, -2.835026264190674, -2.340750217437744, -0.3762146830558777, -0.9805139303207397, -0.733386754989624, -1.7980077266693115, -1.884460210800171, 2.109565019607544, 1.9259772300720215, -5.109854698181152, -2.9563169479370117, -0.6495225429534912, 3.603959560394287, 1.355114221572876, 1.8295751810073853, 1.473549246788025, 0.8075357675552368, 1.0622484683990479, -0.7043789625167847, 0.1861911118030548, -0.9770374298095703, 3.9257004261016846, -3.3704018592834473, 1.025821566581726, -4.30460262298584, -2.4966554641723633, -0.8104358911514282, -5.186094284057617, -8.325912475585938, 3.6768293380737305, -3.1691129207611084, -3.041971206665039, -0.8690392971038818, -4.547025680541992, 2.970460891723633, 4.0540876388549805, -5.305416584014893, -3.258521318435669, 3.907259464263916, -1.3717418909072876, 2.612839460372925, -2.6429128646850586, 1.2037701606750488, -1.792013168334961, -1.7617682218551636, -3.1695070266723633, -5.271356105804443, 1.2165617942810059, 1.3660190105438232, 0.9968111515045166, 0.553655743598938, 1.285595178604126, 2.3227930068969727, 6.428236961364746, -0.6806789636611938, 0.5286771059036255, -2.691509485244751, 2.75237774848938, 1.026430368423462, -3.816455364227295, -2.400787830352783, -0.2118465006351471, -0.8267213106155396, -0.19695895910263062, -0.616012692451477, 2.439621925354004, -0.12983734905719757, -5.713489055633545, -0.612029492855072, -4.001437664031982, 0.9000133275985718, -0.07043471932411194, -4.23764705657959, -0.8309131264686584, -2.559664726257324, -4.652505874633789, -0.7448786497116089, -1.5864754915237427, -3.267130136489868, -2.0579488277435303, -4.293086051940918, -1.1525877714157104, -3.701932668685913, -3.4072160720825195, -1.7575868368148804, -3.430420160293579, -0.898955225944519, 5.243713855743408, -3.5196313858032227, -5.727597713470459, 1.7484521865844727, 5.211582183837891, 3.470852851867676, -0.780487060546875, -1.0309350490570068, 2.1733551025390625, 4.8332648277282715, 1.4628881216049194, -0.5411853194236755, 3.6794729232788086, -5.46923303604126, -1.658402919769287, -0.6316608786582947, -3.3963398933410645, -6.331619739532471, -0.5095562934875488, -2.3392715454101562, 0.344666987657547, 4.989380359649658, -0.09887862950563431, -4.743292331695557, 1.601212501525879, -0.0589514821767807, -0.976873517036438, 2.9457285404205322, -4.566714286804199, 0.42084038257598877, 3.3255958557128906, 1.9586223363876343, 1.8773893117904663, 6.830078125, 2.362490177154541, -1.9747202396392822, -0.42791110277175903, 1.2673171758651733, 3.203444480895996, 5.16562032699585, 0.48535698652267456, 1.3814886808395386, 3.5786075592041016, 3.3868422508239746, 2.9521985054016113, -1.1469358205795288, 3.920781135559082, -3.0911529064178467, 6.669655799865723, -0.6285855174064636, 1.4758057594299316, -2.2982873916625977, 0.5128565430641174, 0.6221878528594971, -0.743847668170929, -1.1955585479736328, -4.824712753295898, 5.327054023742676, -1.4721415042877197, -1.2657018899917603, 1.295050024986267, 2.7226336002349854, 0.1785009503364563, -0.14189675450325012, -2.5009398460388184, -3.654475688934326, -2.121516466140747, -0.37017741799354553, -3.198653221130371, -0.19582021236419678, -1.8385210037231445, -2.740079879760742, 3.501864194869995, 3.1423258781433105, 1.74993896484375, -1.7148551940917969, 1.1818368434906006, 3.495415687561035, 3.064483880996704, -5.28049373626709, 0.3677819073200226, -0.1317766010761261, -1.1381961107254028, -2.8849353790283203, -3.500095844268799, -2.459167242050171, -5.261552333831787, 1.1108198165893555, 1.8122609853744507, -3.518126964569092, -0.12264157831668854, 7.524785041809082, -0.5032176375389099, 5.487339019775391, 0.10680605471134186, -0.2947164475917816, -4.383800029754639, -5.877610206604004, -0.2752227783203125, -0.5664165616035461, 0.21025098860263824, 7.345418930053711, 1.3295304775238037, -2.45114803314209, -1.001460075378418, 0.5263379216194153, 2.734544515609741, -2.4788544178009033, -4.377656936645508, 0.08666054904460907, -3.4593124389648438, 3.305734634399414, -3.447789192199707, -5.732945442199707, -0.7700523138046265, -3.917945384979248, -0.6089161038398743, -0.9695207476615906, 0.3715364933013916, 1.5829439163208008, -1.9456429481506348, 2.546889305114746, 3.0186188220977783, -0.8552994728088379, -7.511013507843018, -4.390707969665527, -1.5361268520355225, -3.9640049934387207, 1.1737271547317505, 3.386638641357422, 1.0328996181488037, 2.1762893199920654, -1.967844843864441, -4.552521705627441, -1.2471128702163696, 1.707610845565796, -5.129275321960449, 1.715322732925415, -2.134000301361084, 1.1586616039276123, 0.7437431812286377, -1.6524364948272705, -1.5345895290374756, 0.11432470381259918, -0.7639685869216919, -3.3996362686157227, -2.8253581523895264, 3.692162036895752, -5.413033485412598, -3.7829697132110596, 0.4342806339263916, -2.133556842803955, 1.1145793199539185, 0.8727756142616272, -2.5640501976013184, -2.77182674407959, -0.39270132780075073, 1.7381186485290527, -1.405837893486023, 2.1671011447906494, -1.1184802055358887, -3.799940586090088, 3.7678308486938477, 0.1731421947479248, 1.5285239219665527, -4.268760681152344, -2.773681640625, 1.9358820915222168, 3.0572454929351807, -0.04870298132300377, -3.6014533042907715, 0.3897731602191925, -1.8106114864349365, 1.24056077003479, -0.9462910890579224, 4.067000389099121, -0.4264085590839386, 3.787297010421753, 0.5987290143966675, 0.5581246614456177, 0.9922559261322021, 2.3570964336395264, -2.6389265060424805, -4.880559921264648, -1.4499515295028687, 1.102914571762085, -1.0319929122924805, -1.4974349737167358, -0.2074010968208313, -1.9607553482055664, -0.6423163414001465, 1.0966682434082031, 0.31519144773483276, -3.290102958679199, -0.477531373500824, 4.289941787719727, 0.07540957629680634, -1.2449809312820435, 0.5466139316558838, 1.1612541675567627, 0.7825310230255127, 0.08721143007278442, 0.7317696809768677, -0.7278247475624084, -8.155353546142578, 0.7109043002128601, -1.270881175994873, 3.793586254119873, -1.44260573387146, 0.7568873763084412, 2.7586748600006104, -2.804542303085327, -2.1395184993743896, 1.3304219245910645, -0.9544813632965088, -0.2232455611228943, 5.902223587036133, 8.105498313903809, -6.935007095336914, -3.5067005157470703, 0.9697028398513794, -0.40690678358078003, -4.281964302062988, -1.0354690551757812, -2.7774791717529297, -0.5640861988067627, 1.0810120105743408, 2.382377862930298, -3.4498400688171387, -2.187899112701416, -1.4601367712020874, 6.516120910644531, 1.3762893676757812, 2.846510648727417, 4.156826019287109, -0.4742509722709656, -3.727389335632324, 0.1397015005350113, 3.610147476196289, 1.4829400777816772, -6.09423828125, -0.7510069012641907, 5.402024745941162, -2.33404278755188, -0.18130478262901306, 1.7540054321289062, -8.28026008605957, 3.072474956512451, -0.050639472901821136, 0.7579910755157471, -0.6031650900840759, 1.1649749279022217, 1.4677423238754272, 0.4945772886276245, -2.6981544494628906, 0.42186474800109863, -1.113931655883789, 1.3149244785308838, -4.3836541175842285, -1.4800939559936523, 2.0865931510925293, -2.7291388511657715, 2.678537607192993, -2.0175070762634277, -4.51533317565918, -2.738818407058716, -2.3009915351867676, 0.3610331416130066, -0.9640496969223022, -3.8181660175323486, 1.5519206523895264, 0.39487096667289734, -4.349752426147461, 1.696251392364502, 0.4804069697856903, -0.44798552989959717, -2.1201391220092773, -3.5158095359802246, 6.921351432800293, -1.6573090553283691, -0.4973446726799011, 2.512117624282837, -0.861211359500885, 1.985618233680725, -3.9061925411224365, -3.226315975189209, 0.997791588306427, -4.0574421882629395, -2.0710692405700684, -4.46793270111084, -4.022948265075684, 0.08156018704175949, 5.346686840057373, 0.5461632013320923, -4.576261043548584, 1.3256263732910156, -0.8039751052856445, 1.7070927619934082, -2.275885581970215, -5.695005893707275, 4.752966403961182, -1.6937990188598633, 1.607547640800476, -0.7178669571876526, -1.672869086265564, -0.8618038892745972, 1.5258994102478027, 1.1510138511657715, 0.47107750177383423, -0.2252109944820404, 2.1891517639160156, -7.622655868530273, -1.4375340938568115, -0.9331163763999939, -2.3679659366607666, -5.0594587326049805, -1.0436241626739502, -1.4954943656921387, 1.837151288986206, -0.6024302244186401, 1.2340441942214966, 3.872868537902832, -3.0760421752929688, -3.724149703979492, 1.3280534744262695, -1.9935625791549683, 0.7488530278205872, 1.444244623184204, 1.2807178497314453, -0.4147983491420746, -4.292881011962891, 2.102116346359253, 0.7846536040306091, -0.7623579502105713, -0.3256317377090454, 2.1792802810668945, -0.2039918750524521, 1.056781530380249, 7.082278251647949, -5.072795867919922, -1.0418832302093506, -2.5809969902038574, -3.1239891052246094, 0.27505290508270264, 3.5901198387145996, -0.5266963243484497, 2.250509738922119, -0.24454857409000397, -1.0509446859359741, -0.639802098274231, 5.930069923400879, -0.9819623231887817, -4.2534403800964355, -1.5332927703857422, 0.9208296537399292, 2.513955593109131, -2.100377321243286, 0.6045882701873779, -3.976396322250366, -1.9207274913787842, 2.6477527618408203, 5.497749328613281, -2.370821475982666, 1.36214017868042, -6.025938034057617, -2.0467333793640137, -4.630425930023193, -2.4575889110565186, 1.8451337814331055, -4.868360996246338, 3.478423595428467, 2.32772159576416, 4.840411186218262, 1.180140733718872, -1.7369037866592407, 0.4158637523651123, -2.090158462524414, 0.49330562353134155, 3.7236862182617188, 0.6076145172119141, 2.3733291625976562, 3.927281141281128, -0.9159942865371704, 4.212605953216553, 3.5411081314086914, -0.1799490600824356, 1.598020076751709, 1.5641067028045654, -0.5277005434036255, 0.2748472988605499, -3.50470232963562, -2.3628196716308594, 0.19529619812965393, -3.7509102821350098, 0.423090398311615, 3.5823779106140137, -3.8507962226867676, 2.9116666316986084, -5.024150371551514, 1.4825787544250488, -3.6645498275756836, 0.6147127747535706, -0.5570687651634216, 3.082953453063965, 2.2806613445281982, 1.0794436931610107, -2.7296111583709717, -3.9307217597961426, 1.8930072784423828, 1.240317702293396, 2.976475238800049, 3.7484731674194336, -2.628920078277588, 4.136951446533203, 4.9641618728637695, -0.5297303199768066, -0.03770485520362854, 3.980672836303711, -1.9114990234375, -3.314518451690674, 0.38590434193611145, -0.5313278436660767, -2.6016294956207275, -1.220605731010437, -2.9660449028015137, -0.8262280225753784, 7.054723739624023, -0.06416374444961548, 3.2724709510803223, 0.5989632606506348, 1.198940396308899, 5.704446792602539, -2.4686501026153564, -5.178817272186279, -1.3228294849395752, -6.449850082397461, -0.6762903928756714, -1.125231385231018, -4.500885486602783, 1.8874543905258179, 3.239445447921753, -0.20972940325737, 1.762709379196167, 3.4944210052490234, 5.854134559631348, 1.6144741773605347, -6.670527458190918, 4.888454437255859, -3.886552333831787, -5.4635162353515625, -1.0972660779953003, 3.9528913497924805, 1.547560453414917, -4.944891452789307, -2.6583309173583984, 3.9607362747192383, 3.024061918258667, 1.384636640548706, -2.406392812728882, -3.511809825897217, 5.121020793914795, 0.5611929893493652, 3.060138702392578, -0.29766106605529785, -0.4806305766105652, -5.855081558227539, 0.9065113067626953, 2.61755633354187, -0.5177686214447021, -2.1121160984039307, -1.5214152336120605, 0.39080220460891724, 0.28567272424697876, -1.3672176599502563, -1.9208016395568848, -0.7718511819839478, 1.934288740158081, -0.20236802101135254, -2.481344699859619, 1.7572227716445923, -2.136289596557617, 2.0472514629364014, 0.16737014055252075, 1.8363850116729736, -2.77764892578125, 2.5249698162078857, 3.1505684852600098, 0.6416155099868774, -1.4642832279205322, -5.92279577255249, -0.9375248551368713, -2.0128917694091797, 2.260087490081787, 0.7649773359298706, -1.0116779804229736, -0.18990033864974976, -1.2456952333450317, 2.245666980743408, -0.8103396892547607, -4.42927360534668, 0.6013883948326111, -2.698031425476074, -1.3532569408416748, 7.3104472160339355, -0.4785955548286438, 1.6812384128570557, -0.47871875762939453, 2.0140702724456787, -1.2536561489105225, 4.886783123016357, 1.9848673343658447, 4.526956558227539, -0.42288824915885925, 1.6623234748840332, 0.4445769488811493, -4.681197643280029, 2.679710865020752, -2.3552350997924805, -0.008313019759953022, -0.08058103919029236, 0.4695902466773987, 0.7487436532974243, 2.5080032348632812, 3.3744277954101562, 0.13463354110717773, 0.05061003193259239, -2.1223859786987305, -1.3552088737487793, 2.5924410820007324, 2.2475671768188477, -4.07037353515625, -0.6055809259414673, -0.5261372327804565, 3.2145988941192627, -2.2172281742095947, -2.413764476776123, 2.2794413566589355, -0.5503592491149902, -2.3107857704162598, -0.9796520471572876, 2.624074935913086, -1.2546738386154175, -3.815370798110962, 0.24403108656406403, -5.071785926818848, 1.8397367000579834, -0.9517031908035278, -2.9950613975524902, -2.0915346145629883, -0.1242416501045227, 4.0685906410217285, -4.761417865753174, 2.522583484649658, 2.041865110397339, -1.1808955669403076, 4.149511337280273, 3.9345154762268066, -0.09527519345283508, -0.23063549399375916, 1.674678087234497, 0.7046580910682678, 4.983105659484863, -2.626452922821045, 0.6806533932685852, 3.463581085205078, -2.6710305213928223, 1.7460527420043945, -2.1428873538970947, 1.5176650285720825, 1.7726759910583496, -0.7539312839508057, 3.650148868560791, 1.2658500671386719, 3.005833148956299, 0.8382171392440796, -2.949915647506714, -1.7404699325561523, 1.8616483211517334, 0.11141044646501541, -0.17813287675380707, -0.3433142304420471, 7.563172340393066, -5.3564453125, 0.5434852838516235, -7.009047508239746, -5.527388572692871, 0.5279994010925293, 0.42325669527053833, -0.08350668847560883, 0.20894721150398254, 0.26290860772132874, -1.523625373840332, 2.1870265007019043, -5.4616289138793945, 3.5496017932891846, 1.9953606128692627, -2.5768918991088867, -1.0103821754455566, 4.055508136749268, -4.113406181335449, -4.189717769622803, 1.030366063117981, 1.1930088996887207, -1.0951368808746338, 2.747110605239868, 0.6751365661621094, -3.4646637439727783, 3.027446746826172, 2.119481086730957, -6.349735736846924, 0.2592509984970093, 2.263176679611206, -6.89265251159668, -4.709574222564697, 2.109846353530884, -3.0730953216552734, -2.509126663208008, -0.6823397874832153, -1.7358158826828003, -1.2668827772140503, 1.786848545074463, 2.132418155670166, 4.391229152679443, -0.1903364062309265, 4.713568210601807, -7.853748321533203, -2.1879916191101074, 0.30043870210647583, -3.7192649841308594, -2.3255577087402344, -3.082446813583374, 2.1357693672180176, 1.1501970291137695, -0.9306827783584595, -3.0971875190734863, -2.8627490997314453, -4.047816753387451, 4.093907356262207, -1.5074447393417358, 0.9202008247375488, 2.754774332046509, -2.190108299255371, -4.494535446166992, -0.7904646992683411, -2.3724536895751953, -3.94893479347229, 6.695095062255859, -3.135335683822632, 2.825876474380493, 1.7426402568817139, 4.961591720581055, -1.0066289901733398, -2.526071786880493, 1.767376184463501, -2.7426719665527344, -0.011512876488268375, 1.164230227470398, -2.015622615814209, 1.7274134159088135, -3.987307548522949, -2.431950569152832, 7.530251502990723, 3.0719454288482666, -5.1747589111328125, -2.1723546981811523, -4.629092693328857, -6.233692646026611, 1.288628101348877, 1.2029119729995728, -4.928726673126221, -7.740784645080566, -12.60879135131836, 0.7405891418457031, -0.5396001935005188, 6.5817155838012695, -5.584031105041504, -0.24828684329986572, -1.7259628772735596, -1.4028699398040771, -0.1361992508172989, -1.929302453994751, -0.7401492595672607, -1.1879440546035767, -0.7343059182167053, -0.5360898971557617, 0.842665433883667, -2.818887710571289, -0.6625425815582275, -5.606226921081543, -2.077784776687622, -2.70438551902771, -0.6732131242752075, -0.7253950834274292, 1.8573907613754272, 2.944251298904419, -3.9862523078918457, -0.5300657749176025, 3.4646286964416504, 1.760448932647705, -4.998524188995361, 0.9764851927757263, 0.6202571392059326, 0.9883716702461243, -4.662949562072754, -6.424590110778809, 4.352237224578857, 0.7529944181442261, 6.154656410217285, -1.517221450805664, -2.03478741645813, 1.1118385791778564, -2.016077995300293, -2.473951578140259, 0.808658242225647, 0.3347458243370056, 0.6281600594520569, -0.17886117100715637, 1.1747076511383057, 5.356657981872559, 1.9687812328338623, -1.817093849182129, -4.109111785888672, -3.5200109481811523, 2.6316463947296143, 0.2424110323190689, -0.112464040517807, 0.15906131267547607, 2.612985134124756, -0.988781213760376, -0.6212248206138611, 2.924253463745117, -0.35181546211242676, 0.3529580533504486, -3.0173885822296143, -0.24253633618354797, 1.9060261249542236, -1.8739659786224365, -0.044564686715602875, -2.704580545425415, -2.07743239402771, -0.5206058025360107, 1.4145054817199707, 0.44846615195274353, 2.975374698638916, 2.081817865371704, 4.845046043395996, -0.8023133277893066, -1.9740933179855347, -4.936906337738037, -0.8872173428535461, -6.281369209289551, 4.749677658081055, -1.406369924545288, -3.7498531341552734, 3.1327600479125977, 3.2159972190856934, 0.330602765083313, 2.590200901031494, 5.7969207763671875, 1.0484672784805298, -0.9214320182800293, -4.042583465576172, 3.544048309326172, -3.506613254547119, 0.6395862102508545, 0.31283795833587646, 3.6545138359069824, 1.805016279220581, -1.4574412107467651, 0.973355233669281, -1.177775502204895, -0.5333475470542908, 4.100848197937012, -4.871306896209717, -2.65692400932312, -0.6448193788528442, 1.750604510307312, -2.2726292610168457, 3.770443916320801, 2.0533580780029297, -1.1969828605651855, -4.027700424194336, 2.153244972229004, -5.483888149261475, 0.20790942013263702, 0.2688840329647064, 0.6748536825180054, -4.237884044647217, -6.027439117431641, 3.577216386795044, 3.4217958450317383, 0.8432735204696655, -1.6289573907852173, -0.32492464780807495, -1.9172919988632202, 2.451176404953003, 2.6993961334228516, -1.2786507606506348, -0.14166449010372162, -0.0917874202132225, 0.505443274974823, -3.062020778656006, 2.1959826946258545, 1.9491517543792725, 3.747196674346924, 1.4732073545455933, 5.020083427429199, -1.4325909614562988, -1.475715160369873, 0.40417128801345825, 4.445288181304932, 4.481821537017822, 2.1120445728302, 3.609705924987793, 1.9441511631011963, 3.407515048980713, -2.19416880607605, -3.446727752685547, 1.1058571338653564, 1.802194595336914, -0.49802160263061523, -0.6183598637580872, 1.4436728954315186, -2.5322859287261963, -6.14727783203125, 0.168537899851799, -1.1281132698059082, 0.3540124297142029, 0.8216598629951477, 5.448634147644043, 1.8842737674713135, -2.317870616912842, -0.3416855037212372, 2.9335241317749023, -1.6741645336151123, -0.07926318794488907, -0.29843592643737793, -2.974719762802124, 1.679255723953247, 5.680917739868164, 2.4408764839172363, -2.346803665161133, -1.191277265548706, 0.47434768080711365, 1.1172846555709839, -0.15395422279834747, -1.1964868307113647, -3.683976173400879, 2.777207374572754, -1.6656215190887451, 3.7353310585021973, -0.4882534146308899, 1.0981754064559937, 2.006953477859497, 1.5355286598205566, 3.724625587463379, 0.9127489328384399, 0.8682987689971924, 4.642948150634766, 3.725071430206299, -2.2601332664489746, -5.854071140289307, 1.3037357330322266, 0.07972069084644318, 1.2045239210128784, -0.21837478876113892, -0.8789387941360474, -6.80318546295166, -3.0370869636535645, -2.6089820861816406, 3.5953798294067383, 0.3122462034225464, 3.32338285446167, 4.111629009246826, -3.737595558166504, -3.2173428535461426, -1.5111110210418701, 5.4299139976501465, 4.110931396484375, -1.5886255502700806, 0.02725861966609955, 2.2172443866729736, 4.311281681060791, 2.3332066535949707, -2.8482394218444824, 0.9076976180076599, 7.566466331481934, 4.1186203956604, -1.410893201828003, -0.650457501411438, -5.615725517272949, -0.5472308397293091, 5.643806457519531, -0.48615512251853943, 2.1614890098571777, 3.573206901550293, 5.148004055023193, 2.6092262268066406, 1.0741851329803467, -2.3253862857818604, -0.64107745885849, -1.317968726158142, -4.054462909698486, -3.0810749530792236, -2.5834805965423584, -1.8028576374053955, -1.4577733278274536, 2.183077096939087, -0.9215894937515259, -4.872589588165283, -0.023881100118160248, 6.07801628112793, 1.864558219909668, -0.552765965461731, 1.1315330266952515, -0.7065819501876831, -0.5481935739517212, -3.8992648124694824, 1.9243533611297607, -1.7651209831237793, -0.10258905589580536, 1.3777005672454834, 0.30698713660240173, -0.9027427434921265, 2.606673240661621, 0.9375066757202148, 1.8590614795684814, 3.354297161102295, 1.6801373958587646, -1.1131176948547363, -0.142293781042099, 1.672658085823059, 1.1369507312774658, -4.156707286834717, 1.7891552448272705, -0.44289296865463257, -2.7121973037719727, -4.0024213790893555, 6.154711723327637, -5.340089797973633, 4.075283050537109, 4.884807109832764, 1.8427280187606812, -1.4639344215393066, -3.4610610008239746, 3.419940948486328, 1.9332358837127686, 3.000074863433838, -0.9891411066055298, -3.089024066925049, 1.369646668434143, -0.5958181619644165, 2.149430274963379, -0.6006530523300171, -3.1283559799194336, -4.195623397827148, 0.044430963695049286, 0.5859959721565247, -2.42792010307312, 1.18337881565094, 1.1802074909210205, -1.0783194303512573, -2.310736656188965, -4.306358337402344, 3.3763368129730225, -4.088292598724365, 1.0009313821792603, 3.689545154571533, 4.208708763122559, 0.9500480890274048, 1.238204002380371, -4.763165473937988, 1.9404709339141846, -1.7429640293121338, -5.362587928771973, -1.5743558406829834, -1.8034098148345947, -1.2281854152679443, 0.18561001121997833, 3.0151333808898926, -3.3554015159606934, -3.5602149963378906, 1.2789372205734253, -0.4981105327606201, 6.720715522766113, -6.054525375366211, -3.9750494956970215, -2.422048330307007, 1.5229614973068237, -2.6489105224609375, 5.2288079261779785, 2.6850247383117676, -1.088419795036316, 3.140657901763916, -1.5477385520935059, 3.4759819507598877, 0.22763368487358093, 3.545454502105713, 6.2533392906188965, 1.7139326333999634, -3.5158603191375732, -3.9598910808563232, -3.115291118621826, -8.48231315612793, 4.3058929443359375, 1.3332884311676025, -0.29296165704727173, -0.8470464944839478, -3.334317684173584, -0.5747388601303101, 6.134281158447266, -0.56813645362854, 0.5021325945854187, 1.289158582687378, 5.035099029541016, -2.038541078567505, 2.4821815490722656, 1.7402567863464355, -5.564855098724365, -1.2525852918624878, 3.9908599853515625, -0.32767826318740845, 1.3277130126953125, 3.4749069213867188, -0.6316686868667603, -0.7010279297828674, 1.3799632787704468, 2.328460693359375, -0.3056687116622925, -3.553581714630127, 0.7337833642959595, -1.0704625844955444, -4.464505195617676, 0.6055185794830322, -2.1265709400177, -2.1715760231018066, 3.4231345653533936, -2.524402141571045, 4.108431816101074, -3.31118106842041, 4.88299560546875, -4.4080119132995605, -0.4136274456977844, 2.7447144985198975, 1.630311369895935, 4.895085334777832, -2.4796135425567627, -3.0078048706054688, 3.300248146057129, 2.4271883964538574, 4.342101097106934, -0.24734556674957275, -4.379369735717773, -1.6489208936691284, -3.0665173530578613, -1.2953441143035889, 0.8405662775039673, 2.3062326908111572, 1.348321557044983, -2.803394079208374, -2.3708105087280273, 0.2384684979915619, -2.7508082389831543, -4.495308876037598, 2.134671211242676, 0.584200918674469, -3.869581460952759, 1.8239550590515137, 1.119977355003357, -1.14268958568573, 1.9729878902435303, 1.149348497390747, 5.790277004241943, -0.20921126008033752, -1.9742655754089355, -4.8186516761779785, -2.745030403137207, -1.7628087997436523, -0.8527227640151978, -2.0850250720977783, 6.028322696685791, 6.55567741394043, -1.1398396492004395, 4.4830641746521, 2.892430305480957, -2.5053372383117676, 1.035971999168396, -2.6016499996185303, 5.061814308166504, -1.4230730533599854, -4.193760871887207, 2.7047841548919678, -0.2756410837173462, -5.598371505737305, -2.1779441833496094, -2.6359024047851562, 3.0776209831237793, -4.984914779663086, -2.6212072372436523, -3.0662102699279785, -1.539395809173584, 7.572877883911133, 0.4047573208808899, 4.430030345916748, 10.767816543579102, 0.16174891591072083, 1.9444832801818848, -3.063415288925171, -3.9779844284057617, 0.6887384057044983, 2.4222335815429688, -5.618210315704346, 2.391120433807373, -0.7971230149269104, 0.6809965372085571, 4.791505813598633, 3.940119981765747, -1.6930756568908691, 0.38485896587371826, 1.332186222076416, 4.4346923828125, -2.8971219062805176, 3.2908923625946045, -2.789797782897949, 1.5122698545455933, 3.7567763328552246, 0.24663925170898438, 0.2380906045436859, 2.7985775470733643, -2.095390558242798, -7.60609245300293, 1.9003477096557617, 5.263182640075684, 2.2089836597442627, -0.2274513691663742, 4.1502461433410645, -2.751347541809082, -2.086987257003784, -4.079102039337158, 1.017906665802002, -0.6952948570251465, 1.0327880382537842, -1.781441330909729, -4.013845443725586, 3.480877161026001, -6.977016448974609, -0.15448133647441864, 2.4471819400787354, -1.0722765922546387, 2.6014068126678467, -5.578123092651367, 4.954252243041992, -1.2622313499450684, 1.1597528457641602, -1.4158200025558472, 2.5179741382598877, 5.806979179382324, -2.7912001609802246, 2.6116597652435303, -0.61342453956604, -1.47977876663208, 0.7844316959381104, 0.08148278295993805, 1.9813737869262695, -1.8105981349945068, 3.7041077613830566, -4.626786708831787, 2.310551881790161, -0.9949687719345093, -5.090176105499268, -3.6558046340942383, -0.8615972995758057, -4.411912441253662, 2.8669180870056152, 3.2188143730163574, 3.403273105621338, 2.4273698329925537, 5.58009147644043, 0.8893425464630127, 0.1651938408613205, 5.696798324584961, -1.4603487253189087, -2.987757921218872, -3.8101468086242676, 0.8039093017578125, 2.0146286487579346, 0.0775141716003418, 3.882291316986084, 0.5140118598937988, -1.8309663534164429, 3.297194004058838, -0.26603689789772034, 1.855374813079834, -0.8461111783981323, 2.0930182933807373, 5.12884521484375, -1.8184444904327393, 2.907533884048462, 2.92559814453125, -0.058525197207927704, -2.4953575134277344, 0.9656679630279541, 0.9429724216461182, 3.9525198936462402, 3.8792574405670166, -0.08869504183530807, -2.559264898300171, 1.8224749565124512, -4.995530128479004, 2.270672082901001, -0.37700942158699036, -2.963460683822632, 5.142580986022949, 7.124115943908691, -2.59224796295166, -7.373407363891602, 3.0195136070251465, -6.175319671630859, 3.322530746459961, -3.2732810974121094, 0.6210252046585083, -1.7064052820205688, -1.9742658138275146, -0.9431156516075134, -0.10955765843391418, 4.4977006912231445, 2.572997570037842, -1.236908197402954, -1.3360682725906372, 3.343327045440674, 1.5584940910339355, 3.3368968963623047, 0.3235603868961334, -0.4579644203186035, 4.545487880706787, 0.5787466168403625, 0.46536779403686523, -0.561344563961029, 0.07254531234502792, -0.9582804441452026, -0.4800095558166504, -3.1195285320281982, 1.8746132850646973, -1.2147228717803955, -3.042393922805786, -0.66014164686203, 1.1154873371124268, -3.249812602996826, -1.2840852737426758, 0.23964393138885498, -0.40768963098526, 0.7034426331520081, -7.470233917236328, -0.15359030663967133, 1.1887747049331665, -0.9991630911827087, 2.8709537982940674, -2.1767542362213135, -3.8187568187713623, 2.0097849369049072, -0.9755327701568604, -0.5531436800956726, 5.452733039855957, -0.424959659576416, -4.939441680908203, -1.5453591346740723, -1.3640786409378052, 2.0835213661193848, 1.1286448240280151, 2.626605987548828, -5.340321063995361, -1.7518303394317627, -2.646827220916748, 0.5066249966621399, 1.6210631132125854, -2.477177858352661, 5.725167274475098, -1.9159445762634277, 3.0231776237487793, 2.315101385116577, 3.7771434783935547, -0.9018858075141907, -2.2803125381469727, 4.082893371582031, 2.7096478939056396, 0.08199060708284378, 0.18246933817863464, -4.368780136108398, -3.1300387382507324, 2.8720641136169434, -4.446290969848633, -5.038863182067871, -1.053244948387146, -0.2914741337299347, 2.4642493724823, -3.868272304534912, -1.2699143886566162, 3.0744314193725586, -0.7111393213272095, -1.5384531021118164, 1.2240676879882812, -2.27517032623291, -3.311422348022461, -1.4256832599639893, 5.04785680770874, -0.3467029631137848, 0.24258235096931458, -0.9950677156448364, 0.565557062625885, 0.3791274428367615, 2.1039328575134277, 3.7874069213867188, -0.7089975476264954, -2.397196054458618, 5.218143463134766, -0.3856491446495056, 2.882995128631592, -2.681419849395752, 0.7641962766647339, -1.3956270217895508, 2.1612532138824463, -3.466866970062256, -4.847498893737793, -1.5313396453857422, 2.497124195098877, -0.45940524339675903, 2.4122989177703857, 1.0475423336029053, -2.493640184402466, -4.511385440826416, -0.2609861195087433, -3.283818483352661, 3.7351956367492676, -0.22468301653862, 1.5635135173797607, -7.238490104675293, -2.5607693195343018, 0.3238798975944519, -3.2606801986694336, 0.17055128514766693, -0.25071796774864197, -1.895309329032898, 0.09303663671016693, -0.41576987504959106, 7.481525421142578, 6.977784156799316, -2.8735032081604004, 3.4019594192504883, -3.305743932723999, 1.1907262802124023, -1.0468541383743286, 4.13427209854126, 2.1410512924194336, 1.5689260959625244, -1.2342115640640259, 1.8454573154449463, -0.28867679834365845, 0.4922226369380951, 2.7840065956115723, 1.4665913581848145, -2.6155340671539307, 2.4288058280944824, 1.3675748109817505, 2.9224178791046143, -0.7026797533035278, -0.38244593143463135, -3.855780601501465, 1.4943418502807617, -0.9580798745155334, 4.529121398925781, 0.6853080987930298, -0.8741352558135986, -1.2003461122512817, -1.7574255466461182, -0.8526869416236877, -2.3195910453796387, -2.2073793411254883, 3.8844032287597656, -3.5410995483398438, -1.418870449066162, 0.5615659952163696, 0.12628766894340515, -1.9523996114730835, -2.049503803253174, -4.06752872467041, 2.387254238128662, -0.49525532126426697, -5.161048889160156, -0.9168646335601807, 5.659593105316162, -3.9801454544067383, -0.7091473340988159, -0.32373180985450745, -6.423520088195801, 0.8011782169342041, -2.256995439529419, 2.6672537326812744, 1.6051472425460815, -6.216487884521484, 2.5632174015045166, 1.714755892753601, -0.2376774251461029, 2.552186965942383, -5.909488201141357, -0.36613383889198303, -2.68087100982666, 2.3949451446533203, 1.0948905944824219, -2.073504686355591, 0.9790834188461304, 4.208993434906006, 4.004305839538574, 3.540039539337158, 1.2907943725585938, 1.3996906280517578, 2.698312759399414, -1.1052602529525757, -3.2001290321350098, 1.8913073539733887, 5.891512393951416, -2.5147032737731934, 1.0491554737091064, 2.5538747310638428, 1.517266035079956, 1.460160493850708, 0.07072526216506958, -1.1199957132339478, -3.8154337406158447, 0.5703880786895752, -0.29101890325546265, -1.5438226461410522, -0.7851934432983398, 3.0781664848327637, -6.012296676635742, -1.8244857788085938, -1.510244369506836, 0.5441144108772278, -2.32186222076416, -3.1394386291503906, -0.2236195206642151, 2.468292474746704, -1.4228971004486084, 4.390027046203613, 1.0515031814575195, -0.2881809175014496, 0.9945511221885681, -2.0197856426239014, 0.9350723028182983, 3.9618337154388428, -0.7666448950767517, -4.950533390045166, 4.065641403198242, -0.9492365121841431, 1.685130000114441, 1.361580491065979, 2.681572914123535, -2.444883346557617, -0.16183750331401825, 3.2621288299560547, 0.37488147616386414, -3.662320375442505, 3.8675427436828613, -1.890324592590332, 3.9023919105529785, 9.232399940490723, 0.8443468809127808, -2.3660802841186523, -0.7155640125274658, -3.501814126968384, -4.170897483825684, 1.1522700786590576, 1.522280216217041, 1.211540937423706, -1.5379180908203125, 3.166623115539551, -2.017422676086426, -4.96829080581665, 2.899306297302246, 4.5374860763549805, 4.695274353027344, -3.465008497238159, -3.052534580230713, 6.776782989501953, -2.9283089637756348, -1.7775882482528687, -1.1097009181976318, -4.757166862487793, 1.6290466785430908, -1.0841025114059448, 3.691128730773926, 2.368705987930298, -3.0240676403045654, -0.5644457936286926, 0.9990295171737671, 3.216367721557617, 2.775871515274048, -1.1685391664505005, -1.144467830657959, 3.181351661682129, -0.9992750883102417, 0.3371654152870178, -2.500450611114502, 0.03584784269332886, -2.4005351066589355, -0.03443847969174385, 1.052490234375, -6.0859808921813965, 1.1269384622573853, -2.821071147918701, 2.4815127849578857, -2.6212379932403564, -1.729690432548523, -5.136703014373779, 3.199321746826172, -1.7773663997650146, -0.938495397567749, 2.2923460006713867, 1.8963305950164795, 1.6255643367767334, -1.2439855337142944, -4.107425689697266, 3.567458152770996, 0.5228669047355652, -1.0294078588485718, 0.3223417103290558, 3.2458629608154297, -0.8751180171966553, 0.6122876405715942, -2.6324760913848877, 1.090294599533081, 0.9808624982833862, 0.031953103840351105, -6.579347610473633, 2.077014923095703, -2.12271785736084, 2.0397415161132812, 0.9269494414329529, 4.367034912109375, 1.0875688791275024, 0.769965648651123, -4.812375068664551, -6.980810165405273, -3.946711778640747, 0.20718792080879211, -3.9074835777282715, 2.451531171798706, 1.5018525123596191, -0.8187527656555176, 2.695706844329834, -0.25049087405204773, 2.5639991760253906, 0.22012820839881897, -0.754888653755188, 3.4679789543151855, -5.093050479888916, 2.5589020252227783, 3.0864644050598145, -1.090582013130188, -1.89267897605896, 6.121033668518066, -8.850071907043457, 2.623325824737549, -6.336361885070801, -5.652487754821777, 2.066892147064209, 4.442532539367676, 2.2347075939178467, 0.1554482877254486, 1.2099413871765137, -3.0496666431427, -1.1793084144592285, 1.373453140258789, 3.465756416320801, 4.9603142738342285, 3.093653440475464, -1.947224736213684, -1.4797917604446411, 0.8503293991088867, -0.2636040449142456, 1.5014244318008423, 2.2516424655914307, 4.027163505554199, 4.4068756103515625, -4.351144313812256, 2.5311291217803955, 2.254307508468628, -1.4887921810150146, 0.5930609703063965, 0.5280981063842773, -1.670730471611023, 0.6395317316055298, 2.0034427642822266, 0.3108641505241394, 1.0680203437805176, 1.6351430416107178, 3.6284780502319336, -3.0724191665649414, 2.57586669921875, -4.784038543701172, -3.147650718688965, -1.0669636726379395, 2.055985450744629, 2.577462673187256, -4.421175956726074, -3.1976816654205322, 0.742672860622406, 1.9586262702941895, -2.956754446029663, 2.4262661933898926, -1.4740028381347656, 5.029775619506836, 1.5385212898254395, -1.3274716138839722, 9.535024642944336, -2.916623592376709, -4.784928798675537, -2.500624179840088, -7.606184959411621, -1.1841199398040771, 2.828376054763794, 1.432709813117981, -1.7119828462600708, 2.037515640258789, -0.42219090461730957, -1.5100572109222412, -2.0648393630981445, 2.593129873275757, -0.7600901126861572, -3.4698407649993896, 0.7814571857452393, 0.6991722583770752, 3.999380588531494, -1.9446847438812256, -1.9311432838439941, -5.981187343597412, -2.190406084060669, -1.0903840065002441, -5.053106784820557, 2.092792272567749, -1.2952380180358887, -0.5868679285049438, -1.8653401136398315, 1.5976985692977905, 4.601848125457764, 0.9387640357017517, 0.4728463590145111, 5.4160003662109375, -0.0006992555572651327, 7.053322792053223, -5.512475967407227, 4.58607816696167, 4.820782661437988, -3.6119728088378906, -2.4160685539245605, -1.6136603355407715, 0.5721176266670227, 7.595837593078613, 2.4796431064605713, -1.362932801246643, 1.8900928497314453, 0.5892834067344666, 3.1388120651245117, -1.9222772121429443, -1.9505778551101685, -2.258709192276001, 1.1362539529800415, 1.3333954811096191, -5.196889877319336, 2.54669189453125, -2.518172264099121, 2.119718551635742, 1.043151617050171, -0.713598906993866, 2.0167112350463867, 3.4366073608398438, -2.821934223175049, -0.4270166754722595, -0.24063415825366974, 0.24410270154476166, 2.0849835872650146, -6.248125076293945, -2.052868604660034, 0.8489971160888672, 3.035135507583618, -1.2368839979171753, 1.6372014284133911, -3.4972734451293945, 1.4144489765167236, 0.8341360092163086, -2.2418155670166016, 0.50022292137146, -2.363668441772461, -2.195082187652588, 0.08751076459884644, -1.1792707443237305, 0.07094654440879822, -3.459660530090332, -4.062914848327637, 3.0535154342651367, -3.6012587547302246, -3.510201930999756, 1.4624816179275513, -3.006626844406128, 0.5117355585098267, 0.9208056926727295, -7.15002965927124, -3.356657028198242, -3.320324420928955, 0.7869454622268677, -3.1915602684020996, -4.656733512878418, 5.507894039154053, -1.9409735202789307, 2.5043468475341797, -1.705351710319519, -1.5960464477539062, -0.6347013711929321, 1.9702188968658447, 0.5338069200515747, 5.033375263214111, -4.5914998054504395, 7.894602298736572, 4.346658229827881, 5.123200416564941, 2.300532579421997, -1.1221741437911987, -0.5785340070724487, -1.1471083164215088, 1.6904199123382568, -2.73421311378479, -0.4121973216533661, -0.7188959121704102, -1.326148271560669, 1.3148795366287231, 3.7754409313201904, 3.852006196975708, 1.2152918577194214, -2.6529059410095215, 3.0319862365722656, -1.6346290111541748, -2.3093814849853516, -0.6379910707473755, 1.1078628301620483, -0.7754541635513306, 2.673553705215454, -0.0012314033228904009, -4.837027549743652, 3.5057265758514404, 5.056018829345703, -2.3210926055908203, 0.584485650062561, 1.958808183670044, 0.28044843673706055, 3.133471965789795, -0.290507435798645, 3.113248825073242, -2.5292370319366455, 2.146707057952881, -0.3526146113872528, 3.397662878036499, 2.174830675125122, 3.285085916519165, -5.6895751953125, -2.721790075302124, -5.328540802001953, -1.1169441938400269, -2.676764965057373, -4.604324817657471, -2.8644895553588867, 2.7338409423828125, -2.803192615509033, -0.12581011652946472, -0.5412231683731079, -0.3529757261276245, 2.0106828212738037, 2.9265589714050293, -1.5238962173461914, 5.18440055847168, 1.7005481719970703, 5.0192108154296875, -2.8279409408569336, -4.3607635498046875, 2.850167751312256, 1.080784797668457, -0.20293596386909485, -1.1729649305343628, -1.1175466775894165, -0.057656481862068176, -1.0605024099349976, -2.3157434463500977, -5.3378119468688965, 0.718537449836731, 1.7730040550231934, 0.5036492347717285, -4.242222785949707, -0.10758164525032043, 0.7144269943237305, -3.3043112754821777, -1.4674957990646362, 1.7948527336120605, 5.872500896453857, -3.8148880004882812, 2.385310649871826, -2.713609457015991, -3.3555359840393066, 1.731805443763733, -7.181209564208984, 0.5328198671340942, -0.8090019226074219, -5.46282434463501, 0.11180432140827179, -0.5694630146026611, -3.569125175476074, -0.6969878673553467, 1.3857821226119995, -1.2221908569335938, 0.6080182790756226, -5.2350239753723145, 2.3021624088287354, -3.7189297676086426, -1.00282883644104, 1.341238021850586, -0.18115727603435516, -4.9547810554504395, 0.0591629296541214, 3.538588047027588, 1.0937960147857666, -2.2046751976013184, -0.25747835636138916, 3.4181149005889893, -3.498422384262085, -2.062258243560791, -3.324540615081787, -0.48523491621017456, 1.1116238832473755, -1.9314926862716675, -0.4210103154182434, -2.670919418334961, 2.7792115211486816, -1.496100902557373, 2.271721601486206, 3.6316916942596436, -0.1027444526553154, -5.5940656661987305, -2.606806755065918, -3.3309617042541504, -1.0734617710113525, -1.6297998428344727, -0.6740676760673523, 1.5182788372039795, 2.18021559715271, 0.014435159973800182, -3.5294151306152344, 1.0327051877975464, 4.93245267868042, 1.3521449565887451, 0.8695966005325317, -0.8012268543243408, -3.8946356773376465, -0.9556827545166016, -2.743569850921631, 4.378294467926025, -0.03955596685409546, -0.09457114338874817, -1.5527641773223877, 0.09092564880847931, 0.532889723777771, -0.41756051778793335, 1.5037164688110352, -2.072535991668701, -0.45766425132751465, 1.1118372678756714, 5.044238090515137, 4.22101354598999, 1.6178388595581055, -0.8535473346710205, -3.22855806350708, -1.850893497467041, -0.8590551614761353, -2.460930347442627, 0.8106746673583984, -7.619712829589844, 3.4010002613067627, 0.05991481989622116, -2.0535898208618164, 0.07123914361000061, 5.588373184204102, -1.873490333557129, -0.058175116777420044, 1.1699848175048828, -1.3463304042816162, 0.6977490186691284, -4.1308465003967285, -5.632152557373047, 4.856701850891113, -4.136807441711426, 5.757081508636475, -4.9099321365356445, 1.2408519983291626, -2.7066099643707275, 1.5436029434204102, -0.791569709777832, 2.3352174758911133, -4.1284685134887695, 0.9708182215690613, 3.727248430252075, 4.5494160652160645, 3.944777488708496, 0.07014355063438416, 0.12835091352462769, -3.715494155883789, -3.858595371246338, -0.37689435482025146, 7.741343021392822, -3.1247363090515137, -0.06874311715364456, 0.7544772624969482, -3.9942703247070312, 1.0675991773605347, 3.1925690174102783, 3.8635518550872803, 4.09116268157959, -0.4443913400173187, 2.3105859756469727, -6.20550537109375, -1.5337316989898682, -1.040418028831482, -0.20591890811920166, 2.20651912689209, -2.437824010848999, 2.274097442626953, -2.7338943481445312, -3.167318344116211, 3.269728183746338, 3.903940200805664, 0.7571216225624084, -0.3852056860923767, -0.835309624671936, -0.6162366271018982, 0.9454999566078186, -9.835792541503906, -7.966170310974121, 1.314249038696289, -0.4469491243362427, -0.47070902585983276, -3.4757633209228516, -0.10413879156112671, 2.9193673133850098, -4.095499038696289, 3.0138580799102783, -2.2343552112579346, -4.214987277984619, 2.90582013130188, -3.93792986869812, -2.1676790714263916, 0.2233259677886963, -6.000894546508789, 5.595553874969482, -7.14578914642334, 1.4646767377853394, 0.8947677612304688, 1.4290118217468262, -2.245548963546753, 3.136229991912842, 1.7220184803009033, -0.0032270927913486958, -0.6095796823501587, 5.008024215698242, -1.712322473526001, -2.715946674346924, -4.368024826049805, -5.870820045471191, 5.125904083251953, -0.6517277956008911, -2.8571765422821045, -1.5300722122192383, 1.0278434753417969, 2.8464932441711426, -3.093879461288452, -2.410763740539551, -6.998774528503418, 5.089273452758789, 1.60984468460083, -1.594897985458374, 0.7681905031204224, 3.5103700160980225, 6.244463920593262, 4.702888011932373, -1.5514404773712158, -0.7721309661865234, 3.404782295227051, 4.030692100524902, 5.058267593383789, -1.1152793169021606, 0.06227254122495651, -1.1522316932678223, 1.7166376113891602, -0.5631523132324219, -4.728752613067627, -0.5088302493095398, -0.7210341691970825, -1.2266377210617065, -1.9946823120117188, 2.5454816818237305, -1.2736339569091797, -5.899570941925049, -1.2964481115341187, -3.9980483055114746, 1.9849278926849365, 0.4280698299407959, -2.2733774185180664, 4.471543312072754, 4.071989059448242, -0.31590282917022705, 3.107370615005493, -1.389798879623413, 2.8052475452423096, 0.5250846743583679, -1.8375396728515625, -2.9075350761413574, -2.0303845405578613, -0.07212815433740616, 0.39472222328186035, 3.151289463043213, -1.4218616485595703, 1.7238603830337524, -2.3496718406677246, -2.7064950466156006, 0.9884823560714722, 0.03707791492342949, -1.041703224182129, 0.46777600049972534, -1.840053677558899, 2.0846340656280518, -0.9853523969650269, 0.40392857789993286, -5.334724426269531, 4.588668346405029, -2.7600111961364746, 1.5367045402526855, -1.9497196674346924, 2.9799959659576416, -1.2566019296646118, 2.930300712585449, -5.9234490394592285, 0.5897901058197021, 3.933765411376953, -1.9916794300079346, 1.2583956718444824, -0.3095521628856659, -0.060705557465553284, -4.4316301345825195, 4.256965160369873, 2.522587299346924, 1.0325422286987305, -0.642304539680481, 3.117063045501709, -4.944579124450684, -2.607367992401123, 0.09913213551044464, 2.026231527328491, 5.160881042480469, -3.291778564453125, 3.8685030937194824, -2.034991502761841, -2.6484079360961914, 2.201890707015991, -1.149451494216919, 3.073474407196045, -0.27354031801223755, -5.936379432678223, 2.744184970855713, -3.922121047973633, -0.10398532450199127, -0.6736047267913818, 1.0684878826141357, -0.2741931676864624, 0.6860102415084839, -4.549689292907715, -0.005646704230457544, 0.9451409578323364, -3.3498940467834473, 1.8250722885131836, -0.9245160818099976, -1.2850680351257324, 2.304157257080078, -1.8261449337005615, 0.9721592664718628, 4.986358165740967, -4.168423175811768, 1.3011523485183716, -1.7325654029846191, -1.1626386642456055, 0.493327796459198, -9.359813690185547, 1.452177882194519, -1.6482079029083252, -2.6462249755859375, 0.14512844383716583, 1.489419937133789, -0.9081178903579712, 2.1433167457580566, -0.20417016744613647, -6.970983982086182, 6.544720649719238, -1.5595639944076538, 4.373887062072754, -4.528041839599609, -2.4720938205718994, -4.169344902038574, 4.038670539855957, -8.772937774658203, 0.11916966736316681, -1.3578635454177856, 1.742659330368042, -3.029621124267578, 0.6074622869491577, -3.216710090637207, 4.629354476928711, 0.2039855271577835, 4.841261386871338, 4.821840286254883, 2.728705406188965, 2.9024393558502197, 1.903283715248108, -0.5511986613273621, -0.20096245408058167, -1.001323938369751, -2.611083507537842, 1.2352564334869385, -4.205581188201904, -5.650619029998779, 2.316439151763916, 0.5460969805717468, 3.646641254425049, 2.869232177734375, 1.7366819381713867, -0.07473969459533691, -2.0262789726257324, -0.45634979009628296, 1.8019269704818726, 4.295563697814941, 2.512328863143921, 1.4537038803100586, -0.7865711450576782, 1.4291837215423584, 2.4332785606384277, -3.9650912284851074, 3.9507758617401123, 0.09758472442626953, 1.5399668216705322, 0.3631317913532257, -3.747873544692993, 0.10798542201519012, -0.07206537574529648, -4.910550117492676, 2.086005210876465, -0.22077414393424988, -2.224699020385742, 2.395643711090088, -0.4478170871734619, 0.41838929057121277, -4.000254154205322, -3.123170852661133, 1.2677932977676392, 1.1597684621810913, -0.4692249298095703, -3.4597864151000977, -1.1365101337432861, -2.642427921295166, -1.0938340425491333, 4.878938674926758, -0.2622396945953369, 1.4437627792358398, -3.330268621444702, -2.1482973098754883, -0.8964452743530273, 6.618679046630859, -1.3748284578323364, 1.195759892463684, 0.5338987112045288, 0.9015132188796997, 6.437705039978027, -3.6911816596984863, 5.253248691558838, -3.14463472366333, -5.200918674468994, 2.586188316345215, -2.7485475540161133, -0.5527251362800598, 0.23329778015613556, -0.46402859687805176, -0.48469796776771545, -0.23752707242965698, 4.766105651855469, 3.0225253105163574, 0.9407947063446045, -1.048612356185913, 1.4393901824951172, 0.5556327104568481, 8.186793327331543, -2.7969415187835693, 1.0119212865829468, 3.775984525680542, -1.879776954650879, 1.4960283041000366, -4.039066314697266, 8.3233642578125, 2.9747841358184814, -3.193986654281616, 0.2430988848209381, -8.38050651550293, -0.021601136773824692, 0.8082258701324463, 4.637686729431152, 4.371232032775879, 2.184359073638916, -2.047597646713257, 1.503220796585083, 1.0289366245269775, 1.0912494659423828, 2.041370391845703, -1.697237253189087, -1.5906590223312378, -1.8950897455215454, -0.7420061826705933, -3.5798027515411377, 0.20613431930541992, 1.3520970344543457, -1.4326006174087524, 3.0215442180633545, 0.5978703498840332, 5.359381198883057, 1.7831041812896729, 0.02656799554824829, -1.7386815547943115, -1.4849363565444946, 4.527935028076172, 0.424504816532135, -5.57773494720459, 0.8266595005989075, 1.7001476287841797, 4.766096591949463, 1.1141732931137085, -0.3366769254207611, -2.183077096939087, -4.572091102600098, -5.397978782653809, -1.1044312715530396, 3.353787899017334, -3.1493561267852783, -2.99473237991333, -2.0281622409820557, -2.8065271377563477, 1.7133736610412598, 0.9998121857643127, 7.300190448760986, 0.2104962319135666, -2.4236209392547607, 1.5992248058319092, 2.5215260982513428, -4.7757744789123535, 1.748506784439087, 2.9616456031799316, -3.433166742324829, -0.09818622469902039, 0.44277632236480713, 2.1539478302001953, 3.322291851043701, 1.743556261062622, 0.391483873128891, -2.7752268314361572, 0.9308791160583496, -0.8933755159378052, 2.2684130668640137, -2.7399210929870605, 0.8561666011810303, 3.6154582500457764, 3.8917484283447266, -1.7847838401794434, 0.038438234478235245, 8.326986312866211, -3.5646657943725586, 0.20568206906318665, 1.9666614532470703, -1.447486162185669, -1.4407339096069336, -2.471076250076294, -2.9681177139282227, -1.7294349670410156, -3.786271333694458, 0.8184636831283569, 3.0404298305511475, 1.439899206161499, 2.7475507259368896, 1.279072642326355, -0.6368587017059326, -0.9973134994506836, 2.358973264694214, -3.4788401126861572, 2.705068588256836, -2.4999685287475586, 1.0039870738983154, -2.140805959701538, -5.613177299499512, -1.444515585899353, 1.2510778903961182, -1.754837989807129, -0.8510509729385376, 0.8044462203979492, -0.5474997758865356, -0.6010521054267883, 0.16321878135204315, -2.6593735218048096, 0.7397509813308716, -0.07009811699390411, -2.2492055892944336, 1.5910130739212036, 0.4525708556175232, -1.2167649269104004, 1.5334539413452148, -4.889029502868652, -1.7068300247192383, 1.6574676036834717, -7.281383514404297, 2.541700839996338, -2.9863834381103516, -1.702856183052063, 8.266729354858398, -2.926180601119995, 0.4004081189632416, -2.256847620010376, 2.298734188079834, -1.8975675106048584, 7.413130760192871, -5.658953666687012, -2.468351364135742, -1.0883433818817139, 1.8952386379241943, -2.544851779937744, -0.4667920470237732, 6.571775436401367, 0.8566733598709106, 3.1370882987976074, 2.6565816402435303, 2.7426416873931885, -1.6029993295669556, 6.562118053436279, -5.665404796600342, -0.6135802865028381, -3.914416790008545, -3.9579012393951416, -4.933599472045898, 5.009847640991211, -1.4764325618743896, 3.2456846237182617, 1.820468544960022, -3.415472984313965, -5.941426753997803, -1.62276029586792, -2.719283103942871, 1.5126957893371582, -2.7720181941986084, 0.8336055278778076, 0.2944536805152893, -6.153997421264648, 3.988462448120117, -1.1704707145690918, -0.19180703163146973, 0.8928308486938477, 0.5666675567626953, -3.2396397590637207, 9.102249145507812, 7.169658660888672, -2.349841356277466, 8.376493453979492, 0.9705456495285034, -0.2974739670753479, 2.645781993865967, -0.7344814538955688, 3.7206225395202637, -3.790665864944458, -1.8773298263549805, 8.182134628295898, 1.135878562927246, -0.6142982244491577, -0.17424583435058594, -2.8382186889648438, 1.1839534044265747, -1.4036250114440918, 1.2093169689178467, 9.99123764038086, 0.9279332160949707, 3.3456921577453613, 1.4862746000289917, -4.74487829208374, 2.137307643890381, 0.6309706568717957, 2.5613014698028564, -7.265558242797852, 0.6713306307792664, 1.4876515865325928, 2.1099555492401123, 0.5973187685012817, 2.549534559249878, -4.186358451843262, 3.698357105255127, -1.662853717803955, -1.778918981552124, 3.8461856842041016, 4.474127769470215, 1.3562172651290894, 3.506978988647461, -1.586247444152832, -0.05999450385570526, -2.134735107421875, -1.4958770275115967, 5.048410415649414, 2.830653429031372, -3.984520435333252, -2.618624210357666, -0.8955590128898621, -1.2504897117614746, 0.24446390569210052, 3.6306703090667725, 0.6069181561470032, -4.8192362785339355, -4.632859230041504, -1.1039316654205322, 7.998388767242432, 2.248814105987549, 1.0049123764038086, -0.775396466255188, 0.5205719470977783, -0.22514484822750092, -1.4470652341842651, -0.23588132858276367, -4.584930419921875, 5.1417388916015625, -0.47615790367126465, -2.87262225151062, 2.2040772438049316, 0.9780130386352539, 3.2500953674316406, 2.0046699047088623, -0.2284516990184784, 2.299511671066284, 0.09425815939903259, -0.5582954287528992, 2.6190054416656494, -5.1520795822143555, -0.5498452186584473, 2.1836042404174805, 0.3463151752948761, 0.5844893455505371, 2.314753770828247, 1.2207581996917725, 2.0552499294281006, 2.454348087310791, 0.7074232697486877, -2.7942888736724854, -1.8094732761383057, -4.079532623291016, 3.109585762023926, 2.0833139419555664, 3.6691110134124756, -0.30022335052490234, 3.742176055908203, 1.3947616815567017, 1.5669620037078857, -0.2780631184577942, 6.144219398498535, -0.44150668382644653, -0.04782533645629883, -3.6526687145233154, -6.945589065551758, 1.9575573205947876, -3.396270275115967, -4.8840227127075195, 1.974501371383667, 0.2455838918685913, 1.4818705320358276, 1.3733004331588745, 3.1601064205169678, 0.5501717925071716, 0.37311458587646484, -2.4790139198303223, -5.011213779449463, 2.2988295555114746, 0.3587004542350769, 4.315816879272461, 3.919654369354248, -3.312272787094116, 1.7595102787017822, -2.1780829429626465, -2.537900924682617, 4.282437324523926, 5.076495170593262, -1.6801135540008545, -4.743566513061523, -3.8115227222442627, 2.928349494934082, -0.5779740214347839, 2.382047414779663, -0.8112630844116211, -2.423673629760742, 2.2891504764556885, -0.9017427563667297, 5.674941539764404, -4.066382884979248, -5.058740615844727, -4.504046440124512, 2.746997356414795, -4.183425426483154, -2.078544855117798, 1.763535737991333, -4.118145942687988, 1.7806298732757568, 4.701873779296875, -2.341789722442627, 1.7976369857788086, 1.021268367767334, -2.711524248123169, -1.9473652839660645, 0.49964654445648193, 3.5097806453704834, -2.4580798149108887, -1.680452823638916, -0.7412476539611816, -1.5269863605499268, 6.588086128234863, -3.957115888595581, 3.3276748657226562, 3.6470723152160645, -0.5073555707931519, 2.9778530597686768, 3.080472707748413, 1.7132511138916016, 0.7536647915840149, -2.2568376064300537, -3.33207368850708, -2.81333589553833, -3.456301689147949, -1.8947696685791016, -2.131012439727783, 2.999154567718506, 0.4128519892692566, 1.0445810556411743, -0.788615345954895, -4.504029273986816, 2.647280693054199, 0.47125494480133057, -0.36225026845932007, -1.9744880199432373, 4.9036865234375, 2.522974729537964, -2.9599719047546387, 5.71722936630249, 1.9333457946777344, -0.31098073720932007, -2.955637216567993, 2.7352192401885986, -0.5027264952659607, -1.1397454738616943, 1.0250052213668823, -0.05348586291074753, -2.5081706047058105, -0.6217716932296753, -3.8486099243164062, 0.5694072842597961, -1.2304346561431885, 0.7784563302993774, 6.058256149291992, 0.732740044593811, -0.4906785488128662, 1.5476800203323364, -3.995720386505127, 0.058418355882167816, -2.2594707012176514, 1.9528415203094482, 4.001404285430908, 0.11881008744239807, 4.414962291717529, 1.642176866531372, 4.052337646484375, 2.4806766510009766, 0.6510937809944153, 3.2575786113739014, -0.7185981869697571, -0.8833066821098328, 0.4954298734664917, -0.8689652681350708, -0.9013317823410034, -3.674408197402954, -2.052595615386963, 1.9046258926391602, 5.962236404418945, -3.373260974884033, 0.8723819255828857, -2.2295079231262207, 2.7653608322143555, 4.34488582611084, -1.2119863033294678, 1.037755012512207, -4.476526260375977, -1.0820872783660889, 0.85107421875, 2.1278738975524902, -1.2474126815795898, -3.5064845085144043, 3.5580875873565674, 1.4462149143218994, -0.07455811649560928, -1.6773808002471924, 1.0893564224243164, -4.160633563995361, -3.7822556495666504, -2.7562875747680664, -1.8483660221099854, 1.4913702011108398, 1.8264702558517456, 5.39284086227417, 3.751059055328369, 2.412760019302368, -0.0350935161113739, 1.6378321647644043, 1.1079002618789673, 0.909349799156189, -1.6665709018707275, 2.0943050384521484, -0.09553775191307068, 0.39258497953414917, 1.9650883674621582, 2.184418201446533, -2.2173590660095215, 3.057827949523926, 2.2213621139526367, 2.3052589893341064, -2.2377195358276367, 0.9928892850875854, -3.1455187797546387, -3.8913803100585938, 3.7600698471069336, -0.3436688184738159, -1.6806492805480957, 1.3692371845245361, -0.7587997913360596, 1.309918999671936, -0.45395877957344055, 2.5431556701660156, 4.569775104522705, 1.2226492166519165, 3.608337163925171, -2.7744922637939453, -1.9239839315414429, 4.049310684204102, 1.6335694789886475, -0.7545406818389893, 0.6742860674858093, 1.4144444465637207, 3.991576671600342, -1.3909317255020142, 1.2209718227386475, -2.418867588043213, 3.483780860900879, 4.9306230545043945, -2.5420007705688477, -1.038947582244873, -0.6238402128219604, 1.529270052909851, -0.1840999722480774, -2.1047604084014893, -1.7364925146102905, 0.9543991684913635, 0.20228734612464905, 1.678018569946289, -5.334249973297119, 2.7838294506073, 2.2540624141693115, -0.5086370706558228, -0.7998623847961426, -0.8845609426498413, 2.6757020950317383, -0.8862560391426086, -1.7160204648971558, 1.3156750202178955, 4.030429363250732, 2.0769166946411133, 0.08392821997404099, 0.06456627696752548, -3.9739816188812256, 3.3659915924072266, 0.5202074646949768, -4.52389669418335, 1.0023508071899414, 1.0239105224609375, 5.172702789306641, -2.5511412620544434, -1.0066213607788086, 0.013964521698653698, 1.3984034061431885, 2.668334722518921, -1.7924494743347168, -0.3800666034221649, -1.3600126504898071, 4.395853519439697, 4.174674034118652, -0.8673311471939087, -1.2789146900177002, -4.342502593994141, 0.5430361032485962, 0.4737841486930847, -3.7010016441345215, 6.7418365478515625, -2.696180820465088, -6.261909484863281, 0.9661787748336792, -0.4425298869609833, -0.14641904830932617, 5.062267303466797, -6.355822563171387, -0.3368256092071533, -0.9978781342506409, 4.499068260192871, -1.5840257406234741, -3.420438766479492, 0.9683377742767334, -0.30626818537712097, -3.109382152557373, 4.406876087188721, -4.21522855758667, -1.2497689723968506, 3.3950071334838867, 3.5351548194885254, 3.27862286567688, -4.586888313293457, -0.030454829335212708, -3.0384840965270996, -7.53144645690918, -3.290616035461426, 3.108060836791992, -0.11835071444511414, 0.42713266611099243, -1.365455985069275, -1.9465551376342773, 2.884882688522339, 1.311692237854004, -4.594852924346924, 1.264517903327942, -2.5198278427124023, 1.3872203826904297, -3.303739070892334, -2.998789072036743, -1.2048113346099854, 1.3517621755599976, -6.70224666595459, 6.303743362426758, 3.1257238388061523, -2.815110445022583, 4.081602573394775, -5.981592178344727, -1.0414025783538818, -2.847585678100586, 1.6574335098266602, -0.39600175619125366, -1.1785264015197754, -3.588881015777588, -1.1861343383789062, 0.9370881915092468, 0.3432062864303589, -0.7977300882339478, 0.30243605375289917, -2.5206570625305176, -4.5160017013549805, -2.1593058109283447, 2.5961217880249023, 4.470336437225342, -0.021585732698440552, -3.450005292892456, -5.7998504638671875, 0.06469932198524475, 2.0446419715881348, -5.539046764373779, -2.1244914531707764, -0.74342942237854, 4.5928802490234375, 4.803136348724365, -4.436128616333008, 9.523242950439453, 1.627200722694397, -2.4075515270233154, 5.0231733322143555, -1.6909973621368408, -0.8699421882629395, 6.918721675872803, 3.482347011566162, 6.083491325378418, 0.6752345561981201, 0.6227138638496399, -2.1255083084106445, 1.1868994235992432, 0.30491912364959717, -5.465923309326172, 0.8122302889823914, -1.1151270866394043, -6.367919921875, -1.2323389053344727, -3.7141964435577393, -1.0151745080947876, -1.636948585510254, -1.8020427227020264, -0.5518746376037598, -2.0708975791931152, -2.628209114074707, -4.2609968185424805, 1.2720718383789062, -2.361018657684326, 0.44089412689208984, -0.49870187044143677, -1.6188501119613647, -6.638209342956543, -3.8075003623962402, -0.38730767369270325, 0.7510619759559631, 3.187448024749756, 1.0321879386901855, 4.879914283752441, 1.942873239517212, 0.03394363820552826, 3.7785887718200684, 0.8400022387504578, -0.8662810325622559, -1.4188461303710938, -0.844968318939209, 1.1681342124938965, -2.4836232662200928, -0.6645582914352417, 0.03363854065537453, -0.10489401966333389, 1.827216625213623, 2.8566689491271973, -3.484374761581421, -1.0916996002197266, 1.2512013912200928, -0.19998297095298767, -3.3094186782836914, -7.245205879211426, 4.476010799407959, 1.9707834720611572, 2.9235823154449463, 1.327826976776123, 0.9319915771484375, -0.8153080940246582, 0.8628926277160645, 0.6449680328369141, -5.684845924377441, 3.2637367248535156, 1.2364145517349243, 0.3635179400444031, 2.8588359355926514, 1.7313055992126465, 4.615241050720215, 0.6826140284538269, 3.7754464149475098, 1.1903046369552612, 1.7615125179290771, -2.4473485946655273, -1.952282190322876, -0.37891870737075806, 1.235365629196167, -4.319654941558838, -3.9558284282684326, 4.155637264251709, 1.349648356437683, -2.1651411056518555, -1.24689519405365, 5.953746795654297, -0.024806272238492966, 0.16061745584011078, 0.26618292927742004, 2.4841766357421875, -2.8613100051879883, -6.625513076782227, -0.10770522058010101, 1.1839823722839355, 2.5152475833892822, -1.278407096862793, 2.2300283908843994, -1.0974195003509521, 0.8748295307159424, -2.105501890182495, 2.579404830932617, -1.5438010692596436, -0.9749244451522827, -3.294355630874634, -4.085874557495117, -1.413156270980835, -3.7655248641967773, 1.2004145383834839, -1.7346479892730713, 4.024402141571045, 0.48373210430145264, -0.7863812446594238, -1.34761643409729, 3.814168930053711, 1.4516263008117676, 2.7201144695281982, 2.52467942237854, -4.118168830871582, -0.8694456815719604, 0.1546701192855835, 2.073700428009033, -2.6416311264038086, 1.26982581615448, 4.232524871826172, 3.5696582794189453, -2.1514108180999756, -2.719172954559326, 1.069433569908142, 2.5980887413024902, 2.6708481311798096, -1.2857418060302734, 4.520035266876221, 3.1767501831054688, -0.5979340076446533, 2.493401050567627, -3.450162649154663, 1.5050077438354492, 0.09282596409320831, 0.15074028074741364, -1.699699878692627, 3.0641934871673584, 2.2082607746124268, 2.877192258834839, 3.55346417427063, 0.728272020816803, -4.252223014831543, 2.9650020599365234, 1.4300217628479004, -0.49387165904045105, -1.541563868522644, -1.200547695159912, 3.846773386001587, 3.7912373542785645, -2.3677594661712646, 2.8949413299560547, -2.147953748703003, 2.7019081115722656, 1.6016287803649902, 2.46272349357605, -1.8543953895568848, -0.7455891370773315, 0.4084452688694, -4.541133880615234, -2.6992673873901367, -0.31292009353637695, 5.673064231872559, -0.9530034065246582, -1.9596301317214966, 3.1828501224517822, 2.826537609100342, -0.5590009689331055, 0.5884318947792053, 1.530759334564209, -1.6878859996795654, 5.5270094871521, 1.5648014545440674, 3.9529223442077637, -0.3260321319103241, -0.17884038388729095, -2.7535126209259033, -1.7694740295410156, -2.9514222145080566, 1.0886720418930054, 4.204730033874512, -0.8645923733711243, 3.6064844131469727, -1.1891860961914062, 1.8764744997024536, -0.829535186290741, -1.4625325202941895, -1.575892686843872, -0.09756013751029968, 4.931418418884277, -4.3481245040893555, 4.724898338317871, 1.8138574361801147, -0.4230887293815613, 4.973380088806152, 2.8655097484588623, -1.2364108562469482, 5.055957794189453, -3.251208782196045, 0.9669625759124756, -0.0014186998596414924, 3.0133087635040283, 1.6887738704681396, 0.037552930414676666, -1.3415817022323608, -1.034592628479004, -1.664362907409668, 1.8186938762664795, 1.3318837881088257, -0.3598504960536957, -4.016584396362305, 9.167016983032227, -1.4164491891860962, -1.2208404541015625, 3.8949174880981445, -0.48333507776260376, -3.785130500793457, 0.3899487257003784, 1.2609684467315674, 2.655022621154785, 1.8650264739990234, -3.049178123474121, 4.261346340179443, -2.6928555965423584, -3.093090534210205, 2.4521186351776123, -1.4905503988265991, 1.2785632610321045, 1.7637029886245728, -4.7704033851623535, -1.0042259693145752, 1.833101749420166, 3.5315866470336914, 0.9588637351989746, 1.071641445159912, -0.5468305945396423, -4.933620929718018, 3.1992368698120117, -3.044186592102051, 3.9135711193084717, -6.245445728302002, 0.5432844161987305, -3.1026740074157715, 2.052776336669922, -3.7734711170196533, 2.9378693103790283, 4.873381614685059, -0.6962680816650391, -0.007198752835392952, 1.1389497518539429, -3.1229088306427, 2.1250205039978027, -4.004943370819092, -0.6108419299125671, -5.037359714508057, -0.4986512064933777, -0.35124868154525757, -0.33628857135772705, -3.6593995094299316, 0.17875435948371887, -3.8907055854797363, -2.1374568939208984, -1.3973183631896973, -1.1450904607772827, 0.013580943457782269, 4.353147029876709, -1.5145623683929443, -0.22586636245250702, 1.9934947490692139, -0.022544700652360916, -5.99139404296875, -2.5826926231384277, 4.575603485107422, -1.505289077758789, 4.133577823638916, -0.6492404937744141, -0.05417994037270546, -1.4064615964889526, 1.957183599472046, -4.574756145477295, -6.608529567718506, -0.8656501770019531, -2.1276609897613525, -0.20568621158599854, 1.120887041091919, -2.1174368858337402, 0.13628755509853363, -3.573030710220337, 3.0092906951904297, -3.1515870094299316, 8.108359336853027, 0.032238464802503586, -1.4360275268554688, -1.6050024032592773, -1.6820650100708008, 2.0030288696289062, -1.242114782333374, -1.7710809707641602, -1.0327644348144531, 0.32739925384521484, 4.859786033630371, 8.903472900390625, -1.9308875799179077, -1.180944800376892, 0.05968155339360237, 4.044057369232178, -3.293105363845825, 0.018229708075523376, -1.3391501903533936, -1.4284127950668335, 1.434393048286438, 1.3190453052520752, 4.542612552642822, -2.9959559440612793, -0.47739750146865845, -1.8334863185882568, -4.536285400390625, 2.0195884704589844, -2.9708240032196045, 4.3379106521606445, -1.046642780303955, 0.8486343622207642, 3.206684112548828, 4.240474700927734, -0.09202602505683899, 5.3395514488220215, 2.9049243927001953, -2.9278080463409424, 2.8331985473632812, -3.854034900665283, -0.4751898944377899, -2.9105732440948486, 0.4977266788482666, -2.0077826976776123, 0.7787686586380005, -0.9408977031707764, 2.214306354522705, -6.8492536544799805, 3.2957704067230225, 3.3944010734558105, -3.2283740043640137, 0.6030719876289368, 1.442073106765747, 0.7870041131973267, 1.8184759616851807, 0.7475483417510986, 1.9346897602081299, -1.438323974609375, -2.522730827331543, -1.6285088062286377, -3.142827033996582, 6.796716690063477, -4.575688362121582, 5.2504096031188965, 1.1518323421478271, -1.9583040475845337, -3.5107407569885254, -1.0243538618087769, -2.0776712894439697, -3.2074193954467773, -2.118974208831787, 0.8723322153091431, -3.4443933963775635, 6.0793561935424805, 2.877932548522949, -0.5259044170379639, 1.5586159229278564, -4.417163372039795, 3.6475114822387695, 0.8334521055221558, 3.0832793712615967, -1.794661521911621, -1.4163461923599243, 0.5490022897720337, -4.7711286544799805, 3.0725183486938477, -3.4513654708862305, -1.2149068117141724, 4.155780792236328, -0.03151705861091614, 4.815299987792969, -0.056947819888591766, -0.6894763708114624, 3.426370143890381, 3.4098987579345703, -3.5481204986572266, -2.935710906982422, 1.531073808670044, 1.2061716318130493, 4.596875190734863, 1.1701767444610596, -0.4549996554851532, -4.9730658531188965, -1.7641514539718628, 0.24166834354400635, -2.1272079944610596, 6.8725762367248535, 2.452859878540039, -1.0055323839187622, 0.0715053379535675, -1.1692559719085693, -3.8861753940582275, -2.014835834503174, -1.7612147331237793, -5.359879970550537, 0.34209656715393066, 1.3205878734588623, -4.0750627517700195, -1.0005247592926025, -4.301239967346191, -2.532288074493408, 0.8443638682365417, -2.5082011222839355, -1.3741180896759033, 6.397214889526367, -2.8280889987945557, -0.5307915210723877, 0.9828612804412842, 1.6435744762420654, -5.166052341461182, 1.3534653186798096, 2.2260031700134277, 0.8707432150840759, 1.8795114755630493, 2.681717872619629, -1.6193852424621582, 0.8325283527374268, 0.754116415977478, 4.317586898803711, 3.8769900798797607, 6.3588995933532715, -4.024242401123047, -6.625077247619629, 2.9766554832458496, 0.9352695941925049, -2.4244792461395264, 2.35431170463562, 2.3720712661743164, 2.7172532081604004, 0.9869518280029297, 1.5160249471664429, -0.049131400883197784, -4.819051742553711, -0.8837748765945435, 5.336541175842285, -1.6240078210830688, -6.215774059295654, -3.714049816131592, 4.892874717712402, 3.472656011581421, -2.1118531227111816, 1.788750171661377, 3.517566204071045, -2.9979960918426514, -3.9007034301757812, -1.4260491132736206, -8.058477401733398, 0.5783181190490723, -2.4394028186798096, 2.3964948654174805, 10.27767276763916, 0.4552869200706482, 0.01904667168855667, 1.4626262187957764, 2.4151220321655273, 4.4898481369018555, 4.350892066955566, -0.6674840450286865, 1.901672124862671, 0.5130650401115417, -0.8413691520690918, -3.4623379707336426, -2.615274429321289, -3.7391347885131836, -0.33199483156204224, 2.1055667400360107, -0.9213511347770691, 1.6093659400939941, -4.10344123840332, -1.8522781133651733, 0.4350316524505615, -3.374250888824463, -1.1307123899459839, -3.3102760314941406, -1.7574652433395386, -0.5975247621536255, 3.721836805343628, -2.966522216796875, 1.0567541122436523, -2.3824503421783447, 0.08320824801921844, -4.142699241638184, -1.2512648105621338, 2.7028369903564453, -1.8559901714324951, -3.717245578765869, -4.635472774505615, -3.6412675380706787, -3.300738573074341, 3.373448610305786, 3.1309547424316406, -0.25636211037635803, -0.27305567264556885, -0.9457731246948242, -0.18116351962089539, -6.236875534057617, -0.8292645215988159, 1.740903377532959, 3.7747957706451416, -0.7696168422698975, -0.006691835820674896, -0.039460621774196625, -7.62105655670166, 1.8318204879760742, 3.2710580825805664, -3.9812088012695312, 2.274285316467285, -1.9780449867248535, 2.180267095565796, 4.850674629211426, -1.2379224300384521, 1.7522621154785156, -2.714627742767334, -2.1691536903381348, -3.4456565380096436, 2.851841926574707, -1.152907371520996, -1.9070773124694824, 3.1525192260742188, -7.020956039428711, -4.5497517585754395, 2.60783314704895, -2.5116353034973145, -1.3686680793762207, -0.7060572504997253, -4.204371452331543, 1.642735242843628, -4.895462989807129, 0.943817675113678, 2.030733585357666, -3.137758255004883, -1.3101375102996826, -0.06503087282180786, -0.2435411810874939, -4.5347089767456055, 1.6022404432296753, 0.43349647521972656, -1.5092341899871826, -0.4754951298236847, 0.9109917879104614, -1.3427828550338745, -3.1275293827056885, -0.07904520630836487, -5.232422828674316, -0.18817327916622162, 1.4269249439239502, -1.6030429601669312, 0.5666191577911377, 2.381598949432373, -0.4142208695411682, 0.6171607971191406, 2.6092991828918457, 2.224931240081787, -2.8530538082122803, 2.203479290008545, 2.86559796333313, 3.092134952545166, 4.89714241027832, -0.4436676502227783, -1.2484327554702759, 0.9178986549377441, 6.387048721313477, 3.0176382064819336, 0.012483903206884861, -2.5663907527923584, -0.18484696745872498, 1.3380295038223267, -0.818291425704956, -2.502861499786377, -0.6085489988327026, 2.8733601570129395, 4.382623195648193, 0.6549477577209473, 0.09070388972759247, -0.46033698320388794, 4.609525680541992, 2.032698631286621, 3.151089668273926, 3.3337512016296387, 3.558243751525879, 6.578997611999512, -2.7223901748657227, 1.448472023010254, -2.653571128845215, -1.793856143951416, 1.4241559505462646, -1.8314011096954346, -3.0641608238220215, -1.452688217163086, 0.18062514066696167, -3.282259225845337, -1.2763426303863525, -3.018989086151123, 0.39549511671066284, 2.6456875801086426, 2.121037483215332, 1.4691176414489746, 1.8655235767364502, 2.6354339122772217, -4.891326904296875, -2.7089500427246094, -2.1447196006774902, 2.596741199493408, 1.5600782632827759, 0.8975434303283691, 0.3773356080055237, 0.5512256026268005, -2.9364633560180664, -2.4776251316070557, -3.3492658138275146, 3.0574569702148438, -5.15396785736084, 2.5782673358917236, -0.39118659496307373, 1.66886305809021, -5.012207508087158, 4.052392959594727, 1.308933973312378, 0.04759715124964714, -0.3469848036766052, -3.670689582824707, 6.576319694519043, -6.520280838012695, 4.590818881988525, -3.8697333335876465, 3.063809394836426, 2.5083534717559814, -2.122042655944824, -2.061410903930664, -2.723052501678467, 0.3608373701572418, -3.7594995498657227, 3.623389720916748, 5.041712760925293, -0.43940043449401855, 2.9788622856140137, -2.8811206817626953, -4.269406318664551, 2.542476177215576, -0.017300203442573547, -0.740799605846405, 1.6523170471191406, 3.0633721351623535, -0.7855010032653809, -0.39034849405288696, -4.492644309997559, -2.3264660835266113, 0.9638717174530029, -1.551887035369873, -3.074445962905884, 5.147965431213379, 0.8492958545684814, 1.2624094486236572, -5.373830318450928, -2.443202495574951, 0.5664933919906616, 0.7538890242576599, -3.7006654739379883, 3.472586154937744, -1.0384074449539185, 5.9586710929870605, 4.177575588226318, 1.503122091293335, 0.6347570419311523, -1.4994618892669678, -5.433738708496094, 0.7209984064102173, 0.8153992295265198, -3.0683298110961914, -0.5583337545394897, 2.777437686920166, -3.887122392654419, 1.3021249771118164, 6.315283298492432, 0.23905523121356964, 0.37400779128074646, 1.3299213647842407, -3.5616259574890137, -3.6058526039123535, 3.087740898132324, -0.2675475478172302, 2.6565492153167725, 0.3284786343574524, 4.268899917602539, -3.3200490474700928, -0.22605568170547485, -1.150861144065857, -3.9328179359436035, 0.6966229677200317, -3.278223991394043, -0.15493366122245789, -2.569053888320923, -3.257589340209961, -0.07289811968803406, -3.5572938919067383, 0.4875287413597107, -1.1859954595565796, -2.743939161300659, 7.122751235961914, 3.737849235534668, -3.7087302207946777, -0.3893308639526367, -1.4992890357971191, 3.252631187438965, 2.9657347202301025, -1.7812625169754028, 0.21510685980319977, 4.114550590515137, -0.5875667333602905, 1.4718190431594849, 2.2246391773223877, 0.2854103446006775, 4.867805004119873, 0.21064357459545135, -2.2791121006011963, 2.2747304439544678, -2.480752468109131, -0.7278051972389221, -0.9150716066360474, 3.1668362617492676, 0.4891301393508911, -1.206084132194519, -3.234522819519043, 3.9263176918029785, -1.460224986076355, -2.7116942405700684, 2.6391408443450928, 0.031749479472637177, -0.11146973073482513, 1.1105594635009766, 3.196385383605957, 0.3968154788017273, -4.620593070983887, -3.3352150917053223, -0.6245879530906677, 0.6554684638977051, -2.8925178050994873, -0.7777597904205322, -0.559371829032898, -3.8182291984558105, -3.5398640632629395, 1.3618741035461426, -1.952678918838501, 4.593578815460205, -0.5432828068733215, -5.485581398010254, -1.8388463258743286, 3.1285839080810547, 0.5558103919029236, -3.240288019180298, -0.6346311569213867, 3.138237953186035, -3.1519694328308105, -2.499708890914917, -0.2710607051849365, 0.9147080183029175, -0.7952183485031128, 2.0662689208984375, 0.9016761779785156, -2.614732265472412, -2.9423110485076904, -6.859715461730957, -0.6263384819030762, 0.0919877290725708, 1.9686293601989746, 1.3182129859924316, -2.1054978370666504, 13.43540096282959, 1.5350284576416016, -0.852086067199707, 0.8757728934288025, 2.5367894172668457, -0.5457642078399658, 2.5429537296295166, -0.48296186327934265, 4.707005977630615, -3.989257335662842, -1.0816758871078491, -0.7096955180168152, 3.679279088973999, -3.0227818489074707, -6.2286882400512695, 2.0925419330596924, -2.4918582439422607, 1.002073049545288, 2.49361515045166, 2.5074596405029297, -4.2625274658203125, 3.477937698364258, 0.8592145442962646, 5.390192985534668, -3.0924735069274902, 6.626448631286621, -0.9595654010772705, 0.3250427544116974, 4.064145088195801, -2.911874532699585, 1.8791577816009521, 4.243929862976074, 1.8134987354278564, -2.0958733558654785, -3.619659662246704, -2.9620566368103027, -0.48036855459213257, -0.5836305618286133, 0.2978432774543762, -1.086641788482666, -2.466418743133545, -1.8893394470214844, 1.5109241008758545, 0.7261712551116943, -5.489126682281494, -4.445061683654785, -2.476921558380127, -0.2117100954055786, -1.4932146072387695, -1.823722243309021, -2.478255271911621, -6.979374885559082, 1.5538090467453003, 2.6866846084594727, -2.998307466506958, -1.8983371257781982, 1.8390834331512451, 0.6237413287162781, 3.0846991539001465, 3.5198864936828613, -1.1694689989089966, -0.5073018670082092, -3.867694616317749, 4.308466911315918, 2.218587636947632, -2.786611557006836, -0.45776093006134033, -3.7826879024505615, 7.126428604125977, -3.011507987976074, -3.497756004333496, -2.582521438598633, -2.0229053497314453, -3.7816994190216064, -2.030099868774414, -1.1004984378814697, -3.4787826538085938, 0.8174165487289429, -0.1701679527759552, -1.079014778137207, -6.900280952453613, 6.125621795654297, 1.383071780204773, -0.8002090454101562, -0.012165270745754242, 2.8614444732666016, -2.453500986099243, 4.847476005554199, 0.11377007514238358, 0.717494785785675, 0.5122181177139282, -4.313648223876953, -3.105553150177002, 2.14806866645813, 2.94915509223938, -4.791731834411621, 4.725795745849609, 3.513413906097412, 2.9872376918792725, -2.107278823852539, 1.4358696937561035, -2.870110034942627, -0.049383580684661865, -2.1227262020111084, -0.45342183113098145, 0.9752264022827148, 1.0404506921768188, 6.702082633972168, 0.6508099436759949, -2.672268867492676, 2.9586918354034424, -1.5551443099975586, -0.2340545952320099, -0.15586967766284943, -2.3504128456115723, -4.241379737854004, 0.7489020228385925, 2.6342720985412598, -1.78013014793396, 1.1161491870880127, 2.578826427459717, 0.21357914805412292, 0.6915832161903381, -6.925959587097168, -7.441977500915527, -0.6782661080360413, -0.2846316695213318, 2.3496742248535156, 0.6715523600578308, 0.9142591953277588, 0.9309766292572021, -1.008434772491455, 1.5032912492752075, -4.231574058532715, 2.42630672454834, -4.77777099609375, -2.927980422973633, 2.4912853240966797, -0.49724090099334717, 2.352973461151123], [-1.6604292392730713, -1.2612829208374023, -1.2369440793991089, 0.5415276885032654, -2.009542942047119, 2.2714757919311523, 3.291654586791992, -0.9788886308670044, -4.233704566955566, -1.9898216724395752, -5.859221458435059, -1.9167927503585815, 2.111175060272217, -1.2080373764038086, 3.025947332382202, 2.19677472114563, -0.4366845488548279, -1.252021074295044, 0.4630235731601715, 0.3321041762828827, -4.8448028564453125, -1.226295828819275, -3.0765786170959473, -0.22511041164398193, -1.2003456354141235, -0.633198082447052, 1.6367257833480835, 0.9181588888168335, 1.9163434505462646, 2.139943838119507, -6.174343109130859, -2.2174224853515625, 2.6655266284942627, 0.4002981185913086, 0.295925498008728, 2.513317584991455, 4.567306995391846, 4.831149101257324, -2.0400044918060303, -1.5614867210388184, -4.943134784698486, -1.696570634841919, -1.4475998878479004, -2.6260392665863037, 3.0674819946289062, 1.6713824272155762, -2.373457193374634, -5.843889236450195, 2.26753568649292, -6.040848255157471, 1.6033422946929932, -4.727992057800293, 3.3409461975097656, -2.8094990253448486, 2.3367440700531006, -1.9966650009155273, -4.425631523132324, 1.9130299091339111, -1.442906379699707, 4.0077714920043945, -2.450016498565674, 1.3427716493606567, -3.859654426574707, -2.3184895515441895, -0.07538293302059174, 1.3981566429138184, -1.2269959449768066, -1.5269176959991455, 1.2172026634216309, -2.581467390060425, -4.796045303344727, -3.8259096145629883, -0.05606931075453758, -6.142346382141113, -1.4038033485412598, 8.198465347290039, -4.0023627281188965, 5.384106636047363, 4.889133930206299, -3.1666650772094727, -4.365992546081543, -2.158188819885254, -3.221499443054199, 2.713258981704712, 0.00328624015673995, 2.2632179260253906, -3.537576913833618, 0.12230560183525085, -5.053590774536133, 1.6503643989562988, 4.959098815917969, 1.1604576110839844, 4.712584495544434, 1.5776288509368896, 0.11519495397806168, -1.850604772567749, 1.278878927230835, -1.9858169555664062, 0.9883609414100647, -0.301551878452301, 3.8595690727233887, -0.9234974384307861, -0.030946439132094383, 2.8717188835144043, -1.2840509414672852, 1.4393634796142578, -1.342814326286316, 2.4778027534484863, -3.028318405151367, 2.728046417236328, 1.4502838850021362, 0.36460405588150024, 3.1323604583740234, -1.866066813468933, 3.3093488216400146, 0.5695276260375977, 0.6460860967636108, 0.6050061583518982, -0.5975454449653625, -0.1362716555595398, -2.2066078186035156, 2.0435421466827393, 1.987416386604309, -2.077709674835205, 1.0796480178833008, 0.055273137986660004, 0.13235093653202057, -0.7282589077949524, 1.130376935005188, -2.6916298866271973, -1.7712783813476562, 2.426224946975708, -0.9701817035675049, -0.6836481690406799, -6.236588478088379, 1.0738835334777832, -2.4133365154266357, 0.16667288541793823, -0.45573341846466064, 1.3468353748321533, -5.211191177368164, 2.747382164001465, -1.2181490659713745, 2.027442693710327, -3.8098955154418945, 1.0186012983322144, -2.742680549621582, -0.8846048712730408, 2.503145217895508, 0.5787175893783569, 3.696625232696533, 0.4419668912887573, 1.5252437591552734, 5.114508628845215, -3.8649895191192627, 2.4586353302001953, 0.2172607183456421, 1.5923771858215332, -1.7633718252182007, -1.9638967514038086, 0.055660538375377655, 3.3561320304870605, 3.1379406452178955, -1.688572883605957, 4.013500690460205, -2.861377239227295, -8.105681419372559, -0.15060919523239136, 0.504486083984375, 1.4039386510849, 2.4073970317840576, 0.2695242762565613, -4.137706756591797, -2.5976202487945557, 9.855489730834961, 3.527082681655884, -4.54353141784668, -1.4959783554077148, -0.6202900409698486, -2.4409236907958984, 0.25536125898361206, -0.484692245721817, 1.6882808208465576, 6.016293525695801, -0.7801337242126465, 0.1193723976612091, -4.171530246734619, -1.354065179824829, -4.011868476867676, 3.5317976474761963, 0.4347253739833832, -0.18157370388507843, 0.21451954543590546, 4.22079610824585, 3.0747084617614746, 0.33036088943481445, 1.8800829648971558, 2.702310562133789, 3.349621295928955, -6.376217365264893, 3.4841065406799316, 2.1300511360168457, 2.3602771759033203, -1.6200177669525146, 2.1738948822021484, -3.8560051918029785, -0.2388189435005188, -4.650535583496094, -0.9513082504272461, 3.4678494930267334, -1.7672319412231445, 1.969306468963623, -2.8178253173828125, -3.2224395275115967, -0.15323761105537415, 0.24937914311885834, -1.00029718875885, 1.071855068206787, -0.07333388924598694, 5.546392440795898, -7.528048515319824, 0.09145908057689667, 0.6156004667282104, 4.195212364196777, -1.648078441619873, 4.424232482910156, -0.226204514503479, -1.3283392190933228, 0.08318828791379929, 4.34079647064209, 0.5693897008895874, 1.3259857892990112, 2.5773158073425293, -4.023406505584717, -8.366401672363281, -5.9827070236206055, 4.911355018615723, -0.8674469590187073, -2.7572710514068604, -3.6479010581970215, 3.151327133178711, -4.849663734436035, 0.7437781095504761, -1.657520055770874, -0.15028585493564606, -1.349308967590332, -2.241898536682129, 2.8245787620544434, 1.2406814098358154, -1.4465010166168213, -0.024486567825078964, -4.251154899597168, -2.0239624977111816, 1.6669399738311768, 0.6062721014022827, -4.520594120025635, -3.302339553833008, -0.897814929485321, -0.19508016109466553, -0.4223296046257019, -3.410277843475342, -3.9603800773620605, 0.5165944695472717, 0.9561499357223511, -2.8728225231170654, 6.692775249481201, 3.0249414443969727, -3.3122801780700684, -4.370837211608887, -2.907352924346924, 3.8369293212890625, 6.9066057205200195, -1.4077630043029785, 3.690502643585205, 2.4778342247009277, 2.625196695327759, 3.092385768890381, -1.3630985021591187, -3.669468879699707, 2.5500259399414062, -5.246741771697998, -1.4367140531539917, -4.512161731719971, -1.6384531259536743, -2.7699942588806152, 3.176234245300293, -1.2311153411865234, 6.20121431350708, 7.410229682922363, 0.8832622766494751, 2.143162965774536, 2.3254473209381104, -4.126375198364258, 2.2857677936553955, -3.7197446823120117, -0.06442034989595413, 2.661888360977173, -2.2286243438720703, -1.0321658849716187, 1.937058687210083, -4.545232772827148, 0.4951288104057312, -1.3506698608398438, -1.8474631309509277, 3.192704677581787, -3.4350264072418213, -0.8494899272918701, 0.9796632528305054, -1.7917280197143555, -3.9430911540985107, 3.9246578216552734, 5.236748218536377, 4.624434947967529, 3.716736316680908, 1.5244624614715576, -5.437870025634766, 1.8461544513702393, -3.119699478149414, -1.02065110206604, 3.400273323059082, -6.195085525512695, 1.2510225772857666, 5.422107219696045, -3.0348782539367676, 2.244666576385498, 1.9408347606658936, -0.7520958185195923, 1.568586826324463, -0.11560005694627762, 5.074916362762451, -5.66353178024292, -4.228541374206543, 1.93656587600708, 0.5492242574691772, -2.7060883045196533, -5.6950554847717285, -2.746400833129883, 0.42089787125587463, 2.0175909996032715, 6.377285957336426, 0.367561936378479, 2.1306121349334717, 0.45353588461875916, 0.6922116875648499, -2.6185805797576904, 5.802837371826172, 6.9699554443359375, 1.3843096494674683, 0.5870900750160217, -1.8236796855926514, -2.0987207889556885, 0.1255439668893814, -2.1793212890625, 0.3980374038219452, 2.241105079650879, 1.6091519594192505, 0.9914590716362, -2.226492404937744, 2.440812587738037, 1.1787023544311523, -1.9750488996505737, 0.8654944896697998, -3.141373634338379, 1.8367433547973633, 2.0078234672546387, 5.728235244750977, -8.789560317993164, 3.4781534671783447, -4.288448333740234, -4.60622501373291, -4.056546211242676, 1.5838229656219482, 3.670164108276367, -0.003953323699533939, 3.5675876140594482, 3.8488125801086426, 2.307713508605957, 4.317882537841797, 1.5821846723556519, -0.5873798131942749, -0.5853272676467896, -1.9619674682617188, 2.7075209617614746, 2.537487030029297, -3.3688764572143555, 0.48297691345214844, -1.9752312898635864, -3.0450148582458496, -0.23115214705467224, 6.56118106842041, -3.765763759613037, -1.8983370065689087, -4.078336715698242, 2.4956178665161133, 1.107102632522583, 0.4585249722003937, -5.588255405426025, 0.6182097792625427, 4.1430816650390625, -4.434392929077148, -4.874177932739258, 1.594893455505371, 5.255632400512695, -0.8984696865081787, 3.175281524658203, -0.41920584440231323, 3.0197362899780273, 3.0734620094299316, 10.215700149536133, -1.0608773231506348, 6.193300247192383, 1.874614953994751, 4.170709609985352, -0.3222843110561371, 2.5115256309509277, 0.6215252876281738, 1.716207504272461, 0.19094368815422058, -1.7249305248260498, 4.025331497192383, -1.3095333576202393, 2.6624536514282227, 0.25266847014427185, -2.0691239833831787, -5.7831268310546875, 0.8555408120155334, -3.7583956718444824, -3.5338282585144043, 4.405087947845459, -0.2978171706199646, -2.083117723464966, 7.1522297859191895, -0.9394813776016235, 8.50847339630127, 0.09357263892889023, -2.471522569656372, -0.3810496926307678, 2.230527400970459, 0.1044386774301529, -1.7543766498565674, 0.2389368712902069, 3.885511636734009, 2.404160261154175, 3.0281972885131836, -1.182054042816162, 6.624605655670166, 0.4792240858078003, 0.6358152627944946, 1.1263222694396973, 3.0357003211975098, -1.7482147216796875, 4.838156700134277, -2.311577320098877, -1.5659458637237549, -5.037073135375977, -4.42966365814209, 1.5670735836029053, -6.729819297790527, -2.948688507080078, -3.3018100261688232, 0.9098986387252808, -3.572173595428467, 1.1483795642852783, 1.0834605693817139, -1.7345516681671143, 4.150533676147461, 3.3413615226745605, 2.9125795364379883, 1.9418960809707642, 0.8140273690223694, 0.7620666027069092, -2.1681339740753174, -4.314919471740723, -0.7885892391204834, -2.2722091674804688, 3.428830862045288, 1.038076639175415, 2.130049228668213, 0.9180079698562622, 2.686070442199707, 0.18927696347236633, -0.686247706413269, 3.984003782272339, -5.706960678100586, -3.0530872344970703, -3.1395986080169678, 3.083087682723999, -6.796818733215332, 0.77769935131073, -2.9523370265960693, -5.854998588562012, 2.0282230377197266, -0.9071810245513916, -0.1463335156440735, 2.111496686935425, 3.830965995788574, -3.605736255645752, -0.49065154790878296, -3.0898256301879883, -3.525418281555176, 0.5983383655548096, -1.4155986309051514, -0.4437900185585022, 1.526134967803955, -1.3580939769744873, 0.751114010810852, -1.6879312992095947, 5.322624206542969, 2.8656773567199707, -0.12149465829133987, 2.058570384979248, -0.6677097082138062, -1.6312977075576782, 0.2986721396446228, -4.242003440856934, -4.076179027557373, -0.1859036237001419, -0.9210091829299927, -2.27496337890625, -6.029473304748535, 3.700101613998413, -2.5587244033813477, -2.5926218032836914, 0.6030106544494629, -3.719089984893799, -2.313854217529297, -6.363153457641602, -2.7272932529449463, 5.218912601470947, -0.6442379951477051, -4.301462173461914, 2.83382511138916, -1.9816408157348633, -3.2873761653900146, 1.3302333354949951, 0.04891592264175415, -2.3344106674194336, 0.8085917830467224, -1.160805583000183, -5.283316135406494, -0.6009324789047241, 0.024849308654665947, 0.836063027381897, 0.34526723623275757, -3.246480941772461, 0.257158100605011, 5.162746429443359, -8.34419059753418, -1.2316670417785645, -2.933206796646118, 2.1942687034606934, -5.252676010131836, 0.2665426433086395, 2.1984972953796387, 0.371050626039505, 1.003535509109497, 6.856599807739258, 3.292515754699707, 1.7625203132629395, 0.4142138361930847, -2.901212215423584, -2.7023751735687256, -2.778958320617676, 1.9998571872711182, 3.0310816764831543, -2.7212212085723877, 0.4492766261100769, -0.67265385389328, 5.452295303344727, 4.534764289855957, 3.932157039642334, -3.2131619453430176, 0.3356058597564697, -7.800423622131348, 5.558396339416504, -5.382685661315918, 1.4757875204086304, 2.2206127643585205, -1.9633688926696777, -1.0477851629257202, 4.308704853057861, -1.4888529777526855, 2.004300594329834, -1.1848020553588867, -2.641068458557129, 0.6873964667320251, 1.0689735412597656, 3.861457586288452, 0.19390565156936646, 1.3984284400939941, 4.125020503997803, -2.8585734367370605, -4.594358921051025, -1.240701675415039, -3.1279067993164062, -0.4146713614463806, -0.007634363137185574, 4.758804798126221, -1.229050636291504, -3.732818603515625, -3.116013765335083, 2.6687545776367188, 1.9494757652282715, 3.2363076210021973, -0.23688729107379913, -3.0024161338806152, 5.86747932434082, 1.772542119026184, 1.420390009880066, -3.309121608734131, -5.813407897949219, 5.290830612182617, 0.9837058782577515, -0.8146181106567383, -7.162788391113281, -4.557279109954834, -2.180039644241333, 3.2091188430786133, -1.3121191263198853, 4.483776092529297, -1.722707986831665, 2.198317050933838, -1.4269325733184814, -6.288517951965332, 3.335693836212158, 6.523220062255859, -4.181334495544434, 2.287153720855713, 0.3977671265602112, -2.0624539852142334, 3.0889039039611816, -1.4869296550750732, -0.37695398926734924, -1.50844144821167, 1.5013461112976074, 3.1519527435302734, 2.6472630500793457, -1.9726297855377197, -2.2108001708984375, -4.609306335449219, 1.6212079524993896, -2.1587510108947754, -0.16001173853874207, 1.433942437171936, -4.793291091918945, -3.290747880935669, -1.1323827505111694, 3.087186098098755, 0.18147006630897522, 1.7880444526672363, -4.658188819885254, 2.6800599098205566, -1.362246036529541, -0.7493804097175598, -3.199125051498413, 1.8444825410842896, 1.0893616676330566, -3.4475302696228027, -1.1101524829864502, 5.28966760635376, 1.031113862991333, -0.4261079430580139, -0.6271727085113525, 1.9451026916503906, 0.2990851402282715, -0.6859655380249023, -0.7649940848350525, -1.335649013519287, 1.0508885383605957, 3.651930809020996, -0.6390277743339539, -1.3932297229766846, -0.09987205266952515, -3.091264247894287, -4.285985469818115, 0.3097335398197174, 4.190403938293457, -0.4865438640117645, 1.3966573476791382, -1.1162281036376953, 3.0324089527130127, -0.4367930293083191, -0.30516284704208374, -3.6282835006713867, -0.3445088863372803, 2.9692955017089844, 2.360698699951172, 0.5839757323265076, 2.47629976272583, 1.1283648014068604, 1.303024411201477, 3.3793883323669434, -4.159590244293213, 6.217310428619385, 2.081192970275879, 0.9550381898880005, -1.6086335182189941, 0.9681553840637207, -2.8756895065307617, -2.431567907333374, -4.351502418518066, -1.0868308544158936, 0.20649057626724243, -3.472510814666748, -4.081151008605957, 1.4887981414794922, -3.3163185119628906, -2.243316173553467, -0.3992522954940796, 0.9272397756576538, 0.9392309188842773, 0.19932010769844055, -2.23435640335083, -2.614874839782715, 2.0463991165161133, 2.349437952041626, -0.28648895025253296, -1.341861605644226, -0.03587505593895912, -0.14840324223041534, 2.5036582946777344, -0.11240661144256592, 3.5552353858947754, 1.8973548412322998, -0.9744912385940552, 0.5865656137466431, 1.6944901943206787, 1.6835668087005615, 3.089566707611084, 1.1885417699813843, -0.5126318335533142, 4.307356834411621, -3.4718828201293945, 2.2567288875579834, 7.3235392570495605, 4.40925931930542, -1.7408756017684937, -8.68025016784668, -3.5911498069763184, 2.7873964309692383, 0.7423055768013, 2.089402437210083, 0.8180354833602905, -1.525970220565796, 1.7104865312576294, 0.35954970121383667, 0.26457837224006653, 1.9188668727874756, -3.2273919582366943, -1.4181982278823853, -5.5179853439331055, 0.409360408782959, -2.5833544731140137, 1.050058126449585, -0.4396645426750183, 0.7063193917274475, -0.9047918915748596, 1.808081030845642, -1.3567136526107788, -0.9485880732536316, 3.9684314727783203, -1.240340232849121, 4.267378330230713, -1.3120702505111694, 0.9274985790252686, 0.4776534140110016, 4.1764912605285645, -3.0674548149108887, 1.3449628353118896, -1.190827488899231, -0.8568332195281982, -2.3882269859313965, 1.4803645610809326, 0.7541810274124146, 4.353711128234863, -4.251888275146484, -0.08920882642269135, 4.627342700958252, -0.13586550951004028, 1.1481866836547852, -0.0775107890367508, 2.1113650798797607, 4.345449447631836, -1.0218384265899658, -5.222556114196777, -0.32105696201324463, -5.396965026855469, 2.787593364715576, -4.171292304992676, 1.0418527126312256, -1.5622426271438599, 0.0360250398516655, -1.616255283355713, 1.9203100204467773, -2.0779736042022705, -1.995815634727478, 1.516472578048706, -3.368046998977661, 0.6012510061264038, -6.744905948638916, 3.1112475395202637, -0.46004849672317505, 0.6115672588348389, -2.31400203704834, 2.454867362976074, 6.411965847015381, -1.0149959325790405, 1.842604398727417, -1.4725769758224487, -3.617082118988037, -1.7921223640441895, 1.8690073490142822, -1.5385282039642334, 5.197936534881592, 0.7933961749076843, -0.6124565601348877, 1.065006971359253, 1.1582127809524536, 4.189826965332031, -0.5120058059692383, -0.3252975344657898, -0.36538422107696533, -7.164586067199707, -3.7257046699523926, 2.0961172580718994, 1.1068806648254395, 2.955893039703369, -2.5392749309539795, 5.184721946716309, 2.632107734680176, 4.970306873321533, 1.4928737878799438, 2.114863395690918, -1.9053375720977783, 1.77372145652771, -1.8153364658355713, -0.1583649218082428, 2.3466434478759766, 3.6039719581604004, 4.6672258377075195, 0.3893149495124817, -4.369451999664307, 5.24024772644043, 6.591635227203369, 1.68020761013031, 3.903762102127075, -6.929872512817383, 2.6701648235321045, -0.9437963962554932, -0.9035890102386475, -5.18891716003418, 2.104370594024658, -3.3980445861816406, -5.688624858856201, 0.5665562152862549, 0.2817753553390503, 1.1495835781097412, -1.5671672821044922, -0.38515564799308777, -1.8782894611358643, -3.950373649597168, 3.559154748916626, 3.816307544708252, -4.663271903991699, 3.763343095779419, 1.614086627960205, 3.6564908027648926, -0.6352320909500122, 2.2871320247650146, -2.485624074935913, 0.051482364535331726, 5.177925109863281, 2.8350002765655518, 0.37720584869384766, 2.2260830402374268, 3.1808159351348877, -4.591560363769531, 1.2393122911453247, 0.08768224716186523, -0.33101803064346313, 2.875040292739868, -1.3100162744522095, 1.1218137741088867, -3.1160106658935547, -0.22597110271453857, 1.7017548084259033, 0.6845545768737793, -1.6617162227630615, -0.8140099048614502, 1.768390417098999, 1.207364797592163, -4.629277229309082, -0.3739480674266815, 1.991931676864624, 0.2678249776363373, -0.30402952432632446, -1.9248864650726318, -2.968717336654663, 2.2878551483154297, -1.117958903312683, 0.27021294832229614, 1.4603979587554932, 5.037272930145264, -0.11378087103366852, 3.9962759017944336, -1.4361371994018555, -1.369200587272644, 0.017862603068351746, -0.6137192845344543, 2.389538526535034, -5.9513959884643555, -1.6013299226760864, -0.5011719465255737, -3.4804530143737793, -0.15004634857177734, 2.9392640590667725, -0.055051833391189575, 6.1120147705078125, -3.296733856201172, 1.4164005517959595, 1.045061707496643, -1.2759555578231812, -0.35735028982162476, 0.3740556240081787, -0.5310816168785095, 1.7300430536270142, -1.763221263885498, 6.190748691558838, 2.8505096435546875, 4.496069431304932, -0.2836501896381378, -1.7360626459121704, 5.190120220184326, -1.8081626892089844, -3.935121774673462, 3.3714942932128906, 2.928109884262085, 0.4373515844345093, 4.532959461212158, -0.6345452070236206, 5.259764194488525, 4.840060234069824, 1.0252786874771118, -2.158193349838257, 6.751898288726807, 2.949249267578125, -3.6882169246673584, -7.3094682693481445, -2.9378609657287598, -0.7804539203643799, -1.784590721130371, 1.920112133026123, -1.6941375732421875, -1.6032546758651733, 0.5144718885421753, -0.45846444368362427, -3.363194704055786, 0.5360445380210876, -1.7760095596313477, -7.821960926055908, 2.3812966346740723, -5.670375347137451, -1.8453902006149292, -5.440881252288818, -0.33520832657814026, -5.845885276794434, -2.1781930923461914, -4.264289379119873, -1.1246721744537354, 5.295194625854492, -1.4436966180801392, 1.8732471466064453, 0.4442213773727417, 1.5429916381835938, -2.970118522644043, 0.18523070216178894, -1.6910909414291382, -0.028894614428281784, -1.260137677192688, -1.796919822692871, -1.0049669742584229, -0.4781060814857483, -3.2027883529663086, 2.3554749488830566, -4.395055294036865, 0.5788952112197876, -0.6258431673049927, -1.034066081047058, 1.3266533613204956, -0.27946358919143677, -6.551081657409668, -0.45941582322120667, 2.8742260932922363, -1.0569889545440674, -2.4570059776306152, -8.647480010986328, 2.2863378524780273, -0.46949636936187744, 1.7892706394195557, -3.426265239715576, -1.2696099281311035, -5.817068576812744, -1.7591571807861328, 2.010891914367676, -2.4507174491882324, 2.8458776473999023, 3.2793867588043213, -0.14542236924171448, -3.794947385787964, -0.8204265832901001, -0.2360675036907196, 1.7414556741714478, 1.530428171157837, -0.49970656633377075, -4.964061737060547, 1.0719046592712402, -2.9403738975524902, -3.084655523300171, -1.2880139350891113, 1.7874587774276733, 2.413517713546753, 6.164538383483887, -0.7458574771881104, 3.640665292739868, -6.381967544555664, 4.48615026473999, 4.0843658447265625, -1.0909500122070312, -1.282394289970398, 1.0938639640808105, 2.3305585384368896, -1.410487413406372, -0.7635725736618042, 4.94531774520874, -1.8636492490768433, 1.8233897686004639, 0.7734583616256714, -0.986915111541748, 2.1939480304718018, 0.15214455127716064, -0.0008204147452488542, -3.7815113067626953, -0.3375421166419983, 4.603438854217529, -0.9304724931716919, -1.0815210342407227, -1.2850244045257568, 1.9823544025421143, 2.729631185531616, -0.3660590648651123, -5.8208723068237305, 4.281782150268555, 0.28459039330482483, 2.3974437713623047, -0.6112656593322754, 3.1444692611694336, 1.3405417203903198, 1.4067732095718384, -3.2166361808776855, -0.6885925531387329, 1.4414039850234985, 4.299591541290283, 2.467928647994995, 5.151793479919434, -3.1777472496032715, -1.6924556493759155, 0.08394668996334076, 2.809901475906372, -0.039734020829200745, -1.1009070873260498, -1.6263368129730225, -1.2009012699127197, 4.843509674072266, 1.359984040260315, -1.6871089935302734, 0.5529801249504089, 2.926833152770996, -0.4738208055496216, -4.651474475860596, -2.152128219604492, 0.3006174564361572, -4.128735542297363, -2.3600990772247314, 1.173038125038147, 3.6563305854797363, -0.20744699239730835, -5.58577823638916, 6.15495491027832, -2.7152504920959473, 1.4234659671783447, -5.433504581451416, -1.6479281187057495, 2.51647686958313, -1.573612928390503, 4.359508514404297, -1.4727598428726196, -1.9431262016296387, 1.4362659454345703, 0.85789954662323, -0.8840410709381104, -3.0374884605407715, 0.6595157384872437, -3.094313144683838, -3.0494885444641113, 2.2770912647247314, -0.8828822374343872, 6.267126083374023, 4.303489685058594, 2.53314208984375, 0.6661105155944824, 0.3876366913318634, -0.30406779050827026, 2.4244308471679688, 4.788495063781738, -2.954289197921753, -0.46834075450897217, -3.1531283855438232, 0.9032754898071289, 1.4188865423202515, 3.409005641937256, 0.6231995820999146, 1.217804193496704, -6.941370010375977, -0.07713921368122101, 4.960679531097412, 1.0382826328277588, -10.070361137390137, -0.45280832052230835, -7.397595405578613, 0.05124109983444214, 1.9496705532073975, -3.2168397903442383, 8.434707641601562, 4.8574371337890625, -5.480832099914551, 1.548809289932251, -5.0371479988098145, 0.3504409193992615, -1.1690280437469482, 1.9215502738952637, 1.08770751953125, 0.9015296697616577, -4.631285190582275, -4.688904285430908, 4.117983818054199, -2.8663554191589355, 1.4160270690917969, -2.7070159912109375, 3.2283644676208496, 2.241082191467285, -1.4822429418563843, -0.24468545615673065, -4.891532897949219, 3.6469149589538574, -1.6141295433044434, 2.375345230102539, 3.922398567199707, 2.653120756149292, -1.4769597053527832, 1.4309449195861816, -0.2042158842086792, -7.030557632446289, -0.7542332410812378, 1.2804640531539917, 2.5483758449554443, 1.4356964826583862, 0.2528609335422516, -0.14714482426643372, -1.975765585899353, -4.6522698402404785, -1.6226873397827148, 0.7188512682914734, 0.4646109342575073, -0.02970302850008011, 1.3546836376190186, -0.2656092643737793, -3.029386281967163, 3.1103343963623047, 1.2545872926712036, -1.6302664279937744, -2.5934548377990723, 2.0755228996276855, 0.43670251965522766, 3.935089588165283, 0.47260552644729614, -3.9000539779663086, -2.324647903442383, 2.372537851333618, -4.27140998840332, -1.261588215827942, -0.6909962892532349, 0.3926904797554016, 2.1541216373443604, 1.6586480140686035, -0.8024888634681702, 3.427042007446289, -1.0980236530303955, 1.287888526916504, -9.387438774108887, -1.2634563446044922, 1.2417736053466797, -3.9456145763397217, -0.8395966291427612, -1.6954820156097412, 4.633539199829102, -2.21130633354187, 5.049673080444336, 0.722065806388855, -0.8019677996635437, -0.09452942758798599, -3.091341972351074, 3.823265552520752, 2.842061758041382, 0.23707342147827148, -0.7248644828796387, -5.9200286865234375, -4.567147254943848, 1.1317181587219238, 1.3752069473266602, 2.2604620456695557, -2.244593620300293, -2.8570542335510254, -3.4369118213653564, -1.5298093557357788, 0.43716365098953247, -1.7189844846725464, 3.937864303588867, -5.023613929748535, -1.298477292060852, -0.06280863285064697, 3.598719835281372, -0.33691444993019104, 1.9196240901947021, 1.9935388565063477, 3.320700168609619, 4.0258989334106445, -0.1740884780883789, -3.840305805206299, 2.695064067840576, -4.339704990386963, -1.485323429107666, -2.7435412406921387, -1.5894711017608643, -1.23291015625, -4.619691848754883, -2.645524740219116, 6.420977592468262, -1.8035763502120972, 5.24291467666626, 0.00838441215455532, -0.1335981786251068, -8.293523788452148, 3.9721994400024414, -3.6069610118865967, 1.775423526763916, 2.610525608062744, -0.4574401080608368, -1.5947988033294678, -3.2044224739074707, -3.908308982849121, 0.644971489906311, -0.07161872088909149, -1.6754326820373535, 3.462021589279175, -0.47667181491851807, -6.297861099243164, 0.1281246393918991, 0.10801193118095398, -2.849287748336792, -0.423458456993103, 1.8048458099365234, -0.44249945878982544, -0.8619253635406494, -0.9816962480545044, -1.4792248010635376, -2.6521127223968506, -0.7078220248222351, -4.638034343719482, -5.003559112548828, -2.5938661098480225, 3.9717297554016113, -0.822986364364624, -0.5726234912872314, 1.6535229682922363, -1.4019627571105957, -1.4593214988708496, 5.175867080688477, -0.8047431707382202, 3.842409133911133, -1.792708158493042, 1.751136064529419, 3.8942627906799316, -1.6670976877212524, -0.5500453114509583, 4.549253940582275, -0.8651116490364075, 0.36986270546913147, 0.8894078731536865, -2.591348648071289, -5.825340747833252, -4.433450698852539, -1.0176246166229248, 0.7520992755889893, -2.0770795345306396, -2.947237968444824, -0.32342392206192017, 0.2800031006336212, -0.06499919295310974, -0.530180037021637, 2.544875144958496, -1.154246211051941, 0.6574245095252991, -0.25246354937553406, -0.8819652795791626, -1.3826525211334229, -8.689151763916016, -3.777831554412842, 1.6374926567077637, 1.3297693729400635, 1.7773230075836182, -1.8377082347869873, 1.7385401725769043, -0.14915813505649567, 0.5243619680404663, 0.6782090067863464, -0.7272167205810547, -3.41552996635437, 1.8281598091125488, -5.287476539611816, 1.2440847158432007, 4.647741794586182, 1.5210410356521606, -0.26439574360847473, -1.0569331645965576, 5.534920692443848, 0.22923246026039124, -0.26973938941955566, 3.821458339691162, 0.8195902705192566, 1.6087372303009033, 6.188223838806152, -5.729789733886719, 1.1320335865020752, -2.128556489944458, 1.3011056184768677, 1.589037537574768, 0.21763509511947632, 1.2694779634475708, 2.5328011512756348, 1.5486514568328857, -1.9684092998504639, 0.36598849296569824, 0.018299713730812073, -1.250718593597412, -2.2092843055725098, 1.4828243255615234, -1.630240559577942, 0.5805433988571167, 1.7292654514312744, 4.643853187561035, 3.056021213531494, 2.5864713191986084, 0.8684919476509094, 1.6416118144989014, -4.086210250854492, 1.7174710035324097, 1.6440894603729248, 1.8046820163726807, -2.784060001373291, -2.9311985969543457, -3.598341941833496, 2.239372968673706, 0.5131162405014038, -0.5775611996650696, 0.7139550447463989, -0.1970001608133316, -3.7149834632873535, 1.0535454750061035, 0.5721906423568726, 1.5826027393341064, 2.174980401992798, 1.3093942403793335, 2.0414133071899414, -4.407952785491943, -0.09669919312000275, -2.338557243347168, -0.9631457328796387, -0.8217621445655823, -5.459375858306885, 0.1873292773962021, 1.514114499092102, -1.2646188735961914, 0.6369204521179199, 1.4011790752410889, -0.5956724286079407, -3.552340507507324, 0.4246739149093628, -3.0963659286499023, -1.7933690547943115, -2.5273818969726562, -2.7920684814453125, 0.9089174866676331, -0.3086261451244354, 2.273158311843872, -2.637040615081787, 0.3821306526660919, -2.6707160472869873, -3.6374895572662354, -1.5422279834747314, 2.1721129417419434, 1.9278528690338135, -1.0220487117767334, -3.015900135040283, -0.536553144454956, -0.31558629870414734, -3.6419715881347656, -1.1266734600067139, 3.511765956878662, -4.287595272064209, -0.7941293716430664, -8.084348678588867, -3.381998062133789, 5.548485279083252, 2.1261026859283447, -2.580319881439209, 0.6086370944976807, -0.6451319456100464, -5.288032531738281, 0.38130319118499756, 3.2098388671875, 6.710241317749023, 0.7513635158538818, 2.6841111183166504, -1.2558658123016357, 3.6816906929016113, -0.12199608981609344, -1.9889042377471924, 0.9735751748085022, 1.5367798805236816, 2.62746524810791, 2.9997007846832275, -1.6497491598129272, 1.0851527452468872, 2.9201579093933105, 3.201383590698242, -2.5920143127441406, 2.277909755706787, -2.670135259628296, 2.1540563106536865, 1.911090612411499, -0.9021559953689575, 5.222128868103027, 2.1821699142456055, -5.28957462310791, 1.5807485580444336, -0.29467618465423584, 4.55190372467041, 3.4820287227630615, -0.7876863479614258, 0.893708348274231, 0.4945288300514221, -2.969923734664917, 0.7436344027519226, 0.483464777469635, -1.0239773988723755, 0.6693500876426697, 0.3955467939376831, 0.08961671590805054, 1.4629831314086914, -1.5605103969573975, -0.9756474494934082, -0.9707148671150208, -0.07169124484062195, -1.284364938735962, -1.765871524810791, 0.6613808274269104, 2.9482779502868652, 6.624441146850586, -3.2151083946228027, 4.874650478363037, -1.3783414363861084, 3.7046055793762207, 0.662752628326416, 5.210861682891846, -2.1648106575012207, 1.2567002773284912, 8.832245826721191, -1.4763600826263428, -1.6967334747314453, -2.378889799118042, -2.1672089099884033, 3.484750986099243, 1.9351190328598022, -2.958090305328369, 0.7934160232543945, 2.402890682220459, 2.758715867996216, -1.0533545017242432, -3.759974479675293, -1.3415093421936035, -6.337499141693115, 0.4942357540130615, 1.2430986166000366, -2.760369062423706, -0.8603284358978271, 3.153285503387451, 1.7037932872772217, 0.9626110792160034, 2.0061538219451904, -3.6086416244506836, 2.2311840057373047, 0.9241046905517578, -0.7757257223129272, -0.10495063662528992, 0.042835015803575516, 2.595757007598877, -2.0708580017089844, 0.5768824815750122, 0.62717205286026, 0.9984049201011658, -0.25454315543174744, -1.024520754814148, -0.7622140645980835, -0.4002121090888977, -2.0890727043151855, 0.756509006023407, 3.9468908309936523, 7.111813545227051, -1.3683828115463257, -0.6304137706756592, 3.0841493606567383, -3.8906972408294678, 1.611628532409668, -7.224450588226318, 5.08499813079834, 5.944465160369873, -1.5824986696243286, -1.0503909587860107, 2.371891975402832, 3.8548707962036133, -0.35445690155029297, -6.0961198806762695, -0.3129822909832001, 3.422088623046875, -1.3526736497879028, 1.2626187801361084, -2.000110387802124, -1.8141262531280518, 1.2362767457962036, -2.511058807373047, 1.576766014099121, 1.8235647678375244, 6.312005043029785, -5.984403610229492, 1.224061369895935, -5.208196640014648, -5.320728302001953, -4.8387346267700195, 2.157528877258301, 2.160421848297119, -1.7423083782196045, 5.972433567047119, 1.5265345573425293, -4.577849388122559, 2.8998401165008545, 3.1132001876831055, 2.905306577682495, 0.0931682214140892, -3.2770004272460938, 1.135009527206421, -2.2007510662078857, -5.438987731933594, 5.9912896156311035, -6.207529067993164, 0.4382322430610657, 3.338057041168213, 0.24006721377372742, -4.194151401519775, 0.8475056290626526, -1.9167907238006592, 4.265694618225098, 2.9784979820251465, -6.114950180053711, -7.195879936218262, -3.1112923622131348, 3.691573143005371, 2.37166690826416, 0.5804531574249268, -2.4496984481811523, -3.8230137825012207, 2.7154393196105957, -3.0601511001586914, -1.1331478357315063, -0.9751445055007935, 0.37138551473617554, -1.7595932483673096, 3.888441324234009, 3.3914852142333984, 3.749880790710449, -2.4782114028930664, -0.1326751708984375, 2.237384080886841, 0.4874427616596222, -1.1934797763824463, -4.527040481567383, -2.2632460594177246, -3.474863052368164, -3.017425537109375, -2.6905364990234375, -0.9532356262207031, 2.3480777740478516, 4.786622047424316, -0.09252670407295227, -4.290655612945557, 0.8888148665428162, 2.876542091369629, 1.6054118871688843, -0.9372968673706055, -2.0399394035339355, -1.8567770719528198, 1.8954366445541382, -2.0910422801971436, -2.6305723190307617, 1.5237104892730713, -1.1990431547164917, 0.4636436104774475, -5.633336067199707, -0.40491747856140137, -0.6763091683387756, -0.7468425631523132, -4.469305992126465, 3.946794033050537, -5.337274551391602, -1.5085346698760986, -0.004401385318487883, 3.7985692024230957, 4.776885986328125, 1.6617605686187744, 0.772671103477478, 2.5707788467407227, -0.6073824167251587, -9.276941299438477, -0.8962811231613159, -0.6246391534805298, 2.7152059078216553, 0.6496150493621826, -1.4864052534103394, -2.141542434692383, 3.645775079727173, -2.089858293533325, -3.681818962097168, 2.48569917678833, -0.6819674968719482, -3.729935646057129, 3.7711894512176514, 1.3819090127944946, -4.864361763000488, -1.6566193103790283, -0.5268661975860596, -0.06131385639309883, -3.6963844299316406, 6.118605136871338, 1.60819411277771, 6.349355697631836, 0.2020816057920456, -0.9981807470321655, -1.6532373428344727, 3.312533140182495, -0.38716620206832886, 0.20296746492385864, 0.2900790274143219, -5.430418014526367, 0.3691440224647522, -1.7618426084518433, -4.659296035766602, -0.47250035405158997, 2.994391918182373, 5.100881099700928, -5.836090087890625, -2.4470672607421875, -2.1971118450164795, -3.970916748046875, -0.3698022961616516, -1.9375823736190796, -0.9222405552864075, 2.2166318893432617, -0.4917852580547333, -5.298707485198975, 1.5864660739898682, -1.0138866901397705, -1.4367337226867676, -0.9279043078422546, -1.9261252880096436, 9.645633697509766, 1.5721189975738525, 0.11093452572822571, -3.4789814949035645, -0.03569183498620987, -0.7438163161277771, -5.286199569702148, -0.18231408298015594, 4.376231670379639, 1.700123906135559, 1.0049641132354736, 1.1810117959976196, 2.7430310249328613, 2.5903563499450684, 3.306612491607666, -1.0061042308807373, -1.3947314023971558, 1.2276325225830078, -1.5489954948425293, -3.906322956085205, 1.7885016202926636, -0.06164609268307686, 0.8081390857696533, 1.211775541305542, 1.4748173952102661, -2.5246713161468506, -2.112549066543579, 0.1627439558506012, 2.8067171573638916, -6.0516862869262695, 1.6520764827728271, 0.5546456575393677, -0.23212766647338867, 2.16440749168396, -1.10270357131958, 7.3918256759643555, -0.6343880891799927, -1.558767318725586, -0.8856452703475952, -3.521453619003296, -0.21812567114830017, -4.440456867218018, 4.512195587158203, 1.078325867652893, 2.907459259033203, 0.23572570085525513, 3.7139883041381836, 1.3664801120758057, 2.2719428539276123, 4.008281707763672, 1.4838947057724, -0.8267335295677185, 6.610355377197266, 4.031682014465332, -1.3273506164550781, -0.36608782410621643, 2.083547830581665, 1.4383177757263184, 0.837642252445221, -2.4136905670166016, 1.0469084978103638, -4.778129577636719, 0.2292141616344452, -1.5391169786453247, 0.5164237022399902, -4.428676128387451, -2.172457218170166, 0.3109286427497864, -2.101550340652466, -0.012801988050341606, 0.15527740120887756, -3.4583687782287598, 4.898416519165039, -1.685051679611206, -3.3447794914245605, 1.8952741622924805, 4.477416038513184, 2.5195446014404297, -0.5033484697341919, -0.14729657769203186, 0.9290969967842102, -0.7225227355957031, 1.0365809202194214, 2.428870439529419, -1.4907045364379883, 1.0693551301956177, 6.469999313354492, -4.11623477935791, 2.8034849166870117, 5.255525588989258, -0.1361762285232544, 2.338113307952881, -0.5085877180099487, 2.8318450450897217, 1.946242094039917, 3.59285831451416, 1.0467305183410645, 3.500751256942749, -1.034705400466919, -1.1105459928512573, -2.6571474075317383, -5.806971549987793, 2.611337900161743, -3.4018425941467285, -2.2559401988983154, 2.161442279815674, 2.1111690998077393, -2.427342414855957, 1.102656602859497, -4.4378461837768555, -1.4452077150344849, 1.1794887781143188, 4.260045528411865, 3.8784971237182617, -4.293350696563721, 2.526460647583008, 1.32045316696167, 4.300497055053711, 1.5320180654525757, -5.96137809753418, -1.7453935146331787, -0.6072427034378052, 1.6690263748168945, 0.6777305006980896, -1.7879490852355957, -8.958368301391602, -0.5199792385101318, 5.070709228515625, 1.2359435558319092, 5.397335052490234, -1.5311203002929688, -4.60017728805542, -3.004304885864258, 1.9619269371032715, -6.558733940124512, -1.439744234085083, 5.815102577209473, 4.379607200622559, -1.245822787284851, 0.9472266435623169, 2.503966808319092, -1.999382734298706, -4.98088264465332, 2.9397029876708984, 2.5459351539611816, 0.5430813431739807, -4.887846946716309, 0.09445803612470627, -1.829460620880127, 0.8720181584358215, -3.161900520324707, -2.479684829711914, -1.0537891387939453, -2.162015438079834, 0.9718345999717712, 4.459238052368164, -1.6137197017669678, 0.8867779970169067, 3.014709949493408, -0.473996639251709, 4.3884077072143555, -3.5020744800567627, -0.8518781661987305, -1.3172945976257324, 3.275472640991211, 0.47180047631263733, -2.33705997467041, 5.372976779937744, 1.8704614639282227, -5.104416370391846, 0.8125560879707336, -2.4693264961242676, -4.860507965087891, -0.14877866208553314, -4.839704513549805, 1.4123238325119019, 0.7496060132980347, 1.9465981721878052, -1.9096616506576538, -2.591451406478882, 4.8575825691223145, -2.8118958473205566, 4.680634021759033, 1.3974435329437256, -0.931164026260376, -4.260379791259766, -0.36252227425575256, -0.48790937662124634, -0.8372142314910889, 2.508960723876953, -0.9772270917892456, 2.696700096130371, 0.15073269605636597, -4.7965545654296875, 1.1388168334960938, -0.5620500445365906, 4.708369255065918, 1.3236430883407593, -4.324565887451172, 3.9343652725219727, 3.395665168762207, -2.5038692951202393, 4.631673336029053, -1.2160193920135498, -1.3778600692749023, 2.3605380058288574, -2.658677577972412, 6.292319297790527, -0.7702807188034058, 1.115537405014038, -2.1196985244750977, 0.28814491629600525, -4.217196464538574, 3.1937592029571533, -0.7445422410964966, 0.6916348338127136, 1.0934264659881592, -1.3882405757904053, 1.9001022577285767, 0.260873019695282, -0.7610715627670288, -1.2837883234024048, 5.799747943878174, 4.65476655960083, 0.07438457012176514, -0.7834985256195068, 6.073106288909912, -1.438905954360962, 4.80147647857666, 5.501219749450684, 2.835345983505249, 1.4998589754104614, -3.161409616470337, -3.322340726852417, -1.2600750923156738, 3.069854736328125, -4.748847007751465, -3.887075185775757, 2.087555408477783, 0.3381488621234894, -3.6538474559783936, 5.895984649658203, -0.7840642929077148, -4.87420654296875, -4.099309921264648, -1.311328649520874, -1.1842321157455444, 3.1358470916748047, -2.9298136234283447, 0.5022059679031372, 0.6496495604515076, -1.1212619543075562, 0.22538426518440247, -0.7697380781173706, 4.2037248611450195, -4.027194976806641, -5.296526908874512, -2.055708408355713, -1.4239729642868042, 2.131805896759033, 3.0262136459350586, -3.4369890689849854, -3.29841947555542, -3.48771333694458, -2.0382883548736572, 3.536548614501953, -5.581827640533447, -1.7415471076965332, 1.3908381462097168, 1.5886892080307007, -3.991823673248291, -2.333791971206665, -0.6107505559921265, 2.1560516357421875, -1.9817826747894287, -3.411487340927124, 3.262751579284668, 5.594494342803955, 4.942140579223633, -4.154098033905029, -5.312668800354004, -4.760417461395264, 2.510499954223633, -0.8849550485610962, 2.066145181655884, -1.0527864694595337, -3.9411349296569824, 2.187141180038452, -4.899956226348877, -1.83950936794281, -0.427301287651062, 1.6501295566558838, -0.726889967918396, -1.5733416080474854, 2.9882800579071045, 1.1067698001861572, -4.618295669555664, -1.0626530647277832, 7.123902797698975, -2.658473014831543, 2.674264669418335, -1.4116811752319336, -2.1356167793273926, -1.038902759552002, 0.3608046770095825, -6.436842918395996, -2.8448190689086914, 3.744502544403076, -0.9475938677787781, 1.5008536577224731, -3.734431028366089, 3.714754581451416, -0.28372466564178467, 0.20171771943569183, -0.038749128580093384, 4.5150146484375, -3.260230779647827, 2.693490982055664, 2.123032569885254, 0.641438901424408, 7.309850692749023, 2.549708843231201, 1.268153429031372, 0.3635765314102173, -2.2032690048217773, -0.35846227407455444, -0.2662307322025299, -2.3838109970092773, -2.7967140674591064, 2.1895809173583984, 1.09702730178833, 1.785742163658142, 0.09051861613988876, 2.066699504852295, -4.530113220214844, -2.339538097381592, 4.234448432922363, -0.595356822013855, 2.835897207260132, 3.5339863300323486, 3.183940887451172, 2.1332292556762695, 5.242183208465576, 3.402193307876587, -0.17148232460021973, 2.849838972091675, -4.718613624572754, 2.507974863052368, -0.15029267966747284, -1.5983023643493652, 0.14127828180789948, 0.692438542842865, 0.1512976437807083, -2.9428181648254395, 2.1927051544189453, 6.790855407714844, -1.612842321395874, 1.866052508354187, -2.8168985843658447, -0.1101541817188263, -2.7115678787231445, -3.0146842002868652, -3.940906047821045, -2.1562893390655518, -1.4954463243484497, -0.947737991809845, -7.25161075592041, -2.4951701164245605, 2.4254140853881836, -5.404951572418213, 1.5341455936431885, -6.801540374755859, -4.633033752441406, 2.54121994972229, -0.4669378995895386, 2.771928548812866, 2.305548667907715, 1.3292409181594849, 1.1602336168289185, -1.1535824537277222, 1.3606752157211304, -1.8007861375808716, -0.9345258474349976, -4.3813066482543945, 0.45195263624191284, -1.540487289428711, -1.035152792930603, -2.8398032188415527, -0.8181129693984985, 2.36893367767334, 1.617775797843933, -5.2190752029418945, 2.8090755939483643, 0.9139047861099243, -2.2909750938415527, 0.9101746082305908, 0.3251287341117859, 1.8368693590164185, -2.7491836547851562, 1.1062558889389038, 1.5223610401153564, 5.3107404708862305, -6.938764572143555, 3.661458969116211, 3.1537604331970215, -2.592547655105591, -0.00891395565122366, 2.7440407276153564, -1.8290765285491943, 2.4114091396331787, -1.2146700620651245, 1.216429352760315, -2.1941468715667725, 0.34700146317481995, -2.66904616355896, 0.5531806945800781, -2.5226845741271973, -0.12686589360237122, 2.154772996902466, -1.1028612852096558, 1.9254684448242188, -3.463644027709961, -7.54283332824707, 1.7003703117370605, -0.6273290514945984, 0.1681598573923111, 0.10530209541320801, -1.0560704469680786, -7.1025390625, -2.3749566078186035, -3.4407801628112793, 4.449212074279785, -5.37471342086792, 0.14265701174736023, -5.646739959716797, -6.162075996398926, 1.385337471961975, 3.4174153804779053, -1.1794865131378174, -2.803114891052246, 5.864967346191406, 1.5254520177841187, 0.5269644856452942, -3.5011467933654785, 1.4482288360595703, 1.6622114181518555, -7.253960609436035, 2.0452334880828857, -5.548039436340332, -1.6176373958587646, -4.719675540924072, 0.26543986797332764, 4.439276695251465, 3.0659518241882324, 0.3519613742828369, -1.165851354598999, 1.4815247058868408, -4.822485446929932, 3.1874842643737793, -0.28844785690307617, 0.8624521493911743, 0.4298357367515564, -6.123984336853027, -2.027944326400757, -2.007888078689575, 0.7739152908325195, -5.880787372589111, 6.681262969970703, 4.585963249206543, 3.2895984649658203, 4.5468926429748535, 5.927176475524902, 0.16513411700725555, -0.1384715586900711, 1.3095319271087646, 2.261049747467041, 4.299545764923096, -7.827887535095215, 0.1939525455236435, -2.1665313243865967, 2.291288375854492, -0.2517598569393158, -4.318984508514404, 0.9954759478569031, -2.12058424949646, -0.9294589161872864, -0.6039595603942871, -2.737983465194702, -0.38924646377563477, -0.9111866354942322, 3.5567288398742676, -3.0409865379333496, -2.56192946434021, -1.4137569665908813, 0.07564616203308105, 3.394552707672119, -0.679275631904602, 1.4273180961608887, 2.502131462097168, 0.7997817993164062, -0.879713773727417, -1.6164418458938599, 4.080435276031494, 2.555250644683838, 1.6372615098953247, 4.843878269195557, -2.583580493927002, 3.494056463241577, -0.38397422432899475, -4.906432151794434, -2.7634949684143066, 0.5952578186988831, 3.8912711143493652, -6.279148578643799, 3.3355531692504883, -0.7237699627876282, 0.2679898142814636, 1.576831340789795, -4.58922815322876, 0.7414686679840088, -0.8520035147666931, -0.0300624817609787, 6.707130432128906, 0.8849655389785767, -0.4518962502479553, 0.3363726735115051, 6.761326789855957, 4.657812595367432, 2.853586435317993, 0.3867729902267456, 0.222915381193161, -3.4947760105133057, -4.189114093780518, 3.931473731994629, 2.3820128440856934, 0.4424867033958435, -0.07046429812908173, -3.198140859603882, 3.0805230140686035, 1.076623797416687, 1.0269191265106201, -3.6620121002197266, 5.697139739990234, -0.896999180316925, -2.447213888168335, -0.378510057926178, -0.9411646127700806, 3.0113935470581055, 0.5604882836341858, -2.934664249420166, -4.637887954711914, -5.679287910461426, -2.689237117767334, 0.8268474340438843, -3.6634750366210938, 0.1096903532743454, -4.788392066955566, -0.16949555277824402, -1.070136308670044, -3.41677188873291, 1.5437053442001343, 3.586127758026123, -0.7560727000236511, -0.6218475103378296, -1.5168567895889282, 0.3159138560295105, 1.6317358016967773, 2.8415448665618896, -0.2865455150604248, 6.526796340942383, 9.213939666748047, 7.657547950744629, -6.115492343902588, -4.066313743591309, -1.9971332550048828, 0.3514634966850281, 3.1164326667785645, -1.3105942010879517, -2.3801848888397217, -1.3910061120986938, -3.245201826095581, -2.824974775314331, 0.09865105152130127, -9.580596923828125, 6.111485481262207, 1.4361107349395752, 1.58878755569458, 4.224964618682861, -1.813356876373291, -0.7055575251579285, 0.6052432060241699, -2.519561767578125, 0.13000056147575378, 0.3920761048793793, 1.404686450958252, 3.4052610397338867, 1.893483281135559, -1.5522682666778564, -3.165001630783081, -1.816460371017456, 2.0341248512268066, -6.639839172363281, 0.8384370803833008, -3.4451756477355957, -2.2258777618408203, -0.45870763063430786, -4.590888500213623, 0.3539324998855591, -0.10046426951885223, -0.025934353470802307, -2.8380126953125, 6.861627101898193, 0.21950849890708923, 3.468952178955078, 2.0644986629486084, -3.9146790504455566, 1.1916782855987549, 3.3153157234191895, -2.786290407180786, 2.4768178462982178, 4.725324630737305, -1.7103168964385986, -0.6503553986549377, -1.996305227279663, -1.7460776567459106, 3.65236759185791, -2.8986148834228516, -5.059564113616943, -1.9756336212158203, 5.801868915557861, -0.480030357837677, 0.8286161422729492, 2.234127998352051, 1.6263201236724854, 2.9359278678894043, 5.414802074432373, -0.012976925820112228, 1.4410984516143799, 3.351945400238037, -3.51080060005188, 5.078972339630127, -2.705934524536133, 5.937100410461426, -2.1747286319732666, 2.1949892044067383, -2.9675912857055664, -0.5501344203948975, -1.0849002599716187, -0.7781398296356201, 0.6151518821716309, -1.7192387580871582, -2.169647216796875, 4.149344444274902, -2.9720776081085205, -4.954697608947754, 5.225790977478027, 6.6217803955078125, -1.4883379936218262, 2.510793447494507, -0.19221608340740204, 2.6829752922058105, 0.5974192023277283, -2.3210415840148926, 2.1196017265319824, 7.4555864334106445, -2.3138766288757324, 4.4957733154296875, -6.11478853225708, -1.0803438425064087, 3.335145950317383, 4.057267189025879, -1.6506938934326172, 3.1138148307800293, 0.7275469303131104, 2.458080530166626, 3.82027530670166, -3.220846176147461, -0.7236292958259583, -4.94119930267334, 5.068312644958496, -0.9914754629135132, -2.8415660858154297, -6.121260166168213, -0.03276529908180237, -3.763526439666748, -1.5295610427856445, 4.037024021148682, 0.9282242059707642, 0.10922761261463165, 4.104660987854004, 5.323550701141357, 0.1095728650689125, -2.3948893547058105, -2.350198984146118, -0.43981462717056274, 2.594074010848999, 3.619900703430176, -5.203954219818115, 0.3456474542617798, 1.4483065605163574, 2.7201223373413086, 6.263339996337891, 1.9859156608581543, -0.4118906557559967, -2.9415950775146484, -4.662807464599609, 3.726454734802246, -2.3800976276397705, -1.522261381149292, 2.614164352416992, 1.975606918334961, 0.874121904373169, 3.1481661796569824, -3.6203389167785645, 0.15086835622787476, 1.975696086883545, 0.18814301490783691, -3.4606781005859375, -5.197767734527588, 1.8766340017318726, 1.4174126386642456, -2.502305507659912, -2.8281168937683105, 1.7139239311218262, 1.5246951580047607, -2.4251809120178223, -2.2100303173065186, -0.018433302640914917, -3.151906967163086, 1.4693231582641602, -4.105386257171631, -5.117330551147461, 4.189347267150879, -5.336429595947266, -1.881548523902893, 1.414919376373291, -3.473907709121704, 6.565271377563477, -1.0507280826568604, 1.2416884899139404, -4.088145732879639, -1.3712356090545654, -4.043404579162598, 4.285739898681641, -2.42130708694458, 0.9176686406135559, -1.833383321762085, 1.6525487899780273, 0.5032258033752441, 1.0205310583114624, 7.0109663009643555, -2.2898597717285156, 0.6727561950683594, 0.5498191118240356, 5.393062114715576, -2.316175937652588, 3.6832315921783447, 1.8548104763031006, -0.4527873992919922, 4.502674102783203, -0.03894206881523132, 0.4739799499511719, 1.0752230882644653, -2.7805910110473633, -0.6675429344177246, -0.595906138420105, -1.0175001621246338, -0.02717127650976181, 1.9933393001556396, 1.8751981258392334, -0.582770586013794, 0.5618071556091309, -0.7556484937667847, -3.2263402938842773, -0.3493460416793823, 0.29562416672706604, -1.4521381855010986, -7.18875789642334, 5.172523498535156, -1.5201079845428467, -2.296555519104004, -3.252542495727539, 1.373572826385498, -1.693800687789917, -2.8587019443511963, -2.554332733154297, 1.4721567630767822, -3.4004430770874023, -1.849743366241455, -0.14656558632850647, 0.496565580368042, -1.3804723024368286, -1.6856013536453247, -1.432045817375183, 1.6030622720718384, 3.6851556301116943, -1.7876468896865845, 1.0877859592437744, 1.8919520378112793, -0.006062587723135948, 1.147713541984558, -4.04475212097168, 2.990029811859131, -3.345853805541992, -4.098528861999512, 0.36215871572494507, -0.9920758008956909, 5.85826301574707, 0.23848769068717957, 5.706652641296387, -2.96838641166687, 9.694360733032227, -0.981589674949646, 3.220534324645996, -5.466437339782715, 0.6891636848449707, -3.1853957176208496, -0.015110489912331104, -4.482978343963623, 5.514034271240234, 2.5051751136779785, 4.270695686340332, 3.6940879821777344, -2.1151106357574463, -0.8104453682899475, 1.409859299659729, 7.807215690612793, 1.060456395149231, 1.5703752040863037, -2.8829307556152344, 0.45588046312332153, -2.5777268409729004, 2.594783067703247, 3.1228187084198, 2.2256250381469727, -1.277785301208496, 1.201030969619751, 0.8689096570014954, -3.76050066947937, 4.795969009399414, -2.5070347785949707, -1.4656957387924194, -5.9759979248046875, 0.8506238460540771, -2.311598300933838, 1.68629789352417, 4.402631759643555, 5.746086597442627, -0.04778596758842468, 0.8158296346664429, 5.4117631912231445, -2.4196412563323975, -4.058434963226318, -6.26181697845459, -3.4564290046691895, 0.13600759208202362, 2.3234667778015137, -1.2826435565948486, -0.4428367614746094, 2.7706220149993896, 3.458820343017578, -1.2489986419677734, -4.363785743713379, 0.7240837216377258, 5.974166393280029, 0.28179895877838135, -2.3529229164123535, -3.8622541427612305, -3.0998003482818604, -0.06877449154853821, 0.7962946891784668, -2.381324291229248, -5.1767072677612305, 1.914273738861084, 0.8014640808105469, 1.832436203956604, -5.6871256828308105, -1.9263670444488525, 1.3029813766479492, -1.321473240852356, -3.0921759605407715, 5.7664666175842285, -0.4039433002471924, 0.8173189759254456, -1.6633424758911133, -0.2534920573234558, -3.7877440452575684, 0.990393340587616, -0.858043909072876, 4.389296531677246, -1.3332996368408203, -1.4289484024047852, -2.059619426727295, 3.340196371078491, 2.6663670539855957, 0.003811085131019354, -1.74433171749115, -5.879395484924316, 1.0167579650878906, 0.7267889976501465, 1.1667145490646362, -8.261362075805664, 0.41054439544677734, -0.7967103719711304, -3.279921770095825, 1.4211640357971191, 4.611578464508057, 4.662902355194092, -6.065940856933594, 0.8335585594177246, 4.226919174194336, 1.4189019203186035, -1.9028372764587402, -0.8517864346504211, -0.5133281946182251, 0.5179228186607361, -3.6373257637023926, -5.279638767242432, 2.480794668197632, 1.1976358890533447, -0.5950589776039124, 3.168149948120117, -2.2577223777770996, -1.9340453147888184, 0.36324232816696167, -1.8463172912597656, 2.2377803325653076, 4.342608451843262, -2.348658323287964, 0.5466324090957642, 0.39509332180023193, 1.1593111753463745, -3.6370420455932617, -3.7857162952423096, -3.5122480392456055, 0.9462858438491821, -1.3382138013839722, -0.22867810726165771, -1.7094573974609375, -1.3643970489501953, -0.3630737066268921, 4.368619918823242, 0.49200284481048584, -0.5516804456710815, 0.07166354358196259, 2.9283394813537598, -3.8365039825439453, -3.0415282249450684, -4.1774373054504395, 4.171264171600342, 0.6792226433753967, 0.739141583442688, 4.063889980316162, -0.33638331294059753, 2.099442720413208, 0.7776381969451904, -1.91168212890625, 4.938811302185059, 5.798791885375977, -2.996894598007202, 0.15749365091323853, -4.237866401672363, 5.773591995239258, -0.36655938625335693, -3.6779613494873047, 0.04643537104129791, 2.8832712173461914, 0.7225849628448486, 1.8765335083007812, 0.09885899722576141, 1.8856147527694702, -0.8007153868675232, -1.0662920475006104, -3.419281005859375, -4.321699142456055, 1.7644374370574951, -1.0337469577789307, 1.7640806436538696, 0.3722492754459381, 0.07110606133937836, 0.6608385443687439, -4.436606407165527, 0.6111514568328857, -0.269209086894989, -7.142775535583496, -3.121823787689209, 0.30863526463508606, 2.6974737644195557, -2.0420639514923096, 2.1241297721862793, 3.149789810180664, 2.725864887237549, 1.5728895664215088, -1.4592891931533813, -3.474926233291626, -2.135850667953491, 0.9395797848701477, -8.671024322509766, -0.43074285984039307, -0.5615839958190918, 2.532479763031006, -2.181732654571533, 3.6722025871276855, -1.2536206245422363, 0.24240288138389587, -2.461411476135254, -2.1348769664764404, -2.913651466369629, -2.3925139904022217, -0.2932801842689514, -4.465190887451172, 2.478276252746582, 2.3682916164398193, -1.9889506101608276, -0.7288305759429932, -4.9121904373168945, 0.17612898349761963, 5.207095623016357, -1.121407151222229, 1.8823460340499878, -3.1440789699554443, 9.952299118041992, 0.2949509620666504, 4.947611331939697, -1.8811817169189453, -1.0184690952301025, 4.694687843322754, 0.09453854709863663, -0.7386150360107422, 0.8085249066352844, -5.097832679748535, -0.015366634353995323, -1.502925157546997, -3.841989040374756, 0.7717045545578003, -0.6111900210380554, 0.09573903679847717, -2.6619462966918945, 1.7649425268173218, 1.8010833263397217, 0.40687352418899536, -1.2094461917877197, -7.2668681144714355, -1.9854168891906738, -1.251105546951294, -4.740931510925293, 5.647840976715088, -2.5604467391967773, -1.8837995529174805, -4.155018329620361, -5.816798210144043, -0.7058531045913696, -0.519883930683136, 1.3769841194152832, 1.969125747680664, 1.0945825576782227, -2.27089786529541, 2.0777745246887207, -3.8317270278930664, 5.54641056060791, 0.6710804104804993, 4.582602024078369, 6.4814887046813965, -8.810240745544434, -1.4282995462417603, 1.5039091110229492, -1.2323217391967773, -0.6196039915084839, 2.1890037059783936, 2.198106288909912, 3.016932964324951, -1.1721371412277222, 1.226958990097046, 0.03397176042199135, 0.19310510158538818, 0.4715126156806946, -2.497349977493286, 1.029459834098816, -2.1348068714141846, 3.005095958709717, -0.30161339044570923, 1.250938057899475, -3.3872787952423096, -1.0598658323287964, -5.4444684982299805, -5.419081687927246, 1.128481388092041, -2.265730381011963, -2.4348158836364746, 4.235577583312988, -3.50944447517395, -0.38497745990753174, -2.0408496856689453, 0.6381365060806274, -2.5637176036834717, -5.870695114135742, -4.383051872253418, -0.011874265037477016, 3.1859946250915527, -3.045173168182373, -0.17573675513267517, 0.09798936545848846, -0.7961538434028625, 3.2946643829345703, -0.609978437423706, 3.484137535095215, 4.004799842834473, -2.1526732444763184, -2.7595057487487793, -3.2538833618164062, 1.6074635982513428, -1.8105504512786865, -2.460815191268921, 2.380185604095459, -0.7825589179992676, 8.549161911010742, 1.7678265571594238, 3.6532421112060547, 0.45147112011909485, 2.4037206172943115, 0.2834632396697998, 1.4629610776901245, -0.32364246249198914, 4.215919017791748, 0.9251768589019775, 0.08278574794530869, -2.188396453857422, 3.54783296585083, 1.585723876953125, -3.6468353271484375, 0.5546015501022339, 7.484875679016113, -0.5964143872261047, 0.9704668521881104, 3.3081912994384766, -0.9454860687255859, 2.7095913887023926, 2.5968329906463623, 0.41156113147735596, -3.694265842437744, 4.188513278961182, -2.074481964111328, -3.9744458198547363, -2.234098196029663, 5.576118469238281, -2.136016845703125, -2.609463691711426, 0.0021625435911118984, -1.1611263751983643, -2.6104652881622314, -3.219043254852295, 0.08958999812602997, -1.606551170349121, -3.7312824726104736, 6.1742448806762695, 2.70501971244812, -2.3062610626220703, 1.4096763134002686, 4.087343692779541, 1.0542916059494019, -1.1216180324554443, 3.9381773471832275, 0.33583661913871765, 2.0352835655212402, -1.0212006568908691, 0.18448114395141602, 6.104202747344971, 2.94336199760437, -0.4370381236076355, 2.5110538005828857, 0.9156914949417114, -0.6644483208656311, -0.022731848061084747, -0.40181589126586914, 0.19327190518379211, -0.5854097604751587, 2.001392126083374, -0.12175463140010834, -2.0720388889312744, 1.8974393606185913, 0.5935940146446228, -5.3034563064575195, -2.881985664367676, 1.4932982921600342, 2.466442108154297, 3.5669331550598145, -3.2252979278564453, 3.750889778137207, 1.126231074333191, 0.8501949906349182, -1.636094570159912, 1.925892949104309, -0.629108190536499, -0.03327362984418869, 6.899110794067383, -1.2926288843154907, -0.3532930016517639, -0.6252952218055725, -3.963657855987549, -1.3332713842391968, -1.6460376977920532, -2.690106153488159, 2.6605615615844727, 5.364470481872559, 0.14071817696094513, -0.7773360013961792, 1.5320281982421875, 0.14029709994792938, 3.1550259590148926, -2.771122455596924, -0.7543224096298218, 6.290440559387207, 0.5759705305099487, 4.050841331481934, 5.071614742279053, 0.9372007846832275, -1.191939353942871, -1.7412972450256348, -8.86584758758545, 2.058248996734619, -3.2907333374023438, 6.194933891296387, -0.09976184368133545, -10.652351379394531, -0.21867656707763672, -3.2557625770568848, 1.597204327583313, -0.6118672490119934, 0.5083144903182983, -1.5784521102905273, -1.9396709203720093, -5.232915878295898, -4.324095249176025, -5.408143043518066, 1.3642401695251465, 2.9177606105804443, 0.7650154829025269, -0.9781578779220581, -1.7067716121673584, -3.30574893951416, 3.197093963623047, -8.513440132141113, -0.4454769194126129, -4.972431182861328, 4.453258991241455, -4.108213424682617, -3.834404945373535, 0.010359330102801323, 2.815598964691162, 0.12731514871120453, -1.8833813667297363, -3.0027716159820557, -3.3040952682495117, -3.193389892578125, -0.19621042907238007, 1.8741247653961182, -1.7862200736999512, -2.1989076137542725, -0.30100950598716736, -3.3756890296936035, -3.4760494232177734, 4.263794898986816, -1.5929471254348755, 3.3834943771362305, 2.113621234893799, -3.8291308879852295, 4.4954681396484375, 0.6512438058853149, 1.7560994625091553, 3.4300410747528076, -1.85519278049469, -1.9417290687561035, -3.639721155166626, 1.5078989267349243, -8.106422424316406, 0.08213702589273453, -0.22695767879486084, -1.042771816253662, -2.0143113136291504, -3.8929731845855713, 0.22573497891426086, 1.9612541198730469, -0.8482648730278015, 6.096590995788574, 1.4919811487197876, 0.07605697959661484, 1.0117151737213135, -1.2370383739471436, -2.219600200653076, -1.792917251586914, 1.3180474042892456, 3.0476510524749756, 0.7503873109817505, 3.3410820960998535, -1.4794268608093262, -1.498473882675171, -2.4321703910827637, 1.3967205286026, -4.154445648193359, 0.6925358772277832, 0.30396997928619385, -1.1930171251296997, 0.520946204662323, 1.748859167098999, 2.8875601291656494, -4.312161445617676, 1.192697286605835, -0.864578127861023, -4.036286354064941, -1.9192595481872559, -0.9358330965042114, -1.5192300081253052, -6.346042156219482, -3.859602451324463, -1.1100965738296509, 0.5664860606193542, 2.9112815856933594, -3.8844213485717773, 3.717430830001831, -0.765321671962738, 1.3806976079940796, 1.2494304180145264, 1.3368332386016846, -4.8412909507751465, -0.8942098617553711, 0.964800238609314, -0.026679422706365585, 3.78893780708313, 3.7920398712158203, -0.6851009130477905, 2.858530282974243, -2.591825485229492, -6.014206409454346, 2.1020426750183105, 3.0825090408325195, 3.925713062286377, 1.860701560974121, -2.5124592781066895, 1.829451560974121, -2.5459227561950684, 3.3525147438049316, 1.7150477170944214, -1.2045291662216187, 4.4560136795043945, 0.7625192999839783, 2.473498582839966, -4.567268371582031, -0.12651580572128296, 1.279308795928955, 3.8918228149414062, 2.037508010864258, -0.17125612497329712, -1.146479606628418, -2.845710515975952, 1.1732177734375, 6.0075578689575195, -1.429671287536621, 0.11304941773414612, -5.13748836517334, 0.7789304852485657, -3.9480972290039062, -3.023390769958496, -1.0427758693695068, -5.399559020996094, 3.46783447265625, 5.166165351867676, 0.5794001817703247, 1.4199875593185425, -1.846756935119629, 4.744424819946289, 2.7385215759277344, -4.6416521072387695, -2.9345779418945312, 0.41973626613616943, 6.338825225830078, 1.3058708906173706, -3.9611287117004395, -2.5608131885528564, 4.635225772857666, 4.387650966644287, 2.88611102104187, -0.5748227834701538, 0.7322008609771729, -0.11782827973365784, -1.381299614906311, -5.142638683319092, -6.908048629760742, 2.5544087886810303, 1.1169681549072266, 3.409214973449707, 0.15420284867286682, 3.7236328125, -3.1159701347351074, 3.1756174564361572, -2.1266539096832275, 0.8322348594665527, 2.086555004119873, -0.007296137511730194, 2.5840091705322266, -0.8645419478416443, 5.118927955627441, 1.8256241083145142, -0.5753027200698853, 7.061765670776367, -1.7127437591552734, 0.9408987164497375, 3.5012145042419434, 1.3838000297546387, -2.21610426902771, 2.7766165733337402, -6.254639625549316, -1.4680004119873047, 1.6498916149139404, 0.5246229767799377, -1.6106786727905273, -0.5865719318389893, 0.40982645750045776, 0.6478426456451416, 1.6420801877975464, -2.7206966876983643, -3.2903032302856445, 4.4887542724609375, -2.3160746097564697, -3.159640312194824, 1.1162163019180298, 3.909710168838501, -1.0220781564712524, 3.6048479080200195, 1.7489292621612549, 3.027906894683838, 2.150582790374756, -2.966496467590332, -0.6526979804039001, -0.33538690209388733, 5.005347728729248, 3.908963441848755, 2.982433795928955, -0.22072294354438782, -0.40606510639190674, -5.1098551750183105, 4.010977745056152, -0.8324000835418701, -0.19675080478191376, -3.7845892906188965, -0.793397068977356, 8.64443302154541, 3.302004337310791, 4.173025131225586, 1.2856204509735107, -0.43645864725112915, -6.645902633666992, -1.8071866035461426, -2.8330559730529785, -0.29053637385368347, 1.447073221206665, 3.0563573837280273, 1.7317109107971191, 0.8727030158042908, 2.1478731632232666, 1.5794216394424438, 0.0886286124587059, 3.072225570678711, -4.727729320526123, 9.534643173217773, -3.73313045501709, 1.550997257232666, 4.248488903045654, -0.2276425063610077, 2.7345681190490723, -1.456899881362915, 0.11528107523918152, 6.513790130615234, -3.2022204399108887, -1.3892974853515625, 2.576327323913574, 6.01356315612793, 1.4037139415740967, -3.0112929344177246, -2.580995559692383, 3.5483736991882324, -3.3137834072113037, -0.39217090606689453, 0.6474626660346985, 3.577631950378418, 4.3360161781311035, -1.715072751045227, -2.192084550857544, 7.995051383972168, 1.6601731777191162, -1.5333747863769531, 4.778877258300781, 3.0710673332214355, 0.6086263656616211, -2.6441102027893066, -7.027989864349365, 2.355788469314575, 2.056391954421997, 2.2005183696746826, -2.165832042694092, 0.909568727016449, 1.8297454118728638, 0.798373281955719, -5.1094183921813965, -3.510080337524414, -0.20465697348117828, 0.6071234941482544, 4.312413215637207, -0.7757640480995178, -3.6774215698242188, -3.5061049461364746, 3.0927839279174805, -3.304197311401367, -0.5166633129119873, -3.7987613677978516, 1.4862754344940186, 2.9792325496673584, -4.546021461486816, -3.4933691024780273, 6.664407730102539, 2.4654979705810547, -3.4899580478668213, -1.1061062812805176, 3.9041147232055664, 1.1346421241760254, 3.920473575592041, 1.4288839101791382, 3.9895169734954834, 1.2551774978637695, -0.8840521574020386, 0.9597891569137573, 0.11389903724193573, -0.5895940065383911, -0.0006511477404274046, -1.0201340913772583, 3.3198938369750977, -1.3020638227462769, -0.5099359750747681, -0.41013967990875244, 0.2345511019229889, -2.074507236480713, 2.635800838470459, 0.53067547082901, 1.265788197517395, -5.515835285186768, -0.06059139221906662, -0.2131979763507843, -2.1087088584899902, 2.2029974460601807, -0.8543428182601929, 0.5840020775794983, -1.0268559455871582, 1.42451810836792, 1.8208197355270386, 3.902097702026367, 3.6966209411621094, 3.078306198120117, -1.561903476715088, -1.2903459072113037, 2.6034398078918457, 0.12273100018501282, -2.074523687362671, 1.2687745094299316, 1.5562572479248047, 4.689216136932373, -0.7615215182304382, -1.4787352085113525, -3.8155665397644043, -0.2729666829109192, -2.5959529876708984, -2.2992682456970215, 2.2031288146972656, 3.4078481197357178, 1.208809733390808, 1.2996388673782349, -2.800598382949829, 3.0602681636810303, -3.5697665214538574, 1.7771310806274414, -0.5120818614959717, -2.43070125579834, -4.826549530029297, 3.3046045303344727, 5.849968910217285, 3.801187038421631, -4.428194999694824, 0.7016980051994324, -2.8183040618896484, 6.4404497146606445, -3.2347283363342285, -1.8005733489990234, -4.681722640991211, -1.7926445007324219, -2.3249740600585938, 3.298621416091919, 0.333379328250885, 1.3040308952331543, -1.466141700744629, 0.8708561658859253, 0.718159556388855, 2.6821439266204834, 3.214510202407837, 1.4733774662017822, 3.295391321182251, 2.5403339862823486, 5.536852836608887, 2.0877394676208496, -5.696861267089844, -0.5904927253723145, -2.4340062141418457, 0.6316958069801331, 2.289609670639038, -1.4984756708145142, 5.761686325073242, -2.203592300415039, 3.165606737136841, -1.5811119079589844, -1.8855068683624268, -0.28610843420028687, 3.0016472339630127, -0.6104274392127991, -2.8428616523742676, -6.852743148803711, 0.48514604568481445, 5.975762844085693, -1.308730125427246, -0.25320255756378174, -0.045764412730932236, -2.199587821960449, 4.222761631011963, 2.6546995639801025, 2.78255558013916, -6.202995300292969, 2.286619186401367, 2.156919002532959, 3.4221529960632324, 6.561108112335205, -3.525041341781616, -0.09933869540691376, -1.6277880668640137, 1.4002243280410767, -1.979548454284668, 4.842837333679199, -1.1361896991729736, -2.602205276489258, -4.317489147186279, 0.7420351505279541, 0.7082493901252747, 2.964263916015625, -2.451110363006592, 1.2393152713775635, 4.479050636291504, -3.505331516265869, 0.2551974952220917, 1.8179402351379395, -0.6498321294784546, 2.9925811290740967, 2.0677261352539062, -0.7020738124847412, 2.4634530544281006, -1.9756054878234863, -0.12490317225456238, 3.601499319076538, 0.23796795308589935, 1.3288644552230835, 2.981372594833374, 0.5657341480255127, -0.848270058631897, 1.0513696670532227, 1.9625476598739624, -7.639296531677246, -0.02169574238359928, 1.2940922975540161, 3.316988945007324, -1.6529064178466797, -1.5574626922607422, 2.219771385192871, -3.410181999206543, 0.45647746324539185, -2.4318196773529053, -3.1874020099639893, -4.66579008102417, 0.13857345283031464, 1.185075044631958, -1.0556848049163818, -1.8315269947052002, -1.8610128164291382, -0.033206067979335785, -3.5059869289398193, -0.26019683480262756, 2.710775375366211, 3.2000732421875, -2.6718740463256836, 1.1986181735992432, 0.5533902645111084, 4.326894283294678, 2.3732118606567383, -0.4150916039943695, 0.2037811577320099, 3.9803287982940674, -2.8618392944335938, -1.0853452682495117, 3.580512285232544, 3.05291748046875, 1.7255656719207764, -2.311115264892578, -0.33631712198257446, -5.420318603515625, -2.898348093032837, -0.8239901065826416, 4.86406135559082, 5.602369785308838, 1.761160135269165, -0.426092267036438, 0.010340520180761814, -3.6575870513916016, 1.5095734596252441, -2.7551887035369873, -2.2058815956115723, -1.9854357242584229, -0.7840780019760132, -1.9503841400146484, -2.961090087890625, -4.053424835205078, 1.3643299341201782, -2.671816110610962, 3.0547232627868652, 4.975643157958984, -1.639858603477478, -0.27131810784339905, -4.858983039855957, 4.032681465148926, -4.19783353805542, 0.39454108476638794, 1.4112963676452637, -1.2348785400390625, -2.322420835494995, -2.5559303760528564, -2.458057403564453, -4.375658988952637, -2.668534278869629, -0.18768896162509918, 0.7690737247467041, 2.2424123287200928, 1.344021201133728, -4.831136226654053, -2.0831923484802246, 5.200857162475586, -1.972306251525879, 0.42923596501350403, 6.516909599304199, 1.3629686832427979, -2.03242564201355, -0.5393879413604736, -5.039758682250977, -3.9506242275238037, -2.019699811935425, -5.948232650756836, 6.727295875549316, 2.4334821701049805, 0.010834961198270321, 2.612006664276123, -0.3920605480670929, -4.449403762817383, -2.2527318000793457, -3.676102638244629, 2.0837669372558594, 0.36765068769454956, -2.4729883670806885, -3.945521831512451, -0.1733933687210083, -0.4724377691745758, -1.37019944190979, -1.9107987880706787, -3.637817859649658, 1.8166532516479492, -4.759685039520264, 1.6360673904418945, 0.1782546043395996, -0.23278047144412994, -1.0419667959213257, 0.7907530069351196, 5.068986892700195, 3.9980902671813965, 1.719496250152588, 1.1136671304702759, 0.3191857933998108, 1.2115304470062256, -5.84989595413208, 2.016221046447754, -4.062676429748535, 1.3253071308135986, -5.5978193283081055, 0.9921678304672241, -1.5425159931182861, -1.183065414428711, -0.4381083846092224, 3.166239023208618, -3.605787992477417, -2.743117570877075, 2.2864534854888916, 5.372171401977539, -4.060497760772705, 4.2831501960754395, -2.0827507972717285, -0.0020359144546091557, -3.558316230773926, -2.9480340480804443, 0.23404541611671448, 0.38784098625183105, -2.8313980102539062, 2.652383327484131, 6.755768775939941, 3.134817123413086, -5.387700080871582, -3.0404040813446045, -0.5218502283096313, -0.52874755859375, -1.9727787971496582, -2.5257599353790283, 2.0606565475463867, -4.90348482131958, -0.36767327785491943, 1.5327790975570679, -4.765641212463379, 0.16643786430358887, 2.58573055267334, -2.494659900665283, 3.1276726722717285, 1.0166815519332886, -0.3685833215713501, -0.241356760263443, -0.7167947292327881, -0.5571087002754211, 2.1846261024475098, 4.6895294189453125, -4.9414143562316895, 0.1684829294681549, 3.930239677429199, -3.6064887046813965, -0.33556365966796875, 0.7026146650314331, -2.503152847290039, 0.18851110339164734, -4.025790214538574, 3.462301254272461, 1.2782841920852661, -5.383493900299072, -5.366480827331543, 0.1968279629945755, 5.451389312744141, -2.0586562156677246, 6.435900688171387, 2.357689619064331, 3.056028127670288, 5.218336582183838, 6.030895233154297, -1.2175908088684082, 2.181788444519043, 3.252767562866211, -0.9520906209945679, -4.436241149902344, 0.2512795627117157, -4.354612350463867, -1.2946885824203491, -3.314615249633789, 3.2458362579345703, -2.4114811420440674, 0.2876848578453064, -1.9298367500305176, -2.7355217933654785, -4.502936363220215, 4.284822463989258, 2.773805618286133, 5.487174987792969, -0.6532743573188782, 1.5781090259552002, -4.937211990356445, 4.695333957672119, 3.632321357727051, -4.7730302810668945, -2.3622031211853027, 1.2078665494918823, 0.8675073981285095, -7.442409038543701, -0.34108638763427734, -0.6693023443222046, 1.2944774627685547, 1.5975892543792725, -6.326848983764648, 2.8831515312194824, 5.539577007293701, -1.4251874685287476, -1.0733429193496704, -1.2038323879241943, -5.158648490905762, 1.765005350112915, -1.8390955924987793, -2.1006128787994385, -3.860428810119629, 0.4862062633037567, -1.3897258043289185, -0.04233874753117561, 1.1360206604003906, -1.259562373161316, -0.9427152872085571, 1.7060654163360596, -2.1123688220977783, -3.352687358856201, -1.4041659832000732, 0.5690972805023193, -1.1481797695159912, 3.2910823822021484, 1.0696994066238403, -0.4116252064704895, 0.7117605805397034, -1.3521904945373535, 0.7481342554092407, -2.354691743850708, 3.5047831535339355, 2.1983323097229004, -7.063215255737305, 1.1634145975112915, -4.513838291168213, -1.5402355194091797, -1.6987992525100708, 0.8949742317199707, 0.7936060428619385, -3.642282485961914, -0.9091010093688965, 2.2376961708068848, -1.2075128555297852, -0.9264199733734131, -0.6131104230880737, 2.6060807704925537, -0.6911303997039795, 0.8841378688812256, 1.5413429737091064, 5.952558517456055, 1.3415734767913818, 0.6571588516235352, 1.6769453287124634, -0.07343865185976028, -2.9936859607696533, -1.5669032335281372, -0.8965746164321899, 2.3965907096862793, -1.6084678173065186, -1.3757330179214478, 1.4097155332565308, 1.4829561710357666, 0.6363416910171509, 2.1817328929901123, 0.3552156984806061, -1.4552596807479858, 5.296246528625488, -3.0236988067626953, 0.817868709564209, 1.5547478199005127, 1.7617976665496826, -0.6781807541847229, -0.49678754806518555, -1.0319498777389526, -7.192560195922852, 3.482235908508301, -0.5792790651321411, 4.608938694000244, 3.963521957397461, 0.9167838096618652, -2.7483606338500977, -0.7930963039398193, 4.884031772613525, 3.435103416442871, 4.1174516677856445, 3.7770237922668457, -4.6091814041137695, 4.23488712310791, -4.7262349128723145, 3.5809106826782227, -0.4794605076313019, -4.145599365234375, -3.021634340286255, -4.142356872558594, 6.863074779510498, -0.09268231689929962, -0.5913004279136658, 0.30154430866241455, 3.6623001098632812, -0.39048469066619873, 3.055274248123169, 3.031097888946533, 3.291813373565674, -0.981041669845581, -0.26904353499412537, -3.08516263961792, -0.30149045586586, 1.6879842281341553, -1.9549802541732788, -0.0748654380440712, 2.884920597076416, 0.5995154976844788, -2.9231722354888916, -2.4838833808898926, 3.7361292839050293, 0.6747188568115234, 4.215158939361572, 3.749302625656128, 2.567387104034424, -1.029043436050415, -2.498075485229492, 2.137563467025757, -0.046331845223903656, 1.8562570810317993, 4.342578887939453, -0.9661880135536194, -6.858875274658203, 1.6122018098831177, -0.5751519203186035, 1.6082098484039307, -0.2974593937397003, -2.3162856101989746, 2.020129680633545, -0.08939318358898163, -0.4749845862388611, -0.9289312362670898, 2.6744792461395264, 3.8535728454589844, -0.9227873682975769, -1.5465919971466064, 0.3024735450744629, -3.7104835510253906, -3.600774049758911, -2.720137119293213, 0.4763108491897583, -0.37378886342048645, -1.551002025604248, 2.074190139770508, -1.8782069683074951, -2.118253469467163, -3.85322642326355, 0.20233818888664246, 1.9190447330474854, -5.345428466796875, -0.9660049676895142, 0.5996076464653015, 2.5724220275878906, 0.6199172735214233, -2.5778863430023193, 2.6475181579589844, -0.6011084318161011, 5.641449928283691, -1.059647560119629, 4.007910251617432, -1.5095726251602173, 3.4885172843933105, -3.329129695892334, 1.1843390464782715, 0.5945504903793335, 3.953784465789795, -0.442801833152771, 2.3962693214416504, -0.9472249746322632, 2.1571130752563477, -0.4788619875907898, -1.0598076581954956, 1.0712740421295166, -0.1145993247628212, -3.050614833831787, -0.011303221806883812, -2.657590866088867, -0.5767697095870972, -1.2409840822219849, 5.146501541137695, 1.5769025087356567, -4.817591667175293, 3.134934425354004, 2.776668071746826, 3.716947078704834, 3.213736057281494, 0.2761105000972748, 2.2366039752960205, -2.9567136764526367, 4.538476467132568, 3.378500461578369, 3.429391622543335, -0.362554132938385, -0.4154309034347534, -1.0118706226348877, 2.6031389236450195, -6.292293548583984, -3.5565500259399414, 1.8879542350769043, 4.003548622131348, 0.3701252341270447, 2.197793483734131, -1.6737828254699707, -0.66355299949646, -3.7158100605010986, 0.5708176493644714, 0.23675492405891418, -9.105478286743164, -2.9994852542877197, -1.5823177099227905, -0.4512239694595337, 3.0714120864868164, 1.8414599895477295, -0.9065406322479248, -0.013432684354484081, 1.0363413095474243, 2.463120698928833, -1.9937732219696045, 2.88313889503479, -3.0299429893493652, 0.08387153595685959, 0.9237394332885742, 0.48563799262046814, -1.4087705612182617, -3.860999584197998, -4.403036594390869, -2.0866405963897705, -7.114816188812256, -1.783921241760254, 2.2058234214782715, 1.3791390657424927, 0.6263952255249023, -1.5680677890777588, 6.543896675109863, 2.8826169967651367, 0.2723389267921448, -0.23556920886039734, 3.269551992416382, -1.9773364067077637, -2.2738876342773438, 0.29920390248298645, -1.4883238077163696, -3.9478836059570312, -0.6473165154457092, -5.524354457855225, -4.896718502044678, 0.019807375967502594, 0.41782116889953613, -2.559321880340576, 0.168259397149086, -1.7406575679779053, -0.6248828172683716, 1.8019455671310425, -0.10723666101694107, -0.8415430784225464, 0.7071349620819092, 3.637505054473877, 2.005398988723755, -4.224809646606445, 1.2298359870910645, 4.915070533752441, -0.3956395983695984, -0.21798381209373474, -0.20679235458374023, -1.3033528327941895, 1.9551994800567627, 0.17221948504447937, -3.990496873855591, 1.815019130706787, 1.1900651454925537, 0.3323189318180084, -4.045121192932129, 0.0616707019507885, -0.8196737766265869, -1.0160771608352661, 2.8434276580810547, -2.248100996017456, -6.404704570770264, 0.68888258934021, 2.4274051189422607, -1.272766351699829, -5.861337184906006, -4.872457504272461, -2.3309617042541504, 0.8835455179214478, -1.670437216758728, -5.1101226806640625, -8.101311683654785, 4.879604339599609, -0.10557346045970917, 3.171090602874756, -2.088167905807495, -2.793849468231201, 0.95591139793396, 3.8638572692871094, -0.6402400135993958, 0.4143654406070709, -2.672635316848755, 1.984837532043457, 4.322676181793213, 1.2370108366012573, 0.9184962511062622, 6.903998374938965, 3.7853951454162598, -5.263288497924805, -0.4162309765815735, -2.490696907043457, 1.637434720993042, -1.7571816444396973, -3.2608442306518555, -3.683964252471924, -3.2758383750915527, -0.5161188244819641, 0.9736427664756775, 0.9402852058410645, -6.001066207885742, 2.2795190811157227, -1.5507333278656006, 0.2957237958908081, -5.270332336425781, 1.7941057682037354, 0.1761728823184967, 3.0502617359161377, 1.8128578662872314, -4.281055927276611, 2.903019428253174, 5.9726409912109375, -2.8711659908294678, -5.949746131896973, -4.792623519897461, 0.47614914178848267, -1.6448725461959839, 4.457801342010498, -5.551795482635498, -2.751230239868164, -1.3398174047470093, -1.9865995645523071, -3.4893226623535156, 3.570591449737549, 0.8464130759239197, 2.3408379554748535, -1.9360220432281494, -1.009929895401001, 2.9028735160827637, 1.3528679609298706, 2.734018325805664, 0.35063835978507996, -2.464479923248291, 1.3971679210662842, 2.5490846633911133, 1.396352767944336, 3.067678928375244, 0.63611900806427, 2.7218527793884277, -1.1492812633514404, -0.8842261433601379, -1.4973355531692505, 0.5959524512290955, -1.5754441022872925, 2.6947643756866455, -1.7442731857299805, -5.045984745025635, -2.3429999351501465, 1.9966957569122314, 3.3508567810058594, 3.1838669776916504, 1.578353762626648, 2.6036534309387207, -5.326432704925537, -6.772404670715332, -4.961377143859863, -0.6571846008300781, 2.6931631565093994, -1.3081250190734863, 2.674661159515381, -0.34565871953964233, 1.0985816717147827, -2.711205005645752, -5.698486804962158, 1.5251758098602295, 1.161779522895813, 1.7587099075317383, -1.3998982906341553, -0.8630552887916565, 2.5944695472717285, 3.8704304695129395, 0.22812384366989136, -4.246860980987549, -1.6476171016693115, 1.2546474933624268, 2.042013645172119, 1.8111094236373901, -2.4314255714416504, -0.5136746168136597, -1.1095443964004517, -1.8030810356140137, 4.448831558227539, -1.0491920709609985, -0.7689344882965088, 0.4580445885658264, 0.7620681524276733, -1.0964747667312622, 0.4368804395198822, -1.3790534734725952, 0.46770697832107544, -3.3082618713378906, 4.222159385681152, -3.380666971206665, -3.8461928367614746, 3.092312812805176, -2.5772056579589844, -6.299409866333008, 4.052559852600098, -2.3784332275390625, -1.041792392730713, 4.1469831466674805, -3.0051188468933105, -3.4551405906677246, -4.3828535079956055, -1.1129567623138428, -1.8581538200378418, -0.7840179204940796, -4.068119525909424, -0.20121434330940247, 6.037810325622559, -2.583939552307129, 1.5718894004821777, -1.0817140340805054, -1.152541995048523, -4.403861045837402, 3.4335970878601074, 3.5322253704071045, 0.3480018377304077, 1.784256935119629, -1.9220662117004395, 1.8949402570724487, -3.399913787841797, 2.8543760776519775, 0.17773742973804474, -1.9878203868865967, -1.3619722127914429, -0.22222143411636353, 2.8322529792785645, -1.5854008197784424, -3.5260984897613525, 2.5211234092712402, -1.8553643226623535, 5.420580863952637, -2.472909927368164, 1.2707034349441528, 1.366152286529541, 0.6308773159980774, -3.176351547241211, -3.6808769702911377, -2.8126931190490723, -4.543778896331787, 5.088320255279541, -5.527972221374512, -2.488332748413086, -3.077186107635498, 1.6194124221801758, 0.6300854682922363, -2.5593512058258057, -4.32596492767334, -0.15753062069416046, -3.0783867835998535, -3.869011402130127, 1.3841747045516968, -0.5675622820854187, 1.9234689474105835, 1.2137398719787598, 2.206005096435547, 3.9728264808654785, -1.0255757570266724, -4.50477409362793, 1.5157644748687744, -3.0291872024536133, -3.999049186706543, 0.953815221786499, -1.3635532855987549, 0.04829912632703781, -1.0598784685134888, 1.316991925239563, -3.342158794403076, 3.1118063926696777, 7.331340789794922, -1.1560273170471191, -2.0085508823394775, 2.0634474754333496, 7.690561294555664, -1.934699296951294, 0.975952684879303, 0.24420413374900818, -2.209106922149658, -5.324820518493652, -4.334935188293457, -0.9280369877815247, 4.329721927642822, -2.161001443862915, -1.8723907470703125, -0.43007445335388184, 5.917934894561768, 0.8642781972885132, 2.350214958190918, 1.6771163940429688, 0.7555323243141174, -2.0009994506835938, 0.6323891878128052, 2.992084503173828, 0.5151535868644714, 2.8723621368408203, -2.093282699584961, 2.0489511489868164, 3.47955322265625, -0.90177321434021, -0.8551713228225708, -1.6902272701263428, -7.123688220977783, -0.8933204412460327, 4.65888786315918, -1.1220650672912598, 0.08865652233362198, -1.0006673336029053, -4.586669921875, 0.9215407371520996, -0.38976579904556274, 1.1664674282073975, -3.2152600288391113, 2.589186191558838, 8.077016830444336, -0.8108881115913391, 8.769913673400879, 0.8225562572479248, 0.6908413171768188, -0.14374008774757385, 1.4081237316131592, -0.15395136177539825, -1.5437719821929932, 0.8829240798950195, -3.7942256927490234, -0.28786423802375793, 2.1265709400177, 2.606884002685547, 5.376032829284668, 1.715504765510559, 0.5006417036056519, 0.6613239645957947, -2.3598484992980957, -2.7667031288146973, -2.98683500289917, -2.068270444869995, -1.7369742393493652, 0.6036266088485718, 0.9441136121749878, -0.31396758556365967, 6.152249336242676, -0.308607816696167, 0.9040730595588684, 0.0642881840467453, 4.237912178039551, -2.4419665336608887, -1.0371975898742676, 8.291521072387695, 0.005316910799592733, 2.4692537784576416, 4.1821112632751465, -0.4063915014266968, -4.672794342041016, -2.5835771560668945, 2.9619102478027344, -2.6031494140625, 4.042812824249268, -1.0602713823318481, 4.137204170227051, -1.570824146270752, -1.028339147567749, 5.12392520904541, -3.246359348297119, -2.255462646484375, 3.480738401412964, 1.6344584226608276, 5.167750358581543, 1.048408031463623, 3.0586116313934326, -0.3316459357738495, -0.7154096961021423, 3.5005366802215576, -0.014192520640790462, -5.208797931671143, 4.079858303070068, -1.6860054731369019, -2.3649301528930664, 4.252354621887207, -5.513562202453613, 3.367283821105957, -1.4324414730072021, 4.494348526000977, 1.927290916442871, -3.0398521423339844, 9.827744483947754, 0.6504552364349365, -0.34243664145469666, 7.3196258544921875, -3.6218833923339844, -0.8939964771270752, -0.4067302942276001, -4.06723690032959, 1.0668171644210815, 5.201570510864258, 3.907043933868408, -4.10880184173584, -3.374664783477783, -1.4442620277404785, 2.411313533782959, 3.8739142417907715, 2.713324785232544, -3.335956573486328, -0.30923739075660706, -7.41989803314209, -1.0220365524291992, -0.9677520990371704, -3.1611523628234863, -3.682912588119507, -0.6051194667816162, 0.03241075947880745, 0.8641873598098755, 0.3001396656036377, -1.5940382480621338, -0.8343474864959717, 1.799010992050171, 1.175973892211914, 3.73420786857605, 1.7987068891525269, -1.0507558584213257, -1.2631404399871826, -0.9070925116539001, -0.920262336730957, 4.409037113189697, 0.842013955116272, 2.9455037117004395, 1.4385762214660645, -0.6890147924423218, 0.19659149646759033, 0.5967023372650146, 1.6938862800598145, -3.303325653076172, 0.4875338077545166, -2.1450016498565674, 1.059739351272583, 2.5407426357269287, 0.5055096745491028, 0.21261544525623322, 1.822347640991211, -1.7684288024902344, -1.7557041645050049, 0.2813405394554138, -1.1496007442474365, -2.7199580669403076, -3.2347769737243652, 2.0270304679870605, 1.8534142971038818, 0.7255973815917969, 6.0241851806640625, -2.3647639751434326, -1.300208568572998, -5.426140785217285, -2.5818262100219727, -6.363899230957031, 3.5528674125671387, 2.525759220123291, 3.060702323913574, -0.08268219232559204, 2.783299446105957, 2.557326316833496, 3.4070863723754883, -6.461957931518555, 0.9831174612045288, 2.618619203567505, 0.9420894384384155, -3.974216938018799, 5.635396480560303, 0.47301381826400757, 6.347332000732422, -2.004223585128784, 0.3283567726612091, -1.6840612888336182, -0.9923853278160095, -1.8182799816131592, -1.9984917640686035, 3.782055616378784, -1.7269216775894165, -1.6304980516433716, -0.7476977705955505, -3.673848867416382, -1.9649169445037842, 2.3508448600769043, -2.1229681968688965, 3.583369255065918, 2.501232385635376, 2.6570277214050293, -0.9105727672576904, 5.339902877807617, -4.115318298339844, -1.3777166604995728, -1.9459609985351562, -4.689363956451416, 0.8940869569778442, 3.8156347274780273, 0.17228347063064575, 0.7830628752708435, -1.5843579769134521, 0.716357409954071, -3.224940061569214, -0.059477031230926514, 2.904278039932251, -3.0536048412323, -1.349852442741394, 4.907092571258545, -1.7749946117401123, -3.0387375354766846, 1.7628953456878662, 1.265607237815857, -1.3561010360717773, -0.47434002161026, -1.9187734127044678, -0.9385050535202026, 1.7928335666656494, 2.3598875999450684, 3.8698344230651855, -5.775235176086426, -0.6485755443572998, -2.4144508838653564, 1.613169550895691, -2.227201223373413, 3.7649450302124023, -1.7865657806396484, 2.626646041870117, 5.955170154571533, -1.5419940948486328, 3.3308606147766113, 4.811893939971924, 3.9155564308166504, 4.260919570922852, 4.958471298217773, 1.1411693096160889, 4.848119258880615, 4.527101039886475, -1.237744688987732, 1.3554097414016724, 5.982850074768066, -3.14821195602417, 0.2840885818004608, 1.336536169052124, 2.8259379863739014, -2.738142967224121, -3.8843679428100586, 0.2131304293870926, 0.663129985332489, 2.6802167892456055, -0.07195930182933807, 0.595539927482605, -1.1411008834838867, 4.9361090660095215, -1.2991957664489746, -0.7040231227874756, -1.8273063898086548, -4.213270664215088, -0.9921578168869019, -2.7260358333587646, -0.4888712763786316, 0.4787978529930115, -1.1720587015151978, -1.3779377937316895, -0.6706532835960388, 0.7658818960189819, 1.3439387083053589, -1.2528164386749268, -0.21392536163330078, -0.6957476139068604, 5.575199127197266, -2.026427745819092, -2.0607407093048096, -0.443905234336853, -6.021355628967285, 1.8403260707855225, 1.1581933498382568, -0.1319032609462738, 0.4646475911140442, 3.5021843910217285, 1.2926685810089111, -2.5653858184814453, -0.20641551911830902, 1.5491397380828857, 7.637720108032227, -2.095126152038574, -5.375184535980225, 0.8818105459213257, 1.4471070766448975, 3.140425205230713, 1.653099536895752, -1.717043161392212, 1.2526696920394897, -1.3256280422210693, 2.9400880336761475, 1.7308659553527832, -1.7702245712280273, -0.636722207069397, -0.0015232163714244962, 2.5692191123962402, 1.3985140323638916, 0.6464558839797974, -4.83409309387207, 4.096196174621582, -1.1632897853851318, -2.121276617050171, -2.266899824142456, -4.383219242095947, -5.923460006713867, -5.022507190704346, 2.9464635848999023, -1.2988719940185547, -4.724021911621094, 0.9524028897285461, -4.1759233474731445, -1.1444053649902344, -2.2367639541625977, 4.362977981567383, 3.6716227531433105, -4.48361873626709, 2.3016061782836914, -3.0451793670654297, 2.390411376953125, 1.8247579336166382, -5.846105098724365, -0.395202100276947, 1.411622405052185, -5.193027973175049, -4.565017223358154, 0.08020858466625214, 5.529652118682861, -0.046233490109443665, -0.4019913077354431, 2.8312549591064453, 2.8104000091552734, 0.11574556678533554, 2.494204521179199, -2.731672525405884, 1.7351980209350586, -0.08288339525461197, 4.079839706420898, 3.4435980319976807, -1.0765941143035889, 0.8742000460624695, 1.0390924215316772, 0.6753896474838257, -0.4030012786388397, -2.5784006118774414, 1.0690865516662598, -0.8692927360534668, -2.295714855194092, 6.011335372924805, 1.9524714946746826, -1.5063800811767578, -4.860428810119629, -0.22737020254135132, -2.7653238773345947, -1.574676275253296, 0.28931260108947754, 6.044838905334473, 2.3377065658569336, 2.1832504272460938, -1.2397931814193726, 3.1903929710388184, 3.0316178798675537, 2.2951714992523193, 4.740162372589111, 0.4474674463272095, 2.9544615745544434, -6.8225250244140625, -2.4175753593444824, -0.866471529006958, 2.952885150909424, -1.988427758216858, -1.4097912311553955, 1.929758071899414, -2.033928155899048, 0.8813750743865967, -3.3231561183929443, 3.21323823928833, -0.7467324137687683, 2.2693898677825928, 3.467395305633545, 0.8047576546669006, -1.063048005104065, 5.390469074249268, -0.8535805940628052, -4.88563871383667, 2.3106961250305176, -3.3380112648010254, 0.7834887504577637, -3.251755714416504, 2.2579007148742676, 0.8277114629745483, -2.4871809482574463, 2.476158618927002, -6.387567520141602, -2.8001487255096436, -1.610327959060669, -1.120697259902954, -2.84023380279541, 4.879270553588867, 2.1304898262023926, 0.7703996300697327, -3.338352680206299, 1.4222320318222046, -1.8729007244110107, 1.2515692710876465, -4.980413913726807, 1.1204509735107422, -0.05106458067893982, 2.9272212982177734, -4.959388732910156, -1.4265574216842651, 0.4676123261451721, -2.0705466270446777, 0.8641009330749512, 6.099423408508301, -0.49102991819381714, 1.5050300359725952, 0.8526955842971802, -3.5306389331817627, -2.1863677501678467, -0.7169256806373596, 0.47703635692596436, -1.279865026473999, 3.9052581787109375, -2.59442400932312, -1.0180522203445435, -0.8955810070037842, 3.2409615516662598, -1.23876953125, 2.9633944034576416, 1.3617922067642212, -1.699724793434143, 1.5938937664031982, -0.6931455135345459, -3.4004769325256348, 2.477660655975342, 0.7339012622833252, -5.590460777282715, -0.22194287180900574, -5.009792327880859, -0.0880402997136116, -4.209924697875977, 2.6755642890930176, -7.447599411010742, -0.8976629376411438, 2.7996771335601807, -0.7869662046432495, 3.0451784133911133, -0.5348643660545349, -3.7647852897644043, 3.4936459064483643, -2.050448417663574, -1.475052833557129, 4.304262161254883, 0.7223005890846252, -5.438779830932617, -1.0802642107009888, -0.9735453128814697, 0.5674659609794617, 0.8935375213623047, 0.4776803255081177, -1.4177947044372559, -1.682744026184082, 3.4818854331970215, 3.7849717140197754, 2.172194004058838, -2.903170585632324, -1.6550531387329102, 2.295823097229004, 0.6583570241928101, 2.1226062774658203, -3.5061042308807373, -0.9814635515213013, -2.8711156845092773, 8.649859428405762, 2.024216413497925, 1.9647340774536133, -4.201716899871826, -2.9958510398864746, 0.664820671081543, -0.06257252395153046, -0.4174959659576416, -0.2809838056564331, 1.1832166910171509, 2.2311434745788574, -1.155653476715088, -1.7247347831726074, 3.500548839569092, 0.07398883998394012, 0.3114599585533142, -3.947288990020752, 5.142300605773926, 2.0139777660369873, -4.832878589630127, 0.11525283753871918, -1.0494630336761475, 6.079895496368408, 0.5522561073303223, 1.7053277492523193, -3.69968318939209, -2.8637800216674805, 3.196514129638672, 1.7898892164230347, 2.180102825164795, 4.6474151611328125, 0.4170262813568115, 4.146864891052246, 1.1312955617904663, 1.9198169708251953, 0.3874235153198242, -4.099137306213379, 1.9883358478546143, -3.180145740509033, 2.265380382537842, 1.7227668762207031, -2.5902175903320312, 1.0381656885147095, -2.3170619010925293, -0.20011447370052338, -2.539759635925293, 4.149823188781738, -1.8352651596069336, -3.291539192199707, -5.432417869567871, 3.776597499847412, -2.3772099018096924, -3.9052984714508057, -2.132422924041748, -1.467355728149414, 0.7491511702537537, 3.269310235977173, 3.031906843185425, -1.8450400829315186, 2.8699026107788086, 2.6830363273620605, -0.4850010275840759, 1.3274264335632324, -0.9201947450637817, -1.163680076599121, -2.8707528114318848, -0.44544243812561035, 1.4704618453979492, 1.2452713251113892, -1.9904098510742188, 3.9798097610473633, -3.4799180030822754, -3.9282467365264893, 2.2598159313201904, -2.8796494007110596, -5.66991662979126, -5.0268073081970215, 0.310727059841156, 4.2490434646606445, -1.2719850540161133, -0.4664843678474426, 4.942430019378662, -1.1462677717208862, -0.9031620025634766, -3.748979091644287, -2.81587553024292, 3.761702299118042, 1.6395996809005737, 1.1183905601501465, -3.389918804168701, -5.662981986999512, 0.7653703093528748, -1.5556508302688599, -2.311023712158203, 1.0034019947052002, 0.5062476992607117, 4.74276876449585, 0.2577895522117615, -2.023617744445801, -0.112034372985363, 6.0258283615112305, -1.335768461227417, 3.669671058654785, -1.7810901403427124, 2.946305990219116, -6.916416168212891, -5.527048110961914, -1.4999253749847412, -3.002957344055176, 1.8321669101715088, -0.18458852171897888, 0.16849151253700256, -2.4198856353759766, 1.5326589345932007, 3.0774340629577637, -4.345737457275391, 6.127660751342773, 1.509570837020874, -1.3014887571334839, -0.17231349647045135, -2.105086088180542, -3.267230987548828, 2.731210708618164, 5.886631011962891, -1.2462599277496338, -0.30798330903053284, 0.8176968097686768, 2.528350591659546, 1.6668245792388916, -5.046399116516113, 0.9451076984405518, 1.9702496528625488, 4.3956990242004395, 2.2383675575256348, 1.6651461124420166, 0.9707616567611694, -3.132404327392578, -3.818882703781128, -0.2490081787109375, 2.4627132415771484, 3.6339333057403564, 1.0862438678741455, 0.9074462652206421, -2.720181465148926, 2.3752191066741943, 1.3927555084228516, 0.9832130670547485, 4.495216369628906, -4.895788669586182, -2.589728593826294, -0.23284046351909637, 0.4835081398487091, -0.48544254899024963, -6.435088634490967, 3.2643821239471436, 2.0256764888763428, -4.716540336608887, 2.869905710220337, 4.723260879516602, -0.6735271215438843, -3.0366735458374023, -1.7793048620224, 0.21677422523498535, -1.3701391220092773, 4.337116718292236, -3.687372922897339, -2.4993886947631836, 0.9743137359619141, -2.312913417816162, -0.25212523341178894, -7.703251838684082, 3.371185302734375, -2.905872106552124, 0.8292054533958435, -0.6157763004302979, 6.3486223220825195, -4.63401460647583, 0.5171707272529602, -3.353208541870117, 0.05186343193054199, 1.8413903713226318, 0.17356182634830475, 2.189941883087158, 3.6241798400878906, -1.5557737350463867, -1.9008302688598633, 1.6008238792419434, -2.2529542446136475, 6.370702743530273, 3.2943615913391113, 0.5346038341522217, -0.16869321465492249, -0.10420446842908859, -1.0039217472076416, 0.3824182152748108, 3.4504547119140625, -0.0920487493276596, 1.1720681190490723, -2.0228936672210693, -1.4977138042449951, -1.1770975589752197, 1.9189070463180542, -1.9363857507705688, 1.6981322765350342, 3.4553351402282715, -4.541236400604248, -2.829846143722534, 1.5052051544189453, 3.888085126876831, 0.6988641619682312, -0.735387921333313, -0.8419026732444763, -1.911421298980713, 2.2080814838409424, -1.1424756050109863, -1.4817757606506348, -3.273843765258789, -2.3390369415283203, -2.106874942779541, 0.406322181224823, -2.090545654296875, 4.3977460861206055, 0.24242748320102692, 1.789177656173706, -7.052736282348633, 4.830780982971191, -6.4414825439453125, 1.3064122200012207, -1.5431244373321533, 4.968204975128174, -2.4253969192504883, -3.2772064208984375, 1.235571026802063, 1.056408166885376, 3.3371894359588623, 1.5499725341796875, 1.3688194751739502, 7.680909156799316, 2.1221883296966553, 0.10317574441432953, 2.467877149581909, 3.6752209663391113, -0.7567441463470459, 4.651989936828613, 4.4673309326171875, -0.23235918581485748, -7.200692176818848, 2.5408849716186523, 4.624842643737793, -2.101651191711426, -0.3669406771659851, -1.1115907430648804, 2.998063802719116, -4.506462574005127, -0.1828145682811737, 3.23504638671875, -0.003331865184009075, 0.38314735889434814, -1.8969388008117676, -2.263903856277466, 0.14813683927059174, 5.1881794929504395, -0.6164627075195312, -2.677833080291748, -1.8524754047393799, -0.902819037437439, -5.217641353607178, -3.9949097633361816, 1.2903852462768555, -3.2194912433624268, -0.786703884601593, 1.0952708721160889, -1.358778715133667, 0.47495245933532715, -1.1625614166259766, -0.4427585005760193, -1.2576712369918823, 2.257450580596924, 2.780546188354492, -8.284196853637695, 0.7603660821914673, 2.9908828735351562, 0.07341597229242325, -1.161203145980835, 4.631773471832275, 1.2203326225280762, -0.8282326459884644, -0.5290533304214478, 2.2770841121673584, -1.1252397298812866, 5.617474555969238, -0.29446321725845337, -0.39945951104164124, 3.727651596069336, -1.7205966711044312, -1.415151834487915, 1.42216956615448, -3.495547294616699, 1.7227041721343994, 3.168951988220215, 2.139495372772217, 3.6247944831848145, 3.653803825378418, -0.26175999641418457, -2.499742269515991, 1.4143106937408447, 0.9206737875938416, -0.4635319709777832, 0.8717005848884583, 0.8673667907714844, 0.20223209261894226, -2.479398250579834, -1.9833251237869263, -4.273015022277832, -4.390644073486328, -0.8410279750823975, 4.506820201873779, -2.0590174198150635, 0.3438986539840698, 1.0126285552978516, -2.0153238773345947, 1.0710675716400146, -0.2707500457763672, -0.8370712995529175, -4.620236396789551, -0.2623656094074249, -2.9625301361083984, 0.09943456202745438, -2.699154853820801, 5.034265518188477, 6.25514030456543, 5.92465877532959, 2.4745497703552246, -0.9900628328323364, 0.7109901905059814, 2.371480941772461, 2.002244472503662, 6.623125076293945, 1.3940621614456177, -2.089287042617798, 0.10864752531051636, -5.1821489334106445, -6.645105361938477, -5.764448165893555, 0.000862671178765595, 3.2131612300872803, 1.9562442302703857, -1.5416511297225952, -1.3975458145141602, -0.12597718834877014, 0.07102180272340775, 5.014218330383301, 4.326462745666504, -3.343724012374878, 0.6599924564361572, 1.9035841226577759, -3.0639867782592773, 0.42766284942626953, -1.4264614582061768, -0.906878650188446, 1.7014074325561523, 0.38318800926208496, -2.2342710494995117, -1.5930898189544678, 1.6049799919128418, 3.374422550201416, -0.26885151863098145, -2.0748772621154785, -2.282195806503296, -3.740497589111328, -0.3838365375995636, 4.382521152496338, 4.362062931060791, 1.6139776706695557, -3.549818515777588, -8.052001953125, -4.620481014251709, 1.796518325805664, -2.115720748901367, -6.176380634307861, 3.636568546295166, -3.7478232383728027, -2.1095218658447266, 5.329274654388428, -0.4482077360153198, -1.5039056539535522, -0.5051076412200928, -8.111664772033691, 0.5726181864738464, 2.7935588359832764, -0.942746639251709, 3.1071367263793945, 0.019624365493655205, -4.489901542663574, 1.584158182144165, 1.845468282699585, -0.9555145502090454, 1.355696678161621, -0.7868365049362183, 2.573535442352295, 0.4299435615539551, 0.21117711067199707, -4.273579120635986, -3.009882688522339, 1.076501727104187, 3.414764642715454, 1.6832561492919922, 0.7334508895874023, -3.4171929359436035, 0.3285873532295227, -0.26099535822868347, 0.5731712579727173, 0.7190335988998413, 0.5820185542106628, 7.202721118927002, 6.76323938369751, -6.821331977844238, -4.552597999572754, 5.483215808868408, -2.798264741897583, 1.3042480945587158, -1.5641722679138184, 2.7309730052948, 4.024893760681152, -2.8841423988342285, 2.0250301361083984, -0.3776554763317108, -0.4516201615333557, 1.2663301229476929]], "dtype": "float32"}, {"__nd__": [4696, -100, 406, 3123], "dtype": "int64"}, -100], "e": {"__nd__": 13.27554702758789, "dtype": "float32"}}, {"a": [{"__nd__": [[1.8802276849746704, 0.7279006838798523, -1.2836202383041382, 0.16630098223686218, -0.6401433348655701, -1.1498287916183472, -0.7414207458496094, -0.03274552524089813, -0.9341676235198975, 0.33349090814590454, -0.2678622007369995, 1.2895331382751465, -1.0309338569641113, -0.08166421949863434, -0.1818203628063202, 2.669499635696411, 0.5254125595092773, 0.24104808270931244, -2.220839262008667, 0.5903757214546204, -0.6977051496505737, 1.1507103443145752, 1.2439368963241577, -0.8840035200119019, -1.7924838066101074, 1.201959252357483, -0.5924069881439209, -0.47244754433631897, 0.929639995098114, 0.4076049327850342, 0.20736052095890045, 0.9972889423370361, 0.5782565474510193, -1.2211796045303345, 0.47738316655158997, 0.017980117350816727, -1.2045948505401611, 0.21427159011363983, -1.4522923231124878, 0.9850583076477051, 0.08632951229810715, 0.37478360533714294, -0.7586658000946045, 0.4201248288154602, 1.9517905712127686, -0.0884576141834259, -0.05247887223958969, -0.30502137541770935, 0.452706515789032, 1.7802461385726929, 1.2187581062316895, 0.9093239307403564, -1.0841336250305176, 1.897878646850586, 0.14734095335006714, 0.19805409014225006, -0.32303643226623535, 0.653803288936615, 0.8276771306991577, -0.9367374777793884, -0.34098756313323975, 1.124922275543213, 0.9467458128929138, -0.7493454217910767, -0.7160665392875671, 1.2764164209365845, 1.298364281654358, 0.4946674704551697, -1.6667325496673584, 1.408527135848999, -2.3419785499572754, -0.7451599836349487, -1.577298879623413, 0.771357536315918, -1.3530155420303345, -0.7929389476776123, 1.1190224885940552, 0.33814120292663574, -0.9975835680961609, -0.16446487605571747, 0.27442237734794617, 0.24125701189041138, 0.29350948333740234, 0.5596776008605957, 0.804324209690094, -0.002490914659574628, -1.1275933980941772, 1.3371241092681885, -0.0801100805401802, -0.2272825986146927, -0.6965681910514832, 0.20595119893550873, 0.7295881509780884, 0.38203802704811096, -0.4007374048233032, -0.8075998425483704, 0.09288425743579865, 0.11167268455028534, 0.6617985963821411, -0.8515269756317139, -1.8185871839523315, -0.08593116700649261, -1.4210121631622314, 1.3262605667114258, -1.465405821800232, -1.7480934858322144, 0.88616544008255, 0.3020937442779541, 0.7308815717697144, 0.6771508455276489, 1.0485124588012695, -0.33168497681617737, 0.8892022967338562, 0.44493305683135986, -0.5698381662368774, 2.1237072944641113, -0.4557584226131439, -1.3683477640151978, 1.5724244117736816, -0.11690320819616318, 0.6164780855178833, 0.11418978124856949, 0.4687274396419525, 0.4887916147708893, 1.5006921291351318, 0.7067809104919434, -0.1876218318939209, 0.3990492820739746, 0.7435532212257385, -0.18216246366500854, -0.0072419289499521255, 0.9380357265472412, 1.2890233993530273, 0.24292634427547455, -0.6514701247215271, 0.6578987836837769, -0.288219690322876, 1.7549339532852173, 0.16338218748569489, 0.7476201057434082, -0.4993114769458771, -0.05537620559334755, -0.5820910930633545, -0.9932222366333008, 1.1241816282272339, -0.9847780466079712, -0.019199049100279808, 0.6660227179527283, 0.7307944297790527, -0.43462878465652466, 0.03582574427127838, -0.13114812970161438, -0.47713491320610046, -0.31256332993507385, 0.5086595416069031, -0.693587064743042, 0.6653349995613098, -1.1958403587341309, -0.11444132030010223, 0.2314039170742035, -1.193530797958374, -0.6183189153671265, -1.0185515880584717, -0.20659232139587402, -1.2576576471328735, -1.916381597518921, -0.3317171037197113, 0.5328125953674316, 0.35055384039878845, -0.07202596217393875, -0.30937477946281433, 0.7495282292366028, -0.4037073850631714, 0.3343552350997925, 0.9269185662269592, -0.17602111399173737, -0.781470775604248, 1.3767861127853394, 0.14066073298454285, -0.8926085233688354, 1.3698121309280396, 0.37709105014801025, 0.8046841025352478, -0.29420214891433716, 0.05319925397634506, 0.6152202486991882, 0.5492140054702759, 1.3968706130981445, 0.05827629938721657, 1.4616937637329102, -3.6896004676818848, 1.3322917222976685, -0.8440198302268982, 1.0469634532928467, 0.32178664207458496, -2.179790496826172, -1.159698247909546, 0.3100683391094208, -0.7424007654190063, -0.24236778914928436, -0.04049139842391014, -0.1261465847492218, 0.9492821097373962, -1.175756573677063, 0.9324581027030945, -0.19341076910495758, 1.2581552267074585, -1.2155275344848633, -0.06239961460232735, 0.16007107496261597, -0.28666678071022034, -0.5545989274978638, -0.2865179181098938, 0.3747618794441223, -0.030338367447257042, 0.7741289138793945, -0.3173292577266693, -0.3925408720970154, -1.3729304075241089, -0.6313527226448059, -0.42776986956596375, 0.970384955406189, -0.14124272763729095, -0.2559508681297302, 1.52162504196167, 0.8045490980148315, 1.4029055833816528, -0.17601287364959717, -2.0374882221221924, 0.5293704867362976, -1.199532389640808, -1.7083830833435059, 0.0774659514427185, 1.2339805364608765, 0.9046300649642944, -1.321086049079895, -1.2823636531829834, 2.356924057006836, -0.17716684937477112, -0.11303146183490753, 1.0839837789535522, 0.14749731123447418, -0.5162020921707153, -0.1779811829328537, -0.8770143389701843, 1.7210557460784912, -1.5596106052398682, 0.2547398805618286, 0.23377636075019836, 0.26179173588752747, -0.1326926201581955, 1.8370397090911865, -0.7201399207115173, -0.09834930300712585, -0.9350271821022034, 0.42729708552360535, 0.829055666923523, -0.8427156209945679, 1.0720809698104858, 1.7788006067276, -0.9668548703193665, 0.7972527742385864, 0.10337798297405243, 0.4385579824447632, 0.5818081498146057, 1.5194424390792847, 1.2998838424682617, -1.4707226753234863, -1.3741575479507446, -0.4302380383014679, -1.081398844718933, 0.5210397243499756, 1.8004305362701416, 0.23957039415836334, -0.006307593081146479, -0.6675716042518616, -0.9029381275177002, 1.1896145343780518, 0.888384997844696, 1.2670080661773682, 1.0737568140029907, 1.1461679935455322, 0.6375672221183777, -1.5700628757476807, 0.8129110932350159, 0.6963412165641785, 0.5124603509902954, 0.05948036164045334, -1.5324480533599854, -1.3632221221923828, 0.28362053632736206, 0.5725769400596619, 0.9826611280441284, 0.48070451617240906, 0.443205863237381, -1.3685662746429443, 0.3254361152648926, 0.46180447936058044, -0.9492994546890259, 0.7465317249298096, -1.3927000761032104, 1.7019681930541992, 0.32230818271636963, 0.5932306051254272, 0.32644444704055786, -2.0880773067474365, 0.6723798513412476, -0.3514132797718048, 1.1885442733764648, 2.1122047901153564, -1.5492380857467651, -0.9036102890968323, 0.3797670900821686, -1.0263460874557495, 0.3130926489830017, 0.6229499578475952, 0.4180639684200287, 0.10696157068014145, 0.8179551362991333, -1.4876474142074585, -0.29919782280921936, -0.21051135659217834, 0.5924679636955261, 0.1020541563630104, 0.7025929093360901, -0.01438616868108511, 0.43667683005332947, 0.8194416165351868, -0.17478351294994354, 0.19737857580184937, -0.4284749925136566, 0.5746340751647949, -0.5443035960197449, 0.14721117913722992, 1.137425184249878, 0.7923833727836609, 0.3556089997291565, 0.19955213367938995, -0.587882936000824, 1.1063355207443237, -0.3551362454891205, 1.9365841150283813, -0.10588543117046356, -0.6979377269744873, -1.528529167175293, -0.8627429604530334, 1.710229516029358, 1.8395172357559204, 1.0908336639404297, -0.14605014026165009, -1.2180702686309814, -0.5631700158119202, 1.1408218145370483, 0.3989276885986328, 0.09731841832399368, 1.7578140497207642, -0.7090634107589722, 0.3576609492301941, -0.07883669435977936, 1.5406211614608765, 2.2340590953826904, -0.26211923360824585, -0.18077343702316284, 0.5064542293548584, -1.4625638723373413, -1.7991267442703247, 0.15623827278614044, -0.2969284653663635, -0.20097805559635162, -1.073468804359436, -1.378422498703003, 0.4417796730995178, -0.3576429784297943, 1.2149503231048584, 1.7335200309753418, 0.6820739507675171, -0.5691944360733032, -0.2896897792816162, -0.3811061382293701, -0.18369321525096893, 1.6161478757858276, -0.4865129292011261, -0.01706010475754738, -1.2680543661117554, -0.4781041741371155, 0.027052972465753555, -0.9629439115524292, 1.0977784395217896, 1.8624926805496216, -0.7759219408035278, 0.6723229289054871, -0.48831698298454285, -0.27059313654899597, 1.672954797744751, 0.5022755861282349, -0.17891329526901245, -1.57706618309021, 0.301267147064209, 2.0252537727355957, 0.893518328666687, -1.0690449476242065, 0.6615301370620728, 1.9602177143096924, -0.8351007699966431, 1.2739943265914917, -0.33100593090057373, -0.5245862603187561, 2.0495383739471436, -0.5291056632995605, -0.2382439821958542, 0.47230440378189087, 0.8045395612716675, -0.042152684181928635, 0.9902966618537903, 0.38027089834213257, -1.2750200033187866, 0.9638108611106873, 0.962939441204071, -0.27520787715911865, -0.250201553106308, 1.5815107822418213, 1.3256357908248901, 0.6064929366111755, -2.156704902648926, 0.9529226422309875, -0.35929059982299805, -2.604685068130493, -0.3474220037460327, 0.2677646577358246, 1.348839521408081, 0.05514725297689438, -1.053770899772644, -0.35134485363960266, 0.23864491283893585, -2.8832476139068604, 1.1993346214294434, 0.34992527961730957, -0.4391689598560333, 1.2216200828552246, 1.020904779434204, 0.9122698903083801, -2.1342828273773193, 2.0284199714660645, -0.2708207666873932, -0.6742677092552185, -1.1026314496994019, -0.06608365476131439, 0.026041189208626747, 0.21179629862308502, -0.09288583695888519, 0.9003949165344238, 0.5660901665687561, 0.17209984362125397, -0.12606871128082275, -0.271268367767334, 1.0561062097549438, 0.2789769470691681, 0.09389854967594147, -0.8041484951972961, -1.6380916833877563, -0.6707503795623779, -0.7086644172668457, 0.18090559542179108, -1.3981945514678955, 1.860436201095581, -0.6590092182159424, 0.6474562287330627, -0.7911003232002258, -1.801652193069458, -0.7937384247779846, -0.18742185831069946, 0.399185448884964, -1.2423306703567505, 0.1101657822728157, 0.6013698577880859, 0.42052626609802246, 0.5752356648445129, -0.9977626800537109, -1.2269715070724487, 1.1254037618637085, 1.7831202745437622, -0.21448320150375366, -0.7790944576263428, 0.22917130589485168, 1.169926404953003, 0.23537582159042358, 1.7535494565963745, 1.1618510484695435, 0.6210808157920837, -1.1463100910186768, -1.0492357015609741, 0.19570674002170563, 0.6090711951255798, 0.6993672847747803, 0.018660269677639008, -2.8532352447509766, 0.2944400906562805, 1.41696298122406, 1.8427951335906982, -0.8160175085067749, 1.1378300189971924, -0.009138690307736397, -0.8986909985542297, -0.7954040765762329, 0.6073117852210999, -0.4465087652206421, 0.12005667388439178, 1.5133880376815796, 0.09573451429605484, 1.7296212911605835, -0.5040528178215027, 0.14235171675682068, -0.37221452593803406, -1.4152946472167969, 0.4717172682285309, -1.2458980083465576, -0.5539553761482239, 0.291231244802475, 0.6937445402145386, 0.09708858281373978, 0.018225058913230896, -1.4103306531906128, 1.347152590751648, 2.323452949523926, 0.8469614386558533, -0.3347488343715668, -0.20659324526786804, -2.0562424659729004, 1.9995527267456055, -2.0299878120422363, 0.993730366230011, -0.8317184448242188, 0.311801940202713, 1.3701183795928955, 0.6666536331176758, 0.4806155264377594, -0.858144998550415, 0.5183446407318115, 0.06852660328149796, -1.0023536682128906, -0.11088351905345917, 0.6970084309577942, 0.41575726866722107, -1.2670668363571167, -1.9442589282989502, -0.013501872308552265, -0.1378181129693985, 1.503339409828186, -1.6789506673812866, 0.031096769496798515, -0.17991507053375244, 0.4056722819805145, -1.7314772605895996, 0.4920133352279663, 0.5283256769180298, 0.6135600209236145, -0.9119383096694946, 0.023497464135289192, 1.3166658878326416, -0.20845744013786316, -0.0011466688010841608, 1.509313702583313, 1.5428330898284912, -1.3359736204147339, -2.167367935180664, -0.1281355321407318, -1.1467127799987793, -0.5339891314506531, -0.9556260704994202, 0.022318249568343163, -1.0761306285858154, -1.2135086059570312, 0.5843735933303833, -1.1787660121917725, 0.6323224902153015, -0.15695349872112274, -1.2216017246246338, -0.4407859742641449, -2.843463659286499, -1.6677474975585938, 0.05374758318066597, 0.8371498584747314, -1.1770180463790894, -0.3766385316848755, 0.558244526386261, 1.1809841394424438, -0.4838930368423462, -0.8875847458839417, 0.08917383849620819, -0.26994091272354126, 0.5784070491790771, 0.5126310586929321, -1.950796127319336, 2.442920684814453, 0.3319195508956909, -0.7814144492149353, -0.2287522405385971, 1.009596347808838, -0.8035865426063538, 1.2631537914276123, 1.0470455884933472, 0.8793503046035767, 1.43162202835083, 0.9665902853012085, -0.32378074526786804, 0.21291331946849823, 0.7311300039291382, -0.6576437950134277, -0.5814017653465271, 0.3346329927444458, 0.13401146233081818, -0.22383375465869904, -1.17972993850708, 0.7772176861763, -1.4715749025344849, 0.43087026476860046, -0.6253864765167236, 0.7584417462348938, -0.47131699323654175, 1.043225646018982, 0.9758583307266235, 0.5666764974594116, 0.6748603582382202, -0.38991889357566833, 0.8406528830528259, -1.8361114263534546, 0.45754173398017883, 0.4320371747016907, -0.0334378182888031, -1.33966064453125, -0.3946704566478729, 1.0829734802246094, -0.4476736783981323, -1.3260325193405151, 0.5572938323020935, -0.9675045609474182, 0.8764700889587402, -2.6052327156066895, 0.2762206196784973, -1.1390177011489868, 1.328835368156433, -1.2678784132003784, 1.6141221523284912, 0.2904606759548187, 0.10201980918645859, -0.2774590849876404, 0.312003493309021, -0.49493274092674255, -0.09900148957967758, 0.8587612509727478, 1.4860293865203857, -0.47755178809165955, 0.41741788387298584, -1.2540603876113892, -0.8792549967765808, -0.32103803753852844, -1.7907406091690063, 1.0393377542495728, 0.5901261568069458, -0.6934145092964172, 0.7952927947044373, 2.1178088188171387, -0.5423913598060608, -0.14848542213439941, -0.894422173500061, 1.0549243688583374, -1.135704755783081, -0.3610697388648987, -0.41971713304519653, -0.8618658185005188, -0.601970911026001, -0.3393827974796295, -0.048203565180301666, -0.9196115136146545, 0.8428791761398315, -1.2816234827041626, 0.1549903154373169, -1.0787060260772705, 1.3703036308288574, 0.16798606514930725, 2.040540933609009, -0.9483002424240112, -1.881478190422058, -0.41789230704307556, 0.2554454207420349, 0.39026933908462524, -0.6778831481933594, 0.4248957335948944, -0.36638328433036804, -1.549811601638794, 0.6558873057365417, 2.198155164718628, 0.7482948303222656, -1.5467548370361328, 1.7409507036209106, 0.26336032152175903, -1.0640487670898438, 0.883938193321228, 0.7022055387496948, 0.05473184585571289, -0.22035327553749084, 0.273297518491745, 0.4432555139064789, -0.9882465600967407, 0.4156133532524109, -2.801711320877075, 0.002381987404078245, 0.7860181331634521, -0.14896924793720245, 0.36795222759246826, -1.0369302034378052, 0.2484152764081955, 0.10477291792631149, 0.05447183921933174, 0.6560193300247192, 0.09555117785930634, 0.4640560448169708, -0.4009586572647095, -1.1362383365631104, -1.0513794422149658, 1.1057724952697754, 1.1730214357376099, 0.005196722224354744, 0.449638694524765, -0.6722992658615112, -0.7208244800567627, -0.31989169120788574, -0.3027438223361969, -1.5785014629364014, -0.4312453866004944, 0.09139983355998993, -0.3284362852573395, -1.1877003908157349, -0.3579123020172119, 1.7995245456695557, -1.5389363765716553, 0.7293612957000732, -1.0792241096496582, -1.5985934734344482, 0.11488007009029388, -0.11931347846984863, -0.043467141687870026, -0.09234538674354553, -1.158107876777649, 1.2718870639801025, -0.6536540389060974, -0.9826815128326416, 0.9883365631103516, 0.5317148566246033, -1.4521093368530273, -0.9942646622657776, -0.4762536585330963, 0.4533190429210663, -0.7488157153129578, 0.327328622341156, 1.6831227540969849, 1.2568538188934326, 0.03955051675438881, -0.032651323825120926, 0.2825292944908142, -0.30304643511772156, -0.627112627029419, -2.2275421619415283, 0.45598986744880676, -1.0966752767562866, 2.149852991104126, 1.748573660850525, 0.181975319981575, 1.988546371459961, -0.7740027904510498, 0.4889242649078369, -2.4427688121795654, -0.7420938611030579, 0.7796277403831482, 1.021468997001648, -1.1092112064361572, -3.086827039718628, -2.2852678298950195, 0.7657787203788757, 0.5571679472923279, 1.627994418144226, -0.317442387342453, 0.38491275906562805, -0.2859763205051422, -1.4072424173355103, -0.2304823398590088, 1.895390272140503, -0.7630948424339294, 1.2394858598709106, 1.4580262899398804, -0.41837549209594727, 0.23376275599002838, -1.2687091827392578, 0.7787940502166748, -0.308485209941864, 0.3327813148498535, -0.27723416686058044, -0.3517551124095917, -1.3712164163589478, -0.6264439821243286, 1.6414473056793213, -0.7151317596435547, 1.8344323635101318, -1.0042294263839722, -0.5481129884719849, -0.28595924377441406, -0.4302310049533844, -0.5500300526618958, 1.6855472326278687, -1.2092037200927734, 1.2005020380020142, 0.9922617673873901, -2.0994696617126465, -1.1025136709213257, -0.3594506084918976, 1.3757060766220093, -0.33491021394729614, -0.6678354144096375, -1.050399661064148, -0.12267042696475983, 2.351602554321289, 0.08012714236974716, -0.612231433391571, -0.6740942597389221, 0.08725127577781677, -0.845848798751831, -2.3589913845062256, 0.2858598232269287, -0.3628535270690918, -0.3814676105976105, 0.30363190174102783, -0.9484714269638062, -0.10352328419685364, -0.007537387777119875, 2.983144521713257, 0.47626298666000366, 0.4377847909927368, -0.07789339870214462, 2.7145919799804688, 0.6775490045547485, -1.7984654903411865, 1.1021422147750854, 0.3313107192516327, 1.1038362979888916, 0.46240437030792236, -0.31699588894844055, -0.9484958648681641, 0.8877770900726318, -1.3687721490859985, -0.55448979139328, -1.1125613451004028, 0.4618799686431885, -0.024020293727517128, 0.5067133903503418, -1.17561936378479, -0.7384700775146484, 1.1486928462982178, 0.5023213028907776, -0.44209814071655273, 0.12281271070241928, -1.1231070756912231, 0.07853978872299194, -1.6805790662765503, 0.9817181825637817, 0.46654003858566284, 0.10564501583576202, 0.059134241193532944, -0.07575743645429611, 0.24474918842315674, -1.6176915168762207, 1.6204874515533447, -0.464120477437973, 1.5697968006134033, -0.7026621103286743, 0.8675503730773926, -0.3693143129348755, -0.9889519214630127, -0.8659191131591797, 0.9056309461593628, -1.1689348220825195, 0.43561166524887085, -2.0927958488464355, 0.027317054569721222, -0.24018998444080353, 1.1892263889312744, 1.4348175525665283, 1.5181190967559814, 0.0735522210597992, -1.0773820877075195, 0.7124460339546204, 0.5053903460502625, -1.3185285329818726, -1.6813257932662964, -0.6259422898292542, 0.007623222190886736, 0.09635978192090988, 1.0702097415924072, 0.8631770014762878, -0.5278443098068237, -1.167865514755249, 1.48858642578125, -0.018155820667743683, -1.1309716701507568, 1.909393310546875, -0.285421222448349, -2.3891031742095947, -1.6379746198654175, 0.8807010054588318, -0.07807235419750214, -1.3058241605758667, -0.42952290177345276, 1.900055170059204, -0.28694793581962585, 1.7751003503799438, -0.9365084767341614, -1.5398801565170288, -1.9917570352554321, 0.4164886772632599, 0.9567722678184509, -1.473217487335205, 0.5649948716163635, -0.36854884028434753, -0.8637589812278748, -1.6261799335479736, -1.5617616176605225, 0.031055951490998268, 0.24658440053462982, -0.9993363618850708, 1.1305949687957764, -1.0405199527740479, -3.052600622177124, 1.0520964860916138, 0.5484849810600281, 1.7955583333969116, -1.1101250648498535, -1.2914131879806519, -1.5355974435806274, -0.3912678062915802, -1.1580902338027954, 1.588248372077942, -0.5020186901092529, 0.9542187452316284, 1.885123372077942, -0.67886883020401, -1.1825522184371948, -0.4617466628551483, -0.22680498659610748, 1.1895815134048462, -0.3713996708393097, 1.350007176399231, 3.117493152618408, 0.5579873919487, -0.5442939400672913, 1.1172913312911987, -0.09998497366905212, 1.0948456525802612, 0.44096386432647705, -0.5502506494522095, 0.9635292887687683, -0.5700703859329224, 1.0738673210144043, 0.9875538945198059, -0.2733203172683716, 0.5090802311897278, 0.8799405097961426, 1.1912734508514404, 1.5874016284942627, 1.1851282119750977, 1.252692699432373, 1.2388042211532593, -0.842887282371521, 0.18116284906864166, 0.46546047925949097, -0.11687856167554855, -0.011883054859936237, 0.6267075538635254, 1.386290431022644, 0.6534451842308044, -0.5757058262825012, -0.8194864988327026, -0.6389937996864319, 0.3790305256843567, 1.3457931280136108, 0.852904736995697, -0.39303749799728394, 0.7527689933776855, 0.6030824780464172, -0.2674493193626404, 1.0533268451690674, -0.8788332343101501, 1.1815229654312134, 1.597163200378418, -1.2087727785110474, -0.3931840658187866, 0.48013749718666077, 0.45047932863235474, 0.8883625268936157, -0.49553734064102173, 0.6522260308265686, -0.807529628276825], [0.9077800512313843, 1.7556160688400269, 0.6283040642738342, -0.9609622359275818, -0.10427555441856384, -1.2975807189941406, -0.9519420266151428, 0.40659192204475403, -1.79591703414917, -2.552755832672119, 1.171716570854187, -0.5293439626693726, 0.6436038613319397, -0.6217578053474426, 1.3234736919403076, 0.09414509683847427, 0.006110569462180138, -1.1390968561172485, -0.10039631277322769, -0.0972132682800293, -0.6501402854919434, -0.3201594948768616, 1.48544180393219, 0.4630724787712097, -0.1855255663394928, -1.1972566843032837, -0.27825039625167847, 0.025924600660800934, 0.8886898159980774, 1.4944878816604614, -0.0469992496073246, 1.5750442743301392, -0.41327720880508423, -0.7203894257545471, -0.6108707189559937, -0.645233690738678, -1.2133179903030396, -1.7322546243667603, 0.23261862993240356, 0.4676635265350342, 0.9323716163635254, 0.1401929259300232, -0.5864021182060242, -0.6675640940666199, 0.8779018521308899, 0.41143539547920227, -0.8491775393486023, 0.6290664076805115, -0.06481847912073135, -1.150253415107727, 0.8086470365524292, -1.323710322380066, 0.07677073776721954, -1.159978985786438, -0.7356614470481873, 0.1487455666065216, 0.31775057315826416, -1.9292582273483276, 2.019054412841797, 0.5426813364028931, -0.5875821113586426, -0.23125231266021729, 1.4712222814559937, -1.2280479669570923, 0.920240044593811, 0.5630586743354797, -0.03107343427836895, -0.22624659538269043, -0.6033917665481567, 0.6120248436927795, 0.7115068435668945, -0.7806550860404968, -0.6012951135635376, 0.02701517567038536, 0.03447462245821953, -0.36030876636505127, 0.19794927537441254, 2.78590989112854, -0.5301574468612671, -0.6404052376747131, 0.8105363249778748, 1.493530511856079, -0.544700562953949, -0.21139241755008698, 0.3410708010196686, 2.2324564456939697, 0.8285294771194458, -0.17396695911884308, 0.5223828554153442, -0.5251563787460327, 0.5265015363693237, -0.4690752327442169, 1.2574125528335571, 1.271848440170288, -0.10495876520872116, -0.5804773569107056, 0.7799605131149292, 0.19595283269882202, 0.9617790579795837, -0.4917655885219574, 0.5650864839553833, 0.6229747533798218, -0.6949862837791443, 1.2931287288665771, 0.19543084502220154, -0.5253932476043701, -0.6163126826286316, -0.9921830296516418, 0.08350753039121628, -1.5257936716079712, -0.3702426552772522, -0.5231432914733887, -1.492501139640808, 0.4686131179332733, -0.1633128970861435, -0.6517549753189087, -0.9641119241714478, 0.40868669748306274, 0.43776240944862366, -1.1615993976593018, 0.6452373266220093, -0.4597834348678589, 0.9655482172966003, -0.5154799818992615, 0.6192397475242615, 0.41614460945129395, -0.2580152153968811, 0.33530107140541077, -0.1397879421710968, 1.1705878973007202, 0.157823845744133, 0.04287634789943695, 0.9936615228652954, -1.2345845699310303, -0.7306605577468872, 0.4563796818256378, 0.3870713710784912, 0.1871987283229828, -0.06925635039806366, 0.27153992652893066, -0.17829112708568573, -1.1624304056167603, -0.861548900604248, -0.5043119788169861, -1.7487646341323853, 0.9666041135787964, 0.03782202675938606, -1.110281229019165, -0.8231501579284668, 0.9858615398406982, 2.109137535095215, 1.405483365058899, 1.2800589799880981, 0.06264374405145645, 1.1956887245178223, -0.4873211979866028, 0.035399116575717926, 1.6454945802688599, -0.3479563593864441, 0.014532770030200481, -0.019642340019345284, 1.3143603801727295, 1.8215599060058594, -1.9732872247695923, 0.4509996473789215, 1.1129188537597656, 0.9815627336502075, 0.28322356939315796, 0.7084375619888306, -1.829553246498108, -0.9562811851501465, 0.48963502049446106, -2.2472848892211914, 0.5276117324829102, 0.011186981573700905, -1.0336799621582031, -0.5933802723884583, -0.14296512305736542, -0.7032153010368347, 1.8277660608291626, 0.011411573737859726, -0.2549375891685486, -0.67313551902771, -1.292543888092041, -0.7631529569625854, 1.88699209690094, -0.6036509871482849, 0.2755804657936096, -1.1810719966888428, -0.7572512030601501, -2.324876308441162, 0.851018488407135, 0.23593616485595703, -0.5451927781105042, 0.9291726350784302, 0.4427889883518219, -1.5560461282730103, 0.010371895506978035, -1.0636218786239624, -1.1652947664260864, -0.24172179400920868, 0.6539893746376038, 0.464465856552124, -0.06303136795759201, -0.9812638163566589, -0.6013231873512268, -1.450758457183838, 0.9020664691925049, -0.4153865575790405, -1.02162504196167, 0.6749210357666016, -0.7744354605674744, -0.6713040471076965, -0.838720440864563, -0.049843769520521164, 0.5014492273330688, -0.14113537967205048, 0.8746488690376282, 0.5828680396080017, -2.279611349105835, 0.25512585043907166, -0.5347036719322205, 1.030186653137207, 1.1936477422714233, 0.5769443511962891, 0.40861776471138, 1.2079858779907227, -0.07362375408411026, -0.3348393142223358, -0.13171957433223724, 0.3344452381134033, 0.6500437259674072, 0.10730867832899094, 0.48226574063301086, 2.0984067916870117, 0.8661181926727295, 0.7841762900352478, 0.587136447429657, -2.3859152793884277, -0.922257661819458, 1.2251207828521729, -1.1644214391708374, -0.03758881986141205, -0.5995625257492065, 0.36722666025161743, 0.05224137380719185, -1.2900657653808594, -0.04285871982574463, 0.4425010681152344, -0.36877191066741943, 0.5981134176254272, 0.30279994010925293, -1.7827602624893188, -0.15891948342323303, -0.21987396478652954, 1.049161672592163, 0.350821852684021, 0.10380798578262329, -0.548897385597229, 1.3771255016326904, -1.5659468173980713, 1.1375551223754883, 0.6283597946166992, 1.1982219219207764, 1.3976119756698608, 0.6602387428283691, 0.8045106530189514, -1.3053323030471802, -1.788826584815979, 0.8594428300857544, -0.7675606608390808, -0.36607545614242554, -0.7973914742469788, -1.3512802124023438, -2.0552666187286377, -0.034625016152858734, 0.567160427570343, 2.3155620098114014, -1.031559944152832, -2.1432411670684814, -1.1371469497680664, 0.08636672049760818, -1.685890555381775, 0.9125415086746216, 0.8969946503639221, -1.0094645023345947, -0.005296394694596529, 0.4128371477127075, -0.6489253044128418, 0.04377897456288338, -0.8898561596870422, -0.4276058077812195, -1.1541379690170288, 1.640769362449646, 0.34546518325805664, -0.7160206437110901, 0.7467393279075623, 0.19925157725811005, 0.2492542266845703, -1.9699751138687134, 0.7485661506652832, 0.7240801453590393, -1.287366271018982, -1.0227035284042358, -1.4838573932647705, 0.21506637334823608, -1.2696480751037598, -1.0758236646652222, 0.6640255451202393, -0.3994898200035095, -0.9175216555595398, 0.40363526344299316, 0.23180140554904938, 0.5171780586242676, -0.4782083332538605, 0.45903900265693665, -1.0152184963226318, -1.7668380737304688, 0.9405817985534668, 0.866320013999939, 0.27961283922195435, 0.017657695338129997, -0.6563723683357239, -0.47037866711616516, 0.7814960479736328, -2.9142274856567383, 0.6619610786437988, 0.01728837378323078, -2.3373634815216064, -1.158096194267273, 0.05675379931926727, -1.4495412111282349, -0.4637205898761749, -1.5246660709381104, -0.4317263662815094, -0.06664476543664932, -0.2577292025089264, 2.336719512939453, -0.026314392685890198, -2.5645411014556885, -1.972976565361023, 0.7054771184921265, 0.37898755073547363, -0.8187095522880554, 0.9480679035186768, -1.3997089862823486, -0.5779602527618408, -0.5755128860473633, -0.21922044456005096, -0.13230077922344208, 0.2970869243144989, 0.454719215631485, -0.7930809855461121, -0.1266670525074005, -0.10026130825281143, 0.05083741247653961, -0.7561959624290466, 0.05795012041926384, 0.044636741280555725, -1.0229403972625732, 0.6994542479515076, 0.6581281423568726, -1.0370482206344604, -1.8610464334487915, -0.5283253192901611, 1.8724234104156494, -1.1036853790283203, -0.21600593626499176, 0.23767514526844025, 1.0386133193969727, -1.4146592617034912, -1.0250437259674072, -1.1510541439056396, 0.5490779876708984, -0.36067691445350647, -0.9841570258140564, -1.5536978244781494, 0.8667891025543213, 1.0225356817245483, -0.6961015462875366, -0.8691651225090027, -3.161278486251831, 1.4145264625549316, 0.8889188766479492, -0.5939409136772156, -0.8830332159996033, 0.023085113614797592, -1.0256052017211914, -0.15830856561660767, -0.07292728871107101, -0.032559726387262344, 0.044068992137908936, -0.42711341381073, 0.6493098139762878, -0.15560264885425568, 2.2452030181884766, -0.4627144932746887, 0.20749744772911072, 0.39356324076652527, 0.5271111130714417, -0.6631575226783752, -0.5421063899993896, 0.7177698612213135, 0.4593815207481384, 1.5463323593139648, -0.28023457527160645, -0.5802338719367981, 1.1473567485809326, 0.25553107261657715, 0.14142867922782898, 0.8794581294059753, -0.9399523735046387, -0.2629491090774536, 1.1539243459701538, -1.4156081676483154, 0.7226061224937439, -0.08890961855649948, -0.9850577712059021, 0.15509158372879028, 0.956143319606781, -0.14126218855381012, -0.9660553932189941, 0.01700703054666519, -0.7280910611152649, 0.6608155369758606, -0.32855555415153503, -0.6566438674926758, 2.683717966079712, 0.6105748414993286, 0.0007138713845051825, 0.24797821044921875, -0.513799250125885, -2.3627004623413086, 1.190025806427002, -1.8824379444122314, 0.7392211556434631, -0.6654269099235535, 0.08028353750705719, 0.9154883623123169, 0.27958282828330994, 0.8041413426399231, 0.7829177975654602, 1.3868646621704102, 1.889338731765747, 0.3590761125087738, 0.6256190538406372, 0.7409349679946899, 1.480489730834961, 0.8606085777282715, 0.21952871978282928, 0.5406208038330078, -1.1661574840545654, -0.527649462223053, -1.9319733381271362, -0.8819485902786255, 0.8528774380683899, 0.6906915307044983, -0.7134168744087219, 0.3586351275444031, -1.715779185295105, 0.1691886931657791, -1.2922961711883545, -0.19222436845302582, 0.046865515410900116, -0.9863992929458618, -0.25469261407852173, -0.910545825958252, -0.4343615472316742, 0.486891508102417, -1.2536991834640503, -0.1549655944108963, -1.3910176753997803, -1.3345730304718018, -0.8871607780456543, -1.7366410493850708, 0.08069700747728348, 0.05877147242426872, -0.24573495984077454, -0.4075515866279602, -1.3413782119750977, -1.0415362119674683, -0.5955769419670105, -0.3151454031467438, 0.05942252278327942, -0.06425943970680237, -1.025415301322937, 0.35664597153663635, 1.1150093078613281, 0.2687011957168579, 0.6471647620201111, 0.2822421193122864, -0.8769938945770264, -0.7355680465698242, 0.300182580947876, -1.9164034128189087, 0.01700381189584732, -0.8369572162628174, -0.36592230200767517, -1.8953678607940674, -0.06033730134367943, 1.697264313697815, -1.7794851064682007, 1.6709834337234497, 0.34068647027015686, 0.0628437027335167, -1.4055777788162231, -1.6865006685256958, -0.9862378835678101, -1.698633074760437, -0.2740234434604645, -0.29230761528015137, 1.1019867658615112, -0.14618855714797974, 1.006247639656067, 0.05747602507472038, 1.5044913291931152, 0.46866002678871155, -2.538783550262451, 0.27214017510414124, 1.1840988397598267, -0.5875915884971619, -0.030780058354139328, -0.31327512860298157, -0.12976795434951782, 1.4934275150299072, 0.452717661857605, -0.6068728566169739, -0.0062935492023825645, -0.44968685507774353, -0.22769665718078613, 0.1645473837852478, 0.6729192137718201, -0.05337602645158768, -0.3680683374404907, -0.05432957410812378, 0.22216154634952545, -0.026867033913731575, -1.4974244832992554, -1.852669358253479, -1.1285382509231567, 0.26256075501441956, 0.9677467346191406, -0.5714126229286194, 0.4559427499771118, -0.7006901502609253, -0.8125601410865784, 1.2775492668151855, 1.4667121171951294, -0.46411219239234924, 0.44990742206573486, -0.4769239127635956, 0.3276895582675934, 0.0479276180267334, -1.3489550352096558, 0.30703726410865784, -0.5954835414886475, -0.6729979515075684, 1.4024113416671753, 0.10508139431476593, 0.15342608094215393, 0.9074168801307678, 1.6514567136764526, 0.3768952786922455, -0.656406044960022, -0.6258063316345215, 0.524486243724823, -0.3116714358329773, 0.025569766759872437, -0.6380428671836853, 0.0031253090128302574, 0.8844201564788818, 0.9201886653900146, -0.3386162519454956, -1.638757348060608, -0.12494464218616486, -1.3848546743392944, -1.1236045360565186, -1.0181058645248413, -0.7928955554962158, 0.25764501094818115, 0.6836369037628174, -0.8938696980476379, -0.02371896617114544, 0.7022407054901123, -0.5168822407722473, 0.8956632614135742, -0.5836119651794434, 0.8535731434822083, 0.7172467112541199, -0.19747015833854675, 0.8551895022392273, -0.2534962296485901, -0.24456869065761566, 2.4577183723449707, -0.11096370220184326, -0.5406619906425476, 0.1016382947564125, -2.436309337615967, 0.34253114461898804, -1.4521503448486328, 0.22930262982845306, 0.4117569327354431, 0.5907361507415771, 0.5634835362434387, 1.3770866394042969, -1.6104117631912231, -1.0414491891860962, -0.8623737096786499, -0.06179886311292648, -0.5427431464195251, 1.4681951999664307, 0.621213436126709, 0.8117952346801758, 0.6060369610786438, -1.059609055519104, -1.2185760736465454, -0.12010238319635391, -0.050220612436532974, 1.1263717412948608, -1.448426365852356, 1.9101060628890991, -0.27613499760627747, -0.04341201111674309, 0.15521661937236786, 1.2311259508132935, 0.2192712426185608, -1.4257104396820068, -0.9963250756263733, 0.857581615447998, 1.6620995998382568, -0.5671488046646118, -0.9323227405548096, -0.5818582773208618, 0.1855473667383194, 1.0357849597930908, -0.4170887768268585, 0.6952689290046692, -0.5069380402565002, -1.454824447631836, 1.0091887712478638, 0.1145954504609108, 1.3374083042144775, -0.12263315171003342, 0.6841726899147034, -1.097777247428894, -0.1449463814496994, 0.7561526894569397, 0.19706401228904724, 0.3507305383682251, 0.5500654578208923, -1.0032073259353638, -0.6044771075248718, 0.8394357562065125, 0.8997707366943359, -0.2897479236125946, 0.29479560256004333, 0.5316106677055359, 1.883399486541748, -0.7339882254600525, 0.30178436636924744, -1.3201967477798462, 0.8997961282730103, 0.48942020535469055, -1.6028633117675781, -1.3458236455917358, 0.6927767395973206, 0.3675766885280609, 0.03482001647353172, 0.11106386035680771, -1.2662041187286377, 0.7009124755859375, 0.5118703842163086, -0.2828308045864105, 1.2498522996902466, -0.6776412725448608, 1.4609054327011108, 0.12701483070850372, -0.11110490560531616, -0.8968508839607239, 0.09245310723781586, 0.35454365611076355, -1.697576880455017, 0.12980511784553528, -1.0991930961608887, -0.34243491291999817, 1.9354029893875122, 1.1232329607009888, -0.2624046504497528, 0.9440948963165283, -1.1790001392364502, 0.07740062475204468, 1.2538877725601196, 0.43636053800582886, 2.2110953330993652, -1.36065673828125, 0.4406740069389343, -1.151883602142334, -0.03878948837518692, 0.04754483699798584, -0.7564898133277893, 0.4211817681789398, -2.093357563018799, -0.5587213039398193, 0.27816852927207947, -0.5349478125572205, 0.5295448899269104, 0.9541310667991638, -0.08891170471906662, 1.685168743133545, 0.2571277916431427, -0.3207630217075348, -0.23540213704109192, -0.4788561165332794, 1.2179477214813232, -0.2524661123752594, -0.22936497628688812, -0.30457091331481934, -0.9470067620277405, -0.9540919661521912, -1.2049224376678467, -0.47914376854896545, 0.4143091142177582, -0.34948310256004333, 0.4116511940956116, 0.35694360733032227, -0.007335781119763851, -0.8308058381080627, 0.20633259415626526, 0.5114479064941406, 1.0120782852172852, 1.411410927772522, 0.2621013820171356, -0.9013822078704834, 0.3206045627593994, -1.023445725440979, -1.1436290740966797, 0.9509499669075012, 0.6305738687515259, 2.7060296535491943, 0.9884545803070068, 0.48313525319099426, -0.6207050681114197, -0.6848803758621216, -0.7930598258972168, 1.2826931476593018, -0.9514222145080566, -0.33321595191955566, -0.20310020446777344, 1.0493568181991577, -0.5540373921394348, -0.1016429215669632, 0.8473776578903198, 0.5306090712547302, 0.9689002633094788, -2.416306734085083, -1.6403348445892334, 0.4266800582408905, 0.7723069190979004, 0.17452700436115265, 1.058598518371582, -0.05362696573138237, 0.6403262615203857, -0.1724093109369278, 0.7004328370094299, 0.7339633107185364, 1.4897135496139526, 0.536809504032135, -0.5195921659469604, 0.5304266214370728, 1.4468817710876465, 0.16296948492527008, 0.4677656888961792, 0.8540647625923157, -0.6277816891670227, -0.8548394441604614, -0.8560183048248291, 0.6943073868751526, -0.0883474126458168, -0.7173181772232056, 0.6889382004737854, -0.5808669924736023, 0.03169727325439453, 1.4864099025726318, -0.09877543896436691, -0.42448028922080994, 0.015256619080901146, -0.4507116675376892, 1.301655888557434, -0.9468323588371277, 1.6839412450790405, 0.612697958946228, 1.399013876914978, 0.11484558135271072, 0.8978897333145142, -0.3238418698310852, 1.5813024044036865, -0.24387229979038239, -1.4202936887741089, 1.3245553970336914, 0.009525665082037449, -0.6389111280441284, 0.5341455340385437, -0.8747695684432983, -0.23000696301460266, -0.6633919477462769, -0.40344443917274475, 0.1770896017551422, -0.38396453857421875, 0.30486568808555603, -1.9586021900177002, 0.4516075849533081, -1.2022192478179932, 0.7150282263755798, -0.28007903695106506, 0.522777795791626, 0.6050422787666321, 0.14855927228927612, -1.106029748916626, 1.0213959217071533, -0.45142433047294617, -0.8024172186851501, -0.3179459571838379, -1.395124912261963, -0.729188859462738, 0.11481653898954391, -0.6885122060775757, 0.04048033803701401, -0.21620945632457733, 0.4649525284767151, 0.22197632491588593, -0.9366338849067688, 0.09266314655542374, 1.2558062076568604, 1.1914042234420776, 2.6494650840759277, -0.37861889600753784, 0.0871565043926239, -0.20396582782268524, 0.4205242991447449, 2.3053648471832275, 1.3044047355651855, 0.8240647912025452, -2.2666168212890625, 0.25561589002609253, -1.1542655229568481, 1.174976110458374, 1.1387648582458496, 1.2857136726379395, 0.8876950144767761, -0.527877688407898, 0.8229743838310242, 0.2552664577960968, 1.0896915197372437, -0.45417243242263794, -1.7975329160690308, 0.6632980704307556, 1.6833226680755615, -0.9025076627731323, -0.49688446521759033, 0.1546602100133896, -0.13537293672561646, -1.278021216392517, -2.899061679840088, 0.6258859634399414, 1.0840331315994263, -1.307646632194519, 0.6933175325393677, 0.2943883538246155, -0.48533937335014343, -0.6285960078239441, -0.11144091188907623, -1.418402075767517, -1.0753816366195679, -0.14297686517238617, -0.7360630631446838, 1.3446688652038574, 2.296727180480957, -0.08514812588691711, -0.22317489981651306, 0.09683883190155029, -1.780672550201416, -0.26153331995010376, 0.1556532233953476, 1.0382627248764038, 0.505488395690918, 0.3370736837387085, -0.45293617248535156, -0.49896591901779175, -1.2956737279891968, 0.8976854085922241, 1.1742517948150635, -1.7918277978897095, 0.14684756100177765, 0.20396167039871216, -0.6661524772644043, 0.4811069667339325, -0.999008059501648, -1.851294994354248, -1.189956784248352, 0.6324893236160278, -1.832513689994812, -1.3447703123092651, 1.8530445098876953, -0.5942183136940002, 0.45845696330070496, 0.031391412019729614, -1.2790443897247314, -1.017486572265625, -0.7464476227760315, -1.1672931909561157, -0.1775742918252945, 1.178060531616211, 0.46466952562332153, -0.7249066829681396, -0.6316699385643005, 1.1040759086608887, 0.6507282853126526, 0.47763121128082275, -1.8947118520736694, -1.389431357383728, 0.8766940832138062, 0.962582528591156, -0.2704756259918213, -0.8925169706344604, -1.0075722932815552, -0.6889628171920776, -0.8491866588592529, 0.4587208032608032, -1.441322684288025, 0.987897515296936, -0.1033497229218483, 0.3607955574989319, 0.4897298216819763, 0.6806014776229858, -0.6589815020561218, 1.9440767765045166, -0.04642561823129654, -0.34459206461906433, -1.0556979179382324, -0.5013266801834106, -0.08447664976119995, -0.06753852963447571, 1.018642544746399, 1.8310191631317139, -0.7961593270301819, 0.2658230662345886, -0.05026708170771599, 0.3921244144439697, 0.1883605420589447, -0.5629869103431702, -0.48809024691581726, -0.0052479407750070095, 1.5193850994110107, 1.2762264013290405, 0.3975962698459625, 0.03513076901435852, -0.1722462922334671, -0.4572807848453522, 0.7346091866493225, 1.218123435974121, 1.3726091384887695, -2.2666866779327393, -0.6964607238769531, 2.546997308731079, 2.4560766220092773, -0.3044465482234955, 1.6528816223144531, -0.5347180962562561, -1.8014615774154663, 0.6899145841598511, 0.09790563583374023, 0.3977165222167969, -1.3008053302764893, -0.5875410437583923, -1.3811205625534058, -0.993446409702301, -0.9712936878204346, -1.0650808811187744, 0.2051512897014618, 1.673108696937561, -0.19293589890003204, -1.3521984815597534, -0.8132460713386536, -0.6485452055931091, 0.185686394572258, -1.8763562440872192, -1.5674234628677368, -0.7512935400009155, -0.15257428586483002, 2.5950591564178467, 0.36919212341308594, -0.08708617836236954, 0.6221252083778381, 1.9289512634277344, -0.24170371890068054, 0.6259817481040955, 1.3611277341842651, 1.5925564765930176, 0.1832348257303238, 1.3235054016113281], [-1.9758182764053345, -0.33765944838523865, 2.25126576423645, -0.49845635890960693, 0.6263365745544434, 0.3667249083518982, 2.1436240673065186, 1.2607347965240479, -0.09096608310937881, 1.5450153350830078, -0.6666735410690308, -1.2425955533981323, -1.564988613128662, -1.1370387077331543, -1.225142240524292, -0.001143546891398728, -0.14265134930610657, -0.06356751918792725, 0.2763320505619049, 0.3913480341434479, -2.8275716304779053, -1.1515207290649414, 0.5511866211891174, 0.01437748409807682, -0.15182125568389893, 0.6587173342704773, 1.3308228254318237, 0.03249475359916687, 1.495007872581482, 0.5957090854644775, -0.16044223308563232, 0.8426550030708313, -1.7256484031677246, 0.19121277332305908, 0.29462796449661255, -0.7353639006614685, -0.38701218366622925, -0.15118514001369476, 1.0616477727890015, -0.3607851564884186, 1.462241768836975, -1.72086501121521, -0.4946896433830261, 0.18577943742275238, -1.3320623636245728, 0.13072463870048523, -0.27592507004737854, -0.1601775884628296, 0.6011609435081482, 1.237699031829834, -0.18893297016620636, 0.9385492205619812, -0.7405223250389099, -0.7491301894187927, -0.08456243574619293, 0.3044910430908203, 1.3475407361984253, -0.7616375088691711, -0.6105441451072693, 0.5891053676605225, 0.9504708051681519, -0.1557038277387619, -0.6922545433044434, 0.993761420249939, 0.48880717158317566, 0.01779012940824032, 0.14068503677845, 0.915636420249939, 0.9444862008094788, 0.19010421633720398, 0.9838821887969971, -0.5202401876449585, -2.0382347106933594, -0.40395262837409973, 0.46675312519073486, -0.7953665256500244, 1.235093593597412, -0.1520129293203354, -0.8901571035385132, 0.7466321587562561, -0.2861544191837311, 1.5543807744979858, 1.4684646129608154, 0.35130876302719116, -1.8706755638122559, 0.050382763147354126, 1.1854885816574097, -1.7596830129623413, 1.0905437469482422, 0.7655494213104248, 1.0494482517242432, 1.3057341575622559, -0.640600323677063, -0.6134029030799866, 0.8479790091514587, 0.851362943649292, 0.4588572680950165, 0.5215279459953308, -0.31943756341934204, -0.6132885813713074, 0.832648754119873, 0.285715252161026, -0.24485284090042114, -1.5975992679595947, -1.0582635402679443, 0.11559014767408371, -0.35342949628829956, 2.5099263191223145, 1.782746434211731, 0.06472446769475937, -0.23288093507289886, 0.00985251646488905, 0.46843481063842773, -1.0760499238967896, 0.16703076660633087, 0.584092378616333, -1.0743340253829956, 1.3946130275726318, 0.1578451246023178, -1.3780006170272827, 1.0004252195358276, -0.004244582261890173, -1.2940951585769653, 0.45775970816612244, -1.1318962574005127, 0.9214571714401245, 0.2513064742088318, -1.5657527446746826, 0.20119093358516693, 1.459475040435791, 0.6856747269630432, 0.7079823613166809, 1.2141464948654175, 0.5190271735191345, 0.01988275907933712, -0.7147614359855652, 2.2764644622802734, -0.6442464590072632, -0.5105655789375305, -0.17898601293563843, 0.4300384223461151, -1.2257298231124878, 0.7844504714012146, -2.0517334938049316, 0.4589979946613312, 0.09177865087985992, -0.9715980291366577, 1.4332071542739868, -1.6886483430862427, 0.7492409944534302, 0.11333778500556946, 0.735115110874176, -0.6577579379081726, 0.36251550912857056, 0.171864315867424, -0.1381722390651703, 0.24744628369808197, 0.2783569097518921, -1.2539721727371216, -0.04892485588788986, 1.3481390476226807, -0.12509658932685852, 0.8327540755271912, 0.19994968175888062, 2.074065685272217, -0.26735788583755493, -0.7208018898963928, -1.3954442739486694, 1.077919363975525, 0.33728596568107605, -0.6197435855865479, 1.2802141904830933, -0.267315536737442, -0.5356199145317078, -0.28777846693992615, 0.23195146024227142, -0.9707297086715698, -0.5522128939628601, -0.2245197296142578, -0.48277679085731506, 0.5454279780387878, -1.8389993906021118, -1.4616202116012573, 0.46329593658447266, -2.634445905685425, 1.8522971868515015, -0.3754684031009674, 0.16936534643173218, 0.8485592007637024, 0.10293203592300415, 0.12455787509679794, 1.477528691291809, 0.8924054503440857, -1.1763843297958374, 0.9636744260787964, 0.8966714143753052, 0.9203941226005554, 0.020668359473347664, -0.05208636075258255, 2.1863415241241455, 0.36185505986213684, 2.0942223072052, -2.1855714321136475, 0.8449390530586243, 0.8360273838043213, 1.8730902671813965, 1.8317114114761353, 0.22878973186016083, 0.7922870516777039, 0.6839419603347778, 0.907591700553894, 1.858241319656372, -1.1720722913742065, -0.21086320281028748, -0.599968433380127, 0.06860693544149399, -0.9690653085708618, -0.7608664035797119, 0.26842132210731506, 1.8994386196136475, 0.7421080470085144, -1.2767976522445679, 0.7004880309104919, 0.19757534563541412, 1.0548588037490845, 1.8785103559494019, -1.3394880294799805, -0.00031507067615166306, 1.296066403388977, -2.0291991233825684, 0.304526150226593, -0.09619572013616562, -0.6369010806083679, -1.5454261302947998, -0.8117263317108154, 0.5014474391937256, -0.2358982264995575, -0.08887197077274323, 0.6375076770782471, 0.5379540920257568, -1.2448314428329468, -0.07393897324800491, -0.41088488698005676, -1.0125007629394531, 0.5094711780548096, -1.0755178928375244, -1.4489364624023438, -2.80488657951355, -1.2589856386184692, 1.7693575620651245, -1.6501080989837646, 0.7552359700202942, 0.2334623634815216, 0.0076895044185221195, -0.37826424837112427, 0.5638781189918518, -1.061263084411621, 0.2761066257953644, 0.37529754638671875, -0.305575966835022, -0.45493409037590027, 0.504310131072998, -0.5657707452774048, 0.36423271894454956, 0.07412442564964294, 1.7704780101776123, -1.1986408233642578, -0.6045770049095154, 0.03905482962727547, 0.8567313551902771, 0.5005295872688293, -0.5118411183357239, 0.8105936050415039, -0.24844175577163696, 0.034941934049129486, -0.20151396095752716, -0.7196882963180542, 0.1315680891275406, -1.9041858911514282, 1.248964548110962, 0.45551151037216187, -0.8318330645561218, 0.13977281749248505, -0.6175900101661682, -1.0956921577453613, 1.6193428039550781, -1.0880768299102783, -0.196168452501297, 1.0080841779708862, -0.03601810336112976, -0.06986424326896667, -0.037972938269376755, 1.0516340732574463, 1.380037784576416, -0.5027112364768982, 0.2864994406700134, 0.8018968105316162, -0.2740611433982849, 2.307681083679199, -1.7745214700698853, -0.8628849387168884, 0.6886604428291321, -0.15562798082828522, -0.2738766074180603, 0.5057624578475952, 0.16791285574436188, 1.1890895366668701, -0.8768747448921204, 0.7392125129699707, -0.8160619735717773, -1.620261788368225, 0.3586018979549408, -1.7538542747497559, 0.24596305191516876, 0.5322529673576355, 0.8689993023872375, 0.6068383455276489, -0.6626294851303101, 1.5732436180114746, -1.6106970310211182, 1.2014296054840088, 0.37792012095451355, -0.6423830389976501, -0.6462956070899963, 0.22263631224632263, -0.2686363458633423, 0.5000560879707336, 0.6572467684745789, 0.6792353987693787, 0.6556532979011536, -0.8151885867118835, -0.551318883895874, 1.3314491510391235, 0.10386770963668823, 0.6903232932090759, 0.5857378840446472, 0.9815085530281067, -0.017431914806365967, -0.6761208772659302, 0.720749020576477, 0.8930704593658447, -0.2541082799434662, -0.7996834516525269, 0.5725839734077454, -1.8775148391723633, -0.2496747523546219, -0.3808504045009613, 1.204683542251587, -1.8114513158798218, -0.4117276668548584, 0.11594812572002411, -2.000933885574341, -0.6845307946205139, 1.8960212469100952, -0.4144836664199829, -0.09022119641304016, -1.990248203277588, -0.7547627687454224, 0.4196120798587799, -0.27104344964027405, -1.332110047340393, 0.7698577046394348, 0.2860681116580963, 0.3211004137992859, 0.8194143176078796, -0.4281742572784424, 0.7332022786140442, -2.3112080097198486, 1.6445021629333496, -1.6304423809051514, -1.6899237632751465, -0.8204432725906372, 0.6892983913421631, -0.0751088410615921, 1.2753088474273682, 0.5718979835510254, -0.02250104770064354, -1.522464632987976, 0.7847821116447449, -0.07289887219667435, 1.1103662252426147, -0.232952281832695, 0.5499227643013, 0.17421653866767883, 1.5347504615783691, 0.5606763362884521, 1.201960563659668, -0.41769856214523315, 0.5983911156654358, 1.969070315361023, -0.25885194540023804, 0.13251599669456482, -0.25776904821395874, 1.450959324836731, -0.15814554691314697, -0.774554967880249, 0.7724443674087524, 0.26073157787323, -0.610378623008728, 0.0782894566655159, -0.35248804092407227, 0.4680001437664032, -0.2195974439382553, -0.891920268535614, -2.178043842315674, -0.16657766699790955, -0.8526672720909119, 0.5010319352149963, -0.8212279081344604, -1.0004209280014038, -1.3548840284347534, 0.4160190522670746, -1.6645848751068115, -0.29516908526420593, 1.6266213655471802, 0.06384411454200745, -1.0330644845962524, -0.11037592589855194, 0.6329290270805359, 0.13117055594921112, 1.3258434534072876, -1.9263252019882202, -0.43154481053352356, -0.8750520944595337, 1.1548376083374023, 0.9062613844871521, -0.12794633209705353, -0.41157761216163635, -0.4234973192214966, 0.4978649616241455, -1.1116175651550293, 0.606782853603363, 0.04852491244673729, 0.0020026566926389933, -0.8044466376304626, 0.8901655077934265, 1.96608567237854, 1.752583384513855, 0.20489726960659027, 0.2975606918334961, -0.6223971247673035, -2.7393550872802734, 0.9399062395095825, -0.4382932484149933, 0.8361562490463257, -0.07770103961229324, 1.9817590713500977, 0.21186713874340057, -0.31977328658103943, -0.22750253975391388, 0.7137048244476318, 1.0585564374923706, 0.6262229681015015, -0.09070488810539246, -0.388776034116745, -0.007354261819273233, -3.2533109188079834, -0.7672701478004456, -0.46982845664024353, -1.3068418502807617, -0.6961978673934937, 1.2076175212860107, -0.9176244139671326, -0.6677616834640503, 1.3808590173721313, -0.7632386684417725, 0.3301275372505188, -0.21291497349739075, -1.5065754652023315, 1.0993931293487549, -0.2670150101184845, -1.0515810251235962, -0.6342755556106567, -0.4090978503227234, -0.6095861792564392, -0.7655274271965027, 0.03356028348207474, 1.8619730472564697, -0.15288135409355164, -0.945958137512207, -0.9577043652534485, 1.8806251287460327, 0.09776949137449265, 0.42026111483573914, -0.30161312222480774, 0.7288384437561035, 0.81405109167099, -1.0429173707962036, -2.0984561443328857, -0.7605220079421997, 0.4284106194972992, 1.4586548805236816, -1.7077088356018066, 1.6376038789749146, 1.0861001014709473, 1.420753836631775, -2.1434566974639893, -1.649936556816101, -0.6503209471702576, -0.5767516493797302, 0.21396572887897491, -1.1647356748580933, 0.017814049497246742, 0.17584699392318726, -1.3431570529937744, -0.16366782784461975, -0.30877190828323364, -0.2952755391597748, 0.4366667568683624, -0.898949384689331, 0.029362808912992477, -0.953692615032196, -0.06678716093301773, -1.1861766576766968, 0.863257884979248, 0.5255902409553528, -0.3270544409751892, -0.07609666138887405, -0.517041802406311, -0.3561154007911682, 0.622569739818573, 0.14645245671272278, -0.7225713133811951, 1.0930780172348022, 0.3815551996231079, 1.3124020099639893, 0.5585516691207886, -1.7280882596969604, -0.6374596953392029, 1.0794093608856201, -0.7888523936271667, 0.9874245524406433, -0.32920515537261963, 0.6206609010696411, -0.05474776402115822, -1.2837635278701782, -1.8705755472183228, -0.5291350483894348, 0.08084136992692947, -1.6097581386566162, -1.3370593786239624, -1.2134270668029785, 0.4572349488735199, -0.7214440703392029, 1.3031684160232544, 2.0898079872131348, 0.4585084021091461, -1.0326966047286987, 0.24902696907520294, 1.0275555849075317, 0.9439836144447327, -0.2629704177379608, 0.655944287776947, 1.0529245138168335, 0.3092934191226959, -0.3521251976490021, -1.4344362020492554, 1.2126637697219849, 0.6265981793403625, -1.648165225982666, 0.48575106263160706, 0.55660480260849, -2.282557487487793, -1.2468657493591309, 0.7829039096832275, 0.6703072786331177, -0.3017939031124115, -1.0199530124664307, -1.3705568313598633, 0.36806827783584595, -0.39206138253211975, 0.023836882784962654, -0.4069793224334717, 0.9497712254524231, 0.3838243782520294, -0.8112274408340454, 0.11874563246965408, -1.7403639554977417, 0.7341314554214478, -1.0056666135787964, -0.9288924932479858, 0.6569300293922424, 0.6268343329429626, 1.0563781261444092, -1.0069061517715454, -1.9292070865631104, 0.6302676200866699, 0.46332183480262756, 1.267311453819275, -0.8683499097824097, -0.04778217151761055, 0.5173978209495544, -1.1845570802688599, -1.1307786703109741, 2.016634941101074, 0.10326571017503738, 0.6417840719223022, 0.037287499755620956, -1.328716516494751, 1.2820137739181519, 1.300395131111145, -1.4130593538284302, 0.29010021686553955, -0.08371427655220032, 0.11861090362071991, 0.9852839112281799, -1.0653196573257446, 0.28137293457984924, 0.5456415414810181, 2.103569269180298, 1.9105290174484253, -0.39813557267189026, -1.049198865890503, -0.42939043045043945, -0.5935726165771484, 0.9668108820915222, 0.22664974629878998, 1.500956654548645, -0.2895863354206085, -1.4265061616897583, 0.5515380501747131, -0.45783424377441406, 0.1777462661266327, -0.2376580834388733, -1.1819987297058105, -0.5683982968330383, -1.2440670728683472, -0.46280503273010254, -0.2008523941040039, -1.194455862045288, 1.9029014110565186, 0.38952189683914185, -0.1211632639169693, -0.7369838953018188, -0.37834256887435913, -1.9471969604492188, -1.495072364807129, -0.5617315769195557, -1.0404058694839478, -0.24395352602005005, -1.2991583347320557, 1.1621849536895752, -0.9297138452529907, -1.0867042541503906, -0.12632329761981964, -1.28855299949646, 2.8957018852233887, -0.3284059762954712, -0.779195249080658, 1.323084831237793, -0.6997243762016296, 0.6152480840682983, -0.8058939576148987, -0.6879407167434692, 0.7445254325866699, -1.5214406251907349, -0.7431166172027588, -1.3435673713684082, 1.6197274923324585, -1.3255395889282227, 0.6953170895576477, -0.24694804847240448, 0.7382339835166931, 0.6699392199516296, -0.17051416635513306, -1.3227554559707642, 0.5235187411308289, 0.5665895938873291, -0.20706059038639069, -1.4098633527755737, 0.8470888733863831, 1.576253890991211, 1.1647028923034668, -1.3162386417388916, 1.4485135078430176, -0.7082805037498474, 0.30382871627807617, -1.8005316257476807, -0.22513911128044128, 1.152286171913147, -0.47142118215560913, 1.2096809148788452, 0.11847694963216782, 0.579400360584259, 0.10858780890703201, 0.4919354021549225, 1.8737258911132812, 0.6014360785484314, -2.6147775650024414, -0.5860440731048584, 1.4848730564117432, -0.10953736305236816, -1.517604112625122, -0.9130610823631287, 0.2945959270000458, -1.7471683025360107, 1.5572725534439087, 1.0063118934631348, 0.3854321241378784, -0.8131246566772461, 0.924822211265564, -1.5228354930877686, 1.1383483409881592, -0.25918635725975037, -0.6818698048591614, -0.06241147592663765, 1.2724260091781616, 0.979288637638092, 0.06784186512231827, 0.08286955207586288, 0.6534160375595093, -0.06667795032262802, 0.621321439743042, -1.0253499746322632, -0.5432565808296204, -0.6439741253852844, 0.4476500451564789, -0.6158198118209839, -0.7974462509155273, 0.4490280747413635, -0.5673390626907349, 0.8887084722518921, 0.4424893856048584, 0.10904264450073242, 0.1922605186700821, -0.7745990753173828, -1.8877595663070679, 0.11346922814846039, -0.2640759348869324, 1.271144986152649, 0.5175037384033203, -1.0652570724487305, 0.32136327028274536, 0.2429703176021576, -1.4661471843719482, 0.9890866279602051, -0.15747390687465668, 0.5830230116844177, 0.6051129102706909, 1.3210885524749756, 0.8946967124938965, 1.3347365856170654, 0.48386460542678833, 0.48882555961608887, -0.9620864987373352, 0.427295446395874, -1.7522786855697632, -0.3059718608856201, -0.4572453796863556, 0.1093287542462349, 1.6757069826126099, 0.6247598528862, -0.26502761244773865, -0.00037008896470069885, 1.4121229648590088, 1.84999418258667, 0.37032967805862427, -1.8675622940063477, 0.7846018671989441, 1.4447625875473022, -0.8730788230895996, -1.1345161199569702, -0.7133874297142029, -0.9507718086242676, -0.7668140530586243, -0.9052941203117371, 0.6660221219062805, -1.6396502256393433, 1.039596438407898, 0.9027122855186462, 0.8953900933265686, -0.18809489905834198, -0.06993167102336884, -1.124714970588684, 2.0958356857299805, -0.5283955931663513, 2.032824754714966, 0.1368960738182068, -1.4454550743103027, -1.1008762121200562, -0.009710829704999924, -1.297033429145813, -0.9157892465591431, -1.065543532371521, 0.6689949631690979, -1.5042870044708252, 0.7492935657501221, 0.2696383595466614, -0.6671488881111145, 0.3364962935447693, -0.36366206407546997, 1.2257343530654907, 1.4292964935302734, 0.5052499771118164, 0.34566566348075867, 0.8901101350784302, -0.291427880525589, 0.10147544741630554, 0.31038588285446167, 0.3566097319126129, 1.4729282855987549, -0.6692338585853577, 0.5731319189071655, 1.3719016313552856, -0.5605034232139587, -1.0510531663894653, -2.068944215774536, 1.1802704334259033, -1.042860984802246, 0.8088939189910889, -0.649996817111969, 0.21906769275665283, -1.6243723630905151, 0.5391159057617188, -0.6161321401596069, -0.6861867904663086, -0.7280823588371277, -0.34815481305122375, 0.02485533617436886, -1.0108848810195923, -0.15762904286384583, 0.511931836605072, 0.4053426682949066, -0.2082650065422058, -0.5892395973205566, -0.742828905582428, 0.3433087170124054, 0.2970368564128876, 0.6697387099266052, 0.032009534537792206, 0.34309354424476624, 0.08326014876365662, 0.2364116609096527, -1.156273603439331, -0.06779107451438904, 0.09388430416584015, 0.37667521834373474, 1.029433250427246, -0.34320539236068726, 0.019983353093266487, 0.41667070984840393, -1.1541980504989624, 1.3188101053237915, -1.0302205085754395, 0.29088929295539856, 0.2614026665687561, -0.39872539043426514, -0.5629047751426697, -0.954235851764679, -0.7866105437278748, -1.5443066358566284, 0.6788440942764282, -1.3102705478668213, -0.3265019655227661, 0.3330775499343872, -0.5962119698524475, 0.6614195108413696, 0.27864792943000793, -0.19874463975429535, -0.700593888759613, 1.0076298713684082, -1.8128514289855957, 1.048879861831665, -0.0730857402086258, 1.8579508066177368, -0.39127594232559204, 0.9048153758049011, 0.2214784026145935, -1.4953582286834717, 0.27616971731185913, -0.6435037851333618, 0.2740914523601532, 0.6888983845710754, 1.0724488496780396, 0.3200124502182007, -1.3254340887069702, -1.0279309749603271, 0.5189248323440552, 0.13477221131324768, 0.9219205975532532, -0.8452770113945007, 0.005116508807986975, -0.5857241749763489, 1.4917911291122437, -0.43844708800315857, -0.5828771591186523, -0.165871262550354, 0.7582870721817017, -1.3295460939407349, -1.185051441192627, -0.14123022556304932, 0.6166366338729858, -0.604348361492157, 0.8762378692626953, -0.6011375784873962, 0.1492590308189392, -0.46552056074142456, 0.3332292437553406, 0.578321099281311, -0.007825911976397038, 0.53870689868927, -0.0008348464034497738, 1.0201208591461182, 0.4367757737636566, 1.0262974500656128, -0.8104349374771118, 0.6076740026473999, 0.5251061320304871, 0.1052202507853508, -0.6382362842559814, 0.5106373429298401, 1.341556191444397, 0.11366095393896103, -1.7110625505447388, 1.0755667686462402, -0.37880146503448486, -1.1829028129577637, 0.6227256655693054, 0.4665732979774475, 0.8356291651725769, -1.1759008169174194, 0.8124416470527649, 1.250694751739502, -0.896423876285553, -0.5297614336013794, -0.7451227307319641, 0.4411826431751251, -1.3294925689697266, -0.6232982277870178, -2.179919481277466, -1.0920732021331787, -0.2662685811519623, 0.5423563718795776, 0.9949151873588562, -0.5943726897239685, 0.3227098286151886, -1.0772709846496582, 1.637406349182129, 0.462424099445343, 0.29448413848876953, -0.5405572652816772, -1.1125938892364502, 1.5595297813415527, -0.46184125542640686, 0.7946930527687073, -0.4789767563343048, 0.45219066739082336, -0.7261629104614258, 1.005005121231079, 1.2074054479599, -0.2610740065574646, 1.2586954832077026, 0.15070746839046478, 0.22662177681922913, -0.2055787295103073, 1.818420648574829, 0.6376239657402039, -0.2234055995941162, 0.18501470983028412, 1.2074049711227417, -0.4381079375743866, 0.12910230457782745, -2.717381000518799, -0.5439448952674866, 1.0757896900177002, -1.2839335203170776, -0.425580233335495, 0.5703672766685486, -0.531420111656189, 1.64090895652771, -0.8393492102622986, -0.29376140236854553, 0.12623119354248047, -0.02420164830982685, 0.28360697627067566, 1.2596900463104248, 0.2818078398704529, -1.2557319402694702, -0.976910412311554, 0.15015487372875214, 0.9649549722671509, 0.37074407935142517, 0.45129072666168213, -0.7752622961997986, 1.5537828207015991, -0.3433617055416107, -1.1829644441604614, -0.2869205176830292, 0.20517690479755402, 0.7878421545028687, -1.0976709127426147, -0.25245147943496704, -0.8212750554084778, -0.47149857878685, -0.020421987399458885, -0.12202223390340805, -0.8159741163253784, -0.7336244583129883, 0.7862635850906372, 1.2462512254714966], [-0.5521820187568665, 0.4404195547103882, 0.6675497889518738, -0.28335607051849365, 0.5092628598213196, -0.2710963189601898, 0.920921266078949, 0.2566886842250824, -0.36472415924072266, 0.581059455871582, -0.5407166481018066, 1.3398985862731934, 0.5585972666740417, -0.752086877822876, -2.3364157676696777, 0.2866004407405853, -0.3380666971206665, -1.1974244117736816, 1.2792612314224243, -0.012463133782148361, -0.9340361952781677, 0.20140422880649567, -0.6384166479110718, -1.1002604961395264, -2.5453708171844482, 0.1774066984653473, 0.21731466054916382, 1.312030553817749, -1.2293674945831299, -0.6433441638946533, 0.8334801197052002, -0.7676539421081543, -0.25016921758651733, -0.9688233137130737, 1.431566834449768, 1.2633322477340698, 1.3908264636993408, 0.7196136116981506, -0.6873846650123596, 1.3784643411636353, -1.1282591819763184, 0.8058350086212158, 0.7058395147323608, -0.6388671398162842, 0.6956793069839478, 0.9912211894989014, -1.2881758213043213, -2.837897539138794, -0.7811488509178162, 0.7181125283241272, 0.15626643598079681, 1.351887822151184, 0.4074535667896271, 1.2974262237548828, -0.5632011294364929, -0.8834456205368042, 0.6785252690315247, 1.004323124885559, 0.00033039352274499834, -1.0640925168991089, 1.425337791442871, -1.8160444498062134, 1.6360193490982056, -1.2842626571655273, -0.4115941822528839, -0.022694189101457596, 1.0796494483947754, 1.6396275758743286, -1.8526246547698975, 0.9546644687652588, -0.5552239418029785, -0.23219549655914307, 0.8466408252716064, -0.7770830392837524, -0.23810936510562897, 0.624823808670044, -1.0854848623275757, 0.7039864659309387, 0.8144163489341736, -1.2455151081085205, -0.40291911363601685, -0.7500998377799988, 1.9114129543304443, 0.25641295313835144, 1.6530394554138184, -0.6105649471282959, 0.03656824305653572, -0.6676331758499146, 0.5147610306739807, -1.1192717552185059, 1.0296800136566162, 0.6355655789375305, 0.9216399788856506, 0.5759181380271912, 0.48199039697647095, -1.4160414934158325, -0.5050277709960938, -0.2290290743112564, 0.6457930207252502, 0.07041633129119873, -0.7270036339759827, -1.4302650690078735, 0.7398636937141418, -0.527440071105957, 0.6051586866378784, 1.2936804294586182, 0.14942428469657898, -0.1574140042066574, -0.44875234365463257, -0.2592228055000305, -0.8598125576972961, 1.0844770669937134, -0.9257831573486328, -0.6497715711593628, -0.3304259479045868, -0.5219284892082214, -0.2558726370334625, -0.24869045615196228, -1.078583836555481, 0.23966974020004272, -0.5885205268859863, -0.6272376775741577, -0.5921729207038879, 0.6055477261543274, 0.8117198944091797, 0.9407144784927368, -2.3547163009643555, -0.9139010906219482, 1.3988269567489624, 0.4400203227996826, 0.3965522050857544, -0.4112865924835205, 0.8861055970191956, 0.7651218771934509, 0.10079716145992279, 0.7773380875587463, -0.31477898359298706, 0.18146058917045593, 0.3014245927333832, 0.9540760517120361, -0.8357365727424622, -0.7144412994384766, -0.8159487247467041, -0.6426416635513306, -0.14945079386234283, 2.521519184112549, -0.44321897625923157, -0.7177216410636902, 0.011693567037582397, -1.3190371990203857, 0.051819756627082825, 0.8060227036476135, -0.0694710984826088, 0.873096764087677, 1.3339478969573975, 0.24312838912010193, 0.20977328717708588, -0.14063899219036102, 0.2997216582298279, -1.15657639503479, 0.8261098861694336, -1.4804115295410156, 0.2769256830215454, -0.5349246859550476, 0.6030675768852234, -0.5684647560119629, -0.7360187768936157, -1.3681631088256836, -1.0402827262878418, 0.02341850847005844, -1.533340334892273, 3.0602848529815674, 0.13950935006141663, -2.3637917041778564, -0.5027695298194885, -0.4188566505908966, -0.6701522469520569, -1.0588027238845825, 1.794429063796997, 2.101390838623047, 1.2095929384231567, -0.026382090523838997, -0.129732146859169, -0.04446965456008911, -0.0383046492934227, 1.0730060338974, 0.38913753628730774, 2.118860960006714, 0.017749672755599022, 1.1275995969772339, -0.5895057320594788, -0.09868700057268143, -0.0001772733376128599, -0.13808031380176544, 0.12243684381246567, 1.5816537141799927, 1.1941813230514526, 0.9952178597450256, -0.009515783749520779, -0.919928789138794, -0.015725992619991302, 0.918734610080719, -0.8836272358894348, -0.2836466431617737, 0.26153364777565, -1.4346463680267334, -0.19536760449409485, -0.027269044890999794, -0.013386473059654236, 2.3729326725006104, -0.2899029552936554, 1.475325584411621, -1.2726987600326538, -0.7339310646057129, 2.373253107070923, -0.5588964819908142, 2.511106491088867, 0.8496505618095398, -0.35005709528923035, 1.9152979850769043, -2.06068754196167, 2.5611073970794678, -0.3668220043182373, -0.18089212477207184, 2.7569050788879395, -0.751308023929596, -0.013337144628167152, 0.45513463020324707, -0.3159906268119812, 1.0481891632080078, -0.9419085383415222, -0.8447465300559998, -1.3324638605117798, 0.7171289920806885, 0.8689121603965759, -0.7524474263191223, 1.4877041578292847, -0.5547845363616943, -0.8130893111228943, 0.2873464822769165, -0.16665931046009064, -2.00099515914917, 0.796930193901062, -0.1461971402168274, -0.19805637001991272, 0.08863038569688797, -0.8534178137779236, 0.9400491714477539, -0.2931601405143738, -1.3506041765213013, -1.0745645761489868, -1.6231701374053955, -1.1659351587295532, -1.0580954551696777, -0.5700279474258423, 1.3610100746154785, -0.3816361427307129, 0.7370155453681946, -1.5316659212112427, -0.3058229684829712, -0.31808167695999146, -0.49270597100257874, 0.10432877391576767, 1.2693873643875122, -0.1692996621131897, -0.9142323136329651, -0.38652148842811584, 0.08096196502447128, -1.1005436182022095, -0.2994806468486786, -0.39259934425354004, -0.8331369161605835, 0.5732501745223999, -1.2302085161209106, 1.2292886972427368, 0.2239757627248764, 0.61884605884552, 1.0652413368225098, 1.2658765316009521, -0.7945094704627991, -0.8206217885017395, 0.3469310998916626, -0.7259803414344788, -0.36406415700912476, 0.0818033218383789, 1.1481411457061768, -1.0898361206054688, 0.2596290707588196, -0.002106398344039917, 1.78285813331604, 1.6044460535049438, -1.6607288122177124, 1.0195486545562744, -0.3271690011024475, -0.23570580780506134, 1.1438039541244507, -0.049829039722681046, -0.09645332396030426, 1.5301241874694824, -1.2972553968429565, -1.470400333404541, 0.9439440369606018, 0.5963988900184631, -1.616661548614502, 0.94413822889328, -0.30892840027809143, 0.7606987953186035, -0.917818546295166, 1.0464670658111572, 1.6468935012817383, 0.9053497910499573, -1.5520118474960327, 0.9066658020019531, 1.372152328491211, 2.1998143196105957, -0.30522316694259644, 0.5783968567848206, -0.45221391320228577, 0.17711196839809418, -0.14139479398727417, 0.5635440349578857, 0.324290931224823, 0.3098946511745453, -0.8410613536834717, 0.9364557266235352, -1.039740800857544, -0.5940802693367004, -1.2404721975326538, -0.22083160281181335, 2.2830376625061035, -0.3324516713619232, -0.19949211180210114, -0.452108234167099, -0.35913142561912537, -1.0878826379776, -1.206811547279358, 0.45105084776878357, -0.884598433971405, 0.18846653401851654, 1.32951819896698, -1.0268809795379639, -0.9621708989143372, -0.459286630153656, 1.1367567777633667, 2.752714157104492, 0.39990150928497314, 1.6562970876693726, -0.1801009327173233, 0.9148479104042053, 1.7186400890350342, 0.7207632660865784, -0.6435478925704956, -0.12451832741498947, 0.23998938500881195, 0.5956692695617676, 1.065854787826538, -1.9550502300262451, 0.09089773893356323, 1.122174620628357, 1.1733427047729492, 0.03555460274219513, 1.2640039920806885, -1.385886788368225, -0.38317808508872986, 0.28409451246261597, 0.38338539004325867, -0.1299520879983902, 1.153996467590332, -1.2544307708740234, -0.7326662540435791, 1.5841699838638306, -0.3842680752277374, 0.4867345690727234, -0.18200092017650604, 0.5940101742744446, -0.06636382639408112, -0.9610793590545654, -0.05577092990279198, 0.009365297853946686, 0.2347700446844101, -0.24854017794132233, -0.09342910349369049, -0.8005773425102234, -2.1303956508636475, 0.5872650146484375, 0.6943308711051941, 0.701051652431488, -1.8865035772323608, -1.1177871227264404, 0.2772791087627411, -0.6622707843780518, 0.4143422544002533, 0.1908322274684906, -0.2658994495868683, -0.9621942639350891, 0.018759431317448616, -1.3214021921157837, -0.3488360047340393, -0.4451596736907959, -0.8520808815956116, 0.420597106218338, -0.41608455777168274, 0.35881081223487854, -2.3225128650665283, -0.38415175676345825, -1.5598154067993164, 1.0565699338912964, -2.3700194358825684, 2.754793405532837, 1.3046436309814453, 1.6352535486221313, 0.41485172510147095, -0.06627842783927917, -1.265977382659912, -0.7846947908401489, 0.8956202864646912, 1.1750215291976929, -0.8675664067268372, -1.2878309488296509, 0.23095360398292542, -0.40355682373046875, -0.49127137660980225, 1.2603179216384888, -0.6354041695594788, -0.3871998190879822, -0.04146028310060501, -1.1936367750167847, -0.32554978132247925, -1.3267408609390259, 0.11496903747320175, -0.5982269048690796, -0.1465853601694107, 0.6502589583396912, 0.4866675138473511, 1.0482070446014404, -1.751068353652954, 1.6491864919662476, -0.14318962395191193, -0.8417213559150696, 0.4320996105670929, 1.089532732963562, 1.0022259950637817, -0.36405542492866516, -1.0171449184417725, 0.9150015711784363, 0.42758479714393616, 1.0618987083435059, 0.13053596019744873, -0.7462647557258606, -2.5861403942108154, -0.3639110028743744, 0.23926299810409546, -0.5775684118270874, 0.7725353240966797, -1.0131287574768066, -0.6248257756233215, -0.20232954621315002, -0.9182228446006775, -0.45669275522232056, -0.11471889168024063, -0.6266430020332336, -0.05354687571525574, -0.4746481478214264, 0.8331442475318909, -0.9793432950973511, 1.3561698198318481, -0.6562288403511047, 1.8641362190246582, -1.1498100757598877, 1.4832631349563599, -1.170902132987976, -1.6411641836166382, -0.6323460340499878, 0.5590924024581909, 0.02518109790980816, -0.0772109180688858, -0.10771400481462479, -1.5663174390792847, 1.3922227621078491, 0.517991840839386, 0.3430623412132263, 1.8705238103866577, 0.29922422766685486, -0.9190250635147095, 0.3856939375400543, -0.10113921016454697, -1.4904873371124268, -0.930755078792572, 1.1021480560302734, -1.1007955074310303, 0.3431074619293213, -0.6574358344078064, -0.7270334362983704, 0.1844218522310257, -1.3343981504440308, 0.7218015193939209, -1.2973737716674805, 0.7058209776878357, 0.25342845916748047, 1.6563254594802856, 1.9551279544830322, -1.6917123794555664, 1.2846760749816895, 1.584058165550232, 1.784305214881897, 0.3488614559173584, 0.5628608465194702, -0.11281751841306686, 0.9035377502441406, -0.3478289544582367, 0.8561500906944275, 0.868578314781189, 1.897132158279419, 0.5057687759399414, -0.35375961661338806, 1.317118763923645, -0.5488330125808716, 0.2848202586174011, 0.7684544920921326, -1.5880688428878784, -2.0671157836914062, 1.042664885520935, 0.21475228667259216, 0.5829967260360718, 0.2676100730895996, 1.128636360168457, 0.27034640312194824, -0.7317790985107422, 0.5976769924163818, -0.055155523121356964, -0.20642289519309998, -0.31290316581726074, -0.8630524277687073, 0.35476428270339966, 1.0664554834365845, 1.4224672317504883, -0.30049681663513184, -1.626259207725525, 1.0567055940628052, 0.5431367754936218, 0.8964272141456604, 1.577823281288147, -2.7074859142303467, 1.023702621459961, -1.3665226697921753, 1.545332431793213, 1.3449652194976807, -1.145455241203308, -0.2903796434402466, 1.4729368686676025, 1.6626594066619873, -0.1854846179485321, 0.14353328943252563, -1.0378038883209229, 0.7624520063400269, 0.18157963454723358, -0.04132020100951195, 0.3629790246486664, -1.5996571779251099, 0.7815330028533936, -0.7308117747306824, -0.7409032583236694, -0.3006875216960907, 0.26258134841918945, -1.047607183456421, 0.9428445100784302, 0.9617103934288025, -0.5232646465301514, -1.868837594985962, 1.3294830322265625, 1.214232087135315, 1.1562566757202148, -0.14685404300689697, 1.0874431133270264, -0.05070824921131134, 0.42302680015563965, 0.004045465961098671, 0.3013438284397125, 2.2683568000793457, 1.5684462785720825, -1.0917176008224487, 0.7707018852233887, -0.6017586588859558, -1.1198588609695435, 0.83676677942276, 0.3837242126464844, -0.018461892381310463, 1.4308559894561768, -0.2493836134672165, -0.5175338387489319, -1.430609107017517, 0.0838027074933052, -0.7419008016586304, 0.4120824635028839, 0.039470914751291275, 0.768659234046936, 0.4852602481842041, -0.38960835337638855, 0.35453808307647705, 0.23842811584472656, 1.5922085046768188, 0.20362462103366852, -1.3524441719055176, 0.1533738672733307, 0.9209703207015991, 1.2254281044006348, -1.1826965808868408, -0.7912999391555786, -0.11347822844982147, 1.4501055479049683, -0.3791557252407074, -0.6179825663566589, -0.46474960446357727, -1.0486727952957153, 0.08970268070697784, 0.07764548063278198, -0.891350507736206, -0.5813508033752441, -0.1378818303346634, -0.8486517071723938, 1.734438180923462, -0.20454582571983337, -0.8561569452285767, -0.7904831171035767, 0.6562877893447876, 0.3777010142803192, 1.6162922382354736, -0.4201280176639557, -1.8462591171264648, -0.8302807211875916, 0.27145811915397644, -1.1706485748291016, 1.0929288864135742, -0.9404855966567993, -0.7491673827171326, -0.7216595411300659, 0.3382183015346527, 0.440201997756958, 0.02125222608447075, -0.5892404317855835, -1.1702204942703247, -0.1420861780643463, -0.7602672576904297, 0.8355486989021301, -1.0756217241287231, 0.48093748092651367, 0.838263213634491, 0.7212694883346558, 0.9719300866127014, -1.5850214958190918, 0.6957374215126038, -1.2308471202850342, -1.7422758340835571, -1.1668648719787598, -0.49056315422058105, 1.0994441509246826, 1.1875708103179932, -0.6139909625053406, -0.6839209198951721, 0.08771461993455887, 0.7692431211471558, 0.2922893464565277, 1.227684497833252, -1.009718656539917, -0.6810534596443176, 0.04161914065480232, -0.42712682485580444, -0.5482668876647949, -0.7819634079933167, -1.5053036212921143, -0.5255098938941956, 1.0100562572479248, 1.004369854927063, -0.23327751457691193, 1.7572152614593506, -0.602311909198761, -0.1836269497871399, -2.17561674118042, 0.6451735496520996, -1.5070724487304688, -1.5471965074539185, -0.46646204590797424, -0.6547154784202576, 0.6535413861274719, 0.0999671146273613, -0.264544278383255, -1.0413122177124023, 1.133126974105835, -1.207890510559082, -2.1378173828125, 1.7131279706954956, -0.5016241669654846, 1.6531332731246948, 0.5083913207054138, -1.2435697317123413, -0.5986221432685852, 0.9319596886634827, -0.06252362579107285, -0.26873552799224854, -1.9495811462402344, -0.358357310295105, 1.9324235916137695, -0.8936177492141724, 0.028050795197486877, 1.291395902633667, 0.2981957793235779, -1.1641324758529663, 0.21562087535858154, 1.254656195640564, -1.460686206817627, 2.209251880645752, 0.24713705480098724, 2.011286735534668, 0.3212805688381195, 0.17523165047168732, 0.20107266306877136, 0.5109445452690125, -0.840375542640686, -0.07641980051994324, 1.46779465675354, -1.178778052330017, -1.3778026103973389, 0.713674783706665, 1.6308265924453735, -0.8828688263893127, -0.28225770592689514, -1.263439416885376, -0.045809272676706314, 0.26775914430618286, 0.015853699296712875, -1.3126084804534912, -0.32289037108421326, -0.1937684416770935, -0.697277307510376, 0.2841998040676117, -0.20715388655662537, -1.1181435585021973, 0.6971318125724792, 0.39473217725753784, 0.34153786301612854, 0.4802088141441345, -1.9293712377548218, 0.10906758159399033, 0.014886857010424137, -1.5886541604995728, -1.3716089725494385, 1.3256914615631104, 0.43839791417121887, 0.36090078949928284, -0.4602721333503723, 0.8152354955673218, -0.16545602679252625, -0.8259159922599792, 0.20964530110359192, -0.01685906946659088, -0.46897655725479126, 0.8314792513847351, 0.9805687069892883, 0.5040677189826965, 0.09781474620103836, -0.7087807059288025, 0.23125877976417542, 0.40791821479797363, 0.9490035772323608, 2.1720850467681885, -1.2496075630187988, -0.40096259117126465, 0.10680454224348068, 0.2036246806383133, -1.4953452348709106, 1.6481534242630005, 0.09252963215112686, 1.1322654485702515, 0.1253027766942978, -0.1795034259557724, 0.5522134900093079, -0.4522280693054199, -0.06715483963489532, 0.8977847695350647, 0.7236791253089905, -0.9195640087127686, -0.01540042832493782, -1.2850397825241089, 1.106360912322998, 0.889842689037323, 0.8435556292533875, 0.3578735589981079, -0.12036777287721634, 0.09680169075727463, 1.0110528469085693, -0.3482988178730011, -1.9134892225265503, 0.2639830410480499, 0.45716819167137146, 2.017286777496338, -1.485329508781433, 1.0487867593765259, 0.32105734944343567, 0.5320712924003601, 0.8953161239624023, -0.903576135635376, -0.5507106781005859, 0.33883702754974365, -2.3344812393188477, 1.13816237449646, -0.5953119397163391, -0.40706533193588257, 0.3551308512687683, 0.8436675667762756, 1.5393134355545044, -0.2449154257774353, 0.34580475091934204, -0.5530408024787903, 1.793993592262268, -0.9043499827384949, -1.6923787593841553, -0.23277434706687927, -0.9348084926605225, 0.5865544080734253, 1.1033051013946533, 0.396002858877182, -0.18388381600379944, -0.5894017219543457, -0.048862431198358536, 0.1644439995288849, -1.499333381652832, -1.2886760234832764, 0.9336272478103638, -0.20160531997680664, -0.5766026377677917, 0.8305167555809021, 0.6269614100456238, 0.49284127354621887, -0.35049086809158325, -1.215872883796692, -1.5504212379455566, -0.2119779735803604, -0.852755606174469, 0.023420335724949837, 0.23447751998901367, -0.28571707010269165, 0.49477630853652954, -0.3110961318016052, 0.497514009475708, -1.8058562278747559, -1.310097336769104, 0.4627511501312256, 1.0427519083023071, 1.4285500049591064, 0.035364773124456406, -0.17208167910575867, -1.243272304534912, -1.4314613342285156, -0.17878516018390656, -0.4317419230937958, -0.6469576954841614, -0.04239126294851303, -0.6511511206626892, -2.268630027770996, 0.11117877066135406, 0.06479016691446304, -0.7106148600578308, 0.06967548280954361, -0.47806406021118164, -0.45321372151374817, 1.1993188858032227, 1.1126439571380615, 1.3801103830337524, -0.8822548985481262, 1.8125797510147095, -1.255211353302002, -0.2938916087150574, 0.2520180642604828, 0.5323200225830078, 1.8008335828781128, -0.08890828490257263, -0.4365701973438263, -1.1194089651107788, -0.5107868909835815, -1.0856878757476807, 1.1390101909637451, -0.07725321501493454, 0.22149211168289185, -2.2771973609924316, -0.6432189345359802, -0.1251126229763031, 2.0767269134521484, 0.9055526852607727, 1.2284293174743652, -1.6879119873046875, 0.2802397608757019, -0.816355288028717, 0.47230279445648193, 0.20561806857585907, 0.45004573464393616, -1.7152365446090698, -0.23976273834705353, -1.9286658763885498, 0.11811546236276627, -0.48890095949172974, -0.48524489998817444, -1.6658869981765747, 0.5072802901268005, 0.3412054181098938, -0.5560189485549927, -0.8628816604614258, -1.1936876773834229, -1.200123906135559, -0.8731826543807983, -0.6308229565620422, -0.44752106070518494, -0.3576482832431793, 0.9834428429603577, 0.9129787683486938, 0.018737444654107094, 0.09102120250463486, 0.3495510518550873, -0.43519866466522217, 0.03190455213189125, 0.5987642407417297, -2.2798216342926025, -1.3745135068893433, 1.2185033559799194, 0.06972288340330124, -0.2144232541322708, -0.25388219952583313, 0.15289632976055145, 0.1968763917684555, -0.2087964415550232, 0.9674544930458069, -1.3879454135894775, -0.4898250699043274, 0.658206582069397, -0.6000685095787048, -0.03794443979859352, 0.48472732305526733, 0.020132925361394882, -0.5721475481987, 1.8164114952087402, 0.08776562660932541, -1.1738947629928589, 0.684790313243866, 1.744627833366394, -1.135928750038147, -0.8962023258209229, -2.8357629776000977, 0.6597336530685425, 2.6020138263702393, 0.3804766833782196, -0.5618119239807129, 1.0996655225753784, 0.7392180562019348, -0.29613709449768066, 0.5922023057937622, 1.0146232843399048, -1.583979845046997, 0.2911393940448761, 0.35877710580825806, -0.3006141781806946, -0.14400723576545715, -1.9023317098617554, 0.6806421279907227, 0.6222552061080933, 0.955675482749939, 1.3296679258346558, 0.9423254728317261, 0.03564582020044327, -1.9690274000167847, 0.7686400413513184, 0.5501127243041992, -0.9172217845916748, 1.330142855644226, -0.620988130569458, 0.9960081577301025, 1.7795014381408691, 1.6366758346557617, 0.47521311044692993, 1.167849063873291, 0.7116740345954895, -1.6328154802322388, -0.606296181678772, 1.009392499923706, -0.9841272830963135, 0.0375644750893116, 2.686387300491333, -0.2678242623806, 0.3179019093513489, 0.9579219818115234, -0.4268491864204407, -0.049970582127571106, -0.46548694372177124, -0.4431968629360199, 0.8227158188819885, 0.2665150463581085, -1.1195406913757324, 0.2828465402126312, 0.21773028373718262, 0.19573426246643066, -0.7910747528076172, 0.5689929127693176], [-0.15570515394210815, 1.0247610807418823, 0.6234637498855591, 1.332761287689209, -0.8334503769874573, 0.8006501793861389, 0.09127441048622131, 0.11039610207080841, 0.016007516533136368, -1.1409192085266113, -0.6294216513633728, -1.9799306392669678, 1.3591755628585815, 0.4104052484035492, -0.36292409896850586, -0.5467777252197266, -0.841224730014801, -0.3560706377029419, -0.9961485266685486, 2.2596333026885986, 0.22257068753242493, 2.551943302154541, 0.22482819855213165, -0.5486348867416382, 0.22948695719242096, 0.8796969056129456, 1.0007661581039429, -0.758227527141571, 0.14120905101299286, 0.5917619466781616, -0.4659411907196045, 0.21568618714809418, 0.4242614209651947, -0.09346690028905869, 1.0744197368621826, -1.8473855257034302, -0.3472042381763458, 1.298499584197998, -0.23569059371948242, -0.2830871045589447, -1.3859894275665283, 0.3298410177230835, -1.0724714994430542, -0.02968103252351284, 0.7814375162124634, 0.6690922379493713, -0.09705352038145065, -0.5601854920387268, -0.4014357626438141, 0.974182665348053, 0.38896527886390686, -1.2813427448272705, -1.829548716545105, 0.2840014100074768, 0.1179378405213356, -1.3527663946151733, 0.10670613497495651, 1.0863537788391113, -0.21964631974697113, -0.47405609488487244, 0.3732301890850067, 0.21050551533699036, -0.5793440341949463, -0.638994574546814, -0.6088849306106567, -0.07635604590177536, 1.8312110900878906, 1.4313902854919434, -0.47662705183029175, -1.2982550859451294, -0.3264666497707367, 0.22368119657039642, -0.4653090238571167, -0.04109678789973259, 0.11241845786571503, -0.6694433093070984, -1.3002257347106934, 0.8295480608940125, 1.3434518575668335, -0.6579213738441467, 0.6487399339675903, 0.3642127811908722, 1.3661266565322876, 1.025161623954773, 0.5675617456436157, -0.1085747629404068, 2.2718241214752197, 1.2455967664718628, 0.8202757239341736, 0.21546658873558044, -0.05472184345126152, -0.844866931438446, -0.14692911505699158, 1.0485175848007202, -1.9070847034454346, 0.26043999195098877, -1.907456398010254, -1.257441759109497, 1.1360492706298828, 2.7174072265625, -0.45767509937286377, 0.4758901596069336, 0.5656223297119141, -1.3487708568572998, -0.7395835518836975, -0.6688650846481323, 0.45206546783447266, 0.47482234239578247, -0.037289272993803024, 0.021332424134016037, 0.7997235655784607, -0.6807405948638916, 0.29134103655815125, 0.9566826820373535, 0.3628982901573181, -0.6908319592475891, -0.09085553139448166, -0.7215682864189148, -0.18380723893642426, 1.099319338798523, 0.9462959170341492, -0.6531242728233337, 1.7169002294540405, 0.9032279253005981, 0.9101783633232117, -0.005453523714095354, 0.7480863332748413, -1.3561921119689941, -0.7199379205703735, -0.4039402902126312, -0.4535713493824005, -1.5771589279174805, -0.40517547726631165, -0.5767797231674194, 0.33924224972724915, 1.3841878175735474, 2.06593656539917, 0.8332508206367493, -0.10840481519699097, 1.28091561794281, -2.190227746963501, 1.9536433219909668, 0.4429619312286377, -1.6052436828613281, -1.1409789323806763, -0.7484248876571655, -0.13927814364433289, 0.5500742793083191, -0.3627829849720001, -0.6094750761985779, 0.9771364331245422, -0.3492501676082611, -1.0328510999679565, -1.9236387014389038, 0.6543844938278198, 1.5388984680175781, 0.24390822649002075, -1.1762479543685913, -1.9862942695617676, 1.0944877862930298, -0.06523929536342621, 0.9616045355796814, 0.7775025963783264, -1.47046959400177, -1.2214012145996094, -0.3923201560974121, 0.3513043224811554, -1.180315613746643, 0.2831554114818573, 0.4692516028881073, 1.2481915950775146, -0.7785083055496216, -0.2261119782924652, -0.5757899880409241, -1.063451886177063, 1.8434560298919678, 0.029878053814172745, 0.9808026552200317, -2.165951728820801, -1.7112102508544922, 0.4087463915348053, -0.2724722623825073, 0.5728554129600525, 0.16015136241912842, 0.8900745511054993, 1.9292001724243164, 2.395982027053833, 0.14263159036636353, -1.3218607902526855, -0.8038536906242371, -0.2756054699420929, 0.0865778848528862, 0.6315118074417114, -1.4273288249969482, -0.8937956094741821, -0.401294082403183, 0.08851663768291473, 1.0769373178482056, 0.7183191776275635, 0.7429221868515015, 2.4691531658172607, 0.4686121940612793, 0.03580271080136299, 0.4236176311969757, 1.3581550121307373, -0.11152574419975281, 0.5103591084480286, 0.2116461545228958, -0.03535141423344612, 1.021086573600769, -0.02528417855501175, 1.6053903102874756, -1.7611076831817627, 0.5321485996246338, 1.5212315320968628, 0.1920422911643982, 1.6662447452545166, -0.9863598942756653, 1.6209149360656738, -1.1610283851623535, -0.14863213896751404, 0.0019309227354824543, 1.271077275276184, 1.0932550430297852, -0.09334975481033325, -0.6490838527679443, 0.4036008417606354, 0.4622228145599365, -0.6923967003822327, 0.9380636811256409, 0.20196396112442017, 1.1962401866912842, -0.7213647961616516, 0.6316905617713928, 1.9339529275894165, 0.6726141571998596, 1.2313650846481323, -0.6533376574516296, 0.33063265681266785, -0.7371848821640015, 0.05757733806967735, 0.6502015590667725, 0.17436325550079346, -1.2121907472610474, 0.0932205319404602, -1.8740761280059814, -0.05425389111042023, -0.4257585108280182, 0.4040880501270294, 0.19592314958572388, 0.3650015592575073, -0.6894429922103882, -0.9532609581947327, 0.5136747360229492, 0.23701699078083038, 0.7231845855712891, 0.11067285388708115, -1.2744892835617065, 0.9653708338737488, 1.6567180156707764, 0.7889715433120728, 0.11537212133407593, -1.3606600761413574, -0.6525821089744568, 1.0694977045059204, 0.8113636374473572, -0.006781639996916056, 0.27542391419410706, -1.053829312324524, -0.13344593346118927, -0.5777509808540344, -0.001792815630324185, -0.7451327443122864, 0.3980432152748108, 1.233254313468933, 1.2964128255844116, 0.1609390825033188, -0.5019046068191528, -0.30347004532814026, -1.1927529573440552, 1.0414652824401855, 0.6948944926261902, -1.0902904272079468, -0.23881636559963226, 0.33501413464546204, -0.05324124917387962, -1.1379371881484985, 0.6748185753822327, 0.9485970139503479, -0.2269970029592514, 0.4844837784767151, 0.15700702369213104, -0.8873201608657837, -0.4125400185585022, 0.7437794804573059, 0.7939119935035706, 0.10129529982805252, 0.9209598302841187, 1.6775591373443604, 0.34950485825538635, -0.6667115688323975, -0.5729479789733887, 0.3129757344722748, 0.5189551711082458, 1.1011594533920288, -1.3212546110153198, -1.1855390071868896, -0.48745545744895935, -1.3478636741638184, -0.4464438855648041, -0.5726984143257141, 0.3221045732498169, -0.527393639087677, 1.2520180940628052, -0.7803398966789246, -0.3884349465370178, -0.309706449508667, 0.6818477511405945, 0.2224363535642624, -0.04424351081252098, 0.04921966791152954, -0.666765034198761, -1.4871264696121216, 0.10250041633844376, -0.2915143668651581, -0.5034598708152771, 0.5026003122329712, 0.8193970322608948, -0.48726940155029297, -1.2547471523284912, 0.3769369125366211, -0.7337450385093689, -0.22633332014083862, 0.9025155305862427, 1.166690707206726, 1.122859001159668, 0.21383216977119446, 0.08206658810377121, 0.20993487536907196, -0.6358902454376221, -1.3481431007385254, -0.15574365854263306, 2.445125102996826, 0.3203117251396179, -0.04330830276012421, 0.08977428078651428, 1.19963800907135, -0.8161391615867615, 1.6629109382629395, -0.959918737411499, 1.242491602897644, 2.2296805381774902, 0.9638499021530151, 1.4671049118041992, -0.710385262966156, 0.10014741122722626, -1.0331929922103882, 0.47974535822868347, 2.307178020477295, 0.37861528992652893, -0.9857476949691772, -1.216685175895691, 1.1752289533615112, -0.25375500321388245, -0.07627036422491074, -1.5129728317260742, 1.5618000030517578, 1.844528317451477, 1.4237107038497925, 0.6188842058181763, -1.3453915119171143, 0.7543250918388367, 1.1632208824157715, 0.3625953793525696, -0.14502353966236115, 2.043680429458618, -0.5774856209754944, 0.007322276942431927, 0.2821156084537506, -0.7156054377555847, -0.28193727135658264, 0.5988289713859558, 0.39234989881515503, 0.9035544991493225, 0.4250766634941101, 0.5814304351806641, 0.37898755073547363, -0.365203857421875, -0.23422734439373016, 1.453989863395691, -0.7307090759277344, 0.35790109634399414, -0.8065701127052307, -0.4666288197040558, 0.26791346073150635, 0.2511805593967438, 0.5791587829589844, 0.9779346585273743, -0.8572009205818176, -0.008385966531932354, -1.3811171054840088, -0.8184415102005005, 1.5410832166671753, 0.1924949586391449, -0.36090409755706787, -0.4453308880329132, -0.6892379522323608, -0.8135924339294434, 1.7796093225479126, 1.6779152154922485, 0.9289261698722839, -0.1897450089454651, 1.5402920246124268, -1.788565754890442, 0.19151851534843445, -0.828795313835144, -0.0733635351061821, -0.7151572704315186, 1.1560803651809692, -0.026685990393161774, 1.7919234037399292, -0.08564145863056183, 0.14581874012947083, 0.010216599330306053, 0.6485304236412048, -0.661837637424469, -0.049853455275297165, 0.5140064358711243, -0.00955685693770647, 0.35173794627189636, 0.5608736872673035, -1.2772425413131714, -0.47242438793182373, 1.7264211177825928, -1.6378344297409058, -1.2477039098739624, -0.4025820195674896, 0.12782806158065796, -0.08936571329832077, -1.4645546674728394, 0.642890989780426, 0.19991637766361237, -0.5837677121162415, -1.0925863981246948, -1.6293271780014038, -1.0783237218856812, -0.6572059988975525, 1.0726637840270996, 0.23069113492965698, 0.22140267491340637, -0.32458001375198364, 0.7395374774932861, -0.4134344756603241, 0.9167472124099731, 2.045740842819214, 0.18009968101978302, 0.5212483406066895, 0.04172740504145622, 0.6496526002883911, 1.070246934890747, 0.8191220760345459, 0.3277071714401245, 0.4060335159301758, 0.3432096540927887, -0.08922300487756729, -0.15167264640331268, -1.7503719329833984, -0.13108870387077332, -0.9964286684989929, -0.5327361226081848, -0.04183787479996681, 0.8863952159881592, 0.5203848481178284, -1.323101282119751, -0.5953932404518127, 0.37022629380226135, 1.0859462022781372, 0.8102493286132812, -0.012835479341447353, -0.572193443775177, 0.2603396475315094, -0.07323337346315384, -0.44398075342178345, 1.1771310567855835, 0.5181750655174255, -2.0107643604278564, 0.922781229019165, 0.6205905079841614, -0.45409294962882996, -1.8577734231948853, -1.586469292640686, -1.8414721488952637, 0.07156385481357574, 1.2603843212127686, 0.25333869457244873, 0.5217159390449524, 0.40251362323760986, 1.0799362659454346, -0.3247935473918915, -0.12451985478401184, 0.5195789933204651, 0.43504798412323, -0.6073065996170044, 0.15161003172397614, -0.5039939284324646, 1.405856966972351, -0.13311198353767395, 0.6826682090759277, 0.1347227245569229, 0.34922969341278076, -1.704757571220398, 1.885327696800232, 0.03873955085873604, -1.6904290914535522, -0.6326557993888855, -1.1470203399658203, -0.6118881106376648, 0.39128050208091736, -0.21655063331127167, 0.3582195043563843, 0.3568951487541199, -1.2379568815231323, 0.29460087418556213, -0.3553016483783722, 0.03173219785094261, 1.0749304294586182, 0.8535712957382202, 0.45479726791381836, 1.3124483823776245, 0.9677676558494568, 1.837519645690918, 0.8743173480033875, -0.5466545820236206, 0.3587675094604492, 0.4418911039829254, -0.4768857955932617, 1.0956534147262573, -0.4085705578327179, 0.4776144325733185, 0.10984860360622406, 0.8884562849998474, 1.9329646825790405, -1.9052820205688477, -1.9316877126693726, 1.7634680271148682, -1.586854100227356, -1.8382588624954224, -0.6492053866386414, 0.9622693061828613, -0.13330169022083282, 1.4158530235290527, 1.0549192428588867, -0.20664386451244354, 0.025284327566623688, 0.08293420076370239, -1.4909831285476685, -1.584899663925171, -0.8035104870796204, 0.1170625314116478, 0.9303343296051025, 0.6762763261795044, 0.6304868459701538, -0.5334874987602234, -0.09512433409690857, 0.30123278498649597, -0.4730406701564789, 1.2341105937957764, 1.0658676624298096, -0.048438843339681625, 1.250518560409546, -0.18735194206237793, 0.6674376726150513, -0.4849168360233307, -0.04321519285440445, 1.8368428945541382, 0.8752009272575378, -0.2578533887863159, -2.0106923580169678, -1.219946265220642, 1.4448758363723755, 1.4061979055404663, 1.83537757396698, 0.3993781805038452, -0.7765805721282959, -0.4148949980735779, -0.6693369746208191, -0.05806842818856239, -0.31087738275527954, 1.16452157497406, 1.0663750171661377, -1.3973795175552368, -0.149583101272583, 0.278462678194046, 0.2846561670303345, 0.28291556239128113, -0.7142258286476135, -1.0974249839782715, -0.5561891794204712, -0.3740830719470978, 1.60220205783844, 1.1535838842391968, -0.7248820066452026, 1.0171070098876953, 0.12140917778015137, -0.6485311388969421, 0.14785079658031464, 0.793400228023529, 2.100243330001831, -0.9575321674346924, -0.726958155632019, 0.4258812665939331, 1.2293224334716797, -0.16659392416477203, 0.6980994939804077, -0.14673325419425964, 0.47147515416145325, -0.26315003633499146, -0.7917528748512268, -0.37010708451271057, 0.19717897474765778, 0.4466952085494995, -0.7043442726135254, 0.520973801612854, 0.20488090813159943, -0.9524596929550171, 0.6571106314659119, -0.19122128188610077, -1.800705909729004, 0.21426662802696228, -1.3731709718704224, -0.7519307136535645, 0.9865490198135376, 0.25875213742256165, -1.534803032875061, -0.3340296149253845, -0.4050094485282898, 0.9571464657783508, 0.6562525033950806, -0.6673415899276733, -1.2836915254592896, 2.0598740577697754, -0.8644987940788269, -0.1194515973329544, -0.8760823607444763, 0.2718174457550049, -1.2663562297821045, 1.0322613716125488, -0.511938214302063, -0.13145609200000763, 0.06216232851147652, 0.34095174074172974, -0.3116944432258606, 0.2571878433227539, -0.3018900156021118, -0.04485351964831352, -0.635276734828949, 1.1602904796600342, 1.961688756942749, 1.7573316097259521, 1.6227142810821533, -0.12804248929023743, -0.45694535970687866, -2.371734380722046, 0.2775023579597473, -0.4299256205558777, -0.8238121271133423, 1.208959937095642, 0.7162879109382629, 0.9860047698020935, 0.2130942940711975, 0.17608384788036346, -0.7492068409919739, -0.5948809385299683, 0.6094236969947815, 0.1669723093509674, 0.7446930408477783, -0.020986076444387436, -0.16544528305530548, 0.313638836145401, -0.8441448211669922, 1.7715418338775635, 1.232885718345642, -0.8949940204620361, 0.24609375, 0.7688133120536804, 0.6766184568405151, 0.6702408194541931, -0.11942847818136215, 0.8707159757614136, 0.7085713148117065, 0.43863582611083984, 1.6419419050216675, 0.8932015895843506, 0.7861902117729187, 0.04545394703745842, -0.3416641652584076, -1.0301183462142944, 0.08096280694007874, -0.7517486810684204, -0.8674634099006653, 0.39351052045822144, 0.6985185146331787, -0.6899460554122925, -0.5476073622703552, 1.1500571966171265, 0.8543163537979126, 0.6740917563438416, -1.505383849143982, 0.39109572768211365, -1.1514564752578735, 2.2502360343933105, -0.09286592155694962, -0.926206648349762, -0.09041277319192886, 1.3199574947357178, -0.598452091217041, -0.27394235134124756, -1.095669150352478, -2.5961530208587646, 0.4420571029186249, -0.9637987017631531, 0.14296677708625793, -1.115572214126587, 0.10521949082612991, 0.9441637396812439, 0.16057243943214417, -0.34531456232070923, 0.7948846220970154, -2.193521022796631, 0.9860188364982605, -0.7423752546310425, 0.6800940036773682, 0.9916889071464539, 0.251510888338089, 0.556145966053009, -1.7113096714019775, 1.618071436882019, -0.9991672039031982, 1.5372074842453003, 0.0720381885766983, -0.7735609412193298, -0.6924697756767273, -1.5401898622512817, -0.6222339272499084, 0.4444301426410675, 0.34383848309516907, -0.2994290590286255, 0.6377736926078796, -0.28336286544799805, -0.7764368653297424, 2.6646971702575684, -0.441254585981369, -0.688778817653656, -0.3547174036502838, -0.37201133370399475, -0.02855340577661991, 0.8701687455177307, 2.8394880294799805, -0.397243857383728, -0.7036979794502258, -0.8327299952507019, 0.9558138251304626, 0.17609982192516327, -0.2657720446586609, 0.04021311178803444, -0.15908996760845184, 0.44471317529678345, 0.9168069362640381, -1.017602562904358, 0.19969049096107483, -1.1949752569198608, -2.552608013153076, -0.23658838868141174, -0.7044514417648315, -0.19710759818553925, 0.6636914014816284, 0.05237039178609848, -0.43081146478652954, 0.5635831952095032, 0.7811967730522156, 1.7848621606826782, 0.06063971295952797, 0.452910840511322, -0.34672537446022034, 0.9968138933181763, -0.39384323358535767, 1.1890350580215454, 1.1934112310409546, -0.6778613328933716, -0.5053971409797668, 2.097853899002075, -0.8287678956985474, 0.6866405606269836, -1.4455488920211792, 1.1148431301116943, -1.1384910345077515, -1.7978345155715942, -1.6399130821228027, -2.9889509677886963, -0.9044185876846313, 0.689170777797699, 0.849005401134491, 0.4567270278930664, -1.7548284530639648, 0.4333941340446472, 0.9979500770568848, 0.018804503604769707, -0.9554002285003662, 2.3451170921325684, -0.39425691962242126, 0.2614612877368927, 1.9140805006027222, -1.5601211786270142, 0.19812066853046417, -0.7685820460319519, 0.09227229654788971, 1.1201504468917847, 1.2049731016159058, -0.5363144874572754, 0.0808885395526886, 1.2227236032485962, -0.662243664264679, -0.30693304538726807, -2.187567710876465, -0.3307534158229828, 0.36915603280067444, -1.3773337602615356, -0.42705798149108887, 1.4487559795379639, -0.679212212562561, -0.09291752427816391, 0.8344704508781433, 0.6084312796592712, -1.0203063488006592, -0.5916889905929565, -0.020711926743388176, 0.5742763876914978, 1.361600637435913, 1.8261134624481201, -0.46464410424232483, 0.8337912559509277, -0.23509809374809265, 0.7579147219657898, -1.7233260869979858, -1.0845696926116943, -0.9352888464927673, 0.8281522393226624, -0.5015639066696167, -0.41463515162467957, -1.1020584106445312, 1.4511116743087769, -0.929894745349884, -0.12121367454528809, 1.3245069980621338, 1.1919479370117188, -0.705528736114502, -0.20550425350666046, 0.8302648663520813, -0.5705574154853821, -1.1304271221160889, -0.799956202507019, -1.7626138925552368, 0.3174262046813965, 0.957512617111206, -0.23436342179775238, -0.9807469844818115, -0.4224640429019928, -0.763556718826294, 0.1687266081571579, -0.14163503050804138, 0.23153690993785858, -0.7049104571342468, -2.065317153930664, -0.6593884825706482, 1.2110540866851807, -0.8292793035507202, 2.102216958999634, -0.02763599157333374, 0.441758394241333, -0.046214889734983444, 1.786444902420044, 2.063400983810425, -1.0943742990493774, -0.9231975674629211, 1.49921452999115, 0.8093756437301636, -0.5434122085571289, 0.5338519215583801, 0.36080414056777954, -0.8205264806747437, 0.6431187391281128, -0.4993317425251007, -2.0534465312957764, -1.7727090120315552, -1.3682482242584229, -0.7407267689704895, 0.8510246276855469, 0.3456989526748657, -1.5364090204238892, 1.04044508934021, -0.9536216259002686, -0.10741612315177917, -0.31383341550827026, 1.148062825202942, 0.10337578505277634, -0.3655834197998047, -1.8992825746536255, -0.594351589679718, 0.10677745193243027, -1.6635291576385498, 0.11063971370458603, -0.47921112179756165, -1.8584998846054077, 1.8171024322509766, 0.6956689357757568, 0.9214674830436707, 0.49540311098098755, -0.4041839838027954, 1.4019564390182495, 1.0019869804382324, -0.017269406467676163, -2.5851190090179443, 0.11030759662389755, 1.8559167385101318, -1.052253007888794, -1.2795261144638062, 3.033404588699341, -0.1915179044008255, 0.9407865405082703, 1.2156563997268677, -1.1558152437210083, 1.7809795141220093, -0.30774641036987305, -0.18475618958473206, -0.3334757685661316, 0.8756535649299622, 1.566057801246643, -0.5010071396827698, -0.10248120129108429, 1.9016259908676147, -1.038818120956421, -1.4182183742523193, -0.1969001591205597, -0.5964105725288391, 0.20835252106189728, 0.536163866519928, 0.9154470562934875, -0.6561666131019592, -1.1750165224075317, -2.0991642475128174, -1.5711792707443237, -0.0634416937828064, -0.4922974109649658, 0.24374361336231232, 1.908088207244873, 0.6942432522773743, 0.03926120325922966, 1.778714656829834, 1.2266699075698853, -1.3685673475265503, -0.2007180005311966, -0.2759941518306732, -1.2500970363616943, 2.058623790740967, -0.05679028853774071, 0.5155318379402161, 0.5944942235946655, -0.46405503153800964, -0.4333547055721283, -0.819568932056427, -0.3992195725440979, 0.19876310229301453, 0.7519094944000244, 0.38684308528900146, -0.46856987476348877, 2.131577968597412, 0.3198360204696655, -0.9152872562408447, -1.5443023443222046, 1.4068022966384888, 0.12087570875883102, -0.6967838406562805, -0.6578632593154907, -1.4660346508026123, -1.314028024673462, -0.3092622756958008, -1.322344183921814, 0.30455100536346436, -0.05391894653439522, 0.9989925622940063, 0.986394464969635, -1.2025235891342163, -0.6849321126937866, 1.0248717069625854, 0.1598803699016571, -1.1278409957885742, 0.8602266311645508, 1.1319506168365479, -0.6035599708557129, -0.3818543553352356], [0.2520020604133606, -0.35729387402534485, 0.5170258283615112, 0.36570045351982117, -1.134966254234314, -0.4665803015232086, -0.07308334857225418, 1.020424246788025, -1.207345724105835, 2.0953197479248047, 0.2572857737541199, -0.12538598477840424, -0.3589783310890198, 0.6600778102874756, -0.31590285897254944, 1.6978960037231445, -0.9971920847892761, 2.5210511684417725, 0.9584380388259888, -1.317475438117981, -0.7836079001426697, -0.07809481024742126, -1.0504558086395264, 0.7275296449661255, 0.10017784684896469, 0.03722742944955826, -0.40796083211898804, 1.5848724842071533, -0.7312317490577698, 1.4212082624435425, -0.6761513948440552, -1.304013729095459, 0.9353344440460205, -0.817085325717926, 0.6259792447090149, 2.0894250869750977, -0.23533812165260315, -0.28825563192367554, -0.6341187953948975, -0.9269925951957703, 0.18163979053497314, 0.6982868909835815, -0.5200912952423096, 0.6742836833000183, 1.7911534309387207, -0.35842451453208923, 0.9831107258796692, 1.635743260383606, 0.6210876703262329, -0.4742797017097473, -0.1448104977607727, 1.044893741607666, 0.37350788712501526, -0.648026704788208, 0.15435653924942017, 0.3020612597465515, -1.2844606637954712, -0.10837937891483307, 0.4004426598548889, -0.21541082859039307, -0.717947781085968, -0.37202346324920654, -0.5926120281219482, -0.3544851839542389, 0.720853328704834, 1.3240089416503906, 0.5426324605941772, -1.041853904724121, -0.5672665238380432, 0.8403893709182739, 0.041917454451322556, -0.1207817867398262, 0.06882808357477188, -0.8823711276054382, 1.8419554233551025, 0.7667428255081177, 0.040438566356897354, -0.3032170832157135, -1.1911063194274902, -1.2829502820968628, -0.9031466245651245, -0.3620437979698181, -0.011523266322910786, 1.4566737413406372, 1.348891019821167, -1.420976161956787, 0.05023844912648201, -0.5792674422264099, -0.25382623076438904, -2.4291865825653076, -0.8153936862945557, -0.45259591937065125, -1.027954339981079, -0.3328735828399658, 0.11819039285182953, 0.06656470149755478, -0.47560855746269226, 0.5180308818817139, 0.44830119609832764, -0.5795494914054871, 1.323240876197815, -1.2155959606170654, 0.26037269830703735, 1.6797071695327759, 0.5807454586029053, -0.2795674204826355, 0.8509941101074219, 0.7253137826919556, -0.6248112916946411, -0.06558600068092346, 0.044296085834503174, -1.4728341102600098, -1.5006647109985352, -1.1468104124069214, -2.046160936355591, -0.19562751054763794, -0.8990943431854248, 1.6625680923461914, 2.1256964206695557, -1.621997356414795, 0.8699222803115845, -0.7416118383407593, -1.4367061853408813, -0.2566790282726288, 2.744797706604004, 0.8230258226394653, -0.15707188844680786, 0.5223736763000488, -0.8050904273986816, 1.6100449562072754, 0.33264219760894775, -0.8392640948295593, 0.3361401855945587, 0.007568026892840862, -0.9048353433609009, -0.6383869647979736, 1.170426368713379, 1.4976787567138672, -0.12192366272211075, -1.1924437284469604, 1.7554116249084473, -1.0960140228271484, 0.8638209700584412, -0.5631171464920044, 1.317827820777893, 1.517791748046875, -1.3224848508834839, 0.0900726243853569, -0.2617013156414032, 0.6406152248382568, -0.4429709315299988, -0.8579576015472412, -0.7016326189041138, -0.4046708345413208, -0.3007615804672241, -1.1941070556640625, -0.08376246690750122, 0.5178377628326416, -0.30736634135246277, -0.4032728672027588, 0.1463869959115982, 0.44508644938468933, 1.1889739036560059, -0.9378756880760193, -1.9572054147720337, 0.34323850274086, -1.0956213474273682, -1.1678770780563354, -0.508990466594696, -0.19436904788017273, -0.4584836959838867, -0.8060958981513977, 0.028799664229154587, -0.47150522470474243, 1.624184489250183, -1.6815365552902222, -0.7421855330467224, -0.8329094052314758, 0.18539227545261383, 0.27432429790496826, -0.7223432064056396, -0.7453135848045349, -0.10762819647789001, -0.7621700763702393, -0.6856745481491089, -0.9506015181541443, -0.07555220276117325, 0.123754121363163, -0.8923404216766357, 0.33838728070259094, 1.0341054201126099, -0.6954596042633057, 0.018957192078232765, -1.3588937520980835, -0.843228280544281, 0.9801265597343445, 1.0420352220535278, -1.5962389707565308, -1.088754653930664, 0.7809375524520874, 0.7126618027687073, 0.9822647571563721, -0.5207985639572144, 0.11965567618608475, -0.1544622778892517, 0.10630661994218826, 0.6335210800170898, -0.38514456152915955, 0.1117982268333435, 0.15062692761421204, 0.6264873743057251, -0.36413776874542236, -1.653817892074585, 0.8471808433532715, 1.5263932943344116, -0.5335570573806763, -0.9518901109695435, -0.7875586748123169, 0.020551973953843117, -1.0290188789367676, -1.7189840078353882, -0.6486467719078064, -1.494661569595337, -0.9895748496055603, 0.48939159512519836, -0.8210529088973999, -1.885303258895874, -0.5026514530181885, 1.2060635089874268, -0.9044784307479858, 0.1772061139345169, 0.5475152730941772, 0.3831798732280731, 0.20110216736793518, -1.7412110567092896, 0.4457283914089203, -2.060274600982666, -0.35942894220352173, -0.11722046881914139, 0.5545193552970886, -1.9293479919433594, 0.03080877475440502, -0.6431764364242554, -0.5921959280967712, 2.583491325378418, -1.935211181640625, -2.2158474922180176, 2.106755256652832, 1.6930896043777466, 0.21140773594379425, -1.3061034679412842, 2.2302799224853516, -0.5806660652160645, -0.5223477482795715, -0.5533873438835144, -0.9953582286834717, 0.15270692110061646, 0.06705088168382645, 1.4869146347045898, 0.8755400776863098, -1.317510724067688, 0.266544908285141, -0.7337801456451416, 0.29881227016448975, 0.8241330981254578, -1.4081729650497437, -0.3695959448814392, 0.7047457695007324, 0.3455181419849396, 0.33655840158462524, -0.038456521928310394, 1.990478754043579, -0.9607520699501038, 0.0021529137156903744, -1.8468586206436157, -1.5623233318328857, -0.19203518331050873, 1.0906022787094116, 0.05133716017007828, 0.4476044774055481, -0.034447912126779556, -0.8969526886940002, 0.46912017464637756, 0.4175603985786438, 0.4603409171104431, 0.5213707089424133, 0.8438919186592102, -0.9650090336799622, 0.18640410900115967, -0.5458246469497681, 0.13043804466724396, 1.557996392250061, -0.2870642840862274, -0.6226786375045776, -0.9628212451934814, 2.5675556659698486, 1.6545987129211426, 1.1431777477264404, 0.7791860103607178, 1.8521814346313477, -0.6420872211456299, -1.6615402698516846, -0.5574725866317749, -0.40498802065849304, 0.41455161571502686, 0.5411879420280457, 1.187990665435791, -0.28820985555648804, -0.8789344429969788, -0.639079749584198, 0.18774349987506866, 2.10170841217041, 0.9563613533973694, -0.4246947765350342, -1.868971586227417, 1.1017385721206665, 1.0204075574874878, -1.2972649335861206, -0.10801708698272705, -0.011156292632222176, -1.6256836652755737, 0.8888651132583618, -1.0624371767044067, -0.9631798267364502, -1.1526074409484863, 0.00446632644161582, 0.23566456139087677, -0.3483438491821289, -0.841234564781189, 0.2970731854438782, -0.1554999053478241, 0.9842233657836914, -0.1767638623714447, 1.6911780834197998, -1.1993989944458008, -0.6215513348579407, 1.271913766860962, 0.9937559366226196, 0.9020588994026184, 1.3405200242996216, -0.6787906885147095, -0.812885046005249, -0.7759225964546204, 1.261521816253662, -0.6232889890670776, 0.3318260610103607, -1.0305607318878174, -1.4010400772094727, -0.6044677495956421, 1.1907098293304443, -0.48829710483551025, 0.7408750057220459, -0.021712280809879303, 0.871414303779602, -0.33687880635261536, -0.46229246258735657, -0.4172300696372986, 0.8783788084983826, -0.9917041063308716, 0.11737119406461716, -0.4853289723396301, -0.45038899779319763, -0.33422747254371643, -0.9314976930618286, 1.0735920667648315, -1.8494454622268677, -1.5927714109420776, 1.8390499353408813, 0.5279820561408997, 0.18700869381427765, -0.728169858455658, 1.409385085105896, 0.4082065224647522, -1.0796512365341187, 0.5417556762695312, -0.9792013168334961, 1.08896005153656, -0.5948910117149353, 1.834189772605896, 0.5409823656082153, 0.5457680821418762, -0.4797878861427307, -0.16799157857894897, -0.17657503485679626, -1.2239171266555786, -0.1055726483464241, -0.6736096739768982, 2.0426151752471924, -0.8911727666854858, -1.6716464757919312, -0.4510907828807831, 0.04023851081728935, 1.648377776145935, -0.7974236607551575, 0.11915073543787003, -0.2816523015499115, -0.8264126777648926, -0.7231029272079468, 1.8394209146499634, 0.3226926922798157, 0.7121541500091553, 0.0877910777926445, 0.1600981503725052, -0.8886013031005859, -0.025357840582728386, 1.9356759786605835, -0.7477263808250427, 0.07019690424203873, 0.2782840132713318, 0.7014722228050232, -1.0237445831298828, 0.7023487687110901, 0.6554901003837585, 0.6050439476966858, -0.5204185843467712, -0.895045280456543, 0.0246136374771595, 0.7525206208229065, 0.35008707642555237, -1.183919072151184, 0.6810911297798157, 0.6962501406669617, 0.010132219642400742, -0.08196483552455902, -0.4469468593597412, 1.293920636177063, -1.0784428119659424, 2.5241100788116455, 0.9276661276817322, 0.829881489276886, -1.6691820621490479, 0.20157010853290558, 0.9652777314186096, 1.7494525909423828, 1.0530611276626587, -0.9993652105331421, -1.2927358150482178, -1.141543984413147, -1.7420251369476318, 0.8399008512496948, 1.2074260711669922, -0.0012158213648945093, 1.699127197265625, -0.7420726418495178, -0.4319747984409332, 0.08556673675775528, -1.0051201581954956, 0.09663810580968857, 0.13129498064517975, -2.0318870544433594, -1.2702186107635498, -0.29340195655822754, 0.22355052828788757, -0.2182418704032898, 1.336049199104309, -0.42033565044403076, -0.4996737837791443, -2.1988065242767334, -0.1655082106590271, 0.17174369096755981, -0.39982959628105164, -1.3457504510879517, 0.26526933908462524, -0.8288108110427856, -1.6719589233398438, 0.268540620803833, 0.747360110282898, 0.9554136395454407, 1.0560755729675293, 0.379401296377182, -0.9433999061584473, -0.6206408143043518, 1.0105425119400024, 1.034490704536438, 0.6226164698600769, -1.7777429819107056, -1.4649885892868042, 0.15076684951782227, -0.09762094169855118, 1.1254066228866577, -1.7888084650039673, 1.8693907260894775, 2.1952896118164062, -0.11838160455226898, 2.2101306915283203, -0.1661016047000885, -1.3405203819274902, 1.7130540609359741, 0.6222521066665649, 0.24156197905540466, -0.5222223401069641, 0.010797717608511448, 1.191709280014038, 1.5360760688781738, 0.07915708422660828, 1.1601715087890625, 0.5317648649215698, 1.168121576309204, -1.2415223121643066, 0.14623749256134033, 0.6581414341926575, 0.9760221242904663, 0.34878677129745483, 0.7413256764411926, 0.3840605318546295, -0.1807672679424286, 1.3693568706512451, 3.6468875408172607, 0.40199440717697144, -1.4960161447525024, -1.5008325576782227, -0.23665566742420197, -0.40746256709098816, -0.5780495405197144, 0.5947779417037964, -1.4417747259140015, 0.8390974998474121, -2.1101043224334717, 1.0902125835418701, -0.008613385260105133, 0.7357270121574402, 0.12176743894815445, 0.24593989551067352, -0.6041247844696045, 1.0397591590881348, 0.4426887333393097, 1.4704898595809937, -0.7320020794868469, -1.2185158729553223, 0.9875856637954712, -1.5179362297058105, -0.22752824425697327, 2.53262996673584, -1.186194896697998, 2.3552792072296143, 0.2279891073703766, 1.129847526550293, -1.2536412477493286, -0.8817628622055054, 2.2367045879364014, -0.6380529403686523, 0.6142238974571228, -0.49094873666763306, 0.8535045385360718, -0.6706013083457947, -0.3125564754009247, -0.2068445235490799, -1.1601165533065796, -0.03761490806937218, -1.3419768810272217, 1.945237636566162, -0.835731029510498, -1.9692939519882202, 0.5572696924209595, -0.03868953883647919, -0.41790908575057983, -1.5585861206054688, 0.7692567706108093, -0.449865460395813, -0.743905246257782, 1.7030469179153442, -0.21677768230438232, 1.0075395107269287, -0.6748859882354736, 0.2965320646762848, 0.5583446621894836, -0.8481485843658447, 0.29929012060165405, -1.3246930837631226, -0.34260129928588867, -0.01797901839017868, 0.1950588971376419, 1.1402140855789185, 0.19548974931240082, 0.5607713460922241, 0.3323163688182831, -1.2516891956329346, -0.8661314249038696, 1.45440673828125, -0.025287482887506485, -1.1682782173156738, -0.5627505779266357, 0.4397939145565033, 0.12920545041561127, 0.14268064498901367, -1.6490503549575806, 0.5035247802734375, -0.9751561284065247, -0.9139377474784851, 0.1213756874203682, 0.5153132081031799, -0.9221950173377991, 0.7936938405036926, -0.9960651993751526, 0.29217326641082764, -0.37000179290771484, -0.9357264637947083, -0.5048763751983643, 0.12199626117944717, -0.03728971630334854, 0.4576335549354553, 0.7433582544326782, -2.0339460372924805, 0.29964467883110046, 0.3275701701641083, 0.11379300802946091, -0.502740740776062, 0.7661260962486267, 0.2923681139945984, -0.1073359027504921, -1.5445743799209595, -0.5497763156890869, 0.3690473139286041, 1.1495361328125, 1.3262900114059448, -0.13205431401729584, -0.20898085832595825, 0.9769304394721985, 1.1899243593215942, -0.5534738898277283, 0.5326864719390869, -1.108337640762329, -0.39146462082862854, -0.058087967336177826, -0.8795369267463684, -0.33585381507873535, -0.8375120759010315, -1.9262142181396484, 2.180774688720703, -1.0493866205215454, 0.6973258256912231, -0.7491618394851685, -0.12266118824481964, 0.8088220953941345, -1.496330738067627, -1.744316577911377, 0.2772877514362335, -0.14303453266620636, 0.5002568960189819, -0.6078908443450928, -0.053215231746435165, -0.029365960508584976, -1.4230625629425049, 0.19952824711799622, 0.5090135335922241, -0.3801969885826111, -0.5666767954826355, 2.1445705890655518, -0.13748586177825928, 0.7854205369949341, 1.1016300916671753, 0.42182421684265137, -1.079247236251831, -1.3946771621704102, -1.389652967453003, 0.07082341611385345, -0.5990989208221436, -0.9233711361885071, -0.4486646354198456, 0.4469650983810425, -1.081575870513916, -0.5911845564842224, -1.228474497795105, 1.312880039215088, 0.5932587385177612, -1.006385326385498, 1.2813949584960938, -0.3427777588367462, -1.2691811323165894, -0.7161173224449158, -0.570831298828125, -0.9352409243583679, 0.5255387425422668, 0.6003109812736511, 0.14961804449558258, -0.803769588470459, -0.6714380979537964, 1.3495275974273682, -0.09734820574522018, 1.9208606481552124, 0.31892573833465576, -0.5598002672195435, -0.47432854771614075, -0.5928568243980408, -0.7925665378570557, 1.483229160308838, -0.4647480845451355, -0.9237391352653503, -0.4585443437099457, 0.2053772658109665, 1.4950752258300781, 0.07309195399284363, 0.037277307361364365, 0.6494298577308655, 1.3130346536636353, -0.055752940475940704, -0.7581930160522461, -0.6443567872047424, -1.4451069831848145, 1.2398698329925537, 1.0798523426055908, -0.13970820605754852, -0.5582632422447205, 1.0269198417663574, -0.3878413140773773, -1.3392338752746582, 0.10526949912309647, 2.0958118438720703, 0.20903334021568298, -1.2134947776794434, 0.9958351254463196, -0.6052513122558594, -0.29868361353874207, 0.7723146677017212, -1.4801616668701172, 2.78098464012146, 0.4022073745727539, 1.3850090503692627, -0.6906275153160095, 0.5483813285827637, 1.0346065759658813, -0.341167688369751, 0.5484968423843384, 1.8080099821090698, 0.64003586769104, 0.12683166563510895, 0.4028414487838745, 0.5615405440330505, -0.8924440741539001, 0.3047727942466736, 1.1147733926773071, -1.5117607116699219, 0.7335320711135864, -0.4071064591407776, 0.4517301023006439, -1.608859658241272, -0.2949644923210144, -0.2711857259273529, 0.08640148490667343, 1.0944992303848267, 0.08698246628046036, -1.858209252357483, -0.9238220453262329, -0.002585930749773979, -0.9838641285896301, 0.12255559116601944, 0.10542099922895432, 0.583984911441803, 0.25123459100723267, 0.8220056891441345, -0.4016135036945343, -0.8397439122200012, -2.0338897705078125, -1.1698026657104492, -0.8031336665153503, -1.3368394374847412, -0.3433936834335327, 0.07915326207876205, -0.057332661002874374, 0.8184205293655396, -1.674919843673706, -0.46547678112983704, 0.44351303577423096, -0.36953452229499817, -1.6145132780075073, -0.6800471544265747, 0.06995835155248642, 0.1636752486228943, 0.24595873057842255, -0.7977290749549866, -0.13199461996555328, -1.0528366565704346, 0.5512912273406982, 1.179012417793274, -0.2792437672615051, -1.4230793714523315, 0.8917686939239502, -0.8315897583961487, -1.1868836879730225, -1.640315055847168, 0.3349240720272064, 0.4649454951286316, 0.6618610620498657, -0.28943973779678345, -1.2963120937347412, 1.242020845413208, 0.01006404124200344, -0.21858486533164978, 1.5604525804519653, 0.21310216188430786, -0.6235515475273132, 1.511126160621643, -0.6079736948013306, 1.1913269758224487, 1.2992184162139893, -0.31111836433410645, 0.07284849137067795, 1.0434561967849731, 0.4149157404899597, -1.7466390132904053, 0.11891547590494156, 0.2257051318883896, 0.22385483980178833, 0.8126935362815857, 0.5829220414161682, -1.0736805200576782, 0.4148918688297272, 1.0458811521530151, 0.34808868169784546, 0.7924643158912659, -0.4276134669780731, 0.33442169427871704, 1.9658725261688232, -0.7250857353210449, -1.539670705795288, 1.1063448190689087, 0.7733014225959778, 0.7633829712867737, 0.060635700821876526, 1.0403825044631958, 0.5199052095413208, -0.30907154083251953, -0.9937140941619873, -0.2858833968639374, -0.5300451517105103, -0.21498997509479523, -0.20824798941612244, -1.273000955581665, 0.4092750549316406, 0.45653679966926575, 2.0892391204833984, -1.5192875862121582, 0.39515572786331177, -0.26916542649269104, -2.257754325866699, 0.923142671585083, -0.02609558217227459, -1.5479331016540527, -0.7756954431533813, 0.2278260439634323, 1.1800892353057861, 0.05690813809633255, -1.1889822483062744, 0.2094859927892685, 1.297662615776062, 0.7775712609291077, 1.8965606689453125, -1.4483520984649658, 0.0938686653971672, 0.6025287508964539, -1.7227351665496826, 0.5194572806358337, -1.3822052478790283, 0.1623053103685379, -1.3262035846710205, -1.296962857246399, 1.8273245096206665, -1.4460033178329468, -0.8942109942436218, 0.0802997350692749, -2.078596353530884, 1.6041468381881714, -0.10080871731042862, 0.7043444514274597, 0.6566315293312073, 0.46821150183677673, 0.4599568545818329, -1.3326807022094727, -0.06741484254598618, -1.121994972229004, 1.4236809015274048, 0.9717382192611694, -1.2898398637771606, -0.38313472270965576, 1.3139196634292603, -1.0039453506469727, 0.9123828411102295, 0.5350092053413391, 0.327131062746048, -0.3271848261356354, -0.2597966492176056, -0.8546438217163086, 1.1881301403045654, 1.7642730474472046, -0.5759466290473938, -0.6767197251319885, -2.0162761211395264, 0.11247051507234573, 0.45252785086631775, -1.2216265201568604, -1.5953888893127441, -0.15231847763061523, 1.2820842266082764, -0.12775878608226776, -0.0924825519323349, -0.4356553256511688, 0.00312583870254457, 0.2103647142648697, -0.15288080275058746, -0.36792218685150146, 1.1354118585586548, -0.2624414265155792, 0.14168767631053925, 2.440962553024292, 0.45095106959342957, -0.40037912130355835, 1.8768646717071533, -0.7203995585441589, 0.0956818237900734, 0.4925324618816376, 1.3183821439743042, -0.39174357056617737, 0.38963159918785095, 1.3083829879760742, -0.07154451310634613, -0.6641647815704346, 1.4420208930969238, -0.4859248399734497, 0.19628523290157318, -0.22413866221904755, -2.4015653133392334, 0.1320880800485611, -0.8253951072692871, -0.6820330619812012, -0.7749242782592773, -0.5137825012207031, -0.12774458527565002, -0.412325918674469, 1.8269591331481934, -0.292443186044693, -0.15238280594348907, -1.0600700378417969, 0.19250456988811493, -1.7881453037261963, 1.1655387878417969, -0.493259072303772, -0.5364303588867188, 0.3785726726055145, 2.033160448074341, 0.4159850776195526, -0.7824046015739441, 1.0127581357955933, -0.7390449047088623, 1.2913137674331665, 0.11621536314487457, 0.9150251746177673, -0.718158483505249, 0.026519808918237686, 0.21400530636310577, -0.1430734246969223, -0.17462517321109772, 1.565006136894226, 1.7304468154907227, 0.6602473855018616, 0.3331640958786011, -1.0943758487701416, 1.1928554773330688, -0.42816197872161865, -1.0979766845703125, -0.5475892424583435, 0.5898603200912476, -0.7344268560409546, 1.4859492778778076, 1.1604022979736328, 0.2280600368976593, 0.19347470998764038, -0.39241257309913635, 0.5056124329566956, -0.5677528977394104, 0.9325963258743286, -0.9903556704521179, 0.41301968693733215, -1.5059858560562134, 0.7027028203010559, 0.8392152786254883, -0.5479278564453125, 0.41735100746154785, -1.465048909187317, -0.6064597368240356, 0.17882968485355377, 0.704804539680481, -0.6482527256011963, -2.5322272777557373, -0.03121841512620449, -0.8052562475204468, -2.068089485168457, -1.8062973022460938, 0.9342592358589172, -0.6818562746047974, 1.8203175067901611, 0.31759896874427795, -0.8507837653160095, 1.1569398641586304, -0.36360469460487366, -0.8065715432167053, 1.215287208557129, 0.11297693848609924, 1.2839796543121338, 0.4219858944416046, -0.9860942363739014], [0.16575968265533447, 0.8769795894622803, -1.337418556213379, -0.6923803091049194, 1.2086321115493774, -0.09756071120500565, 1.0174877643585205, 0.923549473285675, -0.05695110186934471, 0.5299267768859863, 0.05489572882652283, 0.38447752594947815, -0.34503182768821716, 1.2451990842819214, -0.977182149887085, 2.6461007595062256, -0.17681999504566193, 0.8497282266616821, -1.2315517663955688, -0.24796712398529053, 0.3481604754924774, -0.17288386821746826, -1.067665457725525, 0.737683892250061, 0.7499951124191284, -0.5575595498085022, -0.07646622508764267, -1.9988526105880737, -1.0593407154083252, -0.293084979057312, -2.1879210472106934, 0.21165648102760315, -0.8041548132896423, -0.19137288630008698, 1.1628388166427612, -0.21741226315498352, -0.021338284015655518, -0.8719250559806824, -0.062246277928352356, 0.1013362854719162, 0.3252488076686859, 0.47502726316452026, -0.44662126898765564, -1.4875465631484985, -0.5915381908416748, 1.313256859779358, 0.9589108228683472, 0.748310923576355, -0.5037004351615906, 0.2550815939903259, 0.12606282532215118, -1.7210686206817627, -0.8325809836387634, -1.4497032165527344, 0.34141838550567627, -0.22816000878810883, 0.7795760631561279, 1.549020767211914, -1.4618239402770996, -0.4748888313770294, 1.6872144937515259, -0.011229963973164558, 0.14416871964931488, -0.3224577307701111, 0.10526277869939804, 1.7593446969985962, -0.4162028431892395, 0.08194169402122498, 0.22011451423168182, -0.27902552485466003, 0.9158819317817688, -0.8827716708183289, 0.3773280382156372, -2.2268025875091553, 0.9512614607810974, 0.7877227663993835, 0.3969966173171997, -0.44338667392730713, -0.031247615814208984, -0.8903220891952515, -0.6050155758857727, 0.7600467205047607, -0.49325111508369446, 0.3002704083919525, -0.5727987289428711, -1.502224087715149, 0.10813991725444794, -2.9567148685455322, 0.8318713307380676, -1.3330968618392944, 1.5137478113174438, 0.3287111818790436, 0.4377003610134125, 0.342732697725296, -0.5430046916007996, -0.1128864735364914, 0.4015972316265106, 0.10127053409814835, -0.4982204735279083, 0.20446527004241943, -0.4851140081882477, -0.1434672325849533, -1.1761678457260132, -0.12400038540363312, -1.797803521156311, -0.6893587112426758, 0.44029009342193604, 1.2943546772003174, -0.883084774017334, 3.155050754547119, 0.3667353093624115, -0.4366949200630188, -0.009263362735509872, -1.3215246200561523, 2.3860232830047607, -1.061463475227356, -0.0027134548872709274, -0.834357500076294, 1.3814030885696411, -0.5574259161949158, -0.873550295829773, -0.1033191978931427, -0.9383596777915955, 0.006277590524405241, -0.5842437744140625, -0.16362710297107697, -0.5237857699394226, -0.46271276473999023, -1.0267291069030762, 0.12301485240459442, 0.060314372181892395, -1.1112264394760132, -1.1074165105819702, 0.9333760142326355, -1.9895403385162354, 0.8126224279403687, 1.7025457620620728, 0.2927921712398529, 2.322605848312378, -0.7069758176803589, -0.44933074712753296, -0.9462535381317139, -0.0090243024751544, -0.018507489934563637, 0.5148919820785522, -0.13018812239170074, 0.5915175080299377, 0.5270540714263916, 1.5265146493911743, 0.021997423842549324, -0.7740514278411865, 0.06306837499141693, 0.638529896736145, -1.5413095951080322, 1.0065686702728271, -0.20443862676620483, 0.3650287687778473, -1.0902377367019653, 0.5096434354782104, -1.472744107246399, 1.3132843971252441, 0.8159263730049133, -1.0651788711547852, 0.7784942984580994, 0.12627625465393066, -0.5124971270561218, 0.22793059051036835, -1.8844205141067505, 1.0217808485031128, 2.310781955718994, 0.43303126096725464, -1.0062099695205688, -0.41749927401542664, -0.07629736512899399, -1.0037583112716675, 1.5290343761444092, -0.1201784759759903, -0.2753349244594574, -0.7299917340278625, -2.349005937576294, -0.29914629459381104, -0.4281279444694519, -0.8209465146064758, 1.108467936515808, -0.6764394640922546, 1.3203065395355225, -1.1082755327224731, 0.977009117603302, -0.9146295189857483, -0.1398584246635437, 0.3586616814136505, -0.36046072840690613, 0.9095858931541443, 0.8821152448654175, -1.0897308588027954, -0.10496646910905838, 0.148410364985466, 0.24274541437625885, -0.19486205279827118, 0.006163189187645912, -0.6819518804550171, -1.5199098587036133, 1.6826953887939453, -0.860997200012207, 1.0881457328796387, 0.25376003980636597, 1.1004561185836792, 0.5959484577178955, 1.0984476804733276, 0.35268929600715637, -0.26509609818458557, -2.160174608230591, 1.5867277383804321, 1.3177241086959839, -0.9678336977958679, 1.941806435585022, 0.709736168384552, -1.0782142877578735, -1.0634649991989136, 1.1611560583114624, -0.11380906403064728, -0.13195066154003143, 2.1688284873962402, 0.11190947890281677, 1.9723941087722778, 0.07034211605787277, -0.33572670817375183, -1.687207818031311, 1.2021491527557373, -1.9026137590408325, 1.151110053062439, 0.5559253692626953, -0.1288246065378189, -0.6282461285591125, -1.1676006317138672, 0.46703892946243286, -0.48819699883461, 0.017292959615588188, 0.986393392086029, -0.47292596101760864, 1.4771480560302734, 0.19633354246616364, 0.7526986598968506, 0.5407665967941284, 1.4206926822662354, -2.046876907348633, 1.808571219444275, -0.15741828083992004, 1.4261490106582642, -0.5450209975242615, -0.621999204158783, 0.16845965385437012, -0.3165316581726074, -1.0729222297668457, -0.676806628704071, -3.139350414276123, -0.09709508717060089, -0.567980170249939, -0.530307948589325, 0.6653910279273987, -1.3269294500350952, 0.4864898920059204, 1.1160837411880493, -1.561284065246582, -0.7122986912727356, 0.2674831449985504, 0.4344314932823181, -1.9899450540542603, -0.14922529458999634, 0.41673165559768677, -0.5533550381660461, -0.1262107640504837, -0.5482221841812134, -0.04934357851743698, -1.0663199424743652, -0.15215565264225006, 0.5861751437187195, 0.8341662287712097, 0.6746377348899841, 1.212119460105896, 1.2509737014770508, 0.3619459271430969, -0.9997516870498657, 1.3789633512496948, 1.6002278327941895, 0.5273206233978271, 0.3102914094924927, 0.6136100888252258, 2.2217659950256348, -0.04858538508415222, -0.8552729487419128, 2.1959068775177, 0.2929953932762146, 0.6788115501403809, -1.061132550239563, 0.043528489768505096, 0.011274016462266445, -0.1574760228395462, -0.5444614887237549, -0.4637244939804077, -0.12519501149654388, -1.203518271446228, 0.05450114980340004, 0.7528440356254578, -2.8096203804016113, -0.08020202815532684, -0.21249468624591827, 1.4544709920883179, -1.2201348543167114, 0.5235863924026489, -0.02560408227145672, -0.342415988445282, 0.9978464245796204, 0.12757153809070587, 0.21187110245227814, 0.46740975975990295, 0.0009587786043994129, 0.2514081299304962, 0.4252568781375885, 0.41508281230926514, 0.14593946933746338, 0.16941562294960022, 1.5376335382461548, -0.7034251093864441, 2.137131929397583, 0.9706625938415527, 0.18087370693683624, -0.26794570684432983, -0.053635068237781525, 0.4509281814098358, 1.0706603527069092, -0.5932237505912781, -1.3746020793914795, -0.22320325672626495, 0.3736869990825653, -0.28825780749320984, 0.05138784274458885, -1.1372400522232056, -0.9036126732826233, 1.8137985467910767, -0.5235289931297302, 1.2784420251846313, -0.5888314247131348, -0.30752331018447876, 0.25335198640823364, -3.40158748626709, 0.7892398834228516, -0.04480060189962387, -0.519677460193634, 0.11865375936031342, -0.07517461478710175, 0.5925877094268799, 0.6188926696777344, 1.0513321161270142, 1.6551322937011719, -1.7255135774612427, -0.3876119554042816, -0.3664532005786896, 0.19728167355060577, 0.23311857879161835, 1.1215498447418213, -1.6048020124435425, -1.6219226121902466, -0.8939352631568909, 0.5004056096076965, 0.34140583872795105, -0.6698618531227112, 1.6455765962600708, 0.19471949338912964, 2.3249094486236572, 0.5508055686950684, -2.843475580215454, 0.194392129778862, -0.1236887127161026, -0.2849333584308624, -0.3783436715602875, -0.7268940806388855, -2.627823829650879, -0.905634343624115, -2.4021759033203125, 0.6310149431228638, 0.25088709592819214, 0.7642272114753723, 0.499879390001297, -1.3931750059127808, -0.6552059054374695, 0.7574126124382019, 0.03509339317679405, -0.2925284802913666, -0.04566184803843498, 0.9805160760879517, -1.4301707744598389, 0.7200363874435425, 0.16836735606193542, -1.6143007278442383, -0.7336801290512085, -0.49480798840522766, -0.8426394462585449, -1.1504361629486084, 0.24053649604320526, 1.0363355875015259, -0.6366715431213379, -0.4514518976211548, -0.7635674476623535, 0.6837844848632812, -0.4670059084892273, -0.2304910570383072, 0.32208141684532166, -1.311220645904541, -0.6374492645263672, -0.20583267509937286, 1.7410002946853638, 1.6865119934082031, -0.1678037941455841, 0.7506704330444336, 0.5842284560203552, 0.8474669456481934, -0.8196919560432434, -1.6394098997116089, -0.673911452293396, -0.43652012944221497, 0.7769795656204224, -0.41025105118751526, -0.16521456837654114, -1.964926838874817, -1.3486961126327515, 0.1820744425058365, -0.35040614008903503, 0.5114114284515381, -0.3396034240722656, -1.4424974918365479, 0.9012202620506287, 0.4289250075817108, 0.8625713586807251, -0.4571548104286194, -0.6848065257072449, 0.06718138605356216, -1.1813474893569946, 0.14913779497146606, 0.5091395378112793, -1.0027306079864502, 0.2375303953886032, 0.8195307850837708, -0.4127997159957886, 0.4604002833366394, 0.6256900429725647, 1.1588988304138184, -0.7626966238021851, -0.6968790292739868, 0.5276515483856201, -0.5420430898666382, -0.11575914174318314, 0.982658326625824, -0.9508914351463318, 0.32757413387298584, 0.7606087923049927, 0.9382300972938538, -0.7480495572090149, 2.1637821197509766, -0.37897178530693054, -0.026835553348064423, 1.3000456094741821, -0.1572035253047943, 0.4278109073638916, -0.8757580518722534, 0.06742318719625473, 0.8543717265129089, -0.4359501600265503, -0.9985669255256653, 0.05913408845663071, 0.45438164472579956, -0.5052110552787781, -1.4458550214767456, 1.3719196319580078, 0.06121726334095001, -0.49610432982444763, 1.9737287759780884, -1.6505838632583618, 0.19822919368743896, 0.7609286904335022, -0.09242670238018036, 0.371493935585022, -0.36467447876930237, 1.9683952331542969, 0.4917731285095215, 0.29396143555641174, -0.33781448006629944, 0.2439364641904831, 1.4390407800674438, 1.8007346391677856, -1.306801438331604, 0.08002559840679169, -0.46922221779823303, 0.7683477997779846, 0.49824196100234985, -0.5531019568443298, 1.5287286043167114, 0.10080265998840332, -0.7799553275108337, 1.3752470016479492, -0.6419798731803894, -0.8887714743614197, -0.27173104882240295, 0.7800373435020447, -1.553052306175232, 0.5882117748260498, -0.7827365398406982, -0.4688361585140228, 2.4960548877716064, 1.5419400930404663, 0.42965424060821533, -1.7162480354309082, -0.9383313655853271, -0.5209370851516724, -0.7231765389442444, -0.6288771629333496, -1.4110727310180664, -0.23553630709648132, -1.0258355140686035, -0.4470212459564209, 0.1738806813955307, -0.3536851704120636, 1.1229411363601685, -1.4536651372909546, -0.5308263301849365, -0.17188787460327148, -1.5757325887680054, 0.18189948797225952, 0.48518791794776917, -0.6879130005836487, 0.11727071553468704, -0.31478652358055115, -2.280057907104492, -0.44524291157722473, -1.2819061279296875, -1.649224042892456, -0.3423253893852234, 1.0072040557861328, 0.9958296418190002, -1.908998727798462, 2.1222665309906006, 0.5337398648262024, -0.634309709072113, 0.6819908022880554, -0.01225180272012949, 1.1937917470932007, -2.1103742122650146, 0.11876226961612701, 0.8147907853126526, 1.2773054838180542, 1.1198759078979492, 0.39448246359825134, 2.591329336166382, -0.19609998166561127, 0.06592119485139847, -0.6546549797058105, 0.7275310158729553, 0.5234580039978027, 0.33720600605010986, 1.1593221426010132, -0.4066699743270874, 1.2733395099639893, 1.050609827041626, -0.01894722320139408, -0.4571990966796875, 1.7802448272705078, -0.34186428785324097, -0.018177246674895287, -0.5995758175849915, 0.8689560890197754, -0.8531507849693298, 0.111577108502388, 0.35594263672828674, -0.5722301602363586, -0.5771145224571228, -0.2498917430639267, 1.9016458988189697, 0.1717693954706192, 0.3891388773918152, -1.3294135332107544, 1.383360743522644, -0.14460442960262299, -1.4951034784317017, -1.017884373664856, 1.7067166566848755, 0.07132703810930252, 1.0827127695083618, 2.424231767654419, 0.8804642558097839, -1.530857801437378, 0.021385643631219864, 0.24882157146930695, -0.8970485925674438, 0.5962201952934265, -2.3733456134796143, -0.6801890134811401, 0.7869483232498169, 0.060394056141376495, -1.1302658319473267, -0.6068721413612366, 2.8371245861053467, -0.3078959584236145, 1.132010817527771, -0.07645384967327118, 1.2801841497421265, -0.579181969165802, -0.29373687505722046, -1.6493536233901978, 1.4074478149414062, 0.006446496583521366, 1.8382600545883179, 0.23741920292377472, -0.14816230535507202, -0.26639601588249207, 1.143422245979309, -1.180079698562622, 0.5222498774528503, 0.30785027146339417, -1.3352714776992798, 0.7522749304771423, 1.0432701110839844, -1.3666880130767822, 0.8386756777763367, 0.4037584066390991, -1.6749101877212524, 0.9994313716888428, -0.11023145914077759, 0.6757469773292542, 0.50466388463974, 0.6720375418663025, -0.6578052639961243, 1.6208099126815796, -0.9815941452980042, -0.8502922654151917, 0.39788010716438293, -1.3917056322097778, 1.0911247730255127, -1.4499636888504028, -1.1585899591445923, -0.05086704343557358, 0.24748831987380981, -1.487212896347046, 1.4741010665893555, 0.36969736218452454, 0.3027820587158203, 0.42231446504592896, 2.127608299255371, -0.6155273914337158, 0.004811728373169899, 0.25796666741371155, -1.0178841352462769, -1.0532283782958984, 0.5848889946937561, -0.2870570421218872, 0.2568575143814087, 1.1808068752288818, 0.8915626406669617, -0.28484266996383667, 0.8757127523422241, -1.2866040468215942, -2.233795642852783, 0.5476128458976746, -1.047595739364624, 1.4476970434188843, -0.32432588934898376, 2.810997724533081, 0.865247905254364, -1.3414546251296997, -2.0660645961761475, 0.16832497715950012, -0.5068516135215759, -0.5730612874031067, -0.5177401900291443, -0.9579154253005981, -0.2631120979785919, -0.615848183631897, 0.22881050407886505, -1.7865350246429443, -0.3077457845211029, -0.9251492619514465, 0.6007155179977417, 0.36818236112594604, 0.014126409776508808, -0.8340193033218384, 0.7311015725135803, 0.1085902526974678, -0.01861756294965744, -0.6687484979629517, 0.16117431223392487, 1.1629948616027832, -1.5773658752441406, 0.7092272639274597, -1.507776141166687, 0.939113438129425, 0.6873763203620911, -0.35674813389778137, 0.11395936459302902, -1.1788054704666138, 0.29438555240631104, 0.3731748163700104, -0.6635378003120422, -0.5130828619003296, 1.152137279510498, 0.5416797995567322, 0.13025614619255066, 0.0007083459640853107, -0.3828998804092407, -1.5249888896942139, -0.4935859441757202, 0.968737006187439, -0.6256173253059387, 0.22453607618808746, 0.8370299339294434, -1.3525915145874023, -0.018568897619843483, -1.807283639907837, -0.4584331214427948, 0.1344936341047287, 0.4769187867641449, 0.8935293555259705, 0.6586027145385742, 2.083434820175171, -0.43704739212989807, -0.948717474937439, 0.08552274107933044, 2.136389970779419, -1.719429612159729, -0.3876999616622925, 0.7542795538902283, -0.35521838068962097, -2.1638078689575195, -0.27453312277793884, 0.008987585082650185, 0.1475738286972046, 0.48040521144866943, 1.0593725442886353, 2.2823824882507324, 0.1501748412847519, -1.7032134532928467, -1.7331851720809937, -0.5830236673355103, -0.3332122564315796, 0.05924776941537857, -0.5068525671958923, -1.7691254615783691, -0.9159099459648132, -0.42378509044647217, -0.5505466461181641, -1.3048207759857178, -1.492451548576355, 0.7845411896705627, -0.2736576497554779, -1.2258411645889282, 0.41335511207580566, -0.5413601398468018, -0.033704061061143875, 0.8135755658149719, 0.7601595520973206, -0.22799305617809296, 1.6252787113189697, 2.670708656311035, 0.4468230903148651, 0.9372813701629639, -0.5119933485984802, 1.1699146032333374, -0.7615061402320862, -0.07428085058927536, -0.36747193336486816, 1.1812388896942139, -0.9798719882965088, -1.9363136291503906, 0.36279067397117615, 0.7256610989570618, 0.6316186189651489, 0.28730952739715576, -0.14128850400447845, -0.6967542171478271, -0.13618403673171997, -0.019141145050525665, -1.7543485164642334, -0.05537628382444382, 2.1437151432037354, -2.4453511238098145, 0.9665096402168274, -0.02221524529159069, 0.11543751507997513, 0.6435707211494446, 0.7192699909210205, -1.4915778636932373, -0.3426762819290161, 0.6016473770141602, 0.16369588673114777, -1.902693510055542, 1.2873961925506592, 0.6275532245635986, 2.206061840057373, 0.05757110193371773, -1.3249101638793945, 1.3195955753326416, -1.3283430337905884, -1.1250756978988647, -0.14016176760196686, 0.09575881063938141, 0.31766602396965027, -0.956954300403595, -0.13747276365756989, -0.6659002900123596, -1.1735533475875854, 0.25457239151000977, 0.4323608875274658, -0.7935197949409485, 0.3710338771343231, 1.203718900680542, 0.5393702983856201, -0.48986127972602844, 1.180552363395691, -0.17412972450256348, 1.0626165866851807, -0.3824252486228943, 0.7087646126747131, -0.010770132765173912, 1.006846308708191, -1.791734218597412, -1.149712085723877, 1.6342891454696655, -0.6107593178749084, 0.6202791333198547, 0.9064421653747559, -1.3030674457550049, -1.263883113861084, -0.07598283141851425, -0.7225714325904846, -4.369974613189697, -1.027709722518921, -0.5057681202888489, 1.1613616943359375, -0.7342450022697449, 0.1373480260372162, 0.603168249130249, 1.276989459991455, -0.7706511616706848, 0.2736298441886902, 0.33067086338996887, 0.3920842409133911, -0.48981228470802307, 1.6739733219146729, 0.838936984539032, -1.8551055192947388, -0.2772365212440491, -0.6512770056724548, -0.4493614137172699, 0.9404692649841309, 1.2081420421600342, 0.4353773295879364, 0.130095973610878, 0.7082871794700623, 0.661564290523529, -0.4999557435512543, 1.2523210048675537, -0.059215571731328964, 0.8115522265434265, 1.4272146224975586, -0.1455956995487213, -1.348575472831726, 0.524882972240448, 0.8365029096603394, -0.5012164115905762, -1.7732505798339844, -0.8280882239341736, -1.6539667844772339, 0.6095876097679138, 0.69246906042099, 1.5725277662277222, -0.09581717103719711, 1.38609778881073, 0.6004525423049927, -2.0889763832092285, 0.039817679673433304, -0.06359641999006271, -1.0226637125015259, 0.17269288003444672, 0.2538010776042938, 0.8755406141281128, -0.8615620136260986, 0.8740125894546509, 0.8949933052062988, -0.2328478991985321, -0.49809184670448303, -0.5602138042449951, -0.6551316380500793, -0.7390222549438477, -0.7344467639923096, 0.6083266139030457, -0.8322259187698364, 0.11157691478729248, -0.5264741778373718, 1.3223669528961182, -0.1541161984205246, -1.2751903533935547, 0.012470332905650139, 0.06547799706459045, -0.8977395296096802, 0.29742521047592163, 1.207719087600708, 0.22112560272216797, -0.2233259230852127, -2.110795259475708, -0.33687514066696167, -0.2980554401874542, 0.5563144087791443, -1.3724733591079712, 1.4782012701034546, -0.258918434381485, 2.0340583324432373, 0.01811194233596325, -1.1358540058135986, 0.16845928132534027, 1.0943021774291992, -1.813138484954834, 0.5258668661117554, 0.06747417896986008, 1.1031887531280518, 0.2764376699924469, -0.042974844574928284, 1.6262421607971191, -0.4701099693775177, -0.37718963623046875, 0.7893943786621094, 0.7900756001472473, -0.9134109020233154, 0.41201138496398926, -0.10984481871128082, 0.3314301371574402, -0.5915306806564331, -2.510472536087036, -0.8638935089111328, -0.6970623731613159, -0.4841465353965759, -1.0937752723693848, -0.3305951654911041, 1.262464165687561, 1.9844272136688232, -0.5456027388572693, -0.35147976875305176, -0.32093724608421326, -1.875651478767395, -0.32979604601860046, -0.5871545672416687, -0.4783630967140198, 0.1593797653913498, -0.5013504028320312, 2.689964771270752, -0.5412815809249878, -1.1218856573104858, -0.6340920925140381, -0.700892448425293, 0.19027380645275116, -1.01361882686615, 1.5267881155014038, -2.2529664039611816, -0.19129307568073273, -1.549204707145691, -0.27755239605903625, 0.008500120602548122, -0.4512150287628174, -1.9923911094665527, 0.9688937067985535, -1.6194086074829102, -1.9299075603485107, 1.5424180030822754, 0.42925727367401123, -1.319921612739563, -0.8383834362030029, -1.6566097736358643, -2.461188316345215, 2.0484840869903564, 0.9566065073013306, 0.2160651832818985, -0.6609840989112854, -1.0274919271469116, -0.5112696886062622, -0.9849614500999451, -0.43906453251838684, -0.5438015460968018, -1.6071101427078247, 0.2260866016149521, 1.9716891050338745, 1.076216697692871, 0.3407020568847656, 1.7651493549346924, -0.6035240292549133, -1.2016816139221191, -0.659735381603241, 0.016603240743279457, 0.8822314739227295, 0.8557196855545044, -0.45109960436820984, 1.2333053350448608, -1.2439161539077759, 1.6334517002105713], [1.103295922279358, 0.5200631618499756, -0.10437863320112228, -0.5540177822113037, 0.10371332615613937, 0.5921960473060608, -2.229916572570801, 1.7197493314743042, -0.17224527895450592, 1.0207455158233643, -1.6189515590667725, 1.3239209651947021, -0.21045270562171936, 0.6314725279808044, -1.1693576574325562, -0.7104937434196472, 0.8760049939155579, 0.33191347122192383, 0.37791186571121216, 0.45437195897102356, -0.044936127960681915, -0.1737348735332489, 0.8612533807754517, -0.7163979411125183, -0.19775940477848053, 1.424940586090088, 0.904783308506012, -0.18465669453144073, 1.1768238544464111, 0.6293359994888306, 0.5412710905075073, -1.8534717559814453, -1.1396265029907227, -0.5405089855194092, -0.6551311016082764, 1.450087070465088, -1.7101061344146729, 1.0747896432876587, -1.484578251838684, 0.2470448613166809, -0.5795010328292847, -0.8714687824249268, 0.5087782740592957, 1.579413890838623, 0.09905915707349777, -0.01595761626958847, 0.30138301849365234, -0.501188337802887, 0.45286718010902405, 0.20811660587787628, 0.6463749408721924, -0.4784386456012726, -0.8919109106063843, -0.020630408078432083, 1.2470148801803589, -0.028395874425768852, -0.8184166550636292, -0.8193116188049316, -0.3579745590686798, 1.3124873638153076, 1.513450264930725, -0.7769374847412109, -0.3169117569923401, -1.5778464078903198, 1.8718699216842651, 1.1461303234100342, -0.49873223900794983, 0.6398354172706604, -1.896060824394226, -0.2196713089942932, -0.15325310826301575, -0.1767532378435135, -2.0882928371429443, 0.028637127950787544, -0.2116451859474182, -0.27469220757484436, -1.1116961240768433, 0.07405783981084824, -0.018738387152552605, -1.6119027137756348, -0.41287267208099365, 0.6075422167778015, 0.4938661456108093, -0.894203782081604, 2.7897253036499023, 1.4186745882034302, -0.02594531700015068, 0.4054146707057953, 1.613444447517395, -0.3625364303588867, -1.1958808898925781, -0.5261078476905823, 1.2647980451583862, 0.7398186922073364, 1.047500729560852, 1.8942711353302002, 1.4173966646194458, -0.5088721513748169, -0.29900744557380676, -0.5661289095878601, 0.7608345746994019, 1.1929370164871216, 0.12464005500078201, 1.153404951095581, -1.0604225397109985, 0.5304042100906372, -0.6827746629714966, -1.4657533168792725, 0.8084142804145813, 0.42889919877052307, 0.6629562973976135, -1.9657442569732666, -0.34585851430892944, -0.8592694997787476, -0.1748194694519043, 1.061900019645691, -0.6224834322929382, 1.7798720598220825, 1.039193034172058, -0.2813429534435272, -0.11624067276716232, 0.2650304436683655, 0.41333454847335815, 2.9925825595855713, -0.28729894757270813, -0.8293455243110657, 0.004106347449123859, -1.1134605407714844, 0.3464095890522003, 0.8786429166793823, -0.40348076820373535, 0.7248002290725708, -0.3213179409503937, -0.0667300894856453, 1.545292854309082, 1.2422138452529907, 0.3119264841079712, 0.8644365668296814, -0.1782544106245041, -2.044748544692993, 0.5492292642593384, 0.13780613243579865, -0.5026111602783203, 0.16115158796310425, -1.3592615127563477, 0.6688272356987, 0.5936402082443237, -1.9369525909423828, -1.1299651861190796, 0.27886903285980225, 0.6762832403182983, -2.3113365173339844, 0.05300388112664223, 0.5239675045013428, -1.5090043544769287, -0.6991903185844421, -1.2550684213638306, 0.384574294090271, -0.5759673714637756, -2.165051221847534, -0.18823371827602386, 0.10014280676841736, 1.4990187883377075, 0.4429095387458801, -0.5002642869949341, 1.4738174676895142, -0.4190962314605713, 0.1764310747385025, -0.06539043039083481, -1.6590534448623657, 0.21063093841075897, 2.466019868850708, 1.0071420669555664, -0.05648341774940491, 0.44475632905960083, 0.5878178477287292, -0.7894063591957092, -1.4891374111175537, 0.2675071358680725, 1.2595393657684326, 0.7809371948242188, -1.3755379915237427, 0.4839628338813782, 0.4622895121574402, -2.6352155208587646, 0.6333855390548706, -0.15793615579605103, -0.14774273335933685, 0.3315128684043884, -0.16460521519184113, 0.43071311712265015, 1.6244909763336182, -0.009449513629078865, -0.8962709903717041, -0.42805078625679016, 2.1812312602996826, -1.758721113204956, -0.623605489730835, -0.7754755616188049, -1.488368272781372, -0.5085539817810059, -0.27772966027259827, 1.0340707302093506, 0.4830469787120819, -1.1966464519500732, -0.6443399786949158, 0.5510047078132629, 1.8077514171600342, 0.480173796415329, -0.5411818027496338, -1.004522442817688, 0.8398959040641785, -1.5750236511230469, -0.1871197670698166, 2.191688060760498, -1.7018756866455078, 0.37614861130714417, -1.6892317533493042, -1.1736243963241577, 1.1539348363876343, -0.2576199471950531, 0.3595412075519562, -0.40372633934020996, -0.7639791965484619, -0.6163638830184937, 0.41284722089767456, 1.395828127861023, -0.045271217823028564, 2.2239696979522705, 1.447498083114624, -1.67491614818573, -1.2777106761932373, 0.3682921230792999, 0.4206785261631012, -1.1207306385040283, -0.531978189945221, -1.4135570526123047, -0.47179922461509705, 1.1603022813796997, 0.4521785080432892, -0.6386014223098755, 1.7438433170318604, 2.624293804168701, -0.7847993969917297, 0.01521723996847868, 1.3958241939544678, -0.5954511761665344, 1.0811011791229248, -1.1281006336212158, -1.6019681692123413, 1.0001283884048462, 0.851593554019928, 0.7397260665893555, 0.8822598457336426, 0.7123175263404846, 0.6917989253997803, 0.30439549684524536, 1.2256748676300049, 1.2959035634994507, 0.884812593460083, 1.5031468868255615, 1.3098187446594238, -0.49967172741889954, -0.2826748192310333, -0.14376939833164215, -0.6177825331687927, 1.1031594276428223, -0.983745813369751, -1.4351625442504883, -0.4655197262763977, 0.7225720286369324, 0.6498261094093323, -0.6367590427398682, 0.8998369574546814, 0.5642229914665222, 0.12522399425506592, -0.2910899221897125, -0.7937414050102234, 1.3920694589614868, 0.5834004878997803, 1.1816807985305786, 1.5049339532852173, 0.7554730176925659, 0.16363780200481415, -0.3333604633808136, 0.5085932612419128, -0.46521055698394775, -1.4766747951507568, -0.09027747064828873, 0.18471647799015045, -0.9425302147865295, 1.0376832485198975, -0.9449753165245056, -0.3724444806575775, 1.641845703125, -0.5068075656890869, 0.38309213519096375, -1.4842568635940552, -0.3687707781791687, 0.6531330943107605, -0.24083249270915985, 0.2534295916557312, -0.7396763563156128, 0.9334307909011841, -0.6512345671653748, 0.054330531507730484, 0.5688557028770447, 1.1391470432281494, 0.5900259017944336, -0.3648037910461426, 1.0977064371109009, -0.23427864909172058, -0.12261918932199478, -0.16059701144695282, 0.43759429454803467, 0.4984699487686157, 0.7469066381454468, 0.6782324910163879, -1.0203102827072144, -0.6285122632980347, 1.4629225730895996, -1.3154754638671875, 1.2985011339187622, 0.5672112703323364, -1.270590901374817, 0.11860395967960358, 0.6344634890556335, 0.8868770599365234, -0.13304166495800018, -1.4826830625534058, -0.10875975340604782, -0.12585996091365814, -0.4250563979148865, -0.4673275053501129, -1.8744903802871704, 0.9169943332672119, -1.3523203134536743, -0.6594547033309937, -0.4799056053161621, -0.891227662563324, -0.2913469076156616, 0.20271748304367065, -0.6283444166183472, -0.23750361800193787, -0.6373366713523865, 1.7615846395492554, -0.08936980366706848, 3.0344958305358887, 0.9918824434280396, 1.4382658004760742, 1.3069676160812378, 1.2780303955078125, 0.23702718317508698, 0.6414709687232971, 0.11387520283460617, 0.285675585269928, -0.1556924730539322, -0.9215548634529114, -1.03663969039917, -1.2542015314102173, 0.08985786139965057, 0.944291889667511, -0.15117360651493073, -0.17685231566429138, 0.38017162680625916, -0.7090844511985779, 1.6708709001541138, -0.5374549031257629, 0.5942447185516357, -1.0360995531082153, -0.8467826843261719, 0.1193876639008522, 0.2385713905096054, 0.9500237107276917, -0.8424970507621765, 1.0370419025421143, 1.5529197454452515, -3.372758626937866, -1.103822946548462, -1.7390880584716797, 0.3626551628112793, -0.07129260897636414, -0.8968266844749451, 0.6117205023765564, 0.6140753626823425, -1.5053067207336426, -0.11991586536169052, 0.6151352524757385, 1.3351343870162964, 0.7922486662864685, -0.9639993906021118, 1.895677089691162, 0.6977295875549316, -0.6136493682861328, 0.43381786346435547, 0.03319576010107994, -1.283883810043335, 0.8135508298873901, 1.048917293548584, 0.2966150939464569, -1.8281036615371704, 1.1938680410385132, 0.43185484409332275, 1.1681907176971436, 2.505293130874634, -0.7248014211654663, 0.8750400543212891, -0.5667119026184082, -0.2711702883243561, 0.3106658160686493, 0.9061914682388306, -0.41226837038993835, -1.5744572877883911, -0.6403887867927551, 0.6869403719902039, -0.019326189532876015, 0.5788822770118713, 0.7558717131614685, 2.08701753616333, -0.19771216809749603, 2.954216718673706, -0.8300786018371582, 0.3318983316421509, 0.9634917974472046, 0.5776816010475159, 0.21497949957847595, 1.044124960899353, 1.8174670934677124, 0.14075233042240143, -1.897019624710083, -0.5390937328338623, -0.49717724323272705, 0.8633211255073547, 0.7450641393661499, -2.503371238708496, 1.7540189027786255, -0.3948820233345032, -1.4451390504837036, -1.5619888305664062, 1.3429086208343506, -0.6876422762870789, 0.25979170203208923, -0.6565681099891663, 0.3520839214324951, -1.3443163633346558, 1.1525990962982178, -0.3510897755622864, -0.6000792980194092, 1.645463228225708, -0.9990529417991638, 0.39470648765563965, 0.28448033332824707, 1.2024832963943481, -0.7957081198692322, -0.018329251557588577, 0.345913290977478, 0.4182036817073822, 0.683501660823822, 0.10479655861854553, 0.43341153860092163, -0.9758720397949219, -1.0447278022766113, 0.910301923751831, 0.03613827005028725, 0.3042653799057007, -0.12799440324306488, -1.1912946701049805, -1.1623247861862183, -0.2155340313911438, -0.8056463599205017, -2.896919012069702, 0.8354810476303101, -0.9370025992393494, 0.1830182820558548, -2.0142030715942383, 0.08202917128801346, -1.1530156135559082, -0.46276652812957764, 0.4729694426059723, 0.10012466460466385, 1.2439563274383545, -0.44098374247550964, -1.6352022886276245, -0.01739821769297123, 0.8257900476455688, -1.37601900100708, -0.05026950314640999, 0.4388789236545563, 0.3045799434185028, 1.2834763526916504, -0.13018590211868286, 0.5507489442825317, -0.8690950870513916, -0.0441741980612278, -1.442956805229187, -1.3774089813232422, -0.04632951691746712, 0.7265979051589966, -0.22927473485469818, 0.07685306668281555, -0.9037454724311829, 1.1195164918899536, -0.6954962611198425, 0.7446906566619873, 0.5120994448661804, -0.3498861789703369, -1.7089083194732666, 0.759201169013977, -1.9446935653686523, -0.5041182637214661, 0.581764817237854, 0.015834560617804527, 1.881277322769165, 2.117591142654419, 1.2016276121139526, 2.373523473739624, 1.193243384361267, -2.050893783569336, -0.34077274799346924, 2.3321473598480225, 2.5458836555480957, 0.7873186469078064, 0.7426407337188721, -1.4637752771377563, 0.5155339241027832, 0.6453036069869995, -0.9145631194114685, 1.108593463897705, 0.8315205574035645, -0.14871813356876373, 2.0914812088012695, 0.06402479857206345, -0.36199960112571716, 0.4842916429042816, -0.3780221939086914, 1.2151408195495605, 1.4044251441955566, 0.017068523913621902, 2.076145648956299, -0.8262366056442261, 0.41068753600120544, -0.566720724105835, 2.1674139499664307, 1.0466095209121704, 1.0908900499343872, 1.9243048429489136, 0.9862866997718811, 0.1653781533241272, -0.176353320479393, 0.051116131246089935, -0.32220637798309326, -0.5236302614212036, 0.37578144669532776, 0.6990708708763123, -0.15967944264411926, -0.5953059196472168, -0.6123157143592834, -0.23640605807304382, 0.3622846305370331, 1.1055271625518799, 0.0985327661037445, 2.935711622238159, 0.15669448673725128, 0.5823385119438171, 1.8120940923690796, -0.3771282136440277, -0.9837889671325684, -0.08980652689933777, -0.21938490867614746, 0.36911094188690186, -1.0423710346221924, 0.36455821990966797, -1.3166108131408691, 0.3531354069709778, -0.26177549362182617, -1.8128166198730469, 2.1692569255828857, -1.3850489854812622, 0.18612028658390045, -1.4313783645629883, 1.0356128215789795, -0.42844149470329285, 1.0526357889175415, 0.5321640968322754, 0.9095253944396973, 0.5746036767959595, -1.372066855430603, 0.3362293243408203, -0.9534090757369995, 0.8693770170211792, -1.001192569732666, 1.2451939582824707, 0.9166830778121948, 0.5569488406181335, 0.7273663878440857, -0.7536260485649109, -1.7580482959747314, -0.07569097727537155, -0.5102404952049255, 0.39500123262405396, 0.2765154540538788, 0.5874903202056885, 0.4719521105289459, -0.02187291346490383, -1.1742557287216187, 0.8536542057991028, -0.6960806250572205, 0.8700076341629028, 1.7079856395721436, 0.9069038033485413, -1.1146992444992065, -0.7886449098587036, 1.0853159427642822, -0.9040795564651489, -0.6051888465881348, -0.49954840540885925, -1.1355347633361816, 0.7662904262542725, 1.5336602926254272, -1.0962467193603516, 1.2791526317596436, -1.0007522106170654, 0.11344019323587418, 1.0798548460006714, -0.3471822440624237, -0.1499171406030655, -0.3353304862976074, 0.00407529016956687, -0.37398236989974976, -0.42655235528945923, -1.0966657400131226, -1.1628354787826538, 0.2954341769218445, -0.23179174959659576, -0.8296490907669067, -0.9711883664131165, 0.19219113886356354, 1.1332134008407593, 0.024971062317490578, 1.0734272003173828, 0.33053451776504517, -0.7965825200080872, -0.4023420810699463, -2.69130802154541, 0.932404637336731, -0.03698763623833656, -0.49211734533309937, 1.5529073476791382, -0.7591356039047241, 1.2078750133514404, -1.0003117322921753, 0.26362770795822144, 0.28175514936447144, 0.8724279999732971, 1.3298629522323608, -0.9231472611427307, 0.9012811183929443, 0.6398509740829468, 0.43273764848709106, -0.11137750744819641, -2.1187500953674316, -0.026479806751012802, -0.8925144076347351, 0.3175809383392334, -0.657487690448761, -1.0198113918304443, 0.24150991439819336, -2.605619430541992, 0.2815195620059967, 0.9141446352005005, -1.2100292444229126, -0.9193528890609741, -0.6390180587768555, 0.9104109406471252, 0.912950336933136, -0.9488818049430847, 1.1381703615188599, -0.36343643069267273, 0.5632644295692444, 0.08265279233455658, 1.021952748298645, 0.010365929454565048, 0.8184491991996765, 0.31036680936813354, 1.3556513786315918, 1.5125607252120972, 1.4688208103179932, -1.1273654699325562, -0.26154717803001404, -0.07262040674686432, -0.6256514191627502, -0.8693008422851562, -0.05091065913438797, -1.0119009017944336, -0.5569747090339661, -0.6223166584968567, 0.6504639983177185, 0.4357343912124634, -0.6548430919647217, -0.5614890456199646, 0.37631756067276, 2.200418472290039, -0.18271639943122864, 1.4494677782058716, -1.1281498670578003, 2.6857106685638428, 1.2409963607788086, 0.7041129469871521, 0.1793607920408249, 1.16574227809906, -0.04687323048710823, 0.37741032242774963, -1.2153617143630981, -0.1366022378206253, 0.7214000225067139, 0.32965052127838135, 0.4419754445552826, 2.6158318519592285, -2.381702423095703, -1.2666219472885132, 0.5439688563346863, 0.2349247932434082, 0.34880292415618896, -0.32421013712882996, 0.9814620018005371, 1.3935599327087402, 0.46157771348953247, 0.1661982238292694, -1.4759376049041748, 0.2898402214050293, 1.0163133144378662, 0.06694366037845612, 1.2454044818878174, -0.06595365703105927, 0.38458821177482605, 2.288057565689087, -1.0574313402175903, 0.1364648938179016, 2.1300907135009766, 1.1381900310516357, 0.5875759124755859, 0.5089457035064697, -2.925952911376953, 0.8478840589523315, -0.22823981940746307, 0.4141802191734314, -0.3519603908061981, -0.6603893637657166, -2.5967986583709717, 0.12695083022117615, -0.44314107298851013, -0.1353757381439209, -0.39892157912254333, 0.24055498838424683, -0.41492101550102234, 0.18806946277618408, -0.9850341081619263, 0.784153163433075, -0.29589542746543884, 0.190373033285141, -0.3485143184661865, 1.0361560583114624, 1.2309149503707886, 0.16475394368171692, -0.7360278367996216, 1.2666624784469604, 0.6287159323692322, -0.053575675934553146, -0.5522925853729248, 0.8628609776496887, 0.7452136278152466, -2.2044248580932617, 0.06223169341683388, -2.1723451614379883, 0.7313857078552246, -0.31202584505081177, -0.2735750675201416, -0.04109175130724907, -0.032602958381175995, 1.4627984762191772, 0.3343077600002289, -0.07322264462709427, 0.14318688213825226, 0.043485142290592194, 1.6848183870315552, 1.2746661901474, 0.7559975385665894, -0.06713372468948364, -1.0667396783828735, -1.2910712957382202, -1.7074553966522217, -0.6520959138870239, -0.4958671033382416, 0.7547041773796082, 0.9380837678909302, 0.22651173174381256, -0.5777242183685303, -0.2682930827140808, 0.20990021526813507, -0.2608184516429901, 0.5119141936302185, 0.19761307537555695, 1.598963737487793, 0.29560405015945435, 0.3697682321071625, 0.3068886697292328, 0.0877670869231224, 0.17119476199150085, -0.9633450508117676, -0.6640669107437134, -1.4404244422912598, -0.5544434189796448, -1.879268765449524, 1.2556971311569214, 2.41991925239563, 0.6075038313865662, -1.5356618165969849, 0.4484589099884033, 1.6449337005615234, -1.6979175806045532, 0.6652241945266724, 0.4976237118244171, -0.883183479309082, -0.7232987880706787, -0.492236852645874, -1.1697629690170288, -0.14535264670848846, 1.5868655443191528, -2.0263373851776123, -0.40538281202316284, 1.4962389469146729, 0.5189215540885925, 0.9362567067146301, -0.3545013666152954, -1.1757724285125732, -0.8648185133934021, 0.2755577266216278, -0.5367975831031799, 1.307834506034851, 0.702581524848938, 0.5431681275367737, -0.5604262948036194, 1.1998858451843262, 0.04245087131857872, 0.43129441142082214, 1.72380530834198, 0.6080533862113953, 1.538268804550171, 0.9373159408569336, 0.46671581268310547, 1.6142816543579102, 1.5293023586273193, 1.4487286806106567, 0.30239740014076233, 1.2444771528244019, -0.603286862373352, -0.4613112807273865, -0.04164639860391617, -0.5717431306838989, 2.788938522338867, 0.03394881635904312, 0.8899597525596619, -0.11527606099843979, 0.4992552101612091, 0.7824203968048096, 1.07620108127594, -2.0517418384552, 0.06748832017183304, -0.9512947797775269, -1.4104219675064087, 1.0971654653549194, -0.7679635882377625, -0.5352914929389954, -0.4356760084629059, -0.5859048366546631, -1.4036953449249268, -0.008250121027231216, 0.953595757484436, 0.382335901260376, 0.1692020446062088, 0.12175595015287399, -0.0809210017323494, -0.26846006512641907, -0.37073007225990295, -0.671856701374054, -1.554088830947876, 0.4154609739780426, 1.1736299991607666, -2.302199602127075, 0.1658707559108734, 0.32070305943489075, 0.22580406069755554, -1.2364568710327148, -0.8488747477531433, 0.11629562824964523, 0.9288251399993896, 1.2687896490097046, -1.2136991024017334, 1.0123945474624634, -0.8413257002830505, -0.6634191274642944, 0.47559842467308044, -1.2654120922088623, -1.3420350551605225, -1.8476110696792603, -1.1020920276641846, 0.00344241876155138, 1.0274338722229004, -1.6747069358825684, -0.23539067804813385, 0.6061378121376038, -1.8997348546981812, -0.5564542412757874, -0.49249690771102905, 0.12993673980236053, 1.047378420829773, -0.8600516319274902, 0.29529520869255066, -0.012989282608032227, -0.3176814913749695, -1.2727705240249634, -0.12198865413665771, 2.1063878536224365, 1.2393527030944824, 0.12221042066812515, -1.0115944147109985, 1.109315276145935, 2.0340287685394287, -0.5600526332855225, 0.305364727973938, -0.2043798416852951, -1.142575979232788, -1.768117070198059, -0.09754377603530884, 1.3728080987930298, 0.6596331596374512, 0.806978702545166, -1.0874028205871582, 0.227777361869812, -0.9880045056343079, -0.3491472601890564, 0.16870374977588654, 0.01526830717921257, -1.503767967224121, 1.2599713802337646, 0.7137835621833801, 0.4535198211669922, -0.9532556533813477, 0.21508605778217316, -0.5517678260803223, -0.6666579842567444, -0.8511962890625, -0.5980114340782166, -0.6534795165061951, -0.020325075834989548, 0.2600371837615967, 0.856099545955658, 0.6289446949958801, 0.17041361331939697, -0.1939733773469925, -0.27217021584510803, -0.1450173705816269, 0.6914617419242859, 0.71486496925354, 1.1383484601974487, -0.08782089501619339, 0.6617355346679688, 1.6759673357009888, -0.9101557731628418, 0.28266119956970215, 1.3289766311645508, 0.6138903498649597, 1.5119138956069946, -0.1651228368282318, -1.6702351570129395, 0.9434532523155212, -0.46908584237098694, 1.2779443264007568, -1.1480706930160522, 0.20225273072719574, -0.7583227157592773, -1.4794827699661255, -0.5762633681297302, -0.4593967795372009, -0.10580555349588394, -0.11066556721925735, -1.0165977478027344, 0.008219304494559765, -2.040510416030884, -0.316002756357193, -0.2566334903240204, -0.07973530888557434, 1.9912083148956299, 0.16949354112148285, -1.0761157274246216, 0.20019318163394928, 2.226750135421753, -2.110785961151123, -0.3327167332172394, -0.021245278418064117]], "dtype": "float32"}, {"__nd__": [261, 421, 602, 126, 18, 650, 491, 576], "dtype": "int64"}, -100], "e": {"__nd__": 7.382866382598877, "dtype": "float32"}}, {"a": [{"__nd__": [[1.3841307163238525, -1.9460335969924927, 0.96751868724823, -1.572811484336853, 0.3084537386894226, -1.0461066961288452, -0.28703299164772034, 0.5473381876945496, -1.2236744165420532, -0.0077272322960197926, -0.5288199186325073, -0.22647365927696228, 0.13551999628543854, 0.5879553556442261, 0.12248311936855316, 1.5425734519958496, 0.4382134675979614, -0.5745278000831604, 3.264113664627075, -1.6827622652053833, 0.13555169105529785, 1.5691332817077637, 0.2864091694355011, 2.146975517272949, -0.5352144241333008, -1.2028474807739258, 0.14286118745803833, -1.334328055381775, 1.75379478931427, 0.17583709955215454, 0.15522564947605133, 1.2827157974243164, 0.12335144728422165, 0.6583711504936218, -1.0073918104171753, -0.07568659633398056, -0.8408817648887634], [2.0648396015167236, 0.5020332932472229, -0.3204405605792999, -1.2495126724243164, -0.8174194693565369, 0.09729433059692383, 1.6586834192276, -3.3587896823883057, -1.0529268980026245, 0.8031277060508728, 0.23186825215816498, 2.0627598762512207, 1.9119290113449097, 0.19335943460464478, 0.16561155021190643, 0.625220000743866, 0.6510670185089111, -0.2769010663032532, 0.19705729186534882, 1.0703686475753784, -0.49716392159461975, -1.152024507522583, 1.1185296773910522, -0.6406587362289429, -0.8672897219657898, -1.6001344919204712, -1.1287732124328613, 0.36201950907707214, -0.4845152497291565, 0.9413995146751404, -0.33264338970184326, 0.03976057842373848, -0.17547275125980377, -1.4290379285812378, -1.581871509552002, -1.4775656461715698, -1.1731280088424683], [1.2081174850463867, -0.8302786350250244, -0.49786460399627686, -0.7367985248565674, 1.106191635131836, 0.6268600225448608, 0.15210917592048645, 0.27558398246765137, 1.1574957370758057, 1.2776894569396973, -0.8752983212471008, -0.2198779284954071, -1.134865164756775, -0.07971120625734329, 0.5800257921218872, 0.5981355309486389, -0.8538250923156738, 0.26949387788772583, 0.9685765504837036, -0.9682018756866455, -0.453336238861084, 0.2807624340057373, 1.4355686902999878, -0.2345171868801117, -0.20067355036735535, 0.8675317168235779, -2.108942985534668, 0.00459369458258152, 0.8937163949012756, -0.35643795132637024, 0.12398962676525116, 0.3284766674041748, 1.1509164571762085, 1.4008519649505615, -0.2746828496456146, 0.9700408577919006, 1.0984694957733154]], "dtype": "float32"}, {"__nd__": [5, 0, 36], "dtype": "int64"}, 0], "e": {"__nd__": 4.249788761138916, "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(cross_entropy, json.loads(_TESTS), 'triton', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")